# WCEP para descarbonización: IEEE 14, IEEE 39 e IEEE 57

Notebook unificado


# Sistema IEEE 14-bus


# Dataset

## Demanda.csv

In [1]:
# Crear análogo a demanda_nodal_final.csv pero con estos datos

import pandas as pd

# Cargar el archivo
bus_data = pd.read_csv("ieee14_bus_data.csv")

# Ver qué tiene
bus_data.head()

,bus_id,bus_name,base_kv_label,area_number,loss_zone,bus_type,voltage_pu,voltage_angle_deg,load_mw,load_mvar,generation_mw,generation_mvar,base_kv,desired_voltage_pu,max_reactive_mvar,min_reactive_mvar,shunt_conductance_pu,shunt_susceptance_pu,remote_controlled_bus
0,1,Bus 1,HV,1,1,3,1.060,0.00,0.0,0.0,232.4,-16.9,0.0,1.060,0.0,0.0,0.0,0.0,0
1,2,Bus 2,HV,1,1,2,1.045,-4.98,21.7,12.7,40.0,42.4,0.0,1.045,50.0,-40.0,0.0,0.0,0
2,3,Bus 3,HV,1,1,2,1.010,-12.72,94.2,19.0,0.0,23.4,0.0,1.010,40.0,0.0,0.0,0.0,0
3,4,Bus 4,HV,1,1,0,1.019,-10.33,47.8,-3.9,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0
4,5,Bus 5,HV,1,1,0,1.020,-8.78,7.6,1.6,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0


In [2]:
bus_data.columns



Index(['bus_id', 'bus_name', 'base_kv_label', 'area_number', 'loss_zone',
       'bus_type', 'voltage_pu', 'voltage_angle_deg', 'load_mw', 'load_mvar',
       'generation_mw', 'generation_mvar', 'base_kv', 'desired_voltage_pu',
       'max_reactive_mvar', 'min_reactive_mvar', 'shunt_conductance_pu',
       'shunt_susceptance_pu', 'remote_controlled_bus'],
      dtype='object')

In [3]:
# Crear demanda nodal a partir del IEEE 14 bus
demanda = bus_data[["bus_id", "load_mw"]].copy()

# Renombrar columnas al formato que te acomode
demanda.columns = ["bus", "demanda"]

# Asegurar formato numérico
demanda["bus"] = demanda["bus"].astype(int)
demanda["demanda"] = demanda["demanda"].astype(float)

demanda

,bus,demanda
0,1,0.0
1,2,21.7
2,3,94.2
3,4,47.8
4,5,7.6
5,6,11.2
6,7,0.0
7,8,0.0
8,9,29.5
9,10,9.0


In [4]:
# Reemplazar NaN por 0 (por si acaso)
demanda["demanda"] = demanda["demanda"].fillna(0)

# Asegurar tipo numérico
demanda["demanda"] = demanda["demanda"].astype(float)

demanda.to_csv("demanda_nodal_ieee14.csv", index=False)

## Lineas.csv

In [5]:
import pandas as pd

branch_data = pd.read_csv("ieee14_branch_data.csv")

branch_data.head()

,from_bus,to_bus,area_number,loss_zone,circuit,branch_type,resistance_pu,reactance_pu,line_charging_pu,rating_a_mva,...,rating_c_mva,control_bus,side,tap_ratio,phase_shift_deg,min_tap,max_tap,step_size,min_voltage,max_voltage
0,1,2,1,1,1,0,0.01938,0.05917,0.0528,0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,5,1,1,1,0,0.05403,0.22304,0.0492,0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,3,1,1,1,0,0.04699,0.19797,0.0438,0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2,4,1,1,1,0,0.05811,0.17632,0.0340,0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2,5,1,1,1,0,0.05695,0.17388,0.0346,0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
branch_data.columns

Index(['from_bus', 'to_bus', 'area_number', 'loss_zone', 'circuit',
       'branch_type', 'resistance_pu', 'reactance_pu', 'line_charging_pu',
       'rating_a_mva', 'rating_b_mva', 'rating_c_mva', 'control_bus', 'side',
       'tap_ratio', 'phase_shift_deg', 'min_tap', 'max_tap', 'step_size',
       'min_voltage', 'max_voltage'],
      dtype='object')

In [7]:
# Crear dataset de líneas
lineas = branch_data[["from_bus", "to_bus", "reactance_pu", "tap_ratio"]].copy()

# Renombrar columnas
lineas.columns = ["desde", "hacia", "x", "tap"]

# 🔥 Ajuste importante: manejar transformadores
# Si tap = 0 → es línea normal → usar x directo
# Si tap ≠ 0 → ajustar reactancia

lineas["tap"] = lineas["tap"].replace(0, 1.0)

# Ajustar reactancia efectiva (opcional pero más correcto)
lineas["x"] = lineas["x"] / lineas["tap"]

# Crear ID de línea
lineas["linea"] = range(1, len(lineas) + 1)

# Agregar límite de flujo (porque IEEE 14 no trae)
lineas["fmax"] = 100  # puedes cambiarlo

# Reordenar columnas
lineas = lineas[["linea", "desde", "hacia", "x", "fmax"]]

lineas

,linea,desde,hacia,x,fmax
0,1,1,2,0.059170,100
1,2,1,5,0.223040,100
2,3,2,3,0.197970,100
3,4,2,4,0.176320,100
4,5,2,5,0.173880,100
5,6,3,4,0.171030,100
6,7,4,5,0.042110,100
7,8,4,7,0.213824,100
8,9,4,9,0.573973,100
9,10,5,6,0.270408,100


In [8]:
lineas.to_csv("lineas_procesadas_ieee14.csv", index=False)

## Generador.csv

In [9]:
# Seleccionar buses con generador: PV (2) o Slack (3)
generadores = bus_data[bus_data["bus_type"].isin([2, 3])].copy()

# Crear tabla base de generadores
generadores_proc = generadores[[
    "bus_id",
    "bus_type",
    "generation_mw",
    "generation_mvar",
    "max_reactive_mvar",
    "min_reactive_mvar",
    "desired_voltage_pu"
]].copy()

# Crear identificador de generador
generadores_proc["gen_id"] = [f"G{i+1}" for i in range(len(generadores_proc))]

# Reordenar columnas
generadores_proc = generadores_proc[[
    "gen_id",
    "bus_id",
    "bus_type",
    "generation_mw",
    "generation_mvar",
    "max_reactive_mvar",
    "min_reactive_mvar",
    "desired_voltage_pu"
]]

generadores_proc

,gen_id,bus_id,bus_type,generation_mw,generation_mvar,max_reactive_mvar,min_reactive_mvar,desired_voltage_pu
0,G1,1,3,232.4,-16.9,0.0,0.0,1.060
1,G2,2,2,40.0,42.4,50.0,-40.0,1.045
2,G3,3,2,0.0,23.4,40.0,0.0,1.010
5,G4,6,2,0.0,12.2,24.0,-6.0,1.070
7,G5,8,2,0.0,17.4,24.0,-6.0,1.090


In [10]:
generadores_proc.to_csv("generadores_procesados_ieee14_base.csv", index=False)

## Demanda_nodal_final

In [11]:
import pandas as pd

# =========================
# 1. Cargar dataset original
# =========================
df = pd.read_csv("demanda_nodal_ieee14.csv")

# =========================
# 2. Definir periodos y proporciones
# =========================
periodos = {
    1: 0.20,
    7: 0.21,
    13: 0.30,
    19: 0.29
}

# =========================
# 3. Expandir dataset
# =========================
filas = []

for _, row in df.iterrows():
    bus = row["bus"]
    demanda_total = row["demanda"]

    for t, alpha in periodos.items():
        filas.append({
            "bus": int(bus),
            "periodo": int(t),
            "demanda_mw": float(demanda_total * alpha)
        })

df_final = pd.DataFrame(filas)

# =========================
# 4. Guardar CSV final
# =========================
df_final.to_csv("demanda_nodal_ieee14_final.csv", index=False)

# =========================
# 5. Verificación rápida
# =========================
print(df_final.head(12))

# Verificar conservación de demanda por bus
check = df_final.groupby("bus")["demanda_mw"].sum().reset_index()
check = check.merge(df, on="bus")
check["error"] = check["demanda_mw"] - check["demanda"]

print("\nChequeo conservación demanda:")
print(check)

    bus  periodo  demanda_mw
0     1        1       0.000
1     1        7       0.000
2     1       13       0.000
3     1       19       0.000
4     2        1       4.340
5     2        7       4.557
6     2       13       6.510
7     2       19       6.293
8     3        1      18.840
9     3        7      19.782
10    3       13      28.260
11    3       19      27.318

Chequeo conservación demanda:
    bus  demanda_mw  demanda  error
0     1         0.0      0.0    0.0
1     2        21.7     21.7    0.0
2     3        94.2     94.2    0.0
3     4        47.8     47.8    0.0
4     5         7.6      7.6    0.0
5     6        11.2     11.2    0.0
6     7         0.0      0.0    0.0
7     8         0.0      0.0    0.0
8     9        29.5     29.5    0.0
9    10         9.0      9.0    0.0
10   11         3.5      3.5    0.0
11   12         6.1      6.1    0.0
12   13        13.5     13.5    0.0
13   14        14.9     14.9    0.0


## Generadores y perfil solar final .csv

In [12]:
import pandas as pd
from pathlib import Path

# =========================================================
# 1) Definir rutas correctamente
# =========================================================
# Asume que el notebook WCEP_IEEE14.ipynb está dentro de la carpeta IEEE14

ruta_gen_base = Path("base_extra_generadores_procesados.csv")
ruta_solar_base = Path("base_extra_perfil_solar_final.csv")

ruta_gen_final = Path("generadores_procesados_ieee14_final.csv")
ruta_solar_final = Path("perfil_solar_ieee14_final.csv")

# Verificación de existencia
if not ruta_gen_base.exists():
    raise FileNotFoundError(f"No encontré el archivo de generadores en: {ruta_gen_base.resolve()}")

if not ruta_solar_base.exists():
    raise FileNotFoundError(f"No encontré el archivo de perfil solar en: {ruta_solar_base.resolve()}")

# =========================================================
# 2) Cargar archivos base
# =========================================================
df_gen_base = pd.read_csv(ruta_gen_base)
df_solar_base = pd.read_csv(ruta_solar_base)

print("Columnas generadores:", df_gen_base.columns.tolist())
print("Columnas perfil solar:", df_solar_base.columns.tolist())

# =========================================================
# 3) Definir asignación nueva al IEEE14
# =========================================================
mapeo_bus_ieee14 = {
    "223_STEAM_1": 1,
    "223_STEAM_2": 1,
    "313_PV_1": 2,
    "314_PV_3": 2,
    "314_PV_1": 3,
    "314_PV_2": 6,
    "314_PV_4": 8,
}

orden_generadores = [
    "223_STEAM_1",
    "223_STEAM_2",
    "313_PV_1",
    "314_PV_3",
    "314_PV_1",
    "314_PV_2",
    "314_PV_4",
]

gen_seleccionados = list(mapeo_bus_ieee14.keys())

# =========================================================
# 4) Crear generadores_procesados_ieee14_final.csv
# =========================================================
df_gen_final = df_gen_base[df_gen_base["id_gen"].isin(gen_seleccionados)].copy()

# Reasignar barras IEEE14
df_gen_final["bus_id"] = df_gen_final["id_gen"].map(mapeo_bus_ieee14)

# Ordenar en el orden definido
df_gen_final["orden_tmp"] = df_gen_final["id_gen"].apply(lambda x: orden_generadores.index(x))
df_gen_final = (
    df_gen_final
    .sort_values("orden_tmp")
    .drop(columns="orden_tmp")
    .reset_index(drop=True)
)

# Guardar
df_gen_final.to_csv(ruta_gen_final, index=False)

print("\nArchivo creado:", ruta_gen_final.resolve())
print(df_gen_final)

# =========================================================
# 5) Adaptar perfil_solar_ieee14_final.csv
# =========================================================
# Detectar nombre de columna del generador
if "Gen_ID" in df_solar_base.columns:
    col_gen_solar = "Gen_ID"
elif "id_gen" in df_solar_base.columns:
    col_gen_solar = "id_gen"
else:
    raise ValueError(
        "No encontré columna de identificador de generador en perfil_solar_final.csv. "
        f"Columnas disponibles: {df_solar_base.columns.tolist()}"
    )

# Detectar nombre de columna de periodo
if "Period" in df_solar_base.columns:
    col_periodo = "Period"
elif "periodo" in df_solar_base.columns:
    col_periodo = "periodo"
else:
    raise ValueError(
        "No encontré columna de periodo en perfil_solar_final.csv. "
        f"Columnas disponibles: {df_solar_base.columns.tolist()}"
    )

# Obtener solo los PV/IBR que quedaron en el sistema IEEE14 final
pv_seleccionados = df_gen_final.loc[df_gen_final["Es_IBR"] == 1, "id_gen"].tolist()

df_solar_final = df_solar_base[df_solar_base[col_gen_solar].isin(pv_seleccionados)].copy()

# Ordenar por generador y periodo
orden_pv = [g for g in orden_generadores if g in pv_seleccionados]
df_solar_final["orden_gen"] = df_solar_final[col_gen_solar].apply(lambda x: orden_pv.index(x))
df_solar_final = (
    df_solar_final
    .sort_values(["orden_gen", col_periodo])
    .drop(columns="orden_gen")
    .reset_index(drop=True)
)

# Guardar
df_solar_final.to_csv(ruta_solar_final, index=False)

print("\nArchivo creado:", ruta_solar_final.resolve())
print(df_solar_final)

# =========================================================
# 6) Chequeo extra de consistencia
# =========================================================
gens_con_perfil = set(df_solar_final[col_gen_solar].unique())
gens_ibr = set(pv_seleccionados)

faltan_perfiles = gens_ibr - gens_con_perfil
sobran_perfiles = gens_con_perfil - gens_ibr

print("\nChequeo de consistencia:")
print("IBR seleccionados:", sorted(gens_ibr))
print("Generadores con perfil:", sorted(gens_con_perfil))
print("Faltan perfiles para:", sorted(faltan_perfiles) if faltan_perfiles else "Ninguno")
print("Sobran perfiles para:", sorted(sobran_perfiles) if sobran_perfiles else "Ninguno")

Columnas generadores: ['id_gen', 'bus_id', 'tipo_tech', 'fuel_orig', 'Pmax', 'Pmin', 'Costo_Var', 'Inercia_H', 'Emisiones_tCO2_MWh', 'Es_IBR', 'Gamma', 'Kappa', 'Costo_Inv', 'Gamma_max', 'Kappa_max']
Columnas perfil solar: ['Gen_ID', 'Period', 'P_Available_MW']

Archivo creado: C:\Users\felip\Downloads\Descarbonizacion_WCEP_Unificado\Descarbonizacion_WCEP_Unificado\generadores_procesados_ieee14_final.csv
        id_gen  bus_id tipo_tech fuel_orig   Pmax  Pmin  Costo_Var  Inercia_H  \
0  223_STEAM_1       1      COAL      Coal  155.0    62       35.0        3.0   
1  223_STEAM_2       1      COAL      Coal  155.0    62       35.0        3.0   
2     313_PV_1       2        PV     Solar   95.1     0        2.0        0.0   
3     314_PV_3       2        PV     Solar   92.7     0        2.0        0.0   
4     314_PV_1       3        PV     Solar   51.6     0        2.0        0.0   
5     314_PV_2       6        PV     Solar   51.6     0        2.0        0.0   
6     314_PV_4       8   

## Datos extras

Así como el Hreq, Rreq, valores de kappa y gamma (máximos y mínimos), etc.

In [13]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0) CONFIG: usar CSV IEEE14 finales
# ============================================================
PATH = "."

PATH_GEN   = os.path.join(PATH, "generadores_procesados_ieee14_final.csv")
PATH_LINE  = os.path.join(PATH, "lineas_procesadas_ieee14.csv")
PATH_LOAD  = os.path.join(PATH, "demanda_nodal_ieee14_final.csv")
PATH_SOLAR = os.path.join(PATH, "perfil_solar_ieee14_final.csv")

df_gen   = pd.read_csv(PATH_GEN)
df_lines = pd.read_csv(PATH_LINE)
df_load  = pd.read_csv(PATH_LOAD)

df_wind = pd.DataFrame(columns=["Gen_ID", "Period", "P_Available_MW"])
df_solar = pd.read_csv(PATH_SOLAR) if os.path.exists(PATH_SOLAR) else pd.DataFrame(
    columns=["Gen_ID", "Period", "P_Available_MW"]
)

# ============================================================
# 1) LIMPIAR NOMBRES DE COLUMNAS
# ============================================================
df_gen.columns   = df_gen.columns.str.strip()
df_lines.columns = df_lines.columns.str.strip()
df_load.columns  = df_load.columns.str.strip()
df_solar.columns = df_solar.columns.str.strip()

# ============================================================
# 2) RENOMBRAR COLUMNAS PARA DEJARLAS COMPATIBLES
# ============================================================

# ----- DEMANDA -----
df_load = df_load.rename(columns={
    "bus": "Bus ID",
    "periodo": "Period",
    "demanda_mw": "Nodal_Load_MW"
})

# ----- LINEAS -----
df_lines = df_lines.rename(columns={
    "desde": "from_bus",
    "hacia": "to_bus",
    "x": "Reactancia_X",
    "fmax": "F_max"
})

# ----- PERFIL SOLAR -----
df_solar = df_solar.rename(columns={
    "id_gen": "Gen_ID",
    "periodo": "Period",
    "disponibilidad_mw": "P_Available_MW"
})

# ============================================================
# 3) CHEQUEO DE COLUMNAS OBLIGATORIAS
# ============================================================
cols_gen_req = ["id_gen", "bus_id", "Pmax", "Pmin", "Costo_Var", "Inercia_H",
                "Emisiones_tCO2_MWh", "Es_IBR", "Gamma", "Kappa", "Costo_Inv"]
cols_line_req = ["from_bus", "to_bus", "Reactancia_X", "F_max"]
cols_load_req = ["Bus ID", "Period", "Nodal_Load_MW"]

for c in cols_gen_req:
    if c not in df_gen.columns:
        raise ValueError(f"Falta columna en generadores: {c}")

for c in cols_line_req:
    if c not in df_lines.columns:
        raise ValueError(f"Falta columna en líneas: {c}")

for c in cols_load_req:
    if c not in df_load.columns:
        raise ValueError(f"Falta columna en demanda: {c}")

# ============================================================
# 4) SANITIZAR TIPOS
# ============================================================
df_gen["id_gen"] = df_gen["id_gen"].astype(str)
df_gen["bus_id"] = df_gen["bus_id"].astype(int)

df_load["Bus ID"] = df_load["Bus ID"].astype(int)
df_load["Period"] = df_load["Period"].astype(int)
df_load["Nodal_Load_MW"] = df_load["Nodal_Load_MW"].astype(float)

df_lines["from_bus"] = df_lines["from_bus"].astype(int)
df_lines["to_bus"]   = df_lines["to_bus"].astype(int)
df_lines["F_max"] = df_lines["F_max"].astype(float)
df_lines["Reactancia_X"] = df_lines["Reactancia_X"].astype(float)

if not df_solar.empty:
    df_solar["Gen_ID"] = df_solar["Gen_ID"].astype(str)
    df_solar["Period"] = df_solar["Period"].astype(int)
    df_solar["P_Available_MW"] = df_solar["P_Available_MW"].astype(float)

# ============================================================
# 5) DEFINICIÓN DE SETS
# ============================================================
Y = [1, 2, 3]
year_growth = {1: 1.00, 2: 1.05, 3: 1.10}

B = sorted(df_load["Bus ID"].unique().tolist())
T = sorted(df_load["Period"].unique().tolist())

G  = df_gen["id_gen"].tolist()
GS = df_gen[df_gen["Es_IBR"] == 0]["id_gen"].tolist()
M  = df_gen[df_gen["Es_IBR"] == 1]["id_gen"].tolist()
GE = df_gen[df_gen["Emisiones_tCO2_MWh"] > 0]["id_gen"].tolist()

print(f"Sets definidos: |B|={len(B)}, |T|={len(T)}, |G|={len(G)} (|GS|={len(GS)}, |M|={len(M)}, |GE|={len(GE)})")

# ============================================================
# 6) PARÁMETROS DE GENERADORES
# ============================================================
Pmax = df_gen.set_index("id_gen")["Pmax"].to_dict()
Pmin = df_gen.set_index("id_gen")["Pmin"].to_dict()
cg   = df_gen.set_index("id_gen")["Costo_Var"].to_dict()
eg   = df_gen.set_index("id_gen")["Emisiones_tCO2_MWh"].to_dict()
cinv = df_gen.set_index("id_gen")["Costo_Inv"].to_dict()

Hg = {g: float(df_gen.set_index("id_gen")["Inercia_H"].to_dict().get(g, 0.0)) for g in GS}

g_bus = df_gen.set_index("id_gen")["bus_id"].to_dict()

G_en_bus = {i: [] for i in B}
for g in G:
    bi = int(g_bus[g])
    if bi in G_en_bus:
        G_en_bus[bi].append(g)

# ============================================================
# 7) PARÁMETROS IBR
# ============================================================
gamma_base = df_gen.set_index("id_gen")["Gamma"].to_dict()
kappa_base = df_gen.set_index("id_gen")["Kappa"].to_dict()

gamma_max = df_gen.set_index("id_gen")["Gamma_max"].to_dict() if "Gamma_max" in df_gen.columns else {m: 0.5 for m in M}
kappa_max = df_gen.set_index("id_gen")["Kappa_max"].to_dict() if "Kappa_max" in df_gen.columns else {m: 0.5 for m in M}

xbar_mult = 2.0
xbar = {m: float(Pmax[m]) * xbar_mult for m in M}

# ============================================================
# 8) RED
# ============================================================
L, F0, Bline = [], {}, {}

for _, row in df_lines.iterrows():
    i, j = int(row["from_bus"]), int(row["to_bus"])
    line = (i, j)
    L.append(line)
    F0[line] = float(row["F_max"])
    x_val = float(row["Reactancia_X"])
    if abs(x_val) < 1e-6:
        x_val = 1e-4
    Bline[line] = 1.0 / x_val

lineas_entran = {i: [] for i in B}
lineas_salen  = {i: [] for i in B}

for (i, j) in L:
    if i in lineas_salen:
        lineas_salen[i].append((i, j))
    if j in lineas_entran:
        lineas_entran[j].append((i, j))

print(f"Red: |L|={len(L)}")

# ============================================================
# 9) DEMANDA
# ============================================================
raw_demand = df_load.set_index(["Bus ID", "Period"])["Nodal_Load_MW"].to_dict()

d = {}
for y in Y:
    factor = year_growth[y]
    for i in B:
        for t in T:
            d[(i, t, y)] = float(raw_demand.get((i, t), 0.0)) * factor

# ============================================================
# 10) PERFIL RENOVABLE
# ============================================================
P_disp = {(g, t): float(Pmax[g]) for g in G for t in T}

if not df_solar.empty:
    for _, row in df_solar.iterrows():
        g_id = str(row["Gen_ID"])
        t = int(row["Period"])
        val = float(row["P_Available_MW"])
        if (g_id in G) and (t in T):
            P_disp[(g_id, t)] = val

# ============================================================
# 11) ESCALARES
# ============================================================
VOLL = 5000.0
y_star = 3
epsilon_CO2 = 0

# ============================================================
# 12) Hreq y Rreq
# ============================================================
D_peak = {y: max(sum(d[(i, t, y)] for i in B) for t in T) for y in Y}

max_hsyn = sum(float(gamma_base.get(m, 0.0)) * float(xbar[m]) for m in M) if len(M) > 0 else 0.0
max_sync = sum(float(Hg.get(g, 0.0)) for g in GS) if len(GS) > 0 else 0.0
cap_total = max_hsyn + max_sync

alpha_H = 0.35
H_floor = 0.05

k_H = (alpha_H * cap_total / D_peak[1]) if D_peak[1] > 1e-9 else 0.0
Hreq = {}
for y in Y:
    base = k_H * D_peak[y]
    floor = H_floor * cap_total
    Hreq[y] = max(base, floor)

MaxR = sum(float(kappa_base.get(m, 0.0)) * float(xbar[m]) for m in M) if len(M) > 0 else 0.0
beta_R = 0.40
R_floor = 0.05

Rreq = {}
for y in Y:
    base = beta_R * MaxR
    floor = R_floor * MaxR
    Rreq[y] = max(base, floor)

print("cap_total =", round(cap_total, 4), " MaxR =", round(MaxR, 4))
print("D_peak =", {y: round(D_peak[y], 2) for y in Y})
print("Hreq =", {y: round(Hreq[y], 4) for y in Y})
print("Rreq =", {y: round(Rreq[y], 4) for y in Y})

Sets definidos: |B|=14, |T|=4, |G|=7 (|GS|=2, |M|=5, |GE|=2)
Red: |L|=20
cap_total = 12.852  MaxR = 68.52
D_peak = {1: 77.7, 2: 81.58, 3: 85.47}
Hreq = {1: 4.4982, 2: 4.7231, 3: 4.948}
Rreq = {1: 27.408, 2: 27.408, 3: 27.408}


# Forward

## Dimensionalidad

In [14]:
# ============================================================
# Celda: Resumen de dimensionalidad / tamaño del problema
# Adaptada a tu FW actual para IEEE14
# Requiere: B, T, Y, G, GS, M, GE, L
# Opcional: fw_base
# ============================================================

import gurobipy as gp

def safe_len(x):
    try:
        return len(x)
    except Exception:
        return None

def mul(*vals):
    out = 1
    for v in vals:
        out *= int(v)
    return out

# Si existe L_arcs úsalo; si no, usar L
if "L_arcs" in globals():
    _L = L_arcs
else:
    _L = gp.tuplelist(L)

nB  = safe_len(B)
nT  = safe_len(T)
nY  = safe_len(Y)
nG  = safe_len(G)
nGS = safe_len(GS)
nM  = safe_len(M)
nGE = safe_len(GE)
nL  = safe_len(_L)

print("\n==============================")
print("DIMENSIONALIDAD DEL PROBLEMA")
print("==============================")
print(f"|B|  (buses)        = {nB}")
print(f"|L|  (líneas/arcos) = {nL}")
print(f"|T|  (periodos)     = {nT}")
print(f"|Y|  (años)         = {nY}")
print(f"|G|  (generadores)  = {nG}")
print(f"|GS| (síncronos)    = {nGS}")
print(f"|M|  (IBR)          = {nM}")
print(f"|GE| (emisores CO2) = {nGE}")

# ============================================================
# Variables (estimadas según tu formulación actual)
# ============================================================
n_p     = mul(nG,  nT, nY) if None not in [nG,  nT, nY] else None
n_u     = mul(nGS, nT, nY) if None not in [nGS, nT, nY] else None
n_s     = mul(nB,  nT, nY) if None not in [nB,  nT, nY] else None
n_theta = mul(nB,  nT, nY) if None not in [nB,  nT, nY] else None
n_f     = mul(nL,  nT, nY) if None not in [nL,  nT, nY] else None
n_x     = mul(nM,  nY)     if None not in [nM,  nY]     else None
n_h     = mul(nM,  nT, nY) if None not in [nM,  nT, nY] else None
n_r     = mul(nM,  nT, nY) if None not in [nM,  nT, nY] else None

print("\n--- Tamaño (variables) según la formulación ---")
print(f"#p      = {n_p}")
print(f"#u      = {n_u}")
print(f"#s      = {n_s}")
print(f"#theta  = {n_theta}")
print(f"#f      = {n_f}")
print(f"#x      = {n_x}")
print(f"#h_syn  = {n_h}")
print(f"#r_ffr  = {n_r}")

n_vars_est = sum(v for v in [n_p, n_u, n_s, n_theta, n_f, n_x, n_h, n_r] if v is not None)
print(f"TOTAL vars (estimado) = {n_vars_est}")

# ============================================================
# Restricciones (estimadas según tu formulación actual)
# ============================================================
n_bal   = mul(nB,  nT, nY) if None not in [nB,  nT, nY] else None      # balance nodal
n_ensub = mul(nB,  nT, nY) if None not in [nB,  nT, nY] else None      # ENS <= demanda
n_dc    = mul(nL,  nT, nY) if None not in [nL,  nT, nY] else None      # flujo DC
n_line  = 2 * n_dc if n_dc is not None else None                       # limites ±Fmax
n_slack = mul(nT,  nY)     if None not in [nT,  nY]     else None      # barra slack
n_pdisp = mul(nG,  nT, nY) if None not in [nG,  nT, nY] else None      # p <= Pdisp
n_sync  = 2 * mul(nGS, nT, nY) if None not in [nGS, nT, nY] else None  # Pmin/Pmax síncronos
n_co2   = 0                                                            # según tu versión actual, comentada
n_xub   = mul(nM,  nY)     if None not in [nM,  nY]     else None      # x <= xbar
n_hcap  = mul(nM,  nT, nY) if None not in [nM,  nT, nY] else None      # h_syn <= gamma*x
n_rcap  = mul(nM,  nT, nY) if None not in [nM,  nT, nY] else None      # r_ffr <= kappa*x
n_inreq = mul(nT,  nY)     if None not in [nT,  nY]     else None      # req. inercia
n_rreq  = mul(nT,  nY)     if None not in [nT,  nY]     else None      # req. FFR

print("\n--- Tamaño (restricciones) según la formulación ---")
print(f"#balance      = {n_bal}")
print(f"#ens_ub       = {n_ensub}")
print(f"#dcflow       = {n_dc}")
print(f"#line_limits  = {n_line}")
print(f"#slack        = {n_slack}")
print(f"#pdisp        = {n_pdisp}")
print(f"#sync_min/max = {n_sync}")
print(f"#CO2_cap      = {n_co2}")
print(f"#x_ub         = {n_xub}")
print(f"#hsyn_cap     = {n_hcap}")
print(f"#ffr_cap      = {n_rcap}")
print(f"#inertia_req  = {n_inreq}")
print(f"#ffr_req      = {n_rreq}")

n_cons_est = sum(v for v in [
    n_bal, n_ensub, n_dc, n_line, n_slack, n_pdisp,
    n_sync, n_co2, n_xub, n_hcap, n_rcap, n_inreq, n_rreq
] if v is not None)

print(f"TOTAL cons (estimado) = {n_cons_est}")

# ============================================================
# Comparar con el modelo real en Gurobi
# ============================================================
if "fw_base" in globals():
    try:
        fw_base.update()
        print("\n--- Conteo REAL del modelo en Gurobi (fw_base) ---")
        print(f"fw_base.NumVars    = {fw_base.NumVars}")
        print(f"fw_base.NumConstrs = {fw_base.NumConstrs}")
    except Exception as e:
        print("\n(No pude leer fw_base.NumVars/fw_base.NumConstrs):", e)
else:
    print("\n(Nota) No existe 'fw_base' en globals(). Corre primero la celda del Forward Problem.")


DIMENSIONALIDAD DEL PROBLEMA
|B|  (buses)        = 14
|L|  (líneas/arcos) = 20
|T|  (periodos)     = 4
|Y|  (años)         = 3
|G|  (generadores)  = 7
|GS| (síncronos)    = 2
|M|  (IBR)          = 5
|GE| (emisores CO2) = 2

--- Tamaño (variables) según la formulación ---
#p      = 84
#u      = 24
#s      = 168
#theta  = 168
#f      = 240
#x      = 15
#h_syn  = 60
#r_ffr  = 60
TOTAL vars (estimado) = 819

--- Tamaño (restricciones) según la formulación ---
#balance      = 168
#ens_ub       = 168
#dcflow       = 240
#line_limits  = 480
#slack        = 12
#pdisp        = 84
#sync_min/max = 48
#CO2_cap      = 0
#x_ub         = 15
#hsyn_cap     = 60
#ffr_cap      = 60
#inertia_req  = 12
#ffr_req      = 12
TOTAL cons (estimado) = 1359

(Nota) No existe 'fw_base' en globals(). Corre primero la celda del Forward Problem.


# Resolución del WCEP

## Final 100% real

### FW

In [15]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd

# ============================================================
# FINAL IEEE14
# PASO 1) FORWARD PROBLEM - FORMULACIÓN ORIGINAL
# DATASET CORREGIDO
#
# Correcciones incluidas:
#
# 1) Escalamiento de demanda para evitar déficit estructural.
#    El dataset original tenía demanda pico mayor que capacidad instalada,
#    generando ENS artificial.
#
# 2) Hreq queda en su posición original:
#       sum_g Hg[g] u[g,t,y] + sum_m h_syn[m,t,y] >= Hreq[y]
#
#    No se usa zH.
#    No se mueve Hreq hacia A.
#
# 3) Se mantiene el diccionario original d intacto.
#    El modelo usa d_fw.
# ============================================================


# ============================================================
# 0) OPCIONES DE CORRECCIÓN DEL DATASET IEEE14
# ============================================================

# Recomendado para IEEE14:
#   "auto_capacity" calcula un factor para que la demanda pico quede
#   bajo la capacidad instalada del sistema.
#
# También puedes fijar manualmente:
#   DEMAND_SCALE_MODE = "manual"
#   DEMAND_SCALE_MANUAL = 0.713
#
DEMAND_SCALE_MODE = "auto_capacity"   # "auto_capacity", "manual", "none"
DEMAND_SCALE_MANUAL = 0.713

# Margen de seguridad respecto a capacidad instalada.
# 1.00 deja la demanda pico igual a la capacidad.
# 0.98 deja un 2% de margen operativo.
DEMAND_CAP_MARGIN = 0.98

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# T contiene pocas horas representativas del día.
# En vez de expandir T a 365*|T| periodos, ponderamos cada t.
#
# Caso base solicitado:
#   el mismo perfil diario se repite 365 veces.
#   Si cada t representa una hora puntual del día, usar TIME_BLOCK_HOURS = 1.0.
#   Si cada t representa un bloque de 6 horas, usar TIME_BLOCK_HOURS = 6.0.
#
# Con T = {1,7,13,19}:
#   TIME_BLOCK_HOURS = 1.0  -> sum_t omega_t = 1460 h equivalentes
#   TIME_BLOCK_HOURS = 6.0  -> sum_t omega_t = 8760 h equivalentes
N_DAYS_REPRESENTED = 365
TIME_BLOCK_HOURS = 1.0

omega_t = {t: float(N_DAYS_REPRESENTED * TIME_BLOCK_HOURS) for t in T}

print("\n==============================")
print("PESOS TEMPORALES FW IEEE14")
print("==============================")
print(f"Días representados       = {N_DAYS_REPRESENTED}")
print(f"Horas por periodo t      = {TIME_BLOCK_HOURS}")
print(f"Peso omega_t por periodo = {next(iter(omega_t.values())) if len(omega_t) > 0 else 0.0}")
print(f"Horas equivalentes total = {sum(omega_t.values()):.2f}")

# Para mantener el experimento reproducible, no se modifica d.
# Se construye un nuevo diccionario d_fw.
d_original = {k: float(v) for k, v in d.items()}


# ============================================================
# 1) ROBUSTECER LÍNEAS
# ============================================================

L_arcs = gp.tuplelist(L)


# ============================================================
# 2) FIX: reconstruir B si fue pisado
# ============================================================

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))


# ============================================================
# 3) TOPOLOGÍA ROBUSTA
# ============================================================

lineas_entran = {bus: [] for bus in B}
lineas_salen  = {bus: [] for bus in B}

for arc in L_arcs:
    if not (isinstance(arc, tuple) and len(arc) == 2):
        raise TypeError(f"Arco inválido en L_arcs: {arc} (type={type(arc)})")

    a, b = arc

    if b in lineas_entran:
        lineas_entran[b].append((a, b))

    if a in lineas_salen:
        lineas_salen[a].append((a, b))


# ============================================================
# 4) CORRECCIÓN DE DEMANDA IEEE14
# ============================================================

print("\n==============================")
print("CORRECCIÓN DATASET IEEE14")
print("==============================")

# Demanda total original por periodo/año
demanda_original_ty = {}

for t in T:
    for y in Y:
        demanda_original_ty[(t, y)] = sum(
            d_original[(i, t, y)]
            for i in B
            if (i, t, y) in d_original
        )

peak_dem_original = max(demanda_original_ty.values())

# Capacidad instalada total
cap_total_pmax = sum(float(Pmax[g]) for g in G)

# Capacidad disponible por periodo si existe P_disp
cap_disp_ty = {}

for t in T:
    for y in Y:
        cap_disp_ty[(t, y)] = sum(
            float(P_disp.get((g, t), Pmax[g]))
            for g in G
        )

min_cap_disp = min(cap_disp_ty.values())

print(f"Demanda pico original     = {peak_dem_original:.6f}")
print(f"Capacidad instalada Pmax  = {cap_total_pmax:.6f}")
print(f"Capacidad disp. mínima    = {min_cap_disp:.6f}")

if DEMAND_SCALE_MODE == "none":
    demand_scale_factor = 1.0

elif DEMAND_SCALE_MODE == "manual":
    demand_scale_factor = float(DEMAND_SCALE_MANUAL)

elif DEMAND_SCALE_MODE == "auto_capacity":
    if peak_dem_original <= 1e-9:
        demand_scale_factor = 1.0
    else:
        # Usamos Pmax total, consistente con el diagnóstico original.
        # Además aplicamos un pequeño margen para evitar operar exactamente
        # en el borde de capacidad.
        demand_scale_factor = min(
            1.0,
            DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
        )

else:
    raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

print(f"Modo escalamiento demanda = {DEMAND_SCALE_MODE}")
print(f"Factor demanda usado      = {demand_scale_factor:.8f}")

# Construir demanda corregida para el modelo
d_fw = {}

for i in B:
    for t in T:
        for y in Y:
            if (i, t, y) not in d_original:
                raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")

            d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

# Resumen demanda corregida
demanda_fw_ty = {}

for t in T:
    for y in Y:
        demanda_fw_ty[(t, y)] = sum(d_fw[(i, t, y)] for i in B)

peak_dem_fw = max(demanda_fw_ty.values())

print(f"Demanda pico corregida    = {peak_dem_fw:.6f}")
print(f"Margen Pmax - peak demand = {cap_total_pmax - peak_dem_fw:.6f}")

if peak_dem_fw > cap_total_pmax + 1e-6:
    print("⚠️ Aún hay demanda pico mayor que Pmax total.")
else:
    print("✅ Demanda corregida queda dentro de la capacidad instalada.")

if peak_dem_fw > min_cap_disp + 1e-6:
    print("⚠️ La demanda pico corregida puede superar la capacidad disponible P_disp en algún periodo.")
    print("   Esto puede ser normal si P_disp modela renovables variables, pero conviene revisar.")
else:
    print("✅ Demanda corregida queda dentro de la capacidad disponible mínima.")


# ============================================================
# 5) PRE-CHECKS
# ============================================================

print("\n==============================")
print("PRE-CHECKS FORWARD IEEE14 - FORMULACIÓN ORIGINAL")
print("==============================")

missing_d = [
    (i, t, y)
    for i in B
    for t in T
    for y in Y
    if (i, t, y) not in d_fw
]

if missing_d:
    print(f"⚠️ Faltan {len(missing_d)} entradas en d_fw[(i,t,y)]. Ejemplo:", missing_d[:5])
else:
    print("✅ d_fw[(i,t,y)] completo para B,T,Y.")

for y in Y:
    for t in T:
        dem_tot = sum(d_fw[(i, t, y)] for i in B)
        cap_tot = sum(float(Pmax[g]) for g in G)
        cap_disp = sum(float(P_disp.get((g, t), Pmax[g])) for g in G)

        if cap_tot + 1e-6 < dem_tot:
            print(
                f"⚠️ Pmax total {cap_tot:.2f} < demanda corregida {dem_tot:.2f} "
                f"en (t={t}, y={y})."
            )

        if cap_disp + 1e-6 < dem_tot:
            print(
                f"⚠️ Capacidad disponible {cap_disp:.2f} < demanda corregida {dem_tot:.2f} "
                f"en (t={t}, y={y})."
            )

max_sync_H = sum(float(Hg.get(g, 0.0)) for g in GS)

gamma_fwd_check = gamma_base.copy()
max_hsyn = (
    sum(float(gamma_fwd_check[m]) * float(xbar[m]) for m in M)
    if len(M) > 0
    else 0.0
)

max_H_total = max_sync_H + max_hsyn

kappa_fwd_check = kappa_base.copy()
max_R_total = (
    sum(float(kappa_fwd_check[m]) * float(xbar[m]) for m in M)
    if len(M) > 0
    else 0.0
)

print(
    f"Max H posible (sync + hsyn) = {max_H_total:.3f} "
    f"(sync={max_sync_H:.3f}, hsyn={max_hsyn:.3f})"
)

print(f"Max R posible (FFR)         = {max_R_total:.3f}")

for y in Y:
    if Hreq[y] > max_H_total + 1e-6:
        print(
            f"❌ ALERTA: Hreq[{y}]={Hreq[y]:.3f} > MaxH={max_H_total:.3f} "
            f"=> INFEASIBLE por inercia."
        )

    if Rreq[y] > max_R_total + 1e-6:
        print(
            f"❌ ALERTA: Rreq[{y}]={Rreq[y]:.3f} > MaxR={max_R_total:.3f} "
            f"=> INFEASIBLE por FFR."
        )


# ============================================================
# 6) MODELO FORWARD IEEE14 - FORMULACIÓN ORIGINAL
# ============================================================

fw_base = gp.Model("FW_base_IEEE14_original_dataset_corregido_con_pesos")
fw_base.Params.OutputFlag = 1

# Time limit opcional para el Forward. Para dejarlo sin límite, usar None.
FW_TIME_LIMIT_SECONDS = None
if FW_TIME_LIMIT_SECONDS is not None:
    fw_base.Params.TimeLimit = FW_TIME_LIMIT_SECONDS


# ============================================================
# 7) VARIABLES
# ============================================================

p = fw_base.addVars(
    G, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="p"
)

u = fw_base.addVars(
    GS, T, Y,
    lb=0.0,
    ub=1.0,
    vtype=GRB.CONTINUOUS,
    name="u"
)

s = fw_base.addVars(
    B, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="s"
)

theta = fw_base.addVars(
    B, T, Y,
    lb=-GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="theta"
)

f = fw_base.addVars(
    L_arcs, T, Y,
    lb=-GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="f"
)

x = fw_base.addVars(
    M, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="x"
)

h_syn = fw_base.addVars(
    M, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="h_syn"
)

r_ffr = fw_base.addVars(
    M, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="r_ffr"
)


# ============================================================
# 8) RESTRICCIONES
# ============================================================

# ------------------------------------------------------------
# 8.1) Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            gen = gp.quicksum(
                p[g, t, y]
                for g in G_en_bus.get(i, [])
            )

            inflow = gp.quicksum(
                f[k, i, t, y]
                for (k, i) in lineas_entran[i]
            )

            outflow = gp.quicksum(
                f[i, kk, t, y]
                for (i, kk) in lineas_salen[i]
            )

            fw_base.addConstr(
                gen - d_fw[(i, t, y)] + s[i, t, y] + inflow - outflow == 0,
                name=f"balance[{i},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.2) Cota superior ENS
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                s[i, t, y] <= d_fw[(i, t, y)],
                name=f"ens_ub[{i},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.3) Flujo DC
# ------------------------------------------------------------
for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                f[i, j, t, y]
                == Bline[(i, j)] * (theta[i, t, y] - theta[j, t, y]),
                name=f"dcflow[{i},{j},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.4) Límites de flujo
# ------------------------------------------------------------
for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                f[i, j, t, y] <= F0[(i, j)],
                name=f"f_ub[{i},{j},{t},{y}]"
            )

            fw_base.addConstr(
                f[i, j, t, y] >= -F0[(i, j)],
                name=f"f_lb[{i},{j},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.5) Bus slack
# ------------------------------------------------------------
bus_slack = int(sorted(B)[0])

for t in T:
    for y in Y:
        fw_base.addConstr(
            theta[bus_slack, t, y] == 0.0,
            name=f"slack[{t},{y}]"
        )

print(f"Bus slack usado: {bus_slack}")


# ------------------------------------------------------------
# 8.6) Disponibilidad de generación
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_gt = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            fw_base.addConstr(
                p[g, t, y] <= cap_gt,
                name=f"pdisp[{g},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.7) Restricciones de generadores síncronos
# ------------------------------------------------------------
for g in GS:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                p[g, t, y] >= float(Pmin.get(g, 0.0)) * u[g, t, y],
                name=f"sync_min[{g},{t},{y}]"
            )

            fw_base.addConstr(
                p[g, t, y] <= float(Pmax[g]) * u[g, t, y],
                name=f"sync_max[{g},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.8) Capacidad máxima de inversión IBR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        fw_base.addConstr(
            x[m, y] <= float(xbar[m]),
            name=f"x_ub[{m},{y}]"
        )


# ------------------------------------------------------------
# 8.9) Capacidad de inercia sintética
# ------------------------------------------------------------
for m in M:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                h_syn[m, t, y] <= float(gamma_base[m]) * x[m, y],
                name=f"hsyn_cap[{m},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.10) Capacidad de FFR
# ------------------------------------------------------------
for m in M:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                r_ffr[m, t, y] <= float(kappa_base[m]) * x[m, y],
                name=f"ffr_cap[{m},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.11) Requerimiento de inercia - FORMULACIÓN ORIGINAL
# ------------------------------------------------------------
for t in T:
    for y in Y:
        fw_base.addConstr(
            gp.quicksum(
                float(Hg.get(g, 0.0)) * u[g, t, y]
                for g in GS
            )
            + gp.quicksum(
                h_syn[m, t, y]
                for m in M
            )
            >= float(Hreq[y]),
            name=f"inertia_req[{t},{y}]"
        )


# ------------------------------------------------------------
# 8.12) Requerimiento de FFR
# ------------------------------------------------------------
for t in T:
    for y in Y:
        fw_base.addConstr(
            gp.quicksum(
                r_ffr[m, t, y]
                for m in M
            ) >= float(Rreq[y]),
            name=f"ffr_req[{t},{y}]"
        )


# ============================================================
# 9) ACTUALIZAR MODELO
# ============================================================

fw_base.update()

print(f"\nVariables: {fw_base.NumVars} | Restricciones: {fw_base.NumConstrs}")


# ============================================================
# 10) FUNCIÓN OBJETIVO
# ============================================================

obj_oper = gp.quicksum(
    float(omega_t[t]) * float(cg[g]) * p[g, t, y]
    for g in G
    for t in T
    for y in Y
)

obj_ens = float(VOLL) * gp.quicksum(
    float(omega_t[t]) * s[i, t, y]
    for i in B
    for t in T
    for y in Y
)

obj_inv = gp.quicksum(
    float(cinv[m]) * x[m, y]
    for m in M
    for y in Y
)

fw_base.setObjective(
    obj_oper + obj_ens + obj_inv,
    GRB.MINIMIZE
)


# ============================================================
# 11) RESOLVER
# ============================================================

fw_base.optimize()


# ============================================================
# 12) DIAGNÓSTICO BÁSICO
# ============================================================

if fw_base.Status == GRB.INFEASIBLE:
    print("\n❌ Modelo INFEASIBLE.")
    print("No se escribió archivo IIS/ILP.")

elif fw_base.Status == GRB.OPTIMAL or fw_base.SolCount > 0:
    if fw_base.Status == GRB.OPTIMAL:
        print("\n✅ Modelo resuelto óptimamente.")
    else:
        print(f"\n⚠️ Modelo terminó con status {fw_base.Status}, pero tiene solución factible.")

    print(f"Valor objetivo = {fw_base.ObjVal:,.4f}")
    if hasattr(fw_base, 'ObjBound'):
        print(f"Best bound     = {fw_base.ObjBound:,.4f}")
    if hasattr(fw_base, 'MIPGap'):
        print(f"Gap            = {fw_base.MIPGap:.6f}")

    ens_total_no_pond = sum(
        s[i, t, y].X
        for i in B
        for t in T
        for y in Y
    )

    ens_total_pond = sum(
        float(omega_t[t]) * s[i, t, y].X
        for i in B
        for t in T
        for y in Y
    )

    print(f"ENS total no ponderado = {ens_total_no_pond:.6f}")
    print(f"ENS total ponderado    = {ens_total_pond:.6f} MWh-equivalente")

    ens_total = ens_total_pond

    if ens_total <= 1e-6:
        print("✅ ENS corregido: el Forward queda sin déficit artificial.")
    else:
        print("⚠️ Aún existe ENS positivo. Revisar P_disp, red o límites de línea.")

else:
    print(f"\n⚠️ Modelo terminó con status Gurobi = {fw_base.Status}")


# ============================================================
# 13) RESULTADOS FORWARD IEEE14 - FORMULACIÓN ORIGINAL
# ============================================================

if fw_base.Status == GRB.OPTIMAL or fw_base.SolCount > 0:

    print("\n==============================")
    print("RESULTADOS FW_base IEEE14 - DATASET CORREGIDO")
    print("==============================")
    print(f"Valor objetivo total = {fw_base.ObjVal:,.4f}")
    print(f"Factor demanda usado = {demand_scale_factor:.8f}")

    # --------------------------------------------------------
    # 13.1) Resumen demanda original vs corregida
    # --------------------------------------------------------
    filas_dem = []

    for t in T:
        for y in Y:
            dem_orig = sum(d_original[(i, t, y)] for i in B)
            dem_corr = sum(d_fw[(i, t, y)] for i in B)

            filas_dem.append({
                "t": t,
                "y": y,
                "demanda_original": dem_orig,
                "demanda_corregida": dem_corr,
                "omega_t": float(omega_t[t]),
                "demanda_corregida_ponderada_MWh": float(omega_t[t]) * dem_corr,
                "factor": demand_scale_factor,
                "capacidad_Pmax_total": cap_total_pmax,
                "capacidad_disp_total": cap_disp_ty[(t, y)]
            })

    df_dem = pd.DataFrame(filas_dem)

    print("\n--- Demanda original vs demanda corregida ---")
    display(
        df_dem
        .sort_values(["y", "t"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.2) Tabla de generación p[g,t,y]
    #       y emisiones e_g * p[g,t,y]
    # --------------------------------------------------------
    filas_p = []

    for g in G:
        for t in T:
            for y in Y:
                val_p = p[g, t, y].X
                val_ep = float(eg.get(g, 0.0)) * val_p
                wt = float(omega_t[t])

                filas_p.append({
                    "gen": g,
                    "t": t,
                    "y": y,
                    "bus": g_bus[g],
                    "Es_IBR": int(g in M),
                    "emision_factor": float(eg.get(g, 0.0)),
                    "p_gty": val_p,
                    "omega_t": wt,
                    "generacion_ponderada_MWh": wt * val_p,
                    "e_g_p_gty": val_ep,
                    "e_g_p_gty_ponderado": wt * val_ep
                })

    df_p = pd.DataFrame(filas_p)

    df_p_pos = df_p[df_p["p_gty"] > 1e-6].copy()
    df_p_pos = df_p_pos.sort_values(["y", "t", "gen"]).reset_index(drop=True)

    print("\n--- Tabla p[g,t,y] y e_g * p[g,t,y] (solo valores > 0) ---")
    display(df_p_pos)

    # --------------------------------------------------------
    # 13.3) Resumen por periodo y año
    # --------------------------------------------------------
    resumen_ty = (
        df_p
        .groupby(["y", "t"], as_index=False)
        .agg(
            generacion_total_no_pond=("p_gty", "sum"),
            generacion_total_ponderada_MWh=("generacion_ponderada_MWh", "sum"),
            emisiones_totales_intervalo_no_pond=("e_g_p_gty", "sum"),
            emisiones_totales_intervalo_ponderadas=("e_g_p_gty_ponderado", "sum")
        )
    )

    print("\n--- Resumen generación/emisiones por (y,t) ---")
    display(resumen_ty)

    # --------------------------------------------------------
    # 13.4) Emisiones totales por año
    # --------------------------------------------------------
    emisiones_por_anio = (
        df_p
        .groupby("y", as_index=False)
        .agg(
            emisiones_totales_no_ponderadas=("e_g_p_gty", "sum"),
            emisiones_totales_anuales_ponderadas=("e_g_p_gty_ponderado", "sum"),
            generacion_total_no_ponderada=("p_gty", "sum"),
            generacion_total_anual_ponderada_MWh=("generacion_ponderada_MWh", "sum")
        )
    )

    print("\n--- Emisiones totales por año ---")
    display(emisiones_por_anio)

    # --------------------------------------------------------
    # 13.5) Resumen por generador
    # --------------------------------------------------------
    resumen_g = (
        df_p
        .groupby("gen", as_index=False)
        .agg(
            bus=("bus", "first"),
            Es_IBR=("Es_IBR", "first"),
            emision_factor=("emision_factor", "first"),
            generacion_total_no_pond=("p_gty", "sum"),
            generacion_total_ponderada_MWh=("generacion_ponderada_MWh", "sum"),
            emisiones_totales_no_pond=("e_g_p_gty", "sum"),
            emisiones_totales_ponderadas=("e_g_p_gty_ponderado", "sum")
        )
        .sort_values("generacion_total_ponderada_MWh", ascending=False)
        .reset_index(drop=True)
    )

    print("\n--- Resumen por generador ---")
    display(resumen_g)

    # --------------------------------------------------------
    # 13.6) ENS total y por bus
    # --------------------------------------------------------
    filas_s = []

    for i in B:
        for t in T:
            for y in Y:
                filas_s.append({
                    "bus": i,
                    "t": t,
                    "y": y,
                    "ENS": s[i, t, y].X,
                    "omega_t": float(omega_t[t]),
                    "ENS_ponderado_MWh": float(omega_t[t]) * s[i, t, y].X,
                    "demanda_corregida": d_fw[(i, t, y)],
                    "demanda_original": d_original[(i, t, y)]
                })

    df_s = pd.DataFrame(filas_s)

    total_ENS = df_s["ENS"].sum()
    print(f"\nENS total = {total_ENS:.6f}")

    df_s_pos = (
        df_s[df_s["ENS"] > 1e-6]
        .sort_values(["y", "t", "bus"])
        .reset_index(drop=True)
    )

    print("\n--- ENS positivo por bus, periodo y año ---")
    display(df_s_pos)

    # --------------------------------------------------------
    # 13.7) Inversiones IBR
    # --------------------------------------------------------
    filas_x = []

    for m in M:
        for y in Y:
            filas_x.append({
                "gen": m,
                "y": y,
                "x_my": x[m, y].X
            })

    df_x = pd.DataFrame(filas_x)

    print("\n--- Inversión x[m,y] positiva ---")
    display(
        df_x[df_x["x_my"] > 1e-6]
        .sort_values(["y", "gen"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.8) Inercia sintética h_syn
    # --------------------------------------------------------
    filas_h = []

    for m in M:
        for t in T:
            for y in Y:
                filas_h.append({
                    "gen": m,
                    "t": t,
                    "y": y,
                    "h_syn": h_syn[m, t, y].X
                })

    df_h = pd.DataFrame(filas_h)

    print("\n--- h_syn[m,t,y] positivo ---")
    display(
        df_h[df_h["h_syn"] > 1e-6]
        .sort_values(["y", "t", "gen"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.9) FFR r_ffr
    # --------------------------------------------------------
    filas_r = []

    for m in M:
        for t in T:
            for y in Y:
                filas_r.append({
                    "gen": m,
                    "t": t,
                    "y": y,
                    "r_ffr": r_ffr[m, t, y].X
                })

    df_r = pd.DataFrame(filas_r)

    print("\n--- r_ffr[m,t,y] positivo ---")
    display(
        df_r[df_r["r_ffr"] > 1e-6]
        .sort_values(["y", "t", "gen"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.10) Chequeo agregado de inercia y FFR por periodo/año
    # --------------------------------------------------------
    filas_req = []

    for t in T:
        for y in Y:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * u[g, t, y].X
                for g in GS
            )

            H_syn_val = sum(
                h_syn[m, t, y].X
                for m in M
            )

            R_val = sum(
                r_ffr[m, t, y].X
                for m in M
            )

            filas_req.append({
                "t": t,
                "y": y,
                "H_sync": H_sync_val,
                "H_syn": H_syn_val,
                "H_total": H_sync_val + H_syn_val,
                "Hreq": float(Hreq[y]),
                "margen_H": H_sync_val + H_syn_val - float(Hreq[y]),
                "R_total": R_val,
                "Rreq": float(Rreq[y]),
                "margen_R": R_val - float(Rreq[y])
            })

    df_req = pd.DataFrame(filas_req)

    print("\n--- Chequeo de requerimientos Hreq y Rreq ---")
    display(
        df_req
        .sort_values(["y", "t"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.11) Flujos de línea
    # --------------------------------------------------------
    filas_f = []

    for (i, j) in L_arcs:
        for t in T:
            for y in Y:
                val_f = f[i, j, t, y].X
                Fij = float(F0[(i, j)])

                filas_f.append({
                    "from_bus": i,
                    "to_bus": j,
                    "t": t,
                    "y": y,
                    "f_ijty": val_f,
                    "F_max": Fij,
                    "uso_linea_abs": abs(val_f) / Fij if Fij > 1e-9 else None
                })

    df_f = pd.DataFrame(filas_f)

    print("\n--- Líneas más cargadas ---")
    display(
        df_f
        .sort_values("uso_linea_abs", ascending=False)
        .head(20)
        .reset_index(drop=True)
    )

else:
    print("El modelo no quedó óptimo. Status =", fw_base.Status)

# ============================================================
# 14) IMPRIMIR PARÁMETRO X barra: xbar[m]
# ============================================================

print("\n==============================")
print("PARÁMETRO X barra por IBR")
print("==============================")
print("xbar[m] representa la capacidad máxima de inversión permitida para cada IBR m.")
print("Es decir, en el modelo se usa como: x[m,y] <= xbar[m]\n")

filas_xbar = []

for m in M:
    filas_xbar.append({
        "gen_IBR": m,
        "bus": g_bus[m] if m in g_bus else None,
        "xbar_m": float(xbar[m])
    })

df_xbar = pd.DataFrame(filas_xbar)

print("--- Valores de xbar[m] ---")
display(
    df_xbar
    .sort_values("gen_IBR")
    .reset_index(drop=True)
)


PESOS TEMPORALES FW IEEE14
Días representados       = 365
Horas por periodo t      = 1.0
Peso omega_t por periodo = 365.0
Horas equivalentes total = 1460.00

CORRECCIÓN DATASET IEEE14
Demanda pico original     = 85.470000
Capacidad instalada Pmax  = 652.600000
Capacidad disp. mínima    = 310.000000
Modo escalamiento demanda = auto_capacity
Factor demanda usado      = 1.00000000
Demanda pico corregida    = 85.470000
Margen Pmax - peak demand = 567.130000
✅ Demanda corregida queda dentro de la capacidad instalada.
✅ Demanda corregida queda dentro de la capacidad disponible mínima.

PRE-CHECKS FORWARD IEEE14 - FORMULACIÓN ORIGINAL
✅ d_fw[(i,t,y)] completo para B,T,Y.
Max H posible (sync + hsyn) = 12.852 (sync=6.000, hsyn=6.852)
Max R posible (FFR)         = 68.520
Set parameter Username
Set parameter LicenseID to value 2748035
Academic license - for non-commercial use only - expires 2026-12-01
Set parameter OutputFlag to value 1
Bus slack usado: 1

Variables: 819 | Restricciones: 1359
Gu

,t,y,demanda_original,demanda_corregida,omega_t,demanda_corregida_ponderada_MWh,factor,capacidad_Pmax_total,capacidad_disp_total
0,1,1,51.8000,51.8000,365.0,18907.0000,1.0,652.6,310.000000
1,7,1,54.3900,54.3900,365.0,19852.3500,1.0,652.6,405.806831
2,13,1,77.7000,77.7000,365.0,28360.5000,1.0,652.6,552.267486
3,19,1,75.1100,75.1100,365.0,27415.1500,1.0,652.6,310.000000
4,1,2,54.3900,54.3900,365.0,19852.3500,1.0,652.6,310.000000
5,7,2,57.1095,57.1095,365.0,20844.9675,1.0,652.6,405.806831
6,13,2,81.5850,81.5850,365.0,29778.5250,1.0,652.6,552.267486
7,19,2,78.8655,78.8655,365.0,28785.9075,1.0,652.6,310.000000
8,1,3,56.9800,56.9800,365.0,20797.7000,1.0,652.6,310.000000
9,7,3,59.8290,59.8290,365.0,21837.5850,1.0,652.6,405.806831



--- Tabla p[g,t,y] y e_g * p[g,t,y] (solo valores > 0) ---


,gen,t,y,bus,Es_IBR,emision_factor,p_gty,omega_t,generacion_ponderada_MWh,e_g_p_gty,e_g_p_gty_ponderado
0,223_STEAM_1,1,1,1,0,0.95,51.800000,365.0,18907.000000,49.210000,17961.650000
1,223_STEAM_1,7,1,1,0,0.95,36.319600,365.0,13256.654000,34.503620,12593.821300
2,314_PV_2,7,1,6,1,0.00,17.740984,365.0,6475.459016,0.000000,0.000000
3,314_PV_3,7,1,2,1,0.00,0.329416,365.0,120.236984,0.000000,0.000000
4,223_STEAM_1,13,1,1,0,0.95,36.319600,365.0,13256.654000,34.503620,12593.821300
5,313_PV_1,13,1,2,1,0.00,5.400892,365.0,1971.325508,0.000000,0.000000
6,314_PV_2,13,1,6,1,0.00,35.979508,365.0,13132.520492,0.000000,0.000000
7,223_STEAM_1,19,1,1,0,0.95,75.110000,365.0,27415.150000,71.354500,26044.392500
8,223_STEAM_1,1,2,1,0,0.95,54.390000,365.0,19852.350000,51.670500,18859.732500
9,223_STEAM_1,7,2,1,0,0.95,40.967740,365.0,14953.225100,38.919353,14205.563845



--- Resumen generación/emisiones por (y,t) ---


...

,t,y,H_sync,H_syn,H_total,Hreq,margen_H,R_total,Rreq,margen_R
0,1,1,1.75740,2.7408,4.49820,4.49820,-8.881784e-16,27.408,27.408,0.0
1,7,1,1.75740,2.7408,4.49820,4.49820,-8.881784e-16,27.408,27.408,0.0
2,13,1,1.75740,2.7408,4.49820,4.49820,-8.881784e-16,27.408,27.408,0.0
3,19,1,1.75740,2.7408,4.49820,4.49820,-8.881784e-16,27.408,27.408,0.0
4,1,2,1.98231,2.7408,4.72311,4.72311,0.000000e+00,27.408,27.408,0.0
5,7,2,1.98231,2.7408,4.72311,4.72311,0.000000e+00,27.408,27.408,0.0
6,13,2,1.98231,2.7408,4.72311,4.72311,0.000000e+00,27.408,27.408,0.0
7,19,2,1.98231,2.7408,4.72311,4.72311,0.000000e+00,27.408,27.408,0.0
8,1,3,2.20722,2.7408,4.94802,4.94802,0.000000e+00,27.408,27.408,0.0
9,7,3,2.20722,2.7408,4.94802,4.94802,0.000000e+00,27.408,27.408,0.0



--- Líneas más cargadas ---


,from_bus,to_bus,t,y,f_ijty,F_max,uso_linea_abs
0,1,2,19,3,57.881030,100.0,0.578810
1,1,2,19,2,55.250074,100.0,0.552501
2,1,2,19,1,52.619118,100.0,0.526191
3,1,2,1,3,39.917951,100.0,0.399180
4,1,2,1,2,38.103499,100.0,0.381035
5,1,2,1,1,36.289047,100.0,0.362890
6,1,2,13,3,34.637058,100.0,0.346371
7,1,2,7,3,32.949386,100.0,0.329494
8,1,2,13,2,30.575750,100.0,0.305757
9,1,2,7,2,29.827784,100.0,0.298278



PARÁMETRO X barra por IBR
xbar[m] representa la capacidad máxima de inversión permitida para cada IBR m.
Es decir, en el modelo se usa como: x[m,y] <= xbar[m]

--- Valores de xbar[m] ---


,gen_IBR,bus,xbar_m
0,313_PV_1,2,190.2
1,314_PV_1,3,103.2
2,314_PV_2,6,103.2
3,314_PV_3,2,185.4
4,314_PV_4,8,103.2


### Parámetros en b

#### Solo dualidad fuerte

In [16]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import time

# ============================================================
# FINAL IEEE14: SOLO WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO + PESOS TEMPORALES
#
# Mutables:
#   1) Hreq_ce[y]  en b
#   2) xbar_ce[m]  en b
#
# Fijo:
#   - Pmax fijo; NO se muta Pmax.
#   - No hay PADM ni warm start.
# ============================================================

# Medición de tiempo total de celda/script
t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE14
# Si ya calculaste en el FW emisiones ponderadas, puedes poner epsilon_CO2_wcep manual.
# Si lo dejas como None, el código intenta usar el 90% de las emisiones FW guardadas en memoria.
epsilon_CO2_wcep = 80041.259530 * 0.90  # 90% de las emisiones ponderadas del FW corregido (valor de ejemplo, ajustar según tu caso)
EPSILON_CO2_FRACTION = 0.90

# Objetivo J
alpha_H = 2.0
alpha_Xbar = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = True

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE14
# ============================================================
# Este bloque hace que WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# Si T = [1, 7, 13, 19] representa un perfil diario, usamos omega_t
# para expandir costos y emisiones a base anual sin crear 365*|T| periodos.
# Por defecto cada t se repite 365 veces y representa 1 hora.
# Si cada t representa un bloque de 6 horas, define TIME_BLOCK_HOURS = 6.0 antes de correr.
TIME_BLOCK_HOURS = float(globals().get("TIME_BLOCK_HOURS", 1.0))
N_DAYS_EQUIV = int(globals().get("N_DAYS_EQUIV", 365))

if "omega_t" not in globals():
    omega_t = {t: float(N_DAYS_EQUIV) * TIME_BLOCK_HOURS for t in T}
else:
    omega_t = {t: float(omega_t[t]) for t in T}

def omega(t):
    return float(omega_t[t])

print("Pesos temporales omega_t usados:", omega_t)

bus_slack = int(sorted(B)[0])

Hreq_base = {y: float(Hreq[y]) for y in Y}
xbar_base = {m: float(xbar[m]) for m in M}

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_xbar = 2.0 * max(float(cinv[m]) for m in M) if len(M) > 0 else 1e8

# ============================================================
# Límite de emisiones D
# ============================================================
def _get_fw_emissions_by_year():
    candidates = [
        "emisiones_por_anio", "df_emis_fw", "df_emis", "df_emis_wcep"
    ]
    value_cols = [
        "emisiones_anuales_equivalentes",
        "emisiones_totales_ponderadas",
        "emisiones_totales_anuales",
        "emisiones",
    ]
    for name in candidates:
        if name in globals():
            df = globals()[name]
            if hasattr(df, "columns") and "y" in df.columns:
                for col in value_cols:
                    if col in df.columns:
                        return {int(row["y"]): float(row[col]) for _, row in df.iterrows()}
    return None

_fw_emis = _get_fw_emissions_by_year()

if epsilon_CO2_wcep is None:
    if _fw_emis is not None:
        epsilon_CO2_wcep_by_y = {y: EPSILON_CO2_FRACTION * float(_fw_emis[y]) for y in Y if y in _fw_emis}
        print("epsilon_CO2_wcep calculado como 90% de emisiones FW:", epsilon_CO2_wcep_by_y)
    else:
        raise ValueError(
            "epsilon_CO2_wcep=None, pero no encontré emisiones del FW. "
            "Define epsilon_CO2_wcep manualmente o ejecuta antes el FW con tabla de emisiones."
        )
elif isinstance(epsilon_CO2_wcep, dict):
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep[y]) for y in epsilon_CO2_wcep}
else:
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep) for y in Y}

def eps_CO2(y):
    return float(epsilon_CO2_wcep_by_y[int(y)])


print("\n" + "="*70)
print("COTAS DUALES IEEE14")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_xbar = {UB_xbar:.6f}")
print(f"Bus slack usado: {bus_slack}")




# ============================================================
# 2) WCEP FINAL CON DUALIDAD FUERTE
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE14 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE14_DF_solo_dataset_corregido")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Hreq_ce = w.addVars(Y, lb=0.0, name="Hreq_ce")
xbar_ce = w.addVars(M, lb=0.0, name="xbar_ce")

for y in Y:
    base = Hreq_base[y]
    w.addConstr(Hreq_ce[y] >= 0.85 * base, name=f"adm_lb_Hreq[{y}]")
    w.addConstr(Hreq_ce[y] <= 1.15 * base, name=f"adm_ub_Hreq[{y}]")

for m in M:
    base = xbar_base[m]
    w.addConstr(xbar_ce[m] >= 0.85 * base, name=f"adm_lb_xbar[{m}]")
    w.addConstr(xbar_ce[m] <= 1.15 * base, name=f"adm_ub_xbar[{m}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(Pmax[g]) * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -xbar_ce[m], name=name)
        RHS_expr[name] = -xbar_ce[m]

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= Hreq_ce[y], name=name)
        RHS_expr[name] = Hreq_ce[y]

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(omega(t) * float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= eps_CO2(y),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sys_H_"):
        ub_here = UB_H
    elif name.startswith("x_ub_"):
        ub_here = UB_xbar

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dHreq = w.addVars(Y, lb=0.0, name="dHreq")
dXbar = w.addVars(M, lb=0.0, name="dXbar")

for y in Y:
    base = Hreq_base[y]
    diff = Hreq_ce[y] - base

    w.addConstr(dHreq[y] >= diff, name=f"dHreq_pos[{y}]")
    w.addConstr(dHreq[y] >= -diff, name=f"dHreq_neg[{y}]")

for m in M:
    base = xbar_base[m]
    diff = xbar_ce[m] - base

    w.addConstr(dXbar[m] >= diff, name=f"dXbar_pos[{m}]")
    w.addConstr(dXbar[m] >= -diff, name=f"dXbar_neg[{m}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for y in Y:
        base = abs(float(Hreq_base[y]))
        if base > 1e-9:
            obj_wcep += alpha_H * dHreq[y] / base
        else:
            obj_wcep += alpha_H * dHreq[y]

    for m in M:
        base = abs(float(xbar_base[m]))
        if base > 1e-9:
            obj_wcep += alpha_Xbar * dXbar[m] / base
        else:
            obj_wcep += alpha_Xbar * dXbar[m]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_H * dHreq.sum()
        + alpha_Xbar * dXbar.sum(),
        GRB.MINIMIZE
    )




print("\n⚠️ WCEP correrá sin PADM y sin warm start.")

# ============================================================
# 3) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE14")
print("="*70)

w.optimize()


# ============================================================
# 4) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE14")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Hreq_ce ---")
    hubo_cambio = False

    for y in Y:
        val = float(Hreq_ce[y].X)
        base = float(Hreq_base[y])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{y}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- xbar_ce ---")
    hubo_cambio = False

    for m in M:
        val = float(xbar_ce[m].X)
        base = float(xbar_base[m])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{m}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            omega(t) * float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = eps_CO2(y) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {eps_CO2(y):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_ce[y].X)

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_ce={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_xbar = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("x_ub_"):
            if abs(var.X - UB_xbar) <= 1e-6 * max(1.0, UB_xbar):
                hits_xbar += 1

    print(f"Duales sys_H pegadas a cota   : {hits_H}")
    print(f"Duales x_ub pegadas a cota    : {hits_xbar}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# REGISTRO DE RESULTADOS WCEP
# ============================================================

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo total  = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }

else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }
# Tiempo total real de toda la celda/script
t_total_script = time.time() - t0_total_script
print("\n" + "="*70)
print("TIEMPO TOTAL DE CELDA / SCRIPT")
print("="*70)
print(f"Tiempo Gurobi optimize() = {float(w.Runtime):.2f} s")
print(f"Tiempo total script      = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")

# Guardar registro en DataFrame si pandas está disponible
df_registro_wcep = pd.DataFrame([registro_wcep])
display(df_registro_wcep)


✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en WCEP = 85.470000
Pmax total sistema                 = 652.600000
Pesos temporales omega_t usados: {1: 365.0, 7: 365.0, 13: 365.0, 19: 365.0}

COTAS DUALES IEEE14
UB_H    = 324200000.000000
UB_xbar = 3242000.000000
Bus slack usado: 1

CONSTRUYENDO WCEP FINAL IEEE14 CON DUALIDAD FUERTE
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.005
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 18000

⚠️ WCEP correrá sin PADM y sin warm start.

RESOLVIENDO WCEP FINAL IEEE14
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12650H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  18000
MIPGap  0.005
NonConvex  2

Optimize a model with 1836 rows, 3046 columns and 7166 nonzeros (Min)
Model fingerprint: 0xc2476d7d
Model has 8

,status,runtime_seconds,runtime_minutes,runtime_hours,sol_count,obj_val,obj_bound,mip_gap,mip_gap_percent
0,2,0.564,0.0094,0.000157,3,0.225733,0.225672,0.000271,0.027126


#### Dualidad fuerte + PADM

In [17]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import copy
import math

# ============================================================
# FINAL IEEE14: PADM-W + WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO:
#
#   sum_g Hg[g] u[g,t,y] + sum_m h_syn[m,t,y] >= Hreq_ce[y]
#
# Es decir:
#   - NO se usa zH.
#   - Hreq_ce queda en b.
#   - Pmax fijo; NO se muta Pmax.
#   - xbar_ce queda en b.
#
# Flujo:
#   1) PADM-W etapa 1: busca factibilidad.
#   2) PADM-W etapa 2: parte de factibilidad y reduce J(theta).
#   3) WCEP completo con dualidad fuerte usando PADM_WARMSTART.
# ============================================================

# ============================================================
# 0) OPCIONES
# ============================================================

PADM_OUTPUT = 1
RUN_PADM_W = True

MAX_OUTER = 100
MAX_INNER = 100

FEAS_TOL = 1e-5
INNER_TOL = 1e-5

RHO0 = 500.0
MU0 = 500.0

PENALTY_FACTOR = 2.0
MAX_PENALTY = 1e9

TIME_LIMIT_SUBPROBLEM = 5 * 60 * 60  # 5 horas por subproblema PADM
WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE14
# Si ya calculaste en el FW emisiones ponderadas, puedes poner epsilon_CO2_wcep manual.
# Si lo dejas como None, el código intenta usar el 90% de las emisiones FW guardadas en memoria.
epsilon_CO2_wcep = 80041.259530 * 0.90  # 90% de las emisiones ponderadas del FW corregido (valor de ejemplo, ajustar según tu caso)
EPSILON_CO2_FRACTION = 0.90

# Objetivo J
alpha_H = 2.0
alpha_Xbar = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = True

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE14
# ============================================================
# Este bloque hace que PADM-W y WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por PADM-W/WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en PADM-W/WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# Si T = [1, 7, 13, 19] representa un perfil diario, usamos omega_t
# para expandir costos y emisiones a base anual sin crear 365*|T| periodos.
# Por defecto cada t se repite 365 veces y representa 1 hora.
# Si cada t representa un bloque de 6 horas, define TIME_BLOCK_HOURS = 6.0 antes de correr.
TIME_BLOCK_HOURS = float(globals().get("TIME_BLOCK_HOURS", 1.0))
N_DAYS_EQUIV = int(globals().get("N_DAYS_EQUIV", 365))

if "omega_t" not in globals():
    omega_t = {t: float(N_DAYS_EQUIV) * TIME_BLOCK_HOURS for t in T}
else:
    omega_t = {t: float(omega_t[t]) for t in T}

def omega(t):
    return float(omega_t[t])

print("Pesos temporales omega_t usados:", omega_t)

bus_slack = int(sorted(B)[0])

Hreq_base = {y: float(Hreq[y]) for y in Y}
xbar_base = {m: float(xbar[m]) for m in M}

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_xbar = 2.0 * max(float(cinv[m]) for m in M) if len(M) > 0 else 1e8

# ============================================================
# Límite de emisiones D
# ============================================================
def _get_fw_emissions_by_year():
    candidates = [
        "emisiones_por_anio", "df_emis_fw", "df_emis", "df_emis_wcep"
    ]
    value_cols = [
        "emisiones_anuales_equivalentes",
        "emisiones_totales_ponderadas",
        "emisiones_totales_anuales",
        "emisiones",
    ]
    for name in candidates:
        if name in globals():
            df = globals()[name]
            if hasattr(df, "columns") and "y" in df.columns:
                for col in value_cols:
                    if col in df.columns:
                        return {int(row["y"]): float(row[col]) for _, row in df.iterrows()}
    return None

_fw_emis = _get_fw_emissions_by_year()

if epsilon_CO2_wcep is None:
    if _fw_emis is not None:
        epsilon_CO2_wcep_by_y = {y: EPSILON_CO2_FRACTION * float(_fw_emis[y]) for y in Y if y in _fw_emis}
        print("epsilon_CO2_wcep calculado como 90% de emisiones FW:", epsilon_CO2_wcep_by_y)
    else:
        raise ValueError(
            "epsilon_CO2_wcep=None, pero no encontré emisiones del FW. "
            "Define epsilon_CO2_wcep manualmente o ejecuta antes el FW con tabla de emisiones."
        )
elif isinstance(epsilon_CO2_wcep, dict):
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep[y]) for y in epsilon_CO2_wcep}
else:
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep) for y in Y}

def eps_CO2(y):
    return float(epsilon_CO2_wcep_by_y[int(y)])


print("\n" + "="*70)
print("COTAS DUALES IEEE14")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_xbar = {UB_xbar:.6f}")
print(f"Bus slack usado: {bus_slack}")


# ============================================================
# 2) REPRESENTACIÓN SIMBÓLICA PARA PADM-W
# ============================================================

def key_p(g,t,y):       return ("p", g, t, y)
def key_u(g,t,y):       return ("u", g, t, y)
def key_s(i,t,y):       return ("s", i, t, y)
def key_xcap(m,y):      return ("xcap", m, y)
def key_hsyn(m,t,y):    return ("h_syn", m, t, y)
def key_rffr(m,t,y):    return ("r_ffr", m, t, y)
def key_fpos(i,j,t,y):  return ("f_pos", i, j, t, y)
def key_fneg(i,j,t,y):  return ("f_neg", i, j, t, y)
def key_thp(i,t,y):     return ("theta_p", i, t, y)
def key_thm(i,t,y):     return ("theta_m", i, t, y)

def param_coeff(family, idx, mult=1.0):
    return ("PARAM", family, idx, float(mult))

def is_param_coeff(c):
    return isinstance(c, tuple) and len(c) == 4 and c[0] == "PARAM"

def coeff_value(c, theta_vals):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * float(theta_vals[fam][idx])
    return float(c)

def coeff_expr_in_theta_model(c, theta_vars):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * theta_vars[fam][idx]
    return float(c)

def rhs_const(val):
    return [("CONST", None, None, float(val))]

def rhs_param(family, idx, mult=1.0):
    return [("PARAM", family, idx, float(mult))]

def rhs_has_param(rhs_terms):
    return any(typ == "PARAM" for typ, fam, idx, mult in rhs_terms)

def rhs_eval(rhs_terms, theta_vals=None):
    out = 0.0

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            out += float(mult)
        elif typ == "PARAM":
            if theta_vals is None:
                raise ValueError("rhs_eval requiere theta_vals para RHS con parámetros.")
            out += float(mult) * float(theta_vals[fam][idx])
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return out

def rhs_expr_in_theta_model(rhs_terms, theta_vars):
    expr = gp.LinExpr()

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            expr += float(mult)
        elif typ == "PARAM":
            expr += float(mult) * theta_vars[fam][idx]
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return expr

def add_to_terms(terms, key, coeff):
    if key not in terms:
        terms[key] = []
    terms[key].append(coeff)

def neg_coeff(c):
    if is_param_coeff(c):
        return ("PARAM", c[1], c[2], -c[3])
    return -float(c)

def terms_has_param(terms):
    for k, coeffs in terms.items():
        for c in coeffs:
            if is_param_coeff(c):
                return True
    return False

def terms_expr_z_model(terms, z_vars, theta_vals):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        for c in coeffs:
            expr += coeff_value(c, theta_vals) * z_vars[k]

    return expr

def terms_expr_theta_model_with_fixed_z(terms, z_vals, theta_vars):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            expr += zval * coeff_expr_in_theta_model(c, theta_vars)

    return expr

def terms_eval(terms, z_vals, theta_vals):
    out = 0.0

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            out += coeff_value(c, theta_vals) * zval

    return out


# ============================================================
# 3) CREAR FILAS PRIMALES Ax >= b
# ============================================================

rows = {}
row_order = []

def add_row(name, terms, rhs_terms):
    if name in rows:
        raise ValueError(f"Fila repetida: {name}")

    rows[name] = {
        "terms": terms,
        "rhs": rhs_terms
    }

    row_order.append(name)

# ------------------------------------------------------------
# A.1 Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}

            for g in G_en_bus.get(i, []):
                add_to_terms(terms, key_p(g,t,y), 1.0)

            add_to_terms(terms, key_s(i,t,y), 1.0)

            for (k,j) in lineas_entran[i]:
                add_to_terms(terms, key_fpos(k,j,t,y), 1.0)
                add_to_terms(terms, key_fneg(k,j,t,y), -1.0)

            for (i2,j) in lineas_salen[i]:
                add_to_terms(terms, key_fpos(i2,j,t,y), -1.0)
                add_to_terms(terms, key_fneg(i2,j,t,y), 1.0)

            rhs_val = float(d_fw[(i,t,y)])

            add_row(f"bal_p_{i}_{t}_{y}", terms, rhs_const(rhs_val))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"bal_n_{i}_{t}_{y}", terms_n, rhs_const(-rhs_val))

# ------------------------------------------------------------
# A.2 ENS <= demanda
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}
            add_to_terms(terms, key_s(i,t,y), -1.0)
            add_row(f"ens_ub_{i}_{t}_{y}", terms, rhs_const(-float(d_fw[(i,t,y)])))

# ------------------------------------------------------------
# A.3 Flujo DC
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i,j)])

            terms = {}
            add_to_terms(terms, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms, key_fneg(i,j,t,y), -1.0)

            add_to_terms(terms, key_thp(i,t,y), -Bij)
            add_to_terms(terms, key_thm(i,t,y),  Bij)
            add_to_terms(terms, key_thp(j,t,y),  Bij)
            add_to_terms(terms, key_thm(j,t,y), -Bij)

            add_row(f"dc_p_{i}_{j}_{t}_{y}", terms, rhs_const(0.0))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"dc_n_{i}_{j}_{t}_{y}", terms_n, rhs_const(0.0))

# ------------------------------------------------------------
# A.4 Límites térmicos fijos
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i,j)])

            terms_lb = {}
            add_to_terms(terms_lb, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms_lb, key_fneg(i,j,t,y), -1.0)
            add_row(f"flb_{i}_{j}_{t}_{y}", terms_lb, rhs_const(-Fij))

            terms_ub = {}
            add_to_terms(terms_ub, key_fpos(i,j,t,y), -1.0)
            add_to_terms(terms_ub, key_fneg(i,j,t,y), 1.0)
            add_row(f"fub_{i}_{j}_{t}_{y}", terms_ub, rhs_const(-Fij))

# ------------------------------------------------------------
# A.5 Slack bus
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_p = {}
        add_to_terms(terms_p, key_thp(bus_slack,t,y), 1.0)
        add_to_terms(terms_p, key_thm(bus_slack,t,y), -1.0)
        add_row(f"slack_p_{t}_{y}", terms_p, rhs_const(0.0))

        terms_n = {}
        add_to_terms(terms_n, key_thp(bus_slack,t,y), -1.0)
        add_to_terms(terms_n, key_thm(bus_slack,t,y), 1.0)
        add_row(f"slack_n_{t}_{y}", terms_n, rhs_const(0.0))

print(f"Bus slack usado: {bus_slack}")

# ------------------------------------------------------------
# A.6 Límites generación fijos p <= P_disp
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_base = float(P_disp.get((g,t), Pmax[g]))

        for y in Y:
            terms = {}
            add_to_terms(terms, key_p(g,t,y), -1.0)
            add_row(f"p_cap_{g}_{t}_{y}", terms, rhs_const(-cap_base))

# ------------------------------------------------------------
# A.7 Síncronos
# ------------------------------------------------------------
for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            terms_lb = {}
            add_to_terms(terms_lb, key_p(g,t,y), 1.0)
            add_to_terms(terms_lb, key_u(g,t,y), -pmin_g)
            add_row(f"sync_lb_{g}_{t}_{y}", terms_lb, rhs_const(0.0))

            terms_ub = {}
            add_to_terms(terms_ub, key_u(g,t,y), float(Pmax[g]))
            add_to_terms(terms_ub, key_p(g,t,y), -1.0)
            add_row(f"sync_ub_{g}_{t}_{y}", terms_ub, rhs_const(0.0))

            terms_u = {}
            add_to_terms(terms_u, key_u(g,t,y), -1.0)
            add_row(f"u_ub_{g}_{t}_{y}", terms_u, rhs_const(-1.0))

# ------------------------------------------------------------
# A.8-A.10 Inversión, h_syn, FFR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        terms_x = {}
        add_to_terms(terms_x, key_xcap(m,y), -1.0)
        add_row(f"x_ub_{m}_{y}", terms_x, rhs_param("xbar_ce", m, -1.0))

        for t in T:
            terms_h = {}
            add_to_terms(terms_h, key_xcap(m,y), float(gamma_base[m]))
            add_to_terms(terms_h, key_hsyn(m,t,y), -1.0)
            add_row(f"hsyn_lim_{m}_{t}_{y}", terms_h, rhs_const(0.0))

            terms_r = {}
            add_to_terms(terms_r, key_xcap(m,y), float(kappa_base[m]))
            add_to_terms(terms_r, key_rffr(m,t,y), -1.0)
            add_row(f"ffr_lim_{m}_{t}_{y}", terms_r, rhs_const(0.0))

# ------------------------------------------------------------
# A.11 Seguridad: Hreq_ce en b, Rreq fijo
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_H = {}

        for g in GS:
            add_to_terms(terms_H, key_u(g,t,y), float(Hg.get(g, 0.0)))

        for m in M:
            add_to_terms(terms_H, key_hsyn(m,t,y), 1.0)

        add_row(f"sys_H_{t}_{y}", terms_H, rhs_param("Hreq_ce", y, 1.0))

        terms_R = {}

        for m in M:
            add_to_terms(terms_R, key_rffr(m,t,y), 1.0)

        add_row(f"sys_R_{t}_{y}", terms_R, rhs_const(float(Rreq[y])))


# ============================================================
# 4) VARIABLES PRIMALES Y COSTOS c
# ============================================================

primal_keys = []

for g in G:
    for t in T:
        for y in Y:
            primal_keys.append(key_p(g,t,y))

for g in GS:
    for t in T:
        for y in Y:
            primal_keys.append(key_u(g,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_s(i,t,y))

for m in M:
    for y in Y:
        primal_keys.append(key_xcap(m,y))

for m in M:
    for t in T:
        for y in Y:
            primal_keys.append(key_hsyn(m,t,y))
            primal_keys.append(key_rffr(m,t,y))

for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            primal_keys.append(key_fpos(i,j,t,y))
            primal_keys.append(key_fneg(i,j,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_thp(i,t,y))
            primal_keys.append(key_thm(i,t,y))

c_map = {}

for g in G:
    for t in T:
        for y in Y:
            c_map[key_p(g,t,y)] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map[key_s(i,t,y)] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map[key_xcap(m,y)] = float(cinv[m])

for k in primal_keys:
    c_map.setdefault(k, 0.0)


# ============================================================
# 5) CLASIFICACIÓN ACOPLADAS VS SEPARABLES
# ============================================================

coupled_primal_rows = [
    name for name in row_order
    if terms_has_param(rows[name]["terms"]) or rhs_has_param(rows[name]["rhs"])
]

separable_primal_rows = [
    name for name in row_order
    if name not in set(coupled_primal_rows)
]

coupled_dual_cols = set()

for name in row_order:
    terms = rows[name]["terms"]

    for k, coeffs in terms.items():
        if any(is_param_coeff(c) for c in coeffs):
            coupled_dual_cols.add(k)

coupled_dual_cols = sorted(list(coupled_dual_cols), key=str)

separable_dual_cols = [
    k for k in primal_keys
    if k not in set(coupled_dual_cols)
]

print("\n" + "="*70)
print("CLASIFICACIÓN PADM-W IEEE14 - FORMULACIÓN ORIGINAL")
print("="*70)
print(f"N° filas primales acopladas   : {len(coupled_primal_rows)}")
print(f"N° filas primales separables  : {len(separable_primal_rows)}")
print(f"N° columnas duales acopladas  : {len(coupled_dual_cols)}")
print(f"N° columnas duales separables : {len(separable_dual_cols)}")

bad_rows = [
    name for name in coupled_primal_rows
    if not (name.startswith("sys_H_") or name.startswith("x_ub_"))
]

if bad_rows:
    print("\n⚠️ Hay filas acopladas no esperadas:")
    print(bad_rows[:30])
else:
    print("\n✅ Filas acopladas esperadas: sys_H_ y x_ub_.")


# ============================================================
# 6) FUNCIONES PADM-W
# ============================================================

def initial_theta_vals():
    return {
        "Hreq_ce": {y: float(Hreq_base[y]) for y in Y},
        "xbar_ce": {m: float(xbar_base[m]) for m in M},
    }

def compute_J(theta_vals):
    J = 0.0

    for y in Y:
        base = float(Hreq_base[y])
        val = float(theta_vals["Hreq_ce"][y])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_H * abs(val - base) / abs(base)
        else:
            J += alpha_H * abs(val - base)

    for m in M:
        base = float(xbar_base[m])
        val = float(theta_vals["xbar_ce"][m])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_Xbar * abs(val - base) / abs(base)
        else:
            J += alpha_Xbar * abs(val - base)

    return J

def compute_ctx(z_vals):
    return sum(
        float(c_map[k]) * float(z_vals.get(k, 0.0))
        for k in primal_keys
    )

def compute_bty(theta_vals, ydual_vals):
    out = 0.0

    for name in row_order:
        out += rhs_eval(rows[name]["rhs"], theta_vals) * float(ydual_vals.get(name, 0.0))

    return out

def compute_primal_residuals(theta_vals, z_vals):
    res = {}

    for name in row_order:
        lhs = terms_eval(rows[name]["terms"], z_vals, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)
        res[name] = max(0.0, rhs - lhs)

    return res

def compute_dual_residuals(theta_vals, ydual_vals):
    res = {}

    for k in primal_keys:
        lhs = 0.0

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            yd = float(ydual_vals.get(name, 0.0))

            if abs(yd) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd

        rhs = float(c_map.get(k, 0.0))
        res[k] = max(0.0, lhs - rhs)

    return res

def compute_sd_residual(theta_vals, z_vals, ydual_vals):
    return max(0.0, compute_ctx(z_vals) - compute_bty(theta_vals, ydual_vals))

def max_residuals(theta_vals, z_vals, ydual_vals):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)
    sd = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return {
        "max_primal_all": max(pr.values()) if pr else 0.0,
        "max_dual_all": max(dr.values()) if dr else 0.0,
        "sd": sd,
        "max_primal_coupled": max([pr[name] for name in coupled_primal_rows], default=0.0),
        "max_dual_coupled": max([dr[k] for k in coupled_dual_cols], default=0.0),
    }

def compute_phi(theta_vals, z_vals, ydual_vals, rho, mu, use_true_objective=True):
    Jval = compute_J(theta_vals) if use_true_objective else 0.0

    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    pen_p = sum(float(rho[name]) * pr[name] for name in coupled_primal_rows)
    pen_d = sum(float(mu[k]) * dr[k] for k in coupled_dual_cols)

    sd_penalty = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return Jval + pen_p + pen_d + 1e3 * sd_penalty

def diff_theta(theta_a, theta_b):
    vals = []

    for y in Y:
        vals.append(abs(theta_a["Hreq_ce"][y] - theta_b["Hreq_ce"][y]))

    for m in M:
        vals.append(abs(theta_a["xbar_ce"][m] - theta_b["xbar_ce"][m]))

    return max(vals) if vals else 0.0

def diff_dict(a, b):
    keys = set(a.keys()) | set(b.keys())
    return max(
        [abs(float(a.get(k,0.0)) - float(b.get(k,0.0))) for k in keys],
        default=0.0
    )


# ============================================================
# 7) SUBPROBLEMAS PADM-W
# ============================================================

def add_z_variables(model):
    z = {}

    for g in G:
        for t in T:
            for y in Y:
                z[key_p(g,t,y)] = model.addVar(lb=0.0, name=f"p[{g},{t},{y}]")

    for g in GS:
        for t in T:
            for y in Y:
                z[key_u(g,t,y)] = model.addVar(lb=0.0, ub=1.0, name=f"u[{g},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_s(i,t,y)] = model.addVar(lb=0.0, name=f"s[{i},{t},{y}]")

    for m in M:
        for y in Y:
            z[key_xcap(m,y)] = model.addVar(lb=0.0, name=f"xcap[{m},{y}]")

    for m in M:
        for t in T:
            for y in Y:
                z[key_hsyn(m,t,y)] = model.addVar(lb=0.0, name=f"h_syn[{m},{t},{y}]")
                z[key_rffr(m,t,y)] = model.addVar(lb=0.0, name=f"r_ffr[{m},{t},{y}]")

    for (i,j) in L_arcs:
        for t in T:
            for y in Y:
                z[key_fpos(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_pos[{i},{j},{t},{y}]")
                z[key_fneg(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_neg[{i},{j},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_thp(i,t,y)] = model.addVar(lb=0.0, name=f"theta_p[{i},{t},{y}]")
                z[key_thm(i,t,y)] = model.addVar(lb=0.0, name=f"theta_m[{i},{t},{y}]")

    return z

def add_dual_variables(model):
    yd = {}

    for name in row_order:
        ub_here = GRB.INFINITY

        if name.startswith("sys_H_"):
            ub_here = UB_H
        elif name.startswith("x_ub_"):
            ub_here = UB_xbar

        yd[name] = model.addVar(lb=0.0, ub=ub_here, name=f"ydual[{name}]")

    return yd

def add_theta_variables(model, warm_theta=None):
    theta = {
        "Hreq_ce": {},
        "xbar_ce": {},
    }

    for y in Y:
        theta["Hreq_ce"][y] = model.addVar(
            lb=0.85 * float(Hreq_base[y]),
            ub=1.15 * float(Hreq_base[y]),
            name=f"Hreq_ce[{y}]"
        )

    for m in M:
        theta["xbar_ce"][m] = model.addVar(
            lb=0.85 * float(xbar_base[m]),
            ub=1.15 * float(xbar_base[m]),
            name=f"xbar_ce[{m}]"
        )

    if warm_theta is not None:
        for y in Y:
            theta["Hreq_ce"][y].Start = float(warm_theta["Hreq_ce"][y])
        for m in M:
            theta["xbar_ce"][m].Start = float(warm_theta["xbar_ce"][m])

    return theta

def extract_theta(theta_vars):
    return {
        "Hreq_ce": {y: float(theta_vars["Hreq_ce"][y].X) for y in Y},
        "xbar_ce": {m: float(theta_vars["xbar_ce"][m].X) for m in M},
    }

def add_separable_primal_constraints_y(model, z_vars):
    theta_base = initial_theta_vals()

    for name in separable_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z_vars, theta_base)
        rhs = rhs_eval(rows[name]["rhs"], theta_base)
        model.addConstr(lhs >= rhs, name=f"Y_primal[{name}]")

def add_D_constraint_y(model, z_vars):
    for y in Y:
        if y >= y_star:
            model.addConstr(
                gp.quicksum(
                    omega(t) * float(eg[g]) * z_vars[key_p(g,t,y)]
                    for g in GE
                    for t in T
                )
                <= float(eps_CO2(y)),
                name=f"D_CO2_cap_{y}"
            )

def add_separable_dual_constraints_y(model, ydual_vars):
    theta_base = initial_theta_vals()

    for k in separable_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_base) * ydual_vars[name]

        model.addConstr(lhs <= float(c_map.get(k, 0.0)), name=f"Y_dual[{k}]")

def add_strong_duality_y(model, theta_vals, z_vars, ydual_vars):
    ctx = gp.quicksum(float(c_map[k]) * z_vars[k] for k in primal_keys)

    bty = gp.LinExpr()

    for name in row_order:
        bty += rhs_eval(rows[name]["rhs"], theta_vals) * ydual_vars[name]

    model.addConstr(ctx <= bty, name="Y_StrongDuality")

def solve_theta_subproblem(
    z_vals,
    ydual_vals,
    rho,
    mu,
    warm_theta=None,
    log=False,
    use_true_objective=True
):
    m = gp.Model("PADM_theta_subproblem_IEEE14_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    theta = add_theta_variables(m, warm_theta=warm_theta)

    obj = gp.LinExpr()

    if use_true_objective:
        for y in Y:
            dvar = m.addVar(lb=0.0, name=f"dHreq[{y}]")
            diff = theta["Hreq_ce"][y] - float(Hreq_base[y])

            m.addConstr(dvar >= diff, name=f"dHreq_pos[{y}]")
            m.addConstr(dvar >= -diff, name=f"dHreq_neg[{y}]")

            base = abs(float(Hreq_base[y]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_H) * dvar / base
            else:
                obj += float(alpha_H) * dvar

        for mm in M:
            dvar = m.addVar(lb=0.0, name=f"dXbar[{mm}]")
            diff = theta["xbar_ce"][mm] - float(xbar_base[mm])

            m.addConstr(dvar >= diff, name=f"dXbar_pos[{mm}]")
            m.addConstr(dvar >= -diff, name=f"dXbar_neg[{mm}]")

            base = abs(float(xbar_base[mm]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_Xbar) * dvar / base
            else:
                obj += float(alpha_Xbar) * dvar

    for name in coupled_primal_rows:
        lhs = terms_expr_theta_model_with_fixed_z(rows[name]["terms"], z_vals, theta)
        rhs = rhs_expr_in_theta_model(rows[name]["rhs"], theta)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            ydval = float(ydual_vals.get(name, 0.0))

            if abs(ydval) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += ydval * coeff_expr_in_theta_model(c, theta)

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    # Dualidad fuerte dura en theta:
    #   c^T z fijo <= b(theta)^T y fijo
    ctx_fixed = sum(float(c_map[k]) * float(z_vals.get(k, 0.0)) for k in primal_keys)

    bty_theta = gp.LinExpr()

    for name in row_order:
        ydval = float(ydual_vals.get(name, 0.0))

        if abs(ydval) <= 1e-12:
            continue

        bty_theta += ydval * rhs_expr_in_theta_model(rows[name]["rhs"], theta)

    m.addConstr(ctx_fixed <= bty_theta, name="Theta_StrongDuality")

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema THETA falló.")
        print(f"Status = {m.Status}")
        return None, m.Status, None

    theta_vals = extract_theta(theta)
    return theta_vals, m.Status, float(m.ObjVal)

def solve_y_subproblem(theta_vals, rho, mu, warm_z=None, warm_ydual=None, log=False):
    m = gp.Model("PADM_y_subproblem_IEEE14_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    z = add_z_variables(m)
    yd = add_dual_variables(m)

    m.update()

    if warm_z is not None:
        for k, v in z.items():
            if k in warm_z:
                v.Start = float(warm_z[k])

    if warm_ydual is not None:
        for name, v in yd.items():
            if name in warm_ydual:
                v.Start = float(warm_ydual[name])

    add_separable_primal_constraints_y(m, z)
    add_D_constraint_y(m, z)
    add_separable_dual_constraints_y(m, yd)
    add_strong_duality_y(m, theta_vals, z, yd)

    obj = gp.LinExpr()

    for name in coupled_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd[name]

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    if Y_OBJECTIVE_MODE == "ctx":
        obj += TIEBREAKER_CTX_WEIGHT * gp.quicksum(
            float(c_map[k]) * z[k]
            for k in primal_keys
        )

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema Y falló.")
        print(f"Status = {m.Status}")
        return None, None, m.Status, None

    z_vals = {k: float(v.X) for k, v in z.items()}
    yd_vals = {name: float(v.X) for name, v in yd.items()}

    return z_vals, yd_vals, m.Status, float(m.ObjVal)

def initialize_penalties():
    rho = {name: float(RHO0) for name in coupled_primal_rows}
    mu = {k: float(MU0) for k in coupled_dual_cols}
    return rho, mu

def update_penalties(theta_vals, z_vals, ydual_vals, rho, mu):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    n_rho = 0
    n_mu = 0

    for name in coupled_primal_rows:
        if pr[name] > FEAS_TOL:
            rho[name] = min(MAX_PENALTY, PENALTY_FACTOR * rho[name])
            n_rho += 1

    for k in coupled_dual_cols:
        if dr[k] > FEAS_TOL:
            mu[k] = min(MAX_PENALTY, PENALTY_FACTOR * mu[k])
            n_mu += 1

    return n_rho, n_mu


# ============================================================
# 8) LOOP PADM-W
# ============================================================

def run_padm(
    use_true_objective=True,
    theta_init=None,
    z_init=None,
    ydual_init=None,
    label="PADM"
):
    print("\n" + "="*70)
    print(f"INICIO {label}")
    print("="*70)
    print(f"use_true_objective = {use_true_objective}")

    rho, mu = initialize_penalties()

    theta_vals = copy.deepcopy(theta_init) if theta_init is not None else initial_theta_vals()

    if z_init is None or ydual_init is None:
        print("\nInicializando Y=(z,lambda)...")

        z_vals, ydual_vals, st_y, obj_y = solve_y_subproblem(
            theta_vals,
            rho,
            mu,
            warm_z=None,
            warm_ydual=None,
            log=False
        )

        if z_vals is None:
            print("❌ No se pudo inicializar Y.")

            return {
                "best_feasible": None,
                "theta": theta_vals,
                "z": None,
                "ydual": None,
                "history": [],
                "last_residuals": None,
                "rho": rho,
                "mu": mu,
            }

    else:
        z_vals = z_init.copy()
        ydual_vals = ydual_init.copy()

    history = []
    best_feasible = None

    res0 = max_residuals(theta_vals, z_vals, ydual_vals)

    print("\nChequeo inicial:")
    print(f"J(theta)       = {compute_J(theta_vals):.8f}")
    print(f"Max primal     = {res0['max_primal_all']:.3e}")
    print(f"Max dual       = {res0['max_dual_all']:.3e}")
    print(f"Strong duality = {res0['sd']:.3e}")

    for k_outer in range(MAX_OUTER):

        print("\n" + "#"*70)
        print(f"{label} | OUTER k = {k_outer}")
        print("#"*70)

        phi_prev = compute_phi(
            theta_vals,
            z_vals,
            ydual_vals,
            rho,
            mu,
            use_true_objective=use_true_objective
        )

        for l_inner in range(MAX_INNER):

            theta_old = copy.deepcopy(theta_vals)
            z_old = z_vals.copy()
            yd_old = ydual_vals.copy()

            theta_new, st_theta, obj_theta = solve_theta_subproblem(
                z_vals,
                ydual_vals,
                rho,
                mu,
                warm_theta=theta_vals,
                log=False,
                use_true_objective=use_true_objective
            )

            if theta_new is None:
                print("⚠️ Falló subproblema theta.")
                break

            theta_vals = theta_new

            z_new, yd_new, st_y, obj_y = solve_y_subproblem(
                theta_vals,
                rho,
                mu,
                warm_z=z_vals,
                warm_ydual=ydual_vals,
                log=False
            )

            if z_new is None:
                print("⚠️ Falló subproblema Y.")
                break

            z_vals = z_new
            ydual_vals = yd_new

            phi_now = compute_phi(
                theta_vals,
                z_vals,
                ydual_vals,
                rho,
                mu,
                use_true_objective=use_true_objective
            )

            res = max_residuals(theta_vals, z_vals, ydual_vals)

            dtheta = diff_theta(theta_vals, theta_old)
            dz = diff_dict(z_vals, z_old)
            dyd = diff_dict(ydual_vals, yd_old)
            dphi = abs(phi_now - phi_prev) / max(1.0, abs(phi_prev))

            if PADM_OUTPUT:
                print(
                    f"k={k_outer:02d}, l={l_inner:02d} | "
                    f"Phi={phi_now:.6e}, J={compute_J(theta_vals):.6e}, "
                    f"p_all={res['max_primal_all']:.2e}, "
                    f"d_all={res['max_dual_all']:.2e}, "
                    f"sd={res['sd']:.2e}, "
                    f"p_coup={res['max_primal_coupled']:.2e}, "
                    f"d_coup={res['max_dual_coupled']:.2e}, "
                    f"dtheta={dtheta:.2e}, dz={dz:.2e}, dyd={dyd:.2e}, dphi={dphi:.2e}"
                )

            history.append({
                "outer": k_outer,
                "inner": l_inner,
                "phi": phi_now,
                "J": compute_J(theta_vals),
                "max_primal_all": res["max_primal_all"],
                "max_dual_all": res["max_dual_all"],
                "sd": res["sd"],
                "max_primal_coupled": res["max_primal_coupled"],
                "max_dual_coupled": res["max_dual_coupled"],
                "dtheta": dtheta,
                "dz": dz,
                "dyd": dyd,
                "dphi": dphi,
                "rho_max": max(rho.values()) if rho else 0.0,
                "mu_max": max(mu.values()) if mu else 0.0,
            })

            if dphi <= INNER_TOL and dtheta <= INNER_TOL and dz <= INNER_TOL and dyd <= INNER_TOL:
                print(f"Inner loop converge en l={l_inner}.")
                break

            phi_prev = phi_now

        res = max_residuals(theta_vals, z_vals, ydual_vals)

        feasible = (
            res["max_primal_all"] <= FEAS_TOL
            and res["max_dual_all"] <= FEAS_TOL
            and res["sd"] <= FEAS_TOL
        )

        print("\n" + "-"*70)
        print(f"{label} | Chequeo fin outer k={k_outer}")
        print("-"*70)
        print(f"J(theta)             = {compute_J(theta_vals):.8f}")
        print(f"Max primal all       = {res['max_primal_all']:.3e}")
        print(f"Max dual all         = {res['max_dual_all']:.3e}")
        print(f"Strong duality       = {res['sd']:.3e}")
        print(f"Max primal coupled   = {res['max_primal_coupled']:.3e}")
        print(f"Max dual coupled     = {res['max_dual_coupled']:.3e}")
        print(f"rho_max              = {max(rho.values()) if rho else 0.0:.3e}")
        print(f"mu_max               = {max(mu.values()) if mu else 0.0:.3e}")

        if feasible:
            print(f"\n✅ {label} encontró solución factible primal-dual.")

            best_feasible = {
                "theta": copy.deepcopy(theta_vals),
                "z": z_vals.copy(),
                "ydual": ydual_vals.copy(),
                "J": compute_J(theta_vals),
                "residuals": res,
                "outer": k_outer,
                "history": history,
            }

            break

        n_rho, n_mu = update_penalties(theta_vals, z_vals, ydual_vals, rho, mu)

        print("\nActualización penalizaciones:")
        print(f"rho actualizadas: {n_rho}")
        print(f"mu actualizadas : {n_mu}")

    if best_feasible is None:
        print(f"\n⚠️ {label} terminó sin certificar factibilidad.")

    return {
        "best_feasible": best_feasible,
        "theta": theta_vals,
        "z": z_vals,
        "ydual": ydual_vals,
        "history": history,
        "last_residuals": max_residuals(theta_vals, z_vals, ydual_vals) if z_vals is not None else None,
        "rho": rho,
        "mu": mu,
    }


# ============================================================
# 9) EJECUTAR PADM-W
# ============================================================

PADM_WARMSTART = None

if RUN_PADM_W:

    res_ws = run_padm(
        use_true_objective=False,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM-W ETAPA 1: FACTIBILIDAD"
    )

    if res_ws["best_feasible"] is not None:

        theta_start = res_ws["best_feasible"]["theta"]
        z_start = res_ws["best_feasible"]["z"]
        yd_start = res_ws["best_feasible"]["ydual"]

        print("\n" + "="*70)
        print("WARMSTART FACTIBLE ENCONTRADO. INICIANDO ETAPA 2 CON J REAL.")
        print("="*70)

        res_final = run_padm(
            use_true_objective=True,
            theta_init=theta_start,
            z_init=z_start,
            ydual_init=yd_start,
            label="PADM-W ETAPA 2: OBJETIVO REAL"
        )

        if res_final["best_feasible"] is not None:
            final_solution = res_final["best_feasible"]
            final_label = "PADM-W ETAPA 2"
        else:
            print("\n⚠️ Etapa 2 no certificó factibilidad. Se conserva warmstart factible de etapa 1.")
            final_solution = res_ws["best_feasible"]
            final_label = "PADM-W ETAPA 1"

    else:
        print("\n⚠️ Etapa 1 no encontró punto factible.")
        final_solution = None
        final_label = "PADM-W FAILED"
        res_final = res_ws

else:
    res_final = run_padm(
        use_true_objective=True,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM"
    )

    if res_final["best_feasible"] is not None:
        final_solution = res_final["best_feasible"]
        final_label = "PADM"
    else:
        final_solution = None
        final_label = "PADM FAILED"

print("\n" + "="*70)
print("RESULTADOS FINALES PADM-W IEEE14")
print("="*70)

if final_solution is not None:
    sol_theta = final_solution["theta"]
    sol_z = final_solution["z"]
    sol_yd = final_solution["ydual"]

    print("✅ SOLUCIÓN FACTIBLE CERTIFICADA")
    print(f"Método final     : {final_label}")
    print(f"Outer encontrado : {final_solution['outer']}")
    print(f"J                : {final_solution['J']:.8f}")

    PADM_WARMSTART = {
        "theta": copy.deepcopy(sol_theta),
        "z": sol_z.copy(),
        "ydual": sol_yd.copy(),
        "label": final_label,
        "J": final_solution["J"],
        "residuals": final_solution["residuals"],
    }

else:
    print("⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.")

    sol_theta = res_final["theta"]
    sol_z = res_final["z"]
    sol_yd = res_final["ydual"]

    if sol_z is not None and sol_yd is not None:
        res = max_residuals(sol_theta, sol_z, sol_yd)

        PADM_WARMSTART = {
            "theta": copy.deepcopy(sol_theta),
            "z": sol_z.copy(),
            "ydual": sol_yd.copy(),
            "label": "PADM_LAST_POINT_NOT_CERTIFIED",
            "J": compute_J(sol_theta),
            "residuals": res,
        }

        print("⚠️ Se guardó el último punto de PADM-W como Start no certificado.")
    else:
        PADM_WARMSTART = None
        print("❌ No hay punto PADM-W disponible para warm start.")

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("CHECK PADM_WARMSTART IEEE14")
    print("="*70)

    sol_theta = PADM_WARMSTART["theta"]
    sol_z = PADM_WARMSTART["z"]
    sol_yd = PADM_WARMSTART["ydual"]

    res_check = max_residuals(sol_theta, sol_z, sol_yd)

    print(f"Origen          = {PADM_WARMSTART['label']}")
    print(f"J(theta)        = {PADM_WARMSTART['J']:.8f}")
    print(f"Max primal      = {res_check['max_primal_all']:.6e}")
    print(f"Max dual        = {res_check['max_dual_all']:.6e}")
    print(f"Strong duality  = {res_check['sd']:.6e}")


# ============================================================
# 10) WCEP FINAL CON DUALIDAD FUERTE Y WARM START PADM-W
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE14 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE14_DF_warmstart_PADM_dataset_corregido")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Hreq_ce = w.addVars(Y, lb=0.0, name="Hreq_ce")
xbar_ce = w.addVars(M, lb=0.0, name="xbar_ce")

for y in Y:
    base = Hreq_base[y]
    w.addConstr(Hreq_ce[y] >= 0.85 * base, name=f"adm_lb_Hreq[{y}]")
    w.addConstr(Hreq_ce[y] <= 1.15 * base, name=f"adm_ub_Hreq[{y}]")

for m in M:
    base = xbar_base[m]
    w.addConstr(xbar_ce[m] >= 0.85 * base, name=f"adm_lb_xbar[{m}]")
    w.addConstr(xbar_ce[m] <= 1.15 * base, name=f"adm_ub_xbar[{m}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(Pmax[g]) * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -xbar_ce[m], name=name)
        RHS_expr[name] = -xbar_ce[m]

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= Hreq_ce[y], name=name)
        RHS_expr[name] = Hreq_ce[y]

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(omega(t) * float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= eps_CO2(y),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sys_H_"):
        ub_here = UB_H
    elif name.startswith("x_ub_"):
        ub_here = UB_xbar

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dHreq = w.addVars(Y, lb=0.0, name="dHreq")
dXbar = w.addVars(M, lb=0.0, name="dXbar")

for y in Y:
    base = Hreq_base[y]
    diff = Hreq_ce[y] - base

    w.addConstr(dHreq[y] >= diff, name=f"dHreq_pos[{y}]")
    w.addConstr(dHreq[y] >= -diff, name=f"dHreq_neg[{y}]")

for m in M:
    base = xbar_base[m]
    diff = xbar_ce[m] - base

    w.addConstr(dXbar[m] >= diff, name=f"dXbar_pos[{m}]")
    w.addConstr(dXbar[m] >= -diff, name=f"dXbar_neg[{m}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for y in Y:
        base = abs(float(Hreq_base[y]))
        if base > 1e-9:
            obj_wcep += alpha_H * dHreq[y] / base
        else:
            obj_wcep += alpha_H * dHreq[y]

    for m in M:
        base = abs(float(xbar_base[m]))
        if base > 1e-9:
            obj_wcep += alpha_Xbar * dXbar[m] / base
        else:
            obj_wcep += alpha_Xbar * dXbar[m]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_H * dHreq.sum()
        + alpha_Xbar * dXbar.sum(),
        GRB.MINIMIZE
    )


# ============================================================
# 11) APLICAR WARM START DEL PADM-W
# ============================================================

def safe_start(var, value):
    try:
        if value is not None and np.isfinite(float(value)):
            var.Start = float(value)
    except Exception:
        pass

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("APLICANDO PADM_WARMSTART AL WCEP FINAL IEEE14")
    print("="*70)
    print("Origen:", PADM_WARMSTART.get("label", "PADM"))

    ws_theta = PADM_WARMSTART["theta"]
    ws_z = PADM_WARMSTART["z"]
    ws_yd = PADM_WARMSTART["ydual"]

    for y in Y:
        safe_start(Hreq_ce[y], ws_theta["Hreq_ce"][y])
        safe_start(dHreq[y], abs(ws_theta["Hreq_ce"][y] - Hreq_base[y]))

    for m in M:
        safe_start(xbar_ce[m], ws_theta["xbar_ce"][m])
        safe_start(dXbar[m], abs(ws_theta["xbar_ce"][m] - xbar_base[m]))

    for g in G:
        for t in T:
            for y in Y:
                safe_start(p[g, t, y], ws_z.get(("p", g, t, y), None))

    for g in GS:
        for t in T:
            for y in Y:
                safe_start(u[g, t, y], ws_z.get(("u", g, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(s[i, t, y], ws_z.get(("s", i, t, y), None))

    for m in M:
        for y in Y:
            safe_start(xcap[m, y], ws_z.get(("xcap", m, y), None))

    for m in M:
        for t in T:
            for y in Y:
                safe_start(h_syn[m, t, y], ws_z.get(("h_syn", m, t, y), None))
                safe_start(r_ffr[m, t, y], ws_z.get(("r_ffr", m, t, y), None))

    for (i, j) in L_arcs:
        for t in T:
            for y in Y:
                safe_start(f_pos[i, j, t, y], ws_z.get(("f_pos", i, j, t, y), None))
                safe_start(f_neg[i, j, t, y], ws_z.get(("f_neg", i, j, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(theta_p[i, t, y], ws_z.get(("theta_p", i, t, y), None))
                safe_start(theta_m[i, t, y], ws_z.get(("theta_m", i, t, y), None))

    for name, var in ydual.items():
        safe_start(var, ws_yd.get(name, None))

    print("✅ Warm start aplicado al WCEP IEEE14.")
else:
    print("\n⚠️ No existe PADM_WARMSTART. El WCEP se resolverá sin Start.")


# ============================================================
# 12) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE14")
print("="*70)

w.optimize()


# ============================================================
# 13) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE14")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Hreq_ce ---")
    hubo_cambio = False

    for y in Y:
        val = float(Hreq_ce[y].X)
        base = float(Hreq_base[y])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{y}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- xbar_ce ---")
    hubo_cambio = False

    for m in M:
        val = float(xbar_ce[m].X)
        base = float(xbar_base[m])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{m}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            omega(t) * float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = eps_CO2(y) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {eps_CO2(y):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_ce[y].X)

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_ce={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_xbar = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("x_ub_"):
            if abs(var.X - UB_xbar) <= 1e-6 * max(1.0, UB_xbar):
                hits_xbar += 1

    print(f"Duales sys_H pegadas a cota   : {hits_H}")
    print(f"Duales x_ub pegadas a cota    : {hits_xbar}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# REGISTRO DE RESULTADOS WCEP
# ============================================================

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo total  = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }

else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }

✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en PADM-W/WCEP = 85.470000
Pmax total sistema                 = 652.600000
Pesos temporales omega_t usados: {1: 365.0, 7: 365.0, 13: 365.0, 19: 365.0}

COTAS DUALES IEEE14
UB_H    = 324200000.000000
UB_xbar = 3242000.000000
Bus slack usado: 1
Bus slack usado: 1

CLASIFICACIÓN PADM-W IEEE14 - FORMULACIÓN ORIGINAL
N° filas primales acopladas   : 27
N° filas primales separables  : 1776
N° columnas duales acopladas  : 0
N° columnas duales separables : 1227

✅ Filas acopladas esperadas: sys_H_ y x_ub_.

INICIO PADM-W ETAPA 1: FACTIBILIDAD
use_true_objective = False

Inicializando Y=(z,lambda)...

Chequeo inicial:
J(theta)       = 0.00000000
Max primal     = 1.110e+00
Max dual       = 3.074e-11
Strong duality = 0.000e+00

######################################################################
PADM-W ETAPA 1: FACTIBILIDAD | OUTER k = 0
######################################################################
k=00, l=00 | Phi=5.5477

### Parámetros en A

#### Dualidad fuerte

In [18]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import time

# ============================================================
# FINAL IEEE14: SOLO WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO + PESOS TEMPORALES
#
# Mutable único:
#   1) Pmax_sync_ce[g] para g in GS, entrando en A.
#
# Fijo:
#   - Hreq y xbar fijos.
#   - No hay PADM ni warm start.
# ============================================================

# Medición de tiempo total de celda/script
t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE14
# Si ya calculaste en el FW emisiones ponderadas, puedes poner epsilon_CO2_wcep manual.
# Si lo dejas como None, el código intenta usar el 90% de las emisiones FW guardadas en memoria.
epsilon_CO2_wcep = 80041.259530 * 0.90  # 90% de las emisiones ponderadas del FW corregido (valor de ejemplo, ajustar según tu caso)
EPSILON_CO2_FRACTION = 0.90

# Objetivo J
alpha_Psync = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = True

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE14
# ============================================================
# Este bloque hace que WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# Si T = [1, 7, 13, 19] representa un perfil diario, usamos omega_t
# para expandir costos y emisiones a base anual sin crear 365*|T| periodos.
# Por defecto cada t se repite 365 veces y representa 1 hora.
# Si cada t representa un bloque de 6 horas, define TIME_BLOCK_HOURS = 6.0 antes de correr.
TIME_BLOCK_HOURS = float(globals().get("TIME_BLOCK_HOURS", 1.0))
N_DAYS_EQUIV = int(globals().get("N_DAYS_EQUIV", 365))

if "omega_t" not in globals():
    omega_t = {t: float(N_DAYS_EQUIV) * TIME_BLOCK_HOURS for t in T}
else:
    omega_t = {t: float(omega_t[t]) for t in T}

def omega(t):
    return float(omega_t[t])

print("Pesos temporales omega_t usados:", omega_t)

bus_slack = int(sorted(B)[0])

Pmax_sync_base = {g: float(Pmax[g]) for g in GS}

UB_sync = 2.0 * max(max(float(cg[g]) for g in G), float(VOLL))

# ============================================================
# Límite de emisiones D
# ============================================================
def _get_fw_emissions_by_year():
    candidates = [
        "emisiones_por_anio", "df_emis_fw", "df_emis", "df_emis_wcep"
    ]
    value_cols = [
        "emisiones_anuales_equivalentes",
        "emisiones_totales_ponderadas",
        "emisiones_totales_anuales",
        "emisiones",
    ]
    for name in candidates:
        if name in globals():
            df = globals()[name]
            if hasattr(df, "columns") and "y" in df.columns:
                for col in value_cols:
                    if col in df.columns:
                        return {int(row["y"]): float(row[col]) for _, row in df.iterrows()}
    return None

_fw_emis = _get_fw_emissions_by_year()

if epsilon_CO2_wcep is None:
    if _fw_emis is not None:
        epsilon_CO2_wcep_by_y = {y: EPSILON_CO2_FRACTION * float(_fw_emis[y]) for y in Y if y in _fw_emis}
        print("epsilon_CO2_wcep calculado como 90% de emisiones FW:", epsilon_CO2_wcep_by_y)
    else:
        raise ValueError(
            "epsilon_CO2_wcep=None, pero no encontré emisiones del FW. "
            "Define epsilon_CO2_wcep manualmente o ejecuta antes el FW con tabla de emisiones."
        )
elif isinstance(epsilon_CO2_wcep, dict):
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep[y]) for y in epsilon_CO2_wcep}
else:
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep) for y in Y}

def eps_CO2(y):
    return float(epsilon_CO2_wcep_by_y[int(y)])


print("\n" + "="*70)
print("COTAS DUALES IEEE14")
print("="*70)
print(f"UB_sync = {UB_sync:.6f}")
print(f"Bus slack usado: {bus_slack}")




# ============================================================
# 2) WCEP FINAL CON DUALIDAD FUERTE
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE14 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE14_DF_solo_dataset_corregido")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Pmax_sync_ce = w.addVars(GS, lb=0.0, name="Pmax_sync_ce")

for g in GS:
    base = Pmax_sync_base[g]
    w.addConstr(Pmax_sync_ce[g] >= 0.85 * base, name=f"adm_lb_Psync[{g}]")
    w.addConstr(Pmax_sync_ce[g] <= 1.00 * base, name=f"adm_ub_Psync[{g}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            # Bilineal porque Pmax_sync_ce[g] es mutable y u[g,t,y] es variable primal.
            R_primal[name] = w.addQConstr(
                Pmax_sync_ce[g] * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -float(xbar[m]), name=name)
        RHS_expr[name] = -float(xbar[m])

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= float(Hreq[y]), name=name)
        RHS_expr[name] = float(Hreq[y])

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(omega(t) * float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= eps_CO2(y),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sync_ub_"):
        ub_here = UB_sync

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dPsync = w.addVars(GS, lb=0.0, name="dPsync")

for g in GS:
    base = Pmax_sync_base[g]
    diff = Pmax_sync_ce[g] - base

    w.addConstr(dPsync[g] >= diff, name=f"dPsync_pos[{g}]")
    w.addConstr(dPsync[g] >= -diff, name=f"dPsync_neg[{g}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for g in GS:
        base = abs(float(Pmax_sync_base[g]))
        if base > 1e-9:
            obj_wcep += alpha_Psync * dPsync[g] / base
        else:
            obj_wcep += alpha_Psync * dPsync[g]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(alpha_Psync * dPsync.sum(), GRB.MINIMIZE)


# ============================================================
# 3) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE14")
print("="*70)

w.optimize()


# ============================================================
# 4) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE14")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Pmax_sync_ce síncronos ---")
    filas_psync = []

    for g in GS:
        val = float(Pmax_sync_ce[g].X)
        base = float(Pmax_sync_base[g])
        delta = val - base
        pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")

        filas_psync.append({
            "gen": g,
            "Pmax_original": base,
            "Pmax_sync_ce": val,
            "delta": delta,
            "pct": pct
        })

    df_psync = pd.DataFrame(filas_psync)
    df_psync_changes = df_psync[abs(df_psync["delta"]) > tol].copy()

    if len(df_psync_changes) == 0:
        print("Sin cambios relevantes.")
    else:
        display(df_psync_changes.sort_values("pct").reset_index(drop=True))

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            omega(t) * float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = eps_CO2(y) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {eps_CO2(y):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq[y])

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_sync = 0

    for name, var in ydual.items():
        if name.startswith("sync_ub_"):
            if abs(var.X - UB_sync) <= 1e-6 * max(1.0, UB_sync):
                hits_sync += 1

    print(f"Duales sync_ub pegadas a cota : {hits_sync}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# REGISTRO DE RESULTADOS WCEP
# ============================================================

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo total  = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }

else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }
# Tiempo total real de toda la celda/script
t_total_script = time.time() - t0_total_script
print("\n" + "="*70)
print("TIEMPO TOTAL DE CELDA / SCRIPT")
print("="*70)
print(f"Tiempo Gurobi optimize() = {float(w.Runtime):.2f} s")
print(f"Tiempo total script      = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")

# Guardar registro en DataFrame si pandas está disponible
df_registro_wcep = pd.DataFrame([registro_wcep])
display(df_registro_wcep)


✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en WCEP = 85.470000
Pmax total sistema                 = 652.600000
Pesos temporales omega_t usados: {1: 365.0, 7: 365.0, 13: 365.0, 19: 365.0}

COTAS DUALES IEEE14
UB_sync = 10000.000000
Bus slack usado: 1

CONSTRUYENDO WCEP FINAL IEEE14 CON DUALIDAD FUERTE
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.005
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 18000

RESOLVIENDO WCEP FINAL IEEE14
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12650H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  18000
MIPGap  0.005
NonConvex  2

Optimize a model with 1788 rows, 3034 columns and 7055 nonzeros (Min)
Model fingerprint: 0xd11ad15d
Model has 2 linear objective coefficients
Model has 1252 quadratic constraints
Coeff

,status,runtime_seconds,runtime_minutes,runtime_hours,sol_count,obj_val,obj_bound,mip_gap,mip_gap_percent
0,3,0.129,0.00215,0.000036,0,None,None,None,None


#### Dualidad fuerte + PADM

In [19]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import copy
import math
import time

# ============================================================
# FINAL IEEE14: PADM-W + WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO:
#
# Mutable único:
#   - Pmax_sync_ce[g] para g in GS, entrando en A.
#
# Es decir:
#   - Hreq y xbar quedan fijos.
#   - Pmax_sync_ce aparece en la restricción sync_ub.
#   - La fila sync_ub es acoplada y genera bilinealidades.
#
# Flujo:
#   1) PADM-W etapa 1: busca factibilidad.
#   2) PADM-W etapa 2: parte de factibilidad y reduce J(theta).
#   3) WCEP completo con dualidad fuerte usando PADM_WARMSTART.
# ============================================================

# Medición de tiempo total de celda/script
t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

PADM_OUTPUT = 1
RUN_PADM_W = True

MAX_OUTER = 100
MAX_INNER = 100

FEAS_TOL = 1e-5
INNER_TOL = 1e-5

RHO0 = 500.0
MU0 = 500.0

PENALTY_FACTOR = 2.0
MAX_PENALTY = 1e9

TIME_LIMIT_SUBPROBLEM = 5 * 60 * 60  # 5 horas por subproblema PADM
WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE14
# Si ya calculaste en el FW emisiones ponderadas, puedes poner epsilon_CO2_wcep manual.
# Si lo dejas como None, el código intenta usar el 90% de las emisiones FW guardadas en memoria.
epsilon_CO2_wcep = 80041.259530 * 0.90
EPSILON_CO2_FRACTION = 0.90

# Objetivo J
alpha_Psync = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE14
# ============================================================
# Este bloque hace que PADM-W y WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por PADM-W/WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en PADM-W/WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# Si T = [1, 7, 13, 19] representa un perfil diario, usamos omega_t
# para expandir costos y emisiones a base anual sin crear 365*|T| periodos.
# Por defecto cada t se repite 365 veces y representa 1 hora.
# Si cada t representa un bloque de 6 horas, define TIME_BLOCK_HOURS = 6.0 antes de correr.
TIME_BLOCK_HOURS = float(globals().get("TIME_BLOCK_HOURS", 1.0))
N_DAYS_EQUIV = int(globals().get("N_DAYS_EQUIV", 365))

if "omega_t" not in globals():
    omega_t = {t: float(N_DAYS_EQUIV) * TIME_BLOCK_HOURS for t in T}
else:
    omega_t = {t: float(omega_t[t]) for t in T}

def omega(t):
    return float(omega_t[t])

print("Pesos temporales omega_t usados:", omega_t)

bus_slack = int(sorted(B)[0])

Pmax_sync_base = {g: float(Pmax[g]) for g in GS}

UB_sync = 2.0 * max(max(float(cg[g]) for g in G), float(VOLL))

# ============================================================
# Límite de emisiones D
# ============================================================
def _get_fw_emissions_by_year():
    candidates = [
        "emisiones_por_anio", "df_emis_fw", "df_emis", "df_emis_wcep"
    ]
    value_cols = [
        "emisiones_anuales_equivalentes",
        "emisiones_totales_ponderadas",
        "emisiones_totales_anuales",
        "emisiones",
    ]
    for name in candidates:
        if name in globals():
            df = globals()[name]
            if hasattr(df, "columns") and "y" in df.columns:
                for col in value_cols:
                    if col in df.columns:
                        return {int(row["y"]): float(row[col]) for _, row in df.iterrows()}
    return None

_fw_emis = _get_fw_emissions_by_year()

if epsilon_CO2_wcep is None:
    if _fw_emis is not None:
        epsilon_CO2_wcep_by_y = {y: EPSILON_CO2_FRACTION * float(_fw_emis[y]) for y in Y if y in _fw_emis}
        print("epsilon_CO2_wcep calculado como 90% de emisiones FW:", epsilon_CO2_wcep_by_y)
    else:
        raise ValueError(
            "epsilon_CO2_wcep=None, pero no encontré emisiones del FW. "
            "Define epsilon_CO2_wcep manualmente o ejecuta antes el FW con tabla de emisiones."
        )
elif isinstance(epsilon_CO2_wcep, dict):
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep[y]) for y in epsilon_CO2_wcep}
else:
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep) for y in Y}

def eps_CO2(y):
    return float(epsilon_CO2_wcep_by_y[int(y)])


print("\n" + "="*70)
print("COTAS DUALES IEEE14")
print("="*70)
print(f"UB_sync = {UB_sync:.6f}")
print(f"Bus slack usado: {bus_slack}")


# ============================================================
# 2) REPRESENTACIÓN SIMBÓLICA PARA PADM-W
# ============================================================

def key_p(g,t,y):       return ("p", g, t, y)
def key_u(g,t,y):       return ("u", g, t, y)
def key_s(i,t,y):       return ("s", i, t, y)
def key_xcap(m,y):      return ("xcap", m, y)
def key_hsyn(m,t,y):    return ("h_syn", m, t, y)
def key_rffr(m,t,y):    return ("r_ffr", m, t, y)
def key_fpos(i,j,t,y):  return ("f_pos", i, j, t, y)
def key_fneg(i,j,t,y):  return ("f_neg", i, j, t, y)
def key_thp(i,t,y):     return ("theta_p", i, t, y)
def key_thm(i,t,y):     return ("theta_m", i, t, y)

def param_coeff(family, idx, mult=1.0):
    return ("PARAM", family, idx, float(mult))

def is_param_coeff(c):
    return isinstance(c, tuple) and len(c) == 4 and c[0] == "PARAM"

def coeff_value(c, theta_vals):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * float(theta_vals[fam][idx])
    return float(c)

def coeff_expr_in_theta_model(c, theta_vars):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * theta_vars[fam][idx]
    return float(c)

def rhs_const(val):
    return [("CONST", None, None, float(val))]

def rhs_param(family, idx, mult=1.0):
    return [("PARAM", family, idx, float(mult))]

def rhs_has_param(rhs_terms):
    return any(typ == "PARAM" for typ, fam, idx, mult in rhs_terms)

def rhs_eval(rhs_terms, theta_vals=None):
    out = 0.0

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            out += float(mult)
        elif typ == "PARAM":
            if theta_vals is None:
                raise ValueError("rhs_eval requiere theta_vals para RHS con parámetros.")
            out += float(mult) * float(theta_vals[fam][idx])
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return out

def rhs_expr_in_theta_model(rhs_terms, theta_vars):
    expr = gp.LinExpr()

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            expr += float(mult)
        elif typ == "PARAM":
            expr += float(mult) * theta_vars[fam][idx]
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return expr

def add_to_terms(terms, key, coeff):
    if key not in terms:
        terms[key] = []
    terms[key].append(coeff)

def neg_coeff(c):
    if is_param_coeff(c):
        return ("PARAM", c[1], c[2], -c[3])
    return -float(c)

def terms_has_param(terms):
    for k, coeffs in terms.items():
        for c in coeffs:
            if is_param_coeff(c):
                return True
    return False

def terms_expr_z_model(terms, z_vars, theta_vals):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        for c in coeffs:
            expr += coeff_value(c, theta_vals) * z_vars[k]

    return expr

def terms_expr_theta_model_with_fixed_z(terms, z_vals, theta_vars):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            expr += zval * coeff_expr_in_theta_model(c, theta_vars)

    return expr

def terms_eval(terms, z_vals, theta_vals):
    out = 0.0

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            out += coeff_value(c, theta_vals) * zval

    return out


# ============================================================
# 3) CREAR FILAS PRIMALES Ax >= b
# ============================================================

rows = {}
row_order = []

def add_row(name, terms, rhs_terms):
    if name in rows:
        raise ValueError(f"Fila repetida: {name}")

    rows[name] = {
        "terms": terms,
        "rhs": rhs_terms
    }

    row_order.append(name)

# ------------------------------------------------------------
# A.1 Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}

            for g in G_en_bus.get(i, []):
                add_to_terms(terms, key_p(g,t,y), 1.0)

            add_to_terms(terms, key_s(i,t,y), 1.0)

            for (k,j) in lineas_entran[i]:
                add_to_terms(terms, key_fpos(k,j,t,y), 1.0)
                add_to_terms(terms, key_fneg(k,j,t,y), -1.0)

            for (i2,j) in lineas_salen[i]:
                add_to_terms(terms, key_fpos(i2,j,t,y), -1.0)
                add_to_terms(terms, key_fneg(i2,j,t,y), 1.0)

            rhs_val = float(d_fw[(i,t,y)])

            add_row(f"bal_p_{i}_{t}_{y}", terms, rhs_const(rhs_val))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"bal_n_{i}_{t}_{y}", terms_n, rhs_const(-rhs_val))

# ------------------------------------------------------------
# A.2 ENS <= demanda
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}
            add_to_terms(terms, key_s(i,t,y), -1.0)
            add_row(f"ens_ub_{i}_{t}_{y}", terms, rhs_const(-float(d_fw[(i,t,y)])))

# ------------------------------------------------------------
# A.3 Flujo DC
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i,j)])

            terms = {}
            add_to_terms(terms, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms, key_fneg(i,j,t,y), -1.0)

            add_to_terms(terms, key_thp(i,t,y), -Bij)
            add_to_terms(terms, key_thm(i,t,y),  Bij)
            add_to_terms(terms, key_thp(j,t,y),  Bij)
            add_to_terms(terms, key_thm(j,t,y), -Bij)

            add_row(f"dc_p_{i}_{j}_{t}_{y}", terms, rhs_const(0.0))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"dc_n_{i}_{j}_{t}_{y}", terms_n, rhs_const(0.0))

# ------------------------------------------------------------
# A.4 Límites térmicos fijos
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i,j)])

            terms_lb = {}
            add_to_terms(terms_lb, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms_lb, key_fneg(i,j,t,y), -1.0)
            add_row(f"flb_{i}_{j}_{t}_{y}", terms_lb, rhs_const(-Fij))

            terms_ub = {}
            add_to_terms(terms_ub, key_fpos(i,j,t,y), -1.0)
            add_to_terms(terms_ub, key_fneg(i,j,t,y), 1.0)
            add_row(f"fub_{i}_{j}_{t}_{y}", terms_ub, rhs_const(-Fij))

# ------------------------------------------------------------
# A.5 Slack bus
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_p = {}
        add_to_terms(terms_p, key_thp(bus_slack,t,y), 1.0)
        add_to_terms(terms_p, key_thm(bus_slack,t,y), -1.0)
        add_row(f"slack_p_{t}_{y}", terms_p, rhs_const(0.0))

        terms_n = {}
        add_to_terms(terms_n, key_thp(bus_slack,t,y), -1.0)
        add_to_terms(terms_n, key_thm(bus_slack,t,y), 1.0)
        add_row(f"slack_n_{t}_{y}", terms_n, rhs_const(0.0))

print(f"Bus slack usado: {bus_slack}")

# ------------------------------------------------------------
# A.6 Límites generación fijos p <= P_disp
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_base = float(P_disp.get((g,t), Pmax[g]))

        for y in Y:
            terms = {}
            add_to_terms(terms, key_p(g,t,y), -1.0)
            add_row(f"p_cap_{g}_{t}_{y}", terms, rhs_const(-cap_base))

# ------------------------------------------------------------
# A.7 Síncronos
# ------------------------------------------------------------
for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            terms_lb = {}
            add_to_terms(terms_lb, key_p(g,t,y), 1.0)
            add_to_terms(terms_lb, key_u(g,t,y), -pmin_g)
            add_row(f"sync_lb_{g}_{t}_{y}", terms_lb, rhs_const(0.0))

            terms_ub = {}
            add_to_terms(terms_ub, key_u(g,t,y), param_coeff("Pmax_sync_ce", g, 1.0))
            add_to_terms(terms_ub, key_p(g,t,y), -1.0)
            add_row(f"sync_ub_{g}_{t}_{y}", terms_ub, rhs_const(0.0))

            terms_u = {}
            add_to_terms(terms_u, key_u(g,t,y), -1.0)
            add_row(f"u_ub_{g}_{t}_{y}", terms_u, rhs_const(-1.0))

# ------------------------------------------------------------
# A.8-A.10 Inversión, h_syn, FFR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        terms_x = {}
        add_to_terms(terms_x, key_xcap(m,y), -1.0)
        add_row(f"x_ub_{m}_{y}", terms_x, rhs_const(-float(xbar[m])))

        for t in T:
            terms_h = {}
            add_to_terms(terms_h, key_xcap(m,y), float(gamma_base[m]))
            add_to_terms(terms_h, key_hsyn(m,t,y), -1.0)
            add_row(f"hsyn_lim_{m}_{t}_{y}", terms_h, rhs_const(0.0))

            terms_r = {}
            add_to_terms(terms_r, key_xcap(m,y), float(kappa_base[m]))
            add_to_terms(terms_r, key_rffr(m,t,y), -1.0)
            add_row(f"ffr_lim_{m}_{t}_{y}", terms_r, rhs_const(0.0))

# ------------------------------------------------------------
# A.11 Seguridad: Hreq fijo, Rreq fijo
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_H = {}

        for g in GS:
            add_to_terms(terms_H, key_u(g,t,y), float(Hg.get(g, 0.0)))

        for m in M:
            add_to_terms(terms_H, key_hsyn(m,t,y), 1.0)

        add_row(f"sys_H_{t}_{y}", terms_H, rhs_const(float(Hreq[y])))

        terms_R = {}

        for m in M:
            add_to_terms(terms_R, key_rffr(m,t,y), 1.0)

        add_row(f"sys_R_{t}_{y}", terms_R, rhs_const(float(Rreq[y])))


# ============================================================
# 4) VARIABLES PRIMALES Y COSTOS c
# ============================================================

primal_keys = []

for g in G:
    for t in T:
        for y in Y:
            primal_keys.append(key_p(g,t,y))

for g in GS:
    for t in T:
        for y in Y:
            primal_keys.append(key_u(g,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_s(i,t,y))

for m in M:
    for y in Y:
        primal_keys.append(key_xcap(m,y))

for m in M:
    for t in T:
        for y in Y:
            primal_keys.append(key_hsyn(m,t,y))
            primal_keys.append(key_rffr(m,t,y))

for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            primal_keys.append(key_fpos(i,j,t,y))
            primal_keys.append(key_fneg(i,j,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_thp(i,t,y))
            primal_keys.append(key_thm(i,t,y))

c_map = {}

for g in G:
    for t in T:
        for y in Y:
            c_map[key_p(g,t,y)] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map[key_s(i,t,y)] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map[key_xcap(m,y)] = float(cinv[m])

for k in primal_keys:
    c_map.setdefault(k, 0.0)


# ============================================================
# 5) CLASIFICACIÓN ACOPLADAS VS SEPARABLES
# ============================================================

coupled_primal_rows = [
    name for name in row_order
    if terms_has_param(rows[name]["terms"]) or rhs_has_param(rows[name]["rhs"])
]

separable_primal_rows = [
    name for name in row_order
    if name not in set(coupled_primal_rows)
]

coupled_dual_cols = set()

for name in row_order:
    terms = rows[name]["terms"]

    for k, coeffs in terms.items():
        if any(is_param_coeff(c) for c in coeffs):
            coupled_dual_cols.add(k)

coupled_dual_cols = sorted(list(coupled_dual_cols), key=str)

separable_dual_cols = [
    k for k in primal_keys
    if k not in set(coupled_dual_cols)
]

print("\n" + "="*70)
print("CLASIFICACIÓN PADM-W IEEE14 - FORMULACIÓN ORIGINAL")
print("="*70)
print(f"N° filas primales acopladas   : {len(coupled_primal_rows)}")
print(f"N° filas primales separables  : {len(separable_primal_rows)}")
print(f"N° columnas duales acopladas  : {len(coupled_dual_cols)}")
print(f"N° columnas duales separables : {len(separable_dual_cols)}")

bad_rows = [
    name for name in coupled_primal_rows
    if not name.startswith("sync_ub_")
]

if bad_rows:
    print("\n⚠️ Hay filas acopladas no esperadas:")
    print(bad_rows[:30])
else:
    print("\n✅ Filas acopladas esperadas: sync_ub_.")


# ============================================================
# 6) FUNCIONES PADM-W
# ============================================================

def initial_theta_vals():
    return {
        "Pmax_sync_ce": {g: float(Pmax_sync_base[g]) for g in GS},
    }

def compute_J(theta_vals):
    J = 0.0

    for g in GS:
        base = float(Pmax_sync_base[g])
        val = float(theta_vals["Pmax_sync_ce"][g])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_Psync * abs(val - base) / abs(base)
        else:
            J += alpha_Psync * abs(val - base)

    return J

def compute_ctx(z_vals):
    return sum(
        float(c_map[k]) * float(z_vals.get(k, 0.0))
        for k in primal_keys
    )

def compute_bty(theta_vals, ydual_vals):
    out = 0.0

    for name in row_order:
        out += rhs_eval(rows[name]["rhs"], theta_vals) * float(ydual_vals.get(name, 0.0))

    return out

def compute_primal_residuals(theta_vals, z_vals):
    res = {}

    for name in row_order:
        lhs = terms_eval(rows[name]["terms"], z_vals, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)
        res[name] = max(0.0, rhs - lhs)

    return res

def compute_dual_residuals(theta_vals, ydual_vals):
    res = {}

    for k in primal_keys:
        lhs = 0.0

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            yd = float(ydual_vals.get(name, 0.0))

            if abs(yd) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd

        rhs = float(c_map.get(k, 0.0))
        res[k] = max(0.0, lhs - rhs)

    return res

def compute_sd_residual(theta_vals, z_vals, ydual_vals):
    return max(0.0, compute_ctx(z_vals) - compute_bty(theta_vals, ydual_vals))

def max_residuals(theta_vals, z_vals, ydual_vals):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)
    sd = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return {
        "max_primal_all": max(pr.values()) if pr else 0.0,
        "max_dual_all": max(dr.values()) if dr else 0.0,
        "sd": sd,
        "max_primal_coupled": max([pr[name] for name in coupled_primal_rows], default=0.0),
        "max_dual_coupled": max([dr[k] for k in coupled_dual_cols], default=0.0),
    }

def compute_phi(theta_vals, z_vals, ydual_vals, rho, mu, use_true_objective=True):
    Jval = compute_J(theta_vals) if use_true_objective else 0.0

    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    pen_p = sum(float(rho[name]) * pr[name] for name in coupled_primal_rows)
    pen_d = sum(float(mu[k]) * dr[k] for k in coupled_dual_cols)

    sd_penalty = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return Jval + pen_p + pen_d + 1e3 * sd_penalty

def diff_theta(theta_a, theta_b):
    vals = []

    for g in GS:
        vals.append(abs(theta_a["Pmax_sync_ce"][g] - theta_b["Pmax_sync_ce"][g]))

    return max(vals) if vals else 0.0

def diff_dict(a, b):
    keys = set(a.keys()) | set(b.keys())
    return max(
        [abs(float(a.get(k,0.0)) - float(b.get(k,0.0))) for k in keys],
        default=0.0
    )


# ============================================================
# 7) SUBPROBLEMAS PADM-W
# ============================================================

def add_z_variables(model):
    z = {}

    for g in G:
        for t in T:
            for y in Y:
                z[key_p(g,t,y)] = model.addVar(lb=0.0, name=f"p[{g},{t},{y}]")

    for g in GS:
        for t in T:
            for y in Y:
                z[key_u(g,t,y)] = model.addVar(lb=0.0, ub=1.0, name=f"u[{g},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_s(i,t,y)] = model.addVar(lb=0.0, name=f"s[{i},{t},{y}]")

    for m in M:
        for y in Y:
            z[key_xcap(m,y)] = model.addVar(lb=0.0, name=f"xcap[{m},{y}]")

    for m in M:
        for t in T:
            for y in Y:
                z[key_hsyn(m,t,y)] = model.addVar(lb=0.0, name=f"h_syn[{m},{t},{y}]")
                z[key_rffr(m,t,y)] = model.addVar(lb=0.0, name=f"r_ffr[{m},{t},{y}]")

    for (i,j) in L_arcs:
        for t in T:
            for y in Y:
                z[key_fpos(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_pos[{i},{j},{t},{y}]")
                z[key_fneg(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_neg[{i},{j},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_thp(i,t,y)] = model.addVar(lb=0.0, name=f"theta_p[{i},{t},{y}]")
                z[key_thm(i,t,y)] = model.addVar(lb=0.0, name=f"theta_m[{i},{t},{y}]")

    return z

def add_dual_variables(model):
    yd = {}

    for name in row_order:
        ub_here = GRB.INFINITY

        if name.startswith("sync_ub_"):
            ub_here = UB_sync

        yd[name] = model.addVar(lb=0.0, ub=ub_here, name=f"ydual[{name}]")

    return yd

def add_theta_variables(model, warm_theta=None):
    theta = {
        "Pmax_sync_ce": {},
    }

    for g in GS:
        theta["Pmax_sync_ce"][g] = model.addVar(
            lb=0.80 * float(Pmax_sync_base[g]),
            ub=1.00 * float(Pmax_sync_base[g]),
            name=f"Pmax_sync_ce[{g}]"
        )

    if warm_theta is not None:
        for g in GS:
            theta["Pmax_sync_ce"][g].Start = float(warm_theta["Pmax_sync_ce"][g])

    return theta

def extract_theta(theta_vars):
    return {
        "Pmax_sync_ce": {g: float(theta_vars["Pmax_sync_ce"][g].X) for g in GS},
    }

def add_separable_primal_constraints_y(model, z_vars):
    theta_base = initial_theta_vals()

    for name in separable_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z_vars, theta_base)
        rhs = rhs_eval(rows[name]["rhs"], theta_base)
        model.addConstr(lhs >= rhs, name=f"Y_primal[{name}]")

def add_D_constraint_y(model, z_vars):
    for y in Y:
        if y >= y_star:
            model.addConstr(
                gp.quicksum(
                    omega(t) * float(eg[g]) * z_vars[key_p(g,t,y)]
                    for g in GE
                    for t in T
                )
                <= float(eps_CO2(y)),
                name=f"D_CO2_cap_{y}"
            )

def add_separable_dual_constraints_y(model, ydual_vars):
    theta_base = initial_theta_vals()

    for k in separable_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_base) * ydual_vars[name]

        model.addConstr(lhs <= float(c_map.get(k, 0.0)), name=f"Y_dual[{k}]")

def add_strong_duality_y(model, theta_vals, z_vars, ydual_vars):
    ctx = gp.quicksum(float(c_map[k]) * z_vars[k] for k in primal_keys)

    bty = gp.LinExpr()

    for name in row_order:
        bty += rhs_eval(rows[name]["rhs"], theta_vals) * ydual_vars[name]

    model.addConstr(ctx <= bty, name="Y_StrongDuality")

def solve_theta_subproblem(
    z_vals,
    ydual_vals,
    rho,
    mu,
    warm_theta=None,
    log=False,
    use_true_objective=True
):
    m = gp.Model("PADM_theta_subproblem_IEEE14_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    theta = add_theta_variables(m, warm_theta=warm_theta)

    obj = gp.LinExpr()

    if use_true_objective:
        for g in GS:
            dvar = m.addVar(lb=0.0, name=f"dPsync[{g}]")
            diff = theta["Pmax_sync_ce"][g] - float(Pmax_sync_base[g])

            m.addConstr(dvar >= diff, name=f"dPsync_pos[{g}]")
            m.addConstr(dvar >= -diff, name=f"dPsync_neg[{g}]")

            base = abs(float(Pmax_sync_base[g]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_Psync) * dvar / base
            else:
                obj += float(alpha_Psync) * dvar

    for name in coupled_primal_rows:
        lhs = terms_expr_theta_model_with_fixed_z(rows[name]["terms"], z_vals, theta)
        rhs = rhs_expr_in_theta_model(rows[name]["rhs"], theta)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            ydval = float(ydual_vals.get(name, 0.0))

            if abs(ydval) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += ydval * coeff_expr_in_theta_model(c, theta)

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    # Dualidad fuerte dura en theta:
    #   c^T z fijo <= b(theta)^T y fijo
    ctx_fixed = sum(float(c_map[k]) * float(z_vals.get(k, 0.0)) for k in primal_keys)

    bty_theta = gp.LinExpr()

    for name in row_order:
        ydval = float(ydual_vals.get(name, 0.0))

        if abs(ydval) <= 1e-12:
            continue

        bty_theta += ydval * rhs_expr_in_theta_model(rows[name]["rhs"], theta)

    m.addConstr(ctx_fixed <= bty_theta, name="Theta_StrongDuality")

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema THETA falló.")
        print(f"Status = {m.Status}")
        return None, m.Status, None

    theta_vals = extract_theta(theta)
    return theta_vals, m.Status, float(m.ObjVal)

def solve_y_subproblem(theta_vals, rho, mu, warm_z=None, warm_ydual=None, log=False):
    m = gp.Model("PADM_y_subproblem_IEEE14_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    z = add_z_variables(m)
    yd = add_dual_variables(m)

    m.update()

    if warm_z is not None:
        for k, v in z.items():
            if k in warm_z:
                v.Start = float(warm_z[k])

    if warm_ydual is not None:
        for name, v in yd.items():
            if name in warm_ydual:
                v.Start = float(warm_ydual[name])

    add_separable_primal_constraints_y(m, z)
    add_D_constraint_y(m, z)
    add_separable_dual_constraints_y(m, yd)
    add_strong_duality_y(m, theta_vals, z, yd)

    obj = gp.LinExpr()

    for name in coupled_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd[name]

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    if Y_OBJECTIVE_MODE == "ctx":
        obj += TIEBREAKER_CTX_WEIGHT * gp.quicksum(
            float(c_map[k]) * z[k]
            for k in primal_keys
        )

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema Y falló.")
        print(f"Status = {m.Status}")
        return None, None, m.Status, None

    z_vals = {k: float(v.X) for k, v in z.items()}
    yd_vals = {name: float(v.X) for name, v in yd.items()}

    return z_vals, yd_vals, m.Status, float(m.ObjVal)

def initialize_penalties():
    rho = {name: float(RHO0) for name in coupled_primal_rows}
    mu = {k: float(MU0) for k in coupled_dual_cols}
    return rho, mu

def update_penalties(theta_vals, z_vals, ydual_vals, rho, mu):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    n_rho = 0
    n_mu = 0

    for name in coupled_primal_rows:
        if pr[name] > FEAS_TOL:
            rho[name] = min(MAX_PENALTY, PENALTY_FACTOR * rho[name])
            n_rho += 1

    for k in coupled_dual_cols:
        if dr[k] > FEAS_TOL:
            mu[k] = min(MAX_PENALTY, PENALTY_FACTOR * mu[k])
            n_mu += 1

    return n_rho, n_mu


# ============================================================
# 8) LOOP PADM-W
# ============================================================

def run_padm(
    use_true_objective=True,
    theta_init=None,
    z_init=None,
    ydual_init=None,
    label="PADM"
):
    print("\n" + "="*70)
    print(f"INICIO {label}")
    print("="*70)
    print(f"use_true_objective = {use_true_objective}")

    rho, mu = initialize_penalties()

    theta_vals = copy.deepcopy(theta_init) if theta_init is not None else initial_theta_vals()

    if z_init is None or ydual_init is None:
        print("\nInicializando Y=(z,lambda)...")

        z_vals, ydual_vals, st_y, obj_y = solve_y_subproblem(
            theta_vals,
            rho,
            mu,
            warm_z=None,
            warm_ydual=None,
            log=False
        )

        if z_vals is None:
            print("❌ No se pudo inicializar Y.")

            return {
                "best_feasible": None,
                "theta": theta_vals,
                "z": None,
                "ydual": None,
                "history": [],
                "last_residuals": None,
                "rho": rho,
                "mu": mu,
            }

    else:
        z_vals = z_init.copy()
        ydual_vals = ydual_init.copy()

    history = []
    best_feasible = None

    res0 = max_residuals(theta_vals, z_vals, ydual_vals)

    print("\nChequeo inicial:")
    print(f"J(theta)       = {compute_J(theta_vals):.8f}")
    print(f"Max primal     = {res0['max_primal_all']:.3e}")
    print(f"Max dual       = {res0['max_dual_all']:.3e}")
    print(f"Strong duality = {res0['sd']:.3e}")

    for k_outer in range(MAX_OUTER):

        print("\n" + "#"*70)
        print(f"{label} | OUTER k = {k_outer}")
        print("#"*70)

        phi_prev = compute_phi(
            theta_vals,
            z_vals,
            ydual_vals,
            rho,
            mu,
            use_true_objective=use_true_objective
        )

        for l_inner in range(MAX_INNER):

            theta_old = copy.deepcopy(theta_vals)
            z_old = z_vals.copy()
            yd_old = ydual_vals.copy()

            theta_new, st_theta, obj_theta = solve_theta_subproblem(
                z_vals,
                ydual_vals,
                rho,
                mu,
                warm_theta=theta_vals,
                log=False,
                use_true_objective=use_true_objective
            )

            if theta_new is None:
                print("⚠️ Falló subproblema theta.")
                break

            theta_vals = theta_new

            z_new, yd_new, st_y, obj_y = solve_y_subproblem(
                theta_vals,
                rho,
                mu,
                warm_z=z_vals,
                warm_ydual=ydual_vals,
                log=False
            )

            if z_new is None:
                print("⚠️ Falló subproblema Y.")
                break

            z_vals = z_new
            ydual_vals = yd_new

            phi_now = compute_phi(
                theta_vals,
                z_vals,
                ydual_vals,
                rho,
                mu,
                use_true_objective=use_true_objective
            )

            res = max_residuals(theta_vals, z_vals, ydual_vals)

            dtheta = diff_theta(theta_vals, theta_old)
            dz = diff_dict(z_vals, z_old)
            dyd = diff_dict(ydual_vals, yd_old)
            dphi = abs(phi_now - phi_prev) / max(1.0, abs(phi_prev))

            if PADM_OUTPUT:
                print(
                    f"k={k_outer:02d}, l={l_inner:02d} | "
                    f"Phi={phi_now:.6e}, J={compute_J(theta_vals):.6e}, "
                    f"p_all={res['max_primal_all']:.2e}, "
                    f"d_all={res['max_dual_all']:.2e}, "
                    f"sd={res['sd']:.2e}, "
                    f"p_coup={res['max_primal_coupled']:.2e}, "
                    f"d_coup={res['max_dual_coupled']:.2e}, "
                    f"dtheta={dtheta:.2e}, dz={dz:.2e}, dyd={dyd:.2e}, dphi={dphi:.2e}"
                )

            history.append({
                "outer": k_outer,
                "inner": l_inner,
                "phi": phi_now,
                "J": compute_J(theta_vals),
                "max_primal_all": res["max_primal_all"],
                "max_dual_all": res["max_dual_all"],
                "sd": res["sd"],
                "max_primal_coupled": res["max_primal_coupled"],
                "max_dual_coupled": res["max_dual_coupled"],
                "dtheta": dtheta,
                "dz": dz,
                "dyd": dyd,
                "dphi": dphi,
                "rho_max": max(rho.values()) if rho else 0.0,
                "mu_max": max(mu.values()) if mu else 0.0,
            })

            if dphi <= INNER_TOL and dtheta <= INNER_TOL and dz <= INNER_TOL and dyd <= INNER_TOL:
                print(f"Inner loop converge en l={l_inner}.")
                break

            phi_prev = phi_now

        res = max_residuals(theta_vals, z_vals, ydual_vals)

        feasible = (
            res["max_primal_all"] <= FEAS_TOL
            and res["max_dual_all"] <= FEAS_TOL
            and res["sd"] <= FEAS_TOL
        )

        print("\n" + "-"*70)
        print(f"{label} | Chequeo fin outer k={k_outer}")
        print("-"*70)
        print(f"J(theta)             = {compute_J(theta_vals):.8f}")
        print(f"Max primal all       = {res['max_primal_all']:.3e}")
        print(f"Max dual all         = {res['max_dual_all']:.3e}")
        print(f"Strong duality       = {res['sd']:.3e}")
        print(f"Max primal coupled   = {res['max_primal_coupled']:.3e}")
        print(f"Max dual coupled     = {res['max_dual_coupled']:.3e}")
        print(f"rho_max              = {max(rho.values()) if rho else 0.0:.3e}")
        print(f"mu_max               = {max(mu.values()) if mu else 0.0:.3e}")

        if feasible:
            print(f"\n✅ {label} encontró solución factible primal-dual.")

            best_feasible = {
                "theta": copy.deepcopy(theta_vals),
                "z": z_vals.copy(),
                "ydual": ydual_vals.copy(),
                "J": compute_J(theta_vals),
                "residuals": res,
                "outer": k_outer,
                "history": history,
            }

            break

        n_rho, n_mu = update_penalties(theta_vals, z_vals, ydual_vals, rho, mu)

        print("\nActualización penalizaciones:")
        print(f"rho actualizadas: {n_rho}")
        print(f"mu actualizadas : {n_mu}")

    if best_feasible is None:
        print(f"\n⚠️ {label} terminó sin certificar factibilidad.")

    return {
        "best_feasible": best_feasible,
        "theta": theta_vals,
        "z": z_vals,
        "ydual": ydual_vals,
        "history": history,
        "last_residuals": max_residuals(theta_vals, z_vals, ydual_vals) if z_vals is not None else None,
        "rho": rho,
        "mu": mu,
    }


# ============================================================
# 9) EJECUTAR PADM-W
# ============================================================

PADM_WARMSTART = None

if RUN_PADM_W:

    res_ws = run_padm(
        use_true_objective=False,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM-W ETAPA 1: FACTIBILIDAD"
    )

    if res_ws["best_feasible"] is not None:

        theta_start = res_ws["best_feasible"]["theta"]
        z_start = res_ws["best_feasible"]["z"]
        yd_start = res_ws["best_feasible"]["ydual"]

        print("\n" + "="*70)
        print("WARMSTART FACTIBLE ENCONTRADO. INICIANDO ETAPA 2 CON J REAL.")
        print("="*70)

        res_final = run_padm(
            use_true_objective=True,
            theta_init=theta_start,
            z_init=z_start,
            ydual_init=yd_start,
            label="PADM-W ETAPA 2: OBJETIVO REAL"
        )

        if res_final["best_feasible"] is not None:
            final_solution = res_final["best_feasible"]
            final_label = "PADM-W ETAPA 2"
        else:
            print("\n⚠️ Etapa 2 no certificó factibilidad. Se conserva warmstart factible de etapa 1.")
            final_solution = res_ws["best_feasible"]
            final_label = "PADM-W ETAPA 1"

    else:
        print("\n⚠️ Etapa 1 no encontró punto factible.")
        final_solution = None
        final_label = "PADM-W FAILED"
        res_final = res_ws

else:
    res_final = run_padm(
        use_true_objective=True,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM"
    )

    if res_final["best_feasible"] is not None:
        final_solution = res_final["best_feasible"]
        final_label = "PADM"
    else:
        final_solution = None
        final_label = "PADM FAILED"

print("\n" + "="*70)
print("RESULTADOS FINALES PADM-W IEEE14")
print("="*70)

if final_solution is not None:
    sol_theta = final_solution["theta"]
    sol_z = final_solution["z"]
    sol_yd = final_solution["ydual"]

    print("✅ SOLUCIÓN FACTIBLE CERTIFICADA")
    print(f"Método final     : {final_label}")
    print(f"Outer encontrado : {final_solution['outer']}")
    print(f"J                : {final_solution['J']:.8f}")

    PADM_WARMSTART = {
        "theta": copy.deepcopy(sol_theta),
        "z": sol_z.copy(),
        "ydual": sol_yd.copy(),
        "label": final_label,
        "J": final_solution["J"],
        "residuals": final_solution["residuals"],
    }

else:
    print("⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.")

    sol_theta = res_final["theta"]
    sol_z = res_final["z"]
    sol_yd = res_final["ydual"]

    if sol_z is not None and sol_yd is not None:
        res = max_residuals(sol_theta, sol_z, sol_yd)

        PADM_WARMSTART = {
            "theta": copy.deepcopy(sol_theta),
            "z": sol_z.copy(),
            "ydual": sol_yd.copy(),
            "label": "PADM_LAST_POINT_NOT_CERTIFIED",
            "J": compute_J(sol_theta),
            "residuals": res,
        }

        print("⚠️ Se guardó el último punto de PADM-W como Start no certificado.")
    else:
        PADM_WARMSTART = None
        print("❌ No hay punto PADM-W disponible para warm start.")

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("CHECK PADM_WARMSTART IEEE14")
    print("="*70)

    sol_theta = PADM_WARMSTART["theta"]
    sol_z = PADM_WARMSTART["z"]
    sol_yd = PADM_WARMSTART["ydual"]

    res_check = max_residuals(sol_theta, sol_z, sol_yd)

    print(f"Origen          = {PADM_WARMSTART['label']}")
    print(f"J(theta)        = {PADM_WARMSTART['J']:.8f}")
    print(f"Max primal      = {res_check['max_primal_all']:.6e}")
    print(f"Max dual        = {res_check['max_dual_all']:.6e}")
    print(f"Strong duality  = {res_check['sd']:.6e}")


# ============================================================
# 10) WCEP FINAL CON DUALIDAD FUERTE Y WARM START PADM-W
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE14 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE14_DF_warmstart_PADM_Psync_dataset_corregido")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Pmax_sync_ce = w.addVars(GS, lb=0.0, name="Pmax_sync_ce")

for g in GS:
    base = Pmax_sync_base[g]
    w.addConstr(Pmax_sync_ce[g] >= 0.80 * base, name=f"adm_lb_Psync[{g}]")
    w.addConstr(Pmax_sync_ce[g] <= 1.00 * base, name=f"adm_ub_Psync[{g}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            # Bilineal porque Pmax_sync_ce[g] es mutable y u[g,t,y] es variable primal.
            R_primal[name] = w.addQConstr(
                Pmax_sync_ce[g] * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -float(xbar[m]), name=name)
        RHS_expr[name] = -float(xbar[m])

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= float(Hreq[y]), name=name)
        RHS_expr[name] = float(Hreq[y])

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(omega(t) * float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= eps_CO2(y),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sync_ub_"):
        ub_here = UB_sync

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dPsync = w.addVars(GS, lb=0.0, name="dPsync")

for g in GS:
    base = Pmax_sync_base[g]
    diff = Pmax_sync_ce[g] - base

    w.addConstr(dPsync[g] >= diff, name=f"dPsync_pos[{g}]")
    w.addConstr(dPsync[g] >= -diff, name=f"dPsync_neg[{g}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for g in GS:
        base = abs(float(Pmax_sync_base[g]))
        if base > 1e-9:
            obj_wcep += alpha_Psync * dPsync[g] / base
        else:
            obj_wcep += alpha_Psync * dPsync[g]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(alpha_Psync * dPsync.sum(), GRB.MINIMIZE)


# ============================================================
# 11) APLICAR WARM START DEL PADM-W
# ============================================================

def safe_start(var, value):
    try:
        if value is not None and np.isfinite(float(value)):
            var.Start = float(value)
    except Exception:
        pass

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("APLICANDO PADM_WARMSTART AL WCEP FINAL IEEE14")
    print("="*70)
    print("Origen:", PADM_WARMSTART.get("label", "PADM"))

    ws_theta = PADM_WARMSTART["theta"]
    ws_z = PADM_WARMSTART["z"]
    ws_yd = PADM_WARMSTART["ydual"]

    for g in GS:
        safe_start(Pmax_sync_ce[g], ws_theta["Pmax_sync_ce"][g])
        safe_start(dPsync[g], abs(ws_theta["Pmax_sync_ce"][g] - Pmax_sync_base[g]))

    for g in G:
        for t in T:
            for y in Y:
                safe_start(p[g, t, y], ws_z.get(("p", g, t, y), None))

    for g in GS:
        for t in T:
            for y in Y:
                safe_start(u[g, t, y], ws_z.get(("u", g, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(s[i, t, y], ws_z.get(("s", i, t, y), None))

    for m in M:
        for y in Y:
            safe_start(xcap[m, y], ws_z.get(("xcap", m, y), None))

    for m in M:
        for t in T:
            for y in Y:
                safe_start(h_syn[m, t, y], ws_z.get(("h_syn", m, t, y), None))
                safe_start(r_ffr[m, t, y], ws_z.get(("r_ffr", m, t, y), None))

    for (i, j) in L_arcs:
        for t in T:
            for y in Y:
                safe_start(f_pos[i, j, t, y], ws_z.get(("f_pos", i, j, t, y), None))
                safe_start(f_neg[i, j, t, y], ws_z.get(("f_neg", i, j, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(theta_p[i, t, y], ws_z.get(("theta_p", i, t, y), None))
                safe_start(theta_m[i, t, y], ws_z.get(("theta_m", i, t, y), None))

    for name, var in ydual.items():
        safe_start(var, ws_yd.get(name, None))

    print("✅ Warm start aplicado al WCEP IEEE14.")
else:
    print("\n⚠️ No existe PADM_WARMSTART. El WCEP se resolverá sin Start.")


# ============================================================
# 12) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE14")
print("="*70)

w.optimize()


# ============================================================
# 13) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE14")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Pmax_sync_ce síncronos ---")
    hubo_cambio = False

    for g in GS:
        val = float(Pmax_sync_ce[g].X)
        base = float(Pmax_sync_base[g])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{g}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            omega(t) * float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = eps_CO2(y) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {eps_CO2(y):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq[y])

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_sync = 0

    for name, var in ydual.items():
        if name.startswith("sync_ub_"):
            if abs(var.X - UB_sync) <= 1e-6 * max(1.0, UB_sync):
                hits_sync += 1

    print(f"Duales sync_ub pegadas a cota : {hits_sync}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# REGISTRO DE RESULTADOS WCEP
# ============================================================

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP FINAL")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo Gurobi optimize() = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds_gurobi": runtime,
        "runtime_minutes_gurobi": runtime / 60,
        "runtime_hours_gurobi": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }
else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds_gurobi": runtime,
        "runtime_minutes_gurobi": runtime / 60,
        "runtime_hours_gurobi": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }

t_total_script = time.time() - t0_total_script
registro_wcep["runtime_seconds_total_script"] = t_total_script
registro_wcep["runtime_minutes_total_script"] = t_total_script / 60
registro_wcep["runtime_hours_total_script"] = t_total_script / 3600

print(f"Tiempo total script = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")

df_registro_wcep = pd.DataFrame([registro_wcep])
display(df_registro_wcep)


✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en PADM-W/WCEP = 85.470000
Pmax total sistema                 = 652.600000
Pesos temporales omega_t usados: {1: 365.0, 7: 365.0, 13: 365.0, 19: 365.0}

COTAS DUALES IEEE14
UB_sync = 10000.000000
Bus slack usado: 1
Bus slack usado: 1

CLASIFICACIÓN PADM-W IEEE14 - FORMULACIÓN ORIGINAL
N° filas primales acopladas   : 24
N° filas primales separables  : 1779
N° columnas duales acopladas  : 24
N° columnas duales separables : 1203

✅ Filas acopladas esperadas: sync_ub_.

INICIO PADM-W ETAPA 1: FACTIBILIDAD
use_true_objective = False

Inicializando Y=(z,lambda)...

Chequeo inicial:
J(theta)       = 0.00000000
Max primal     = 3.020e-14
Max dual       = 5.686e+07
Strong duality = 0.000e+00

######################################################################
PADM-W ETAPA 1: FACTIBILIDAD | OUTER k = 0
######################################################################
k=00, l=00 | Phi=5.685716e+10, J=0.000000e+00, p_all=3.02

,status,runtime_seconds_gurobi,runtime_minutes_gurobi,runtime_hours_gurobi,sol_count,obj_val,obj_bound,mip_gap,mip_gap_percent,runtime_seconds_total_script,runtime_minutes_total_script,runtime_hours_total_script
0,3,0.139,0.002317,0.000039,0,None,None,None,None,84.91625,1.415271,0.023588


# Sistema IEEE 39-bus


# Dataset

## Demanda Nodal

In [20]:
from pathlib import Path
import pandas as pd

# Ruta al RAW
ruta_raw = Path("IEEE 39 bus.RAW")

# Leer archivo completo
with open(ruta_raw, "r", encoding="utf-8", errors="ignore") as f:
    lineas = f.readlines()

# Buscar inicio y fin de LOAD DATA
idx_ini = None
idx_fin = None

for i, linea in enumerate(lineas):
    if "BEGIN LOAD DATA" in linea:
        idx_ini = i + 1
    elif idx_ini is not None and "END OF LOAD DATA" in linea:
        idx_fin = i
        break

if idx_ini is None or idx_fin is None:
    raise ValueError("No se pudo encontrar correctamente la sección LOAD DATA en el archivo RAW.")

load_lines = lineas[idx_ini:idx_fin]

print(f"Se encontraron {len(load_lines)} líneas en la sección LOAD DATA.")
print("\nPrimeras 5 líneas:")
for l in load_lines[:5]:
    print(l.strip())

Se encontraron 31 líneas en la sección LOAD DATA.

Primeras 5 líneas:
1,'1 ',1,   1,   1,     0.000,     0.000,     0.000,     0.000,     0.000,    -0.000,   1,1
2,'1 ',1,   1,   1,     0.000,     0.000,     0.000,     0.000,     0.000,    -0.000,   1,1
3,'1 ',1,   1,   1,   322.000,     2.400,     0.000,     0.000,     0.000,    -0.000,   1,1
4,'1 ',1,   1,   1,   500.000,   184.000,     0.000,     0.000,     0.000,    -0.000,   1,1
5,'1 ',1,   1,   1,     0.000,     0.000,     0.000,     0.000,     0.000,    -0.000,   1,1


In [21]:
filas = []

for linea in load_lines:
    linea = linea.strip()

    # Saltar líneas vacías o marcadores
    if not linea or linea.startswith("0 /"):
        continue

    partes = [p.strip() for p in linea.split(",")]

    # Verificación mínima
    if len(partes) < 6:
        continue

    bus = int(partes[0])
    pl_mw = float(partes[5])

    filas.append({
        "bus": bus,
        "demanda": pl_mw
    })

df_demanda = pd.DataFrame(filas)

# Si hubiera más de una carga por bus, sumar
df_demanda = df_demanda.groupby("bus", as_index=False)["demanda"].sum()

# Guardar
df_demanda.to_csv("demanda_nodal_ieee39.csv", index=False)

print("Archivo creado: demanda_nodal_ieee39.csv")
display(df_demanda.head(15))
print(f"\nTotal buses con demanda registrada: {len(df_demanda)}")
print(f"Demanda total del sistema: {df_demanda['demanda'].sum():.3f} MW")

Archivo creado: demanda_nodal_ieee39.csv


,bus,demanda
0,1,0.0
1,2,0.0
2,3,322.0
3,4,500.0
4,5,0.0
5,6,0.0
6,7,233.8
7,8,522.0
8,9,0.0
9,10,0.0



Total buses con demanda registrada: 31
Demanda total del sistema: 6149.500 MW


In [22]:
# Cargar demanda base
df = pd.read_csv("demanda_nodal_ieee39.csv")

# Definir periodos y proporciones
periodos = {
    1: 0.20,
    7: 0.21,
    13: 0.30,
    19: 0.29
}

filas_final = []

for _, row in df.iterrows():
    bus = int(row["bus"])
    demanda_total = float(row["demanda"])

    for t, alpha in periodos.items():
        filas_final.append({
            "bus": bus,
            "periodo": int(t),
            "demanda_mw": demanda_total * alpha
        })

df_final = pd.DataFrame(filas_final)

# Guardar
df_final.to_csv("demanda_nodal_ieee39_final.csv", index=False)

print("Archivo creado: demanda_nodal_ieee39_final.csv")
display(df_final.head(12))

Archivo creado: demanda_nodal_ieee39_final.csv


,bus,periodo,demanda_mw
0,1,1,0.00
1,1,7,0.00
2,1,13,0.00
3,1,19,0.00
4,2,1,0.00
5,2,7,0.00
6,2,13,0.00
7,2,19,0.00
8,3,1,64.40
9,3,7,67.62


In [23]:
check = df_final.groupby("bus", as_index=False)["demanda_mw"].sum()
check = check.merge(df, on="bus", how="left")
check["error"] = check["demanda_mw"] - check["demanda"]

print("Chequeo conservación demanda:")
display(check.head(20))

print("\nError máximo absoluto:")
print(check["error"].abs().max())

Chequeo conservación demanda:


,bus,demanda_mw,demanda,error
0,1,0.0,0.0,0.0
1,2,0.0,0.0,0.0
2,3,322.0,322.0,0.0
3,4,500.0,500.0,0.0
4,5,0.0,0.0,0.0
5,6,0.0,0.0,0.0
6,7,233.8,233.8,0.0
7,8,522.0,522.0,0.0
8,9,0.0,0.0,0.0
9,10,0.0,0.0,0.0



Error máximo absoluto:
0.0


## Generadores procesados

In [24]:
import pandas as pd

# --------------------------------------------------
# 1) Cargar archivo base desde reduced
# --------------------------------------------------
df = pd.read_csv("base_reduced_generadores_procesados.csv")

columnas_objetivo = [
    "id_gen", "bus_id", "tipo_tech", "fuel_orig", "Pmax", "Pmin",
    "Costo_Var", "Inercia_H", "Emisiones_tCO2_MWh", "Es_IBR",
    "Gamma", "Kappa", "Costo_Inv", "Gamma_max", "Kappa_max"
]

df = df[columnas_objetivo].copy()

# --------------------------------------------------
# 2) Elegir subconjunto IEEE39: 4 síncronos + 8 IBR
# --------------------------------------------------
generadores_ieee39 = [
    # Síncronos / térmicos
    "223_STEAM_1",
    "223_STEAM_2",
    "223_STEAM_3",
    "223_CT_4",

    # IBR / renovables
    "313_PV_1",
    "313_PV_2",
    "314_PV_1",
    "314_PV_2",
    "314_PV_3",
    "314_PV_4",
    "313_RTPV_1",
    "313_RTPV_2",
]

df39 = df[df["id_gen"].isin(generadores_ieee39)].copy()

# --------------------------------------------------
# 3) Reasignar buses para IEEE39
# --------------------------------------------------
mapa_buses = {
    # Síncronos / térmicos
    "223_STEAM_1": 39,
    "223_STEAM_2": 20,
    "223_STEAM_3": 8,
    "223_CT_4": 16,

    # IBR / renovables
    "313_PV_1": 3,
    "313_PV_2": 15,
    "314_PV_1": 24,
    "314_PV_2": 29,
    "314_PV_3": 27,
    "314_PV_4": 25,
    "313_RTPV_1": 26,
    "313_RTPV_2": 4,
}

df39["bus_id"] = df39["id_gen"].map(mapa_buses)

# --------------------------------------------------
# 4) Chequeos
# --------------------------------------------------
faltantes = sorted(set(generadores_ieee39) - set(df39["id_gen"]))

if faltantes:
    print("⚠️ Faltan estos generadores en el CSV base:")
    print(faltantes)

if df39["bus_id"].isna().any():
    print("⚠️ Hay generadores sin bus asignado:")
    display(df39[df39["bus_id"].isna()])

# Asegurar tipo entero en bus_id
df39["bus_id"] = df39["bus_id"].astype(int)

print("Conteo por tipo:")
print(df39["Es_IBR"].value_counts().rename(index={0: "Síncronos", 1: "IBR"}))

# --------------------------------------------------
# 5) Guardar archivo final
# --------------------------------------------------
df39 = df39[columnas_objetivo].copy()
df39.to_csv("generadores_procesados_ieee39_final.csv", index=False)

print("\nArchivo creado: generadores_procesados_ieee39_final.csv")
print(f"Cantidad de generadores: {len(df39)}")

print("\nGeneradores IEEE39:")
display(
    df39.sort_values(["Es_IBR", "bus_id"])[[
        "id_gen", "bus_id", "tipo_tech", "fuel_orig", "Pmax",
        "Pmin", "Costo_Var", "Inercia_H", "Emisiones_tCO2_MWh", "Es_IBR"
    ]]
)

Conteo por tipo:
Es_IBR
IBR          8
Síncronos    4
Name: count, dtype: int64

Archivo creado: generadores_procesados_ieee39_final.csv
Cantidad de generadores: 12

Generadores IEEE39:


,id_gen,bus_id,tipo_tech,fuel_orig,Pmax,Pmin,Costo_Var,Inercia_H,Emisiones_tCO2_MWh,Es_IBR
2,223_STEAM_3,8,COAL,Coal,350.0,140,35.0,3.0,0.95,0
3,223_CT_4,16,CT,NG,55.0,22,95.0,2.8,0.55,0
1,223_STEAM_2,20,COAL,Coal,155.0,62,35.0,3.0,0.95,0
0,223_STEAM_1,39,COAL,Coal,155.0,62,35.0,3.0,0.95,0
7,313_PV_1,3,PV,Solar,95.1,0,2.0,0.0,0.00,1
13,313_RTPV_2,4,PV,Solar,63.1,0,2.0,0.0,0.00,1
10,313_PV_2,15,PV,Solar,93.3,0,2.0,0.0,0.00,1
5,314_PV_1,24,PV,Solar,51.6,0,2.0,0.0,0.00,1
9,314_PV_4,25,PV,Solar,51.6,0,2.0,0.0,0.00,1
12,313_RTPV_1,26,PV,Solar,101.7,0,2.0,0.0,0.00,1


## Lineas procesadas

In [25]:
import pandas as pd
from pathlib import Path

# ============================================================
# Líneas procesadas IEEE39
# Mismo formato que lineas_procesadas_ieee30.csv
# ============================================================

# -----------------------------
# 1) Cargar archivo RAW IEEE39
# -----------------------------
ruta_raw = Path("IEEE 39 bus.RAW")

if not ruta_raw.exists():
    posibles = list(Path(".").glob("*39*.RAW"))
    if len(posibles) == 0:
        raise FileNotFoundError("No se encontró el archivo RAW del IEEE39.")
    ruta_raw = posibles[0]

print(f"Archivo RAW usado: {ruta_raw}")

with open(ruta_raw, "r", encoding="utf-8", errors="ignore") as f:
    lineas_raw = f.readlines()


# ============================================================
# 2) Extraer sección BRANCH DATA
# ============================================================

idx_ini_branch = None
idx_fin_branch = None

for i, linea in enumerate(lineas_raw):
    if "BEGIN BRANCH DATA" in linea:
        idx_ini_branch = i + 1
    elif idx_ini_branch is not None and "END OF BRANCH DATA" in linea:
        idx_fin_branch = i
        break

if idx_ini_branch is None or idx_fin_branch is None:
    raise ValueError("No se pudo encontrar correctamente la sección BRANCH DATA.")

branch_lines = lineas_raw[idx_ini_branch:idx_fin_branch]

filas_branch = []

for linea in branch_lines:
    linea = linea.strip()

    if not linea or linea.startswith("0 /"):
        continue

    partes = [p.strip() for p in linea.split(",")]

    # Formato típico PSS/E branch:
    # I, J, CKT, R, X, B, RATEA, RATEB, RATEC, ...
    from_bus = int(partes[0])
    to_bus = int(partes[1])
    resistance_pu = float(partes[3])
    reactance_pu = float(partes[4])
    line_charging_pu = float(partes[5])

    filas_branch.append({
        "from_bus": from_bus,
        "to_bus": to_bus,
        "resistance_pu": resistance_pu,
        "reactance_pu": reactance_pu,
        "line_charging_pu": line_charging_pu,
        "tap_ratio": 1.0,
        "phase_shift_deg": 0.0,
        "tipo": "branch"
    })

df_branch = pd.DataFrame(filas_branch)

print(f"Cantidad de líneas normales extraídas: {len(df_branch)}")
display(df_branch.head())


# ============================================================
# 3) Extraer sección TRANSFORMER DATA
# ============================================================

idx_ini_trafo = None
idx_fin_trafo = None

for i, linea in enumerate(lineas_raw):
    if "BEGIN TRANSFORMER DATA" in linea:
        idx_ini_trafo = i + 1
    elif idx_ini_trafo is not None and "END OF TRANSFORMER DATA" in linea:
        idx_fin_trafo = i
        break

if idx_ini_trafo is None or idx_fin_trafo is None:
    raise ValueError("No se pudo encontrar correctamente la sección TRANSFORMER DATA.")

trafo_lines = lineas_raw[idx_ini_trafo:idx_fin_trafo]

filas_trafo = []

i = 0
while i < len(trafo_lines):
    linea1 = trafo_lines[i].strip()

    if not linea1 or linea1.startswith("0 /"):
        i += 1
        continue

    # En este RAW, cada transformador ocupa 4 líneas.
    linea2 = trafo_lines[i + 1].strip()
    linea3 = trafo_lines[i + 2].strip()

    partes1 = [p.strip() for p in linea1.split(",")]
    partes2 = [p.strip() for p in linea2.split(",")]
    partes3 = [p.strip() for p in linea3.split(",")]

    from_bus = int(partes1[0])
    to_bus = int(partes1[1])
    k_bus = int(partes1[2])

    if k_bus != 0:
        raise ValueError("Se encontró un transformador de tres devanados. Revisar parsing.")

    resistance_pu = float(partes2[0])
    reactance_pu = float(partes2[1])

    # En transformadores no usamos susceptancia de línea
    line_charging_pu = 0.0

    # Para no complicar el modelo DC, dejamos tap = 1 y shift = 0.
    # Esto replica la simplificación del IEEE30 cuando tap_ratio viene en 0.
    tap_ratio = 1.0
    phase_shift_deg = 0.0

    filas_trafo.append({
        "from_bus": from_bus,
        "to_bus": to_bus,
        "resistance_pu": resistance_pu,
        "reactance_pu": reactance_pu,
        "line_charging_pu": line_charging_pu,
        "tap_ratio": tap_ratio,
        "phase_shift_deg": phase_shift_deg,
        "tipo": "transformer"
    })

    i += 4

df_trafo = pd.DataFrame(filas_trafo)

print(f"Cantidad de transformadores extraídos: {len(df_trafo)}")
display(df_trafo.head())


# ============================================================
# 4) Unir líneas normales + transformadores
# ============================================================

lineas = pd.concat([df_branch, df_trafo], ignore_index=True)


# ============================================================
# 5) Crear identificador de línea
# ============================================================

lineas["line_id"] = [f"L{i+1}" for i in range(len(lineas))]


# ============================================================
# 6) Seleccionar y reordenar columnas
# ============================================================

lineas_proc = lineas[[
    "line_id",
    "from_bus",
    "to_bus",
    "resistance_pu",
    "reactance_pu",
    "line_charging_pu",
    "tap_ratio",
    "phase_shift_deg"
]].copy()


# ============================================================
# 7) Asegurar tipos
# ============================================================

lineas_proc["from_bus"] = lineas_proc["from_bus"].astype(int)
lineas_proc["to_bus"] = lineas_proc["to_bus"].astype(int)

for col in [
    "resistance_pu",
    "reactance_pu",
    "line_charging_pu",
    "tap_ratio",
    "phase_shift_deg"
]:
    lineas_proc[col] = lineas_proc[col].astype(float)


# ============================================================
# 8) Validaciones rápidas
# ============================================================

if (lineas_proc["reactance_pu"] == 0).any():
    print("Advertencia: hay líneas con reactance_pu = 0.")
    display(lineas_proc[lineas_proc["reactance_pu"] == 0])

print("Dimensión:", lineas_proc.shape)
print("Columnas:")
print(lineas_proc.columns.tolist())

display(lineas_proc.head(10))


# ============================================================
# 9) Guardar
# ============================================================

lineas_proc.to_csv("lineas_procesadas_ieee39.csv", index=False)

print("Archivo creado: lineas_procesadas_ieee39.csv")

Archivo RAW usado: IEEE 39 bus.RAW
Cantidad de líneas normales extraídas: 34


,from_bus,to_bus,resistance_pu,reactance_pu,line_charging_pu,tap_ratio,phase_shift_deg,tipo
0,1,2,0.0035,0.0411,0.6987,1.0,0.0,branch
1,1,39,0.0010,0.0250,0.7500,1.0,0.0,branch
2,2,3,0.0013,0.0151,0.2572,1.0,0.0,branch
3,2,25,0.0070,0.0086,0.1460,1.0,0.0,branch
4,3,4,0.0013,0.0213,0.2214,1.0,0.0,branch


Cantidad de transformadores extraídos: 12


,from_bus,to_bus,resistance_pu,reactance_pu,line_charging_pu,tap_ratio,phase_shift_deg,tipo
0,2,30,0.0000,0.0181,0.0,1.0,0.0,transformer
1,31,6,0.0000,0.0250,0.0,1.0,0.0,transformer
2,10,32,0.0000,0.0200,0.0,1.0,0.0,transformer
3,12,11,0.0016,0.0435,0.0,1.0,0.0,transformer
4,12,13,0.0016,0.0435,0.0,1.0,0.0,transformer


Dimensión: (46, 8)
Columnas:
['line_id', 'from_bus', 'to_bus', 'resistance_pu', 'reactance_pu', 'line_charging_pu', 'tap_ratio', 'phase_shift_deg']


,line_id,from_bus,to_bus,resistance_pu,reactance_pu,line_charging_pu,tap_ratio,phase_shift_deg
0,L1,1,2,0.0035,0.0411,0.6987,1.0,0.0
1,L2,1,39,0.0010,0.0250,0.7500,1.0,0.0
2,L3,2,3,0.0013,0.0151,0.2572,1.0,0.0
3,L4,2,25,0.0070,0.0086,0.1460,1.0,0.0
4,L5,3,4,0.0013,0.0213,0.2214,1.0,0.0
5,L6,3,18,0.0011,0.0133,0.2138,1.0,0.0
6,L7,4,5,0.0008,0.0128,0.1342,1.0,0.0
7,L8,4,14,0.0008,0.0129,0.1382,1.0,0.0
8,L9,5,6,0.0002,0.0026,0.0434,1.0,0.0
9,L10,5,8,0.0008,0.0112,0.1476,1.0,0.0


Archivo creado: lineas_procesadas_ieee39.csv


## Perfil solar

In [26]:
import pandas as pd

# ============================================================
# Perfil solar IEEE39
# ============================================================

# --------------------------------------------------
# 1) Cargar perfil solar base de IEEE14
# --------------------------------------------------
df = pd.read_csv(
    r"perfil_solar_ieee14_final.csv"
)

# Asegurar nombres esperados
df = df[["Gen_ID", "Period", "P_Available_MW"]].copy()


# --------------------------------------------------
# 2) Cargar generadores procesados IEEE39
# --------------------------------------------------
df_gen39 = pd.read_csv("generadores_procesados_ieee39_final.csv")

df_pv39 = df_gen39[
    (df_gen39["Es_IBR"] == 1) &
    (df_gen39["tipo_tech"].str.upper() == "PV")
].copy()

if df_pv39.empty:
    raise ValueError("No se encontraron generadores PV en generadores_procesados_ieee39_final.csv.")

print("Generadores PV encontrados en IEEE39:")
display(df_pv39[["id_gen", "bus_id", "Pmax"]])


# --------------------------------------------------
# 3) Usar perfiles solares base de IEEE14
# --------------------------------------------------
# Estos generadores deben existir en perfil_solar_ieee14_final.csv.
# Se usan como plantillas de forma temporal.
gens_base = [
    "313_PV_1",
    "314_PV_1",
    "314_PV_3",
]

for gen_base in gens_base:
    if gen_base not in df["Gen_ID"].unique():
        raise ValueError(f"No se encontró el generador base {gen_base} en el perfil solar IEEE14.")


# --------------------------------------------------
# 4) Crear perfiles para los PV del IEEE39
# --------------------------------------------------
perfiles_nuevos = []

for k, row in df_pv39.reset_index(drop=True).iterrows():

    gen_nuevo = row["id_gen"]
    pmax_nuevo = float(row["Pmax"])

    # Se asignan perfiles base de forma cíclica
    gen_base = gens_base[k % len(gens_base)]

    # Pmax base según los valores usados en tu adaptación IEEE30/IEEE14
    if gen_base == "313_PV_1":
        pmax_base = 95.1
    elif gen_base == "314_PV_1":
        pmax_base = 95.1
    elif gen_base == "314_PV_3":
        pmax_base = 95.1
    else:
        raise ValueError(f"No se definió Pmax base para {gen_base}.")

    perfil_base = df[df["Gen_ID"] == gen_base].copy()

    if perfil_base.empty:
        raise ValueError(f"No se encontró el generador base {gen_base} en el perfil solar.")

    perfil_nuevo = perfil_base.copy()
    perfil_nuevo["Gen_ID"] = gen_nuevo
    perfil_nuevo["P_Available_MW"] = (
        perfil_nuevo["P_Available_MW"] * (pmax_nuevo / pmax_base)
    )

    perfiles_nuevos.append(perfil_nuevo)

    print(
        f"Perfil creado: {gen_nuevo} usando base {gen_base} "
        f"| Pmax_base = {pmax_base:.2f} MW "
        f"| Pmax_nuevo = {pmax_nuevo:.2f} MW"
    )


# --------------------------------------------------
# 5) Eliminar perfiles antiguos con esos nombres, si existían
# --------------------------------------------------
ids_nuevos = df_pv39["id_gen"].tolist()

df39 = df[~df["Gen_ID"].isin(ids_nuevos)].copy()


# --------------------------------------------------
# 6) Concatenar perfiles nuevos
# --------------------------------------------------
df39 = pd.concat([df39] + perfiles_nuevos, ignore_index=True)

df39 = df39.sort_values(["Gen_ID", "Period"]).reset_index(drop=True)


# --------------------------------------------------
# 7) Guardar como IEEE39
# --------------------------------------------------
df39.to_csv("perfil_solar_ieee39_final.csv", index=False)

print("\nArchivo creado: perfil_solar_ieee39_final.csv")

print("\nPerfiles agregados para IEEE39:")
display(df39[df39["Gen_ID"].isin(ids_nuevos)])


# --------------------------------------------------
# 8) Resumen rápido
# --------------------------------------------------
resumen = (
    df39[df39["Gen_ID"].isin(ids_nuevos)]
    .groupby("Gen_ID", as_index=False)
    .agg(
        Pmax_perfil=("P_Available_MW", "max"),
        Energia_relativa=("P_Available_MW", "sum")
    )
)

display(resumen)

Generadores PV encontrados en IEEE39:


,id_gen,bus_id,Pmax
4,314_PV_1,24,51.6
5,314_PV_2,29,51.6
6,313_PV_1,3,95.1
7,314_PV_3,27,92.7
8,314_PV_4,25,51.6
9,313_PV_2,15,93.3
10,313_RTPV_1,26,101.7
11,313_RTPV_2,4,63.1


Perfil creado: 314_PV_1 usando base 313_PV_1 | Pmax_base = 95.10 MW | Pmax_nuevo = 51.60 MW
Perfil creado: 314_PV_2 usando base 314_PV_1 | Pmax_base = 95.10 MW | Pmax_nuevo = 51.60 MW
Perfil creado: 313_PV_1 usando base 314_PV_3 | Pmax_base = 95.10 MW | Pmax_nuevo = 95.10 MW
Perfil creado: 314_PV_3 usando base 313_PV_1 | Pmax_base = 95.10 MW | Pmax_nuevo = 92.70 MW
Perfil creado: 314_PV_4 usando base 314_PV_1 | Pmax_base = 95.10 MW | Pmax_nuevo = 51.60 MW
Perfil creado: 313_PV_2 usando base 314_PV_3 | Pmax_base = 95.10 MW | Pmax_nuevo = 93.30 MW
Perfil creado: 313_RTPV_1 usando base 313_PV_1 | Pmax_base = 95.10 MW | Pmax_nuevo = 101.70 MW
Perfil creado: 313_RTPV_2 usando base 314_PV_1 | Pmax_base = 95.10 MW | Pmax_nuevo = 63.10 MW

Archivo creado: perfil_solar_ieee39_final.csv

Perfiles agregados para IEEE39:


,Gen_ID,Period,P_Available_MW
0,313_PV_1,1,0.000000
1,313_PV_1,7,21.414481
2,313_PV_1,13,66.959016
3,313_PV_1,19,0.000000
4,313_PV_2,1,0.000000
5,313_PV_2,7,21.009159
6,313_PV_2,13,65.691653
7,313_PV_2,19,0.000000
8,313_RTPV_1,1,0.000000
9,313_RTPV_1,7,22.775896


,Gen_ID,Pmax_perfil,Energia_relativa
0,313_PV_1,66.959016,88.373497
1,313_PV_2,65.691653,86.700813
2,313_RTPV_1,73.538532,96.314428
3,313_RTPV_2,22.947729,34.647840
4,314_PV_1,37.311586,48.867498
5,314_PV_2,18.765496,28.333257
6,314_PV_3,67.030698,87.791028
7,314_PV_4,18.765496,28.333257


## Datos extra

In [27]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0) CONFIG: usar CSV IEEE39 finales
# ============================================================
PATH = "."

PATH_GEN   = os.path.join(PATH, "generadores_procesados_ieee39_final.csv")
PATH_LINE  = os.path.join(PATH, "lineas_procesadas_ieee39.csv")
PATH_LOAD  = os.path.join(PATH, "demanda_nodal_ieee39_final.csv")
PATH_SOLAR = os.path.join(PATH, "perfil_solar_ieee39_final.csv")

df_gen   = pd.read_csv(PATH_GEN)
df_lines = pd.read_csv(PATH_LINE)
df_load  = pd.read_csv(PATH_LOAD)

df_wind = pd.DataFrame(columns=["Gen_ID", "Period", "P_Available_MW"])

df_solar = pd.read_csv(PATH_SOLAR) if os.path.exists(PATH_SOLAR) else pd.DataFrame(
    columns=["Gen_ID", "Period", "P_Available_MW"]
)

# ============================================================
# 1) LIMPIAR NOMBRES DE COLUMNAS
# ============================================================
df_gen.columns   = df_gen.columns.str.strip()
df_lines.columns = df_lines.columns.str.strip()
df_load.columns  = df_load.columns.str.strip()
df_solar.columns = df_solar.columns.str.strip()

# ============================================================
# 2) RENOMBRAR COLUMNAS PARA DEJARLAS COMPATIBLES
# ============================================================

# ----- DEMANDA -----
df_load = df_load.rename(columns={
    "bus": "Bus ID",
    "bus_id": "Bus ID",
    "periodo": "Period",
    "period": "Period",
    "demanda_mw": "Nodal_Load_MW",
    "load_mw": "Nodal_Load_MW"
})

# ----- LÍNEAS -----
# Esta parte acepta tanto el formato viejo:
#   desde, hacia, x, fmax
# como el formato estilo IEEE30:
#   from_bus, to_bus, resistance_pu, reactance_pu, line_charging_pu, tap_ratio, phase_shift_deg

df_lines = df_lines.rename(columns={
    "desde": "from_bus",
    "hacia": "to_bus",
    "x": "Reactancia_X",
    "fmax": "F_max",
    "reactance_pu": "Reactancia_X"
})

# Si no viene F_max, se crea constante
if "F_max" not in df_lines.columns:
    df_lines["F_max"] = 100.0

# ----- PERFIL SOLAR -----
df_solar = df_solar.rename(columns={
    "id_gen": "Gen_ID",
    "periodo": "Period",
    "period": "Period",
    "disponibilidad_mw": "P_Available_MW"
})

# ============================================================
# 3) CHEQUEO DE COLUMNAS OBLIGATORIAS
# ============================================================
cols_gen_req = [
    "id_gen", "bus_id", "Pmax", "Pmin", "Costo_Var", "Inercia_H",
    "Emisiones_tCO2_MWh", "Es_IBR", "Gamma", "Kappa", "Costo_Inv"
]

cols_line_req = ["from_bus", "to_bus", "Reactancia_X", "F_max"]
cols_load_req = ["Bus ID", "Period", "Nodal_Load_MW"]

for c in cols_gen_req:
    if c not in df_gen.columns:
        raise ValueError(f"Falta columna en generadores: {c}")

for c in cols_line_req:
    if c not in df_lines.columns:
        raise ValueError(f"Falta columna en líneas: {c}")

for c in cols_load_req:
    if c not in df_load.columns:
        raise ValueError(f"Falta columna en demanda: {c}")

# ============================================================
# 4) SANITIZAR TIPOS
# ============================================================
df_gen["id_gen"] = df_gen["id_gen"].astype(str)
df_gen["bus_id"] = df_gen["bus_id"].astype(int)

df_load["Bus ID"] = df_load["Bus ID"].astype(int)
df_load["Period"] = df_load["Period"].astype(int)
df_load["Nodal_Load_MW"] = df_load["Nodal_Load_MW"].astype(float)

df_lines["from_bus"] = df_lines["from_bus"].astype(int)
df_lines["to_bus"]   = df_lines["to_bus"].astype(int)
df_lines["F_max"] = df_lines["F_max"].astype(float)
df_lines["Reactancia_X"] = df_lines["Reactancia_X"].astype(float)

if not df_solar.empty:
    df_solar["Gen_ID"] = df_solar["Gen_ID"].astype(str)
    df_solar["Period"] = df_solar["Period"].astype(int)
    df_solar["P_Available_MW"] = df_solar["P_Available_MW"].astype(float)

# ============================================================
# 5) DEFINICIÓN DE SETS
# ============================================================
Y = [1, 2, 3]

year_growth = {
    1: 1.00,
    2: 1.05,
    3: 1.10
}

# Buses con demanda
B_load = set(df_load["Bus ID"].astype(int).unique().tolist())

# Buses que aparecen en la red
B_lines = set(df_lines["from_bus"].astype(int).unique().tolist()).union(
    set(df_lines["to_bus"].astype(int).unique().tolist())
)

# Buses donde hay generadores
B_gen = set(df_gen["bus_id"].astype(int).unique().tolist())

# Conjunto completo de buses
B = sorted(B_load.union(B_lines).union(B_gen))

T = sorted(df_load["Period"].unique().tolist())

G  = df_gen["id_gen"].tolist()
GS = df_gen[df_gen["Es_IBR"] == 0]["id_gen"].tolist()
M  = df_gen[df_gen["Es_IBR"] == 1]["id_gen"].tolist()
GE = df_gen[df_gen["Emisiones_tCO2_MWh"] > 0]["id_gen"].tolist()

print(
    f"Sets definidos: |B|={len(B)}, |T|={len(T)}, "
    f"|G|={len(G)} (|GS|={len(GS)}, |M|={len(M)}, |GE|={len(GE)})"
)

print("Buses sin demanda explícita:")
print(sorted(set(B) - B_load))
# ============================================================
# 6) PARÁMETROS DE GENERADORES
# ============================================================
Pmax = df_gen.set_index("id_gen")["Pmax"].to_dict()
Pmin = df_gen.set_index("id_gen")["Pmin"].to_dict()
cg   = df_gen.set_index("id_gen")["Costo_Var"].to_dict()
eg   = df_gen.set_index("id_gen")["Emisiones_tCO2_MWh"].to_dict()
cinv = df_gen.set_index("id_gen")["Costo_Inv"].to_dict()

Hg = {
    g: float(df_gen.set_index("id_gen")["Inercia_H"].to_dict().get(g, 0.0))
    for g in GS
}

g_bus = df_gen.set_index("id_gen")["bus_id"].to_dict()

G_en_bus = {i: [] for i in B}

for g in G:
    bi = int(g_bus[g])
    if bi in G_en_bus:
        G_en_bus[bi].append(g)
    else:
        print(f"Advertencia: el generador {g} está asignado al bus {bi}, que no está en B.")

# ============================================================
# 7) PARÁMETROS IBR
# ============================================================
gamma_base = df_gen.set_index("id_gen")["Gamma"].to_dict()
kappa_base = df_gen.set_index("id_gen")["Kappa"].to_dict()

gamma_max = (
    df_gen.set_index("id_gen")["Gamma_max"].to_dict()
    if "Gamma_max" in df_gen.columns
    else {m: 0.5 for m in M}
)

kappa_max = (
    df_gen.set_index("id_gen")["Kappa_max"].to_dict()
    if "Kappa_max" in df_gen.columns
    else {m: 0.5 for m in M}
)

xbar_mult = 2.0
xbar = {m: float(Pmax[m]) * xbar_mult for m in M}

# ============================================================
# 8) RED
# ============================================================
L, F0, Bline = [], {}, {}

for _, row in df_lines.iterrows():
    i, j = int(row["from_bus"]), int(row["to_bus"])
    line = (i, j)

    L.append(line)

    F0[line] = float(row["F_max"])

    x_val = float(row["Reactancia_X"])

    if abs(x_val) < 1e-6:
        x_val = 1e-4

    Bline[line] = 1.0 / x_val

lineas_entran = {i: [] for i in B}
lineas_salen  = {i: [] for i in B}

for (i, j) in L:
    if i in lineas_salen:
        lineas_salen[i].append((i, j))
    if j in lineas_entran:
        lineas_entran[j].append((i, j))

print(f"Red: |L|={len(L)}")

# ============================================================
# 9) DEMANDA
# ============================================================
raw_demand = df_load.set_index(["Bus ID", "Period"])["Nodal_Load_MW"].to_dict()

d = {}

for y in Y:
    factor = year_growth[y]

    for i in B:
        for t in T:
            # Si el bus no tiene demanda en el CSV, se asigna 0
            d[(i, t, y)] = float(raw_demand.get((i, t), 0.0)) * factor
# ============================================================
# 10) PERFIL RENOVABLE
# ============================================================
P_disp = {
    (g, t): float(Pmax[g])
    for g in G
    for t in T
}

if not df_solar.empty:
    for _, row in df_solar.iterrows():
        g_id = str(row["Gen_ID"])
        t = int(row["Period"])
        val = float(row["P_Available_MW"])

        if (g_id in G) and (t in T):
            P_disp[(g_id, t)] = val

# ============================================================
# 11) ESCALARES
# ============================================================
VOLL = 5000.0

y_star = 3
epsilon_CO2 = 0

# ============================================================
# 12) Hreq y Rreq
# ============================================================
D_peak = {
    y: max(
        sum(d[(i, t, y)] for i in B)
        for t in T
    )
    for y in Y
}

max_hsyn = (
    sum(float(gamma_base.get(m, 0.0)) * float(xbar[m]) for m in M)
    if len(M) > 0
    else 0.0
)

max_sync = (
    sum(float(Hg.get(g, 0.0)) for g in GS)
    if len(GS) > 0
    else 0.0
)

cap_total = max_hsyn + max_sync

alpha_H = 0.35
H_floor = 0.05

k_H = (alpha_H * cap_total / D_peak[1]) if D_peak[1] > 1e-9 else 0.0

Hreq = {}

for y in Y:
    base = k_H * D_peak[y]
    floor = H_floor * cap_total
    Hreq[y] = max(base, floor)

MaxR = (
    sum(float(kappa_base.get(m, 0.0)) * float(xbar[m]) for m in M)
    if len(M) > 0
    else 0.0
)

beta_R = 0.40
R_floor = 0.05

Rreq = {}

for y in Y:
    base = beta_R * MaxR
    floor = R_floor * MaxR
    Rreq[y] = max(base, floor)

print("cap_total =", round(cap_total, 4), " MaxR =", round(MaxR, 4))
print("D_peak =", {y: round(D_peak[y], 2) for y in Y})
print("Hreq =", {y: round(Hreq[y], 4) for y in Y})
print("Rreq =", {y: round(Rreq[y], 4) for y in Y})

Sets definidos: |B|=39, |T|=4, |G|=12 (|GS|=4, |M|=8, |GE|=4)
Buses sin demanda explícita:
[30, 32, 33, 34, 35, 36, 37, 38]


Red: |L|=46
cap_total = 23.814  MaxR = 120.14
D_peak = {1: 1844.85, 2: 1937.09, 3: 2029.34}
Hreq = {1: 8.3349, 2: 8.7516, 3: 9.1684}
Rreq = {1: 48.056, 2: 48.056, 3: 48.056}


# Forward

## Dimensionalidad

In [28]:
# ============================================================
# Celda: Resumen de dimensionalidad / tamaño del problema
# Adaptada al Forward Problem para IEEE39
# Requiere: B, T, Y, G, GS, M, GE, L
# Opcional: fw_base
# ============================================================

import gurobipy as gp

def safe_len(x):
    try:
        return len(x)
    except Exception:
        return None

def mul(*vals):
    out = 1
    for v in vals:
        out *= int(v)
    return out


# ============================================================
# 1) Seleccionar conjunto de líneas/arcos
# ============================================================
# Si existe L_arcs, úsalo.
# Si no existe, usar L.
# Esto lo hace compatible con distintas versiones del notebook.
if "L_arcs" in globals():
    _L = L_arcs
else:
    _L = gp.tuplelist(L)


# ============================================================
# 2) Cardinalidades de conjuntos
# ============================================================
nB  = safe_len(B)
nT  = safe_len(T)
nY  = safe_len(Y)
nG  = safe_len(G)
nGS = safe_len(GS)
nM  = safe_len(M)
nGE = safe_len(GE)
nL  = safe_len(_L)

print("\n==============================")
print("DIMENSIONALIDAD DEL PROBLEMA - IEEE39")
print("==============================")
print(f"|B|  (buses)        = {nB}")
print(f"|L|  (líneas/arcos) = {nL}")
print(f"|T|  (periodos)     = {nT}")
print(f"|Y|  (años)         = {nY}")
print(f"|G|  (generadores)  = {nG}")
print(f"|GS| (síncronos)    = {nGS}")
print(f"|M|  (IBR)          = {nM}")
print(f"|GE| (emisores CO2) = {nGE}")


# ============================================================
# 3) Variables estimadas según tu formulación forward
# ============================================================
n_p     = mul(nG,  nT, nY) if None not in [nG,  nT, nY] else None
n_u     = mul(nGS, nT, nY) if None not in [nGS, nT, nY] else None
n_s     = mul(nB,  nT, nY) if None not in [nB,  nT, nY] else None
n_theta = mul(nB,  nT, nY) if None not in [nB,  nT, nY] else None
n_f     = mul(nL,  nT, nY) if None not in [nL,  nT, nY] else None
n_x     = mul(nM,  nY)     if None not in [nM,  nY]     else None
n_h     = mul(nM,  nT, nY) if None not in [nM,  nT, nY] else None
n_r     = mul(nM,  nT, nY) if None not in [nM,  nT, nY] else None

print("\n--- Tamaño estimado de variables ---")
print(f"#p      = {n_p}")
print(f"#u      = {n_u}")
print(f"#s      = {n_s}")
print(f"#theta  = {n_theta}")
print(f"#f      = {n_f}")
print(f"#x      = {n_x}")
print(f"#h_syn  = {n_h}")
print(f"#r_ffr  = {n_r}")

n_vars_est = sum(
    v for v in [
        n_p, n_u, n_s, n_theta, n_f, n_x, n_h, n_r
    ]
    if v is not None
)

print(f"TOTAL vars estimado = {n_vars_est}")


# ============================================================
# 4) Restricciones estimadas según tu formulación forward
# ============================================================

# Balance nodal
n_bal = (
    mul(nB, nT, nY)
    if None not in [nB, nT, nY]
    else None
)

# ENS <= demanda
n_ensub = (
    mul(nB, nT, nY)
    if None not in [nB, nT, nY]
    else None
)

# Flujo DC por línea
n_dc = (
    mul(nL, nT, nY)
    if None not in [nL, nT, nY]
    else None
)

# Límites ±Fmax
n_line = (
    2 * n_dc
    if n_dc is not None
    else None
)

# Barra slack
n_slack = (
    mul(nT, nY)
    if None not in [nT, nY]
    else None
)

# p <= Pdisp
n_pdisp = (
    mul(nG, nT, nY)
    if None not in [nG, nT, nY]
    else None
)

# Restricciones síncronas:
# Pmin*u <= p
# p <= Pmax*u
n_sync = (
    2 * mul(nGS, nT, nY)
    if None not in [nGS, nT, nY]
    else None
)

# CO2 cap
# En tu versión actual aparece comentada, por eso queda 0.
# Si después activas la restricción de CO2 por año, cambia esto.
n_co2 = 0

# x <= xbar
n_xub = (
    mul(nM, nY)
    if None not in [nM, nY]
    else None
)

# h_syn <= gamma*x
n_hcap = (
    mul(nM, nT, nY)
    if None not in [nM, nT, nY]
    else None
)

# r_ffr <= kappa*x
n_rcap = (
    mul(nM, nT, nY)
    if None not in [nM, nT, nY]
    else None
)

# Requerimiento de inercia
n_inreq = (
    mul(nT, nY)
    if None not in [nT, nY]
    else None
)

# Requerimiento de FFR
n_rreq = (
    mul(nT, nY)
    if None not in [nT, nY]
    else None
)

print("\n--- Tamaño estimado de restricciones ---")
print(f"#balance      = {n_bal}")
print(f"#ens_ub       = {n_ensub}")
print(f"#dcflow       = {n_dc}")
print(f"#line_limits  = {n_line}")
print(f"#slack        = {n_slack}")
print(f"#pdisp        = {n_pdisp}")
print(f"#sync_min/max = {n_sync}")
print(f"#CO2_cap      = {n_co2}")
print(f"#x_ub         = {n_xub}")
print(f"#hsyn_cap     = {n_hcap}")
print(f"#ffr_cap      = {n_rcap}")
print(f"#inertia_req  = {n_inreq}")
print(f"#ffr_req      = {n_rreq}")

n_cons_est = sum(
    v for v in [
        n_bal, n_ensub, n_dc, n_line, n_slack, n_pdisp,
        n_sync, n_co2, n_xub, n_hcap, n_rcap, n_inreq, n_rreq
    ]
    if v is not None
)

print(f"TOTAL cons estimado = {n_cons_est}")


# ============================================================
# 5) Comparar con el modelo real en Gurobi
# ============================================================
if "fw_base" in globals():
    try:
        fw_base.update()

        print("\n--- Conteo REAL del modelo en Gurobi: fw_base ---")
        print(f"fw_base.NumVars    = {fw_base.NumVars}")
        print(f"fw_base.NumConstrs = {fw_base.NumConstrs}")

        print("\n--- Diferencia estimado vs real ---")
        print(f"Diferencia vars  = {fw_base.NumVars - n_vars_est}")
        print(f"Diferencia cons  = {fw_base.NumConstrs - n_cons_est}")

    except Exception as e:
        print("\nNo pude leer fw_base.NumVars/fw_base.NumConstrs:")
        print(e)

else:
    print("\nNota: no existe 'fw_base' en globals().")
    print("Corre primero la celda del Forward Problem si quieres comparar contra el modelo real.")


DIMENSIONALIDAD DEL PROBLEMA - IEEE39
|B|  (buses)        = 39
|L|  (líneas/arcos) = 20
|T|  (periodos)     = 4
|Y|  (años)         = 3
|G|  (generadores)  = 12
|GS| (síncronos)    = 4
|M|  (IBR)          = 8
|GE| (emisores CO2) = 4

--- Tamaño estimado de variables ---
#p      = 144
#u      = 48
#s      = 468
#theta  = 468
#f      = 240
#x      = 24
#h_syn  = 96
#r_ffr  = 96
TOTAL vars estimado = 1584

--- Tamaño estimado de restricciones ---
#balance      = 468
#ens_ub       = 468
#dcflow       = 240
#line_limits  = 480
#slack        = 12
#pdisp        = 144
#sync_min/max = 96
#CO2_cap      = 0
#x_ub         = 24
#hsyn_cap     = 96
#ffr_cap      = 96
#inertia_req  = 12
#ffr_req      = 12
TOTAL cons estimado = 2148

--- Conteo REAL del modelo en Gurobi: fw_base ---
fw_base.NumVars    = 819
fw_base.NumConstrs = 1359

--- Diferencia estimado vs real ---
Diferencia vars  = -765
Diferencia cons  = -789


# Resolución del WCEP

###  FW

In [29]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd

# ============================================================
# FINAL IEEE39
# PASO 1) FORWARD PROBLEM - FORMULACIÓN ORIGINAL
# DATASET CORREGIDO
#
# Correcciones incluidas:
#
# 1) Escalamiento de demanda para evitar déficit estructural.
#    El dataset original tenía demanda pico mayor que capacidad instalada,
#    generando ENS artificial.
#
# 2) Hreq queda en su posición original:
#       sum_g Hg[g] u[g,t,y] + sum_m h_syn[m,t,y] >= Hreq[y]
#
#    No se usa zH.
#    No se mueve Hreq hacia A.
#
# 3) Se mantiene el diccionario original d intacto.
#    El modelo usa d_fw.
# ============================================================


# ============================================================
# 0) OPCIONES DE CORRECCIÓN DEL DATASET IEEE39
# ============================================================

# Recomendado para IEEE39:
#   "auto_capacity" calcula un factor para que la demanda pico quede
#   bajo la capacidad instalada del sistema.
#
# También puedes fijar manualmente:
#   DEMAND_SCALE_MODE = "manual"
#   DEMAND_SCALE_MANUAL = 0.713
#
DEMAND_SCALE_MODE = "auto_capacity"   # "auto_capacity", "manual", "none"
DEMAND_SCALE_MANUAL = 0.713

# Margen de seguridad respecto a capacidad instalada.
# 1.00 deja la demanda pico igual a la capacidad.
# 0.98 deja un 2% de margen operativo.
DEMAND_CAP_MARGIN = 0.98

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# T contiene pocas horas representativas del día.
# En vez de expandir T a 365*|T| periodos, ponderamos cada t.
#
# Caso base solicitado:
#   el mismo perfil diario se repite 365 veces.
#   Si cada t representa una hora puntual del día, usar TIME_BLOCK_HOURS = 1.0.
#   Si cada t representa un bloque de 6 horas, usar TIME_BLOCK_HOURS = 6.0.
#
# Con T = {1,7,13,19}:
#   TIME_BLOCK_HOURS = 1.0  -> sum_t omega_t = 1460 h equivalentes
#   TIME_BLOCK_HOURS = 6.0  -> sum_t omega_t = 8760 h equivalentes
N_DAYS_REPRESENTED = 365
TIME_BLOCK_HOURS = 1.0

omega_t = {t: float(N_DAYS_REPRESENTED * TIME_BLOCK_HOURS) for t in T}

print("\n==============================")
print("PESOS TEMPORALES FW IEEE39")
print("==============================")
print(f"Días representados       = {N_DAYS_REPRESENTED}")
print(f"Horas por periodo t      = {TIME_BLOCK_HOURS}")
print(f"Peso omega_t por periodo = {next(iter(omega_t.values())) if len(omega_t) > 0 else 0.0}")
print(f"Horas equivalentes total = {sum(omega_t.values()):.2f}")

# Para mantener el experimento reproducible, no se modifica d.
# Se construye un nuevo diccionario d_fw.
d_original = {k: float(v) for k, v in d.items()}


# ============================================================
# 1) ROBUSTECER LÍNEAS
# ============================================================

L_arcs = gp.tuplelist(L)


# ============================================================
# 2) FIX: reconstruir B si fue pisado
# ============================================================

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))


# ============================================================
# 3) TOPOLOGÍA ROBUSTA
# ============================================================

lineas_entran = {bus: [] for bus in B}
lineas_salen  = {bus: [] for bus in B}

for arc in L_arcs:
    if not (isinstance(arc, tuple) and len(arc) == 2):
        raise TypeError(f"Arco inválido en L_arcs: {arc} (type={type(arc)})")

    a, b = arc

    if b in lineas_entran:
        lineas_entran[b].append((a, b))

    if a in lineas_salen:
        lineas_salen[a].append((a, b))


# ============================================================
# 4) CORRECCIÓN DE DEMANDA IEEE39
# ============================================================

print("\n==============================")
print("CORRECCIÓN DATASET IEEE39")
print("==============================")

# Demanda total original por periodo/año
demanda_original_ty = {}

for t in T:
    for y in Y:
        demanda_original_ty[(t, y)] = sum(
            d_original[(i, t, y)]
            for i in B
            if (i, t, y) in d_original
        )

peak_dem_original = max(demanda_original_ty.values())

# Capacidad instalada total
cap_total_pmax = sum(float(Pmax[g]) for g in G)

# Capacidad disponible por periodo si existe P_disp
cap_disp_ty = {}

for t in T:
    for y in Y:
        cap_disp_ty[(t, y)] = sum(
            float(P_disp.get((g, t), Pmax[g]))
            for g in G
        )

min_cap_disp = min(cap_disp_ty.values())

print(f"Demanda pico original     = {peak_dem_original:.6f}")
print(f"Capacidad instalada Pmax  = {cap_total_pmax:.6f}")
print(f"Capacidad disp. mínima    = {min_cap_disp:.6f}")

if DEMAND_SCALE_MODE == "none":
    demand_scale_factor = 1.0

elif DEMAND_SCALE_MODE == "manual":
    demand_scale_factor = float(DEMAND_SCALE_MANUAL)

elif DEMAND_SCALE_MODE == "auto_capacity":
    if peak_dem_original <= 1e-9:
        demand_scale_factor = 1.0
    else:
        # Usamos Pmax total, consistente con el diagnóstico original.
        # Además aplicamos un pequeño margen para evitar operar exactamente
        # en el borde de capacidad.
        demand_scale_factor = min(
            1.0,
            DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
        )

else:
    raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

print(f"Modo escalamiento demanda = {DEMAND_SCALE_MODE}")
print(f"Factor demanda usado      = {demand_scale_factor:.8f}")

# Construir demanda corregida para el modelo
d_fw = {}

for i in B:
    for t in T:
        for y in Y:
            if (i, t, y) not in d_original:
                raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")

            d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

# Resumen demanda corregida
demanda_fw_ty = {}

for t in T:
    for y in Y:
        demanda_fw_ty[(t, y)] = sum(d_fw[(i, t, y)] for i in B)

peak_dem_fw = max(demanda_fw_ty.values())

print(f"Demanda pico corregida    = {peak_dem_fw:.6f}")
print(f"Margen Pmax - peak demand = {cap_total_pmax - peak_dem_fw:.6f}")

if peak_dem_fw > cap_total_pmax + 1e-6:
    print("⚠️ Aún hay demanda pico mayor que Pmax total.")
else:
    print("✅ Demanda corregida queda dentro de la capacidad instalada.")

if peak_dem_fw > min_cap_disp + 1e-6:
    print("⚠️ La demanda pico corregida puede superar la capacidad disponible P_disp en algún periodo.")
    print("   Esto puede ser normal si P_disp modela renovables variables, pero conviene revisar.")
else:
    print("✅ Demanda corregida queda dentro de la capacidad disponible mínima.")


# ============================================================
# 5) PRE-CHECKS
# ============================================================

print("\n==============================")
print("PRE-CHECKS FORWARD IEEE39 - FORMULACIÓN ORIGINAL")
print("==============================")

missing_d = [
    (i, t, y)
    for i in B
    for t in T
    for y in Y
    if (i, t, y) not in d_fw
]

if missing_d:
    print(f"⚠️ Faltan {len(missing_d)} entradas en d_fw[(i,t,y)]. Ejemplo:", missing_d[:5])
else:
    print("✅ d_fw[(i,t,y)] completo para B,T,Y.")

for y in Y:
    for t in T:
        dem_tot = sum(d_fw[(i, t, y)] for i in B)
        cap_tot = sum(float(Pmax[g]) for g in G)
        cap_disp = sum(float(P_disp.get((g, t), Pmax[g])) for g in G)

        if cap_tot + 1e-6 < dem_tot:
            print(
                f"⚠️ Pmax total {cap_tot:.2f} < demanda corregida {dem_tot:.2f} "
                f"en (t={t}, y={y})."
            )

        if cap_disp + 1e-6 < dem_tot:
            print(
                f"⚠️ Capacidad disponible {cap_disp:.2f} < demanda corregida {dem_tot:.2f} "
                f"en (t={t}, y={y})."
            )

max_sync_H = sum(float(Hg.get(g, 0.0)) for g in GS)

gamma_fwd_check = gamma_base.copy()
max_hsyn = (
    sum(float(gamma_fwd_check[m]) * float(xbar[m]) for m in M)
    if len(M) > 0
    else 0.0
)

max_H_total = max_sync_H + max_hsyn

kappa_fwd_check = kappa_base.copy()
max_R_total = (
    sum(float(kappa_fwd_check[m]) * float(xbar[m]) for m in M)
    if len(M) > 0
    else 0.0
)

print(
    f"Max H posible (sync + hsyn) = {max_H_total:.3f} "
    f"(sync={max_sync_H:.3f}, hsyn={max_hsyn:.3f})"
)

print(f"Max R posible (FFR)         = {max_R_total:.3f}")

for y in Y:
    if Hreq[y] > max_H_total + 1e-6:
        print(
            f"❌ ALERTA: Hreq[{y}]={Hreq[y]:.3f} > MaxH={max_H_total:.3f} "
            f"=> INFEASIBLE por inercia."
        )

    if Rreq[y] > max_R_total + 1e-6:
        print(
            f"❌ ALERTA: Rreq[{y}]={Rreq[y]:.3f} > MaxR={max_R_total:.3f} "
            f"=> INFEASIBLE por FFR."
        )


# ============================================================
# 6) MODELO FORWARD IEEE39 - FORMULACIÓN ORIGINAL
# ============================================================

fw_base = gp.Model("FW_base_IEEE39_original_dataset_corregido_con_pesos")
fw_base.Params.OutputFlag = 1

# Time limit opcional para el Forward. Para dejarlo sin límite, usar None.
FW_TIME_LIMIT_SECONDS = None
if FW_TIME_LIMIT_SECONDS is not None:
    fw_base.Params.TimeLimit = FW_TIME_LIMIT_SECONDS


# ============================================================
# 7) VARIABLES
# ============================================================

p = fw_base.addVars(
    G, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="p"
)

u = fw_base.addVars(
    GS, T, Y,
    lb=0.0,
    ub=1.0,
    vtype=GRB.CONTINUOUS,
    name="u"
)

s = fw_base.addVars(
    B, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="s"
)

theta = fw_base.addVars(
    B, T, Y,
    lb=-GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="theta"
)

f = fw_base.addVars(
    L_arcs, T, Y,
    lb=-GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="f"
)

x = fw_base.addVars(
    M, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="x"
)

h_syn = fw_base.addVars(
    M, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="h_syn"
)

r_ffr = fw_base.addVars(
    M, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="r_ffr"
)


# ============================================================
# 8) RESTRICCIONES
# ============================================================

# ------------------------------------------------------------
# 8.1) Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            gen = gp.quicksum(
                p[g, t, y]
                for g in G_en_bus.get(i, [])
            )

            inflow = gp.quicksum(
                f[k, i, t, y]
                for (k, i) in lineas_entran[i]
            )

            outflow = gp.quicksum(
                f[i, kk, t, y]
                for (i, kk) in lineas_salen[i]
            )

            fw_base.addConstr(
                gen - d_fw[(i, t, y)] + s[i, t, y] + inflow - outflow == 0,
                name=f"balance[{i},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.2) Cota superior ENS
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                s[i, t, y] <= d_fw[(i, t, y)],
                name=f"ens_ub[{i},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.3) Flujo DC
# ------------------------------------------------------------
for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                f[i, j, t, y]
                == Bline[(i, j)] * (theta[i, t, y] - theta[j, t, y]),
                name=f"dcflow[{i},{j},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.4) Límites de flujo
# ------------------------------------------------------------
for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                f[i, j, t, y] <= F0[(i, j)],
                name=f"f_ub[{i},{j},{t},{y}]"
            )

            fw_base.addConstr(
                f[i, j, t, y] >= -F0[(i, j)],
                name=f"f_lb[{i},{j},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.5) Bus slack
# ------------------------------------------------------------
bus_slack = int(sorted(B)[0])

for t in T:
    for y in Y:
        fw_base.addConstr(
            theta[bus_slack, t, y] == 0.0,
            name=f"slack[{t},{y}]"
        )

print(f"Bus slack usado: {bus_slack}")


# ------------------------------------------------------------
# 8.6) Disponibilidad de generación
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_gt = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            fw_base.addConstr(
                p[g, t, y] <= cap_gt,
                name=f"pdisp[{g},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.7) Restricciones de generadores síncronos
# ------------------------------------------------------------
for g in GS:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                p[g, t, y] >= float(Pmin.get(g, 0.0)) * u[g, t, y],
                name=f"sync_min[{g},{t},{y}]"
            )

            fw_base.addConstr(
                p[g, t, y] <= float(Pmax[g]) * u[g, t, y],
                name=f"sync_max[{g},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.8) Capacidad máxima de inversión IBR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        fw_base.addConstr(
            x[m, y] <= float(xbar[m]),
            name=f"x_ub[{m},{y}]"
        )


# ------------------------------------------------------------
# 8.9) Capacidad de inercia sintética
# ------------------------------------------------------------
for m in M:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                h_syn[m, t, y] <= float(gamma_base[m]) * x[m, y],
                name=f"hsyn_cap[{m},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.10) Capacidad de FFR
# ------------------------------------------------------------
for m in M:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                r_ffr[m, t, y] <= float(kappa_base[m]) * x[m, y],
                name=f"ffr_cap[{m},{t},{y}]"
            )


# ------------------------------------------------------------
# 8.11) Requerimiento de inercia - FORMULACIÓN ORIGINAL
# ------------------------------------------------------------
for t in T:
    for y in Y:
        fw_base.addConstr(
            gp.quicksum(
                float(Hg.get(g, 0.0)) * u[g, t, y]
                for g in GS
            )
            + gp.quicksum(
                h_syn[m, t, y]
                for m in M
            )
            >= float(Hreq[y]),
            name=f"inertia_req[{t},{y}]"
        )


# ------------------------------------------------------------
# 8.12) Requerimiento de FFR
# ------------------------------------------------------------
for t in T:
    for y in Y:
        fw_base.addConstr(
            gp.quicksum(
                r_ffr[m, t, y]
                for m in M
            ) >= float(Rreq[y]),
            name=f"ffr_req[{t},{y}]"
        )


# ============================================================
# 9) ACTUALIZAR MODELO
# ============================================================

fw_base.update()

print(f"\nVariables: {fw_base.NumVars} | Restricciones: {fw_base.NumConstrs}")


# ============================================================
# 10) FUNCIÓN OBJETIVO
# ============================================================

obj_oper = gp.quicksum(
    float(omega_t[t]) * float(cg[g]) * p[g, t, y]
    for g in G
    for t in T
    for y in Y
)

obj_ens = float(VOLL) * gp.quicksum(
    float(omega_t[t]) * s[i, t, y]
    for i in B
    for t in T
    for y in Y
)

obj_inv = gp.quicksum(
    float(cinv[m]) * x[m, y]
    for m in M
    for y in Y
)

fw_base.setObjective(
    obj_oper + obj_ens + obj_inv,
    GRB.MINIMIZE
)


# ============================================================
# 11) RESOLVER
# ============================================================

fw_base.optimize()


# ============================================================
# 12) DIAGNÓSTICO BÁSICO
# ============================================================

if fw_base.Status == GRB.INFEASIBLE:
    print("\n❌ Modelo INFEASIBLE.")
    print("No se escribió archivo IIS/ILP.")

elif fw_base.Status == GRB.OPTIMAL or fw_base.SolCount > 0:
    if fw_base.Status == GRB.OPTIMAL:
        print("\n✅ Modelo resuelto óptimamente.")
    else:
        print(f"\n⚠️ Modelo terminó con status {fw_base.Status}, pero tiene solución factible.")

    print(f"Valor objetivo = {fw_base.ObjVal:,.4f}")
    if hasattr(fw_base, 'ObjBound'):
        print(f"Best bound     = {fw_base.ObjBound:,.4f}")
    if hasattr(fw_base, 'MIPGap'):
        print(f"Gap            = {fw_base.MIPGap:.6f}")

    ens_total_no_pond = sum(
        s[i, t, y].X
        for i in B
        for t in T
        for y in Y
    )

    ens_total_pond = sum(
        float(omega_t[t]) * s[i, t, y].X
        for i in B
        for t in T
        for y in Y
    )

    print(f"ENS total no ponderado = {ens_total_no_pond:.6f}")
    print(f"ENS total ponderado    = {ens_total_pond:.6f} MWh-equivalente")

    ens_total = ens_total_pond

    if ens_total <= 1e-6:
        print("✅ ENS corregido: el Forward queda sin déficit artificial.")
    else:
        print("⚠️ Aún existe ENS positivo. Revisar P_disp, red o límites de línea.")

else:
    print(f"\n⚠️ Modelo terminó con status Gurobi = {fw_base.Status}")


# ============================================================
# 13) RESULTADOS FORWARD IEEE39 - FORMULACIÓN ORIGINAL
# ============================================================

if fw_base.Status == GRB.OPTIMAL or fw_base.SolCount > 0:

    print("\n==============================")
    print("RESULTADOS FW_base IEEE39 - DATASET CORREGIDO")
    print("==============================")
    print(f"Valor objetivo total = {fw_base.ObjVal:,.4f}")
    print(f"Factor demanda usado = {demand_scale_factor:.8f}")

    # --------------------------------------------------------
    # 13.1) Resumen demanda original vs corregida
    # --------------------------------------------------------
    filas_dem = []

    for t in T:
        for y in Y:
            dem_orig = sum(d_original[(i, t, y)] for i in B)
            dem_corr = sum(d_fw[(i, t, y)] for i in B)

            filas_dem.append({
                "t": t,
                "y": y,
                "demanda_original": dem_orig,
                "demanda_corregida": dem_corr,
                "omega_t": float(omega_t[t]),
                "demanda_corregida_ponderada_MWh": float(omega_t[t]) * dem_corr,
                "factor": demand_scale_factor,
                "capacidad_Pmax_total": cap_total_pmax,
                "capacidad_disp_total": cap_disp_ty[(t, y)]
            })

    df_dem = pd.DataFrame(filas_dem)

    print("\n--- Demanda original vs demanda corregida ---")
    display(
        df_dem
        .sort_values(["y", "t"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.2) Tabla de generación p[g,t,y]
    #       y emisiones e_g * p[g,t,y]
    # --------------------------------------------------------
    filas_p = []

    for g in G:
        for t in T:
            for y in Y:
                val_p = p[g, t, y].X
                val_ep = float(eg.get(g, 0.0)) * val_p
                wt = float(omega_t[t])

                filas_p.append({
                    "gen": g,
                    "t": t,
                    "y": y,
                    "bus": g_bus[g],
                    "Es_IBR": int(g in M),
                    "emision_factor": float(eg.get(g, 0.0)),
                    "p_gty": val_p,
                    "omega_t": wt,
                    "generacion_ponderada_MWh": wt * val_p,
                    "e_g_p_gty": val_ep,
                    "e_g_p_gty_ponderado": wt * val_ep
                })

    df_p = pd.DataFrame(filas_p)

    df_p_pos = df_p[df_p["p_gty"] > 1e-6].copy()
    df_p_pos = df_p_pos.sort_values(["y", "t", "gen"]).reset_index(drop=True)

    print("\n--- Tabla p[g,t,y] y e_g * p[g,t,y] (solo valores > 0) ---")
    display(df_p_pos)

    # --------------------------------------------------------
    # 13.3) Resumen por periodo y año
    # --------------------------------------------------------
    resumen_ty = (
        df_p
        .groupby(["y", "t"], as_index=False)
        .agg(
            generacion_total_no_pond=("p_gty", "sum"),
            generacion_total_ponderada_MWh=("generacion_ponderada_MWh", "sum"),
            emisiones_totales_intervalo_no_pond=("e_g_p_gty", "sum"),
            emisiones_totales_intervalo_ponderadas=("e_g_p_gty_ponderado", "sum")
        )
    )

    print("\n--- Resumen generación/emisiones por (y,t) ---")
    display(resumen_ty)

    # --------------------------------------------------------
    # 13.4) Emisiones totales por año
    # --------------------------------------------------------
    emisiones_por_anio = (
        df_p
        .groupby("y", as_index=False)
        .agg(
            emisiones_totales_no_ponderadas=("e_g_p_gty", "sum"),
            emisiones_totales_anuales_ponderadas=("e_g_p_gty_ponderado", "sum"),
            generacion_total_no_ponderada=("p_gty", "sum"),
            generacion_total_anual_ponderada_MWh=("generacion_ponderada_MWh", "sum")
        )
    )

    print("\n--- Emisiones totales por año ---")
    display(emisiones_por_anio)

    # --------------------------------------------------------
    # 13.5) Resumen por generador
    # --------------------------------------------------------
    resumen_g = (
        df_p
        .groupby("gen", as_index=False)
        .agg(
            bus=("bus", "first"),
            Es_IBR=("Es_IBR", "first"),
            emision_factor=("emision_factor", "first"),
            generacion_total_no_pond=("p_gty", "sum"),
            generacion_total_ponderada_MWh=("generacion_ponderada_MWh", "sum"),
            emisiones_totales_no_pond=("e_g_p_gty", "sum"),
            emisiones_totales_ponderadas=("e_g_p_gty_ponderado", "sum")
        )
        .sort_values("generacion_total_ponderada_MWh", ascending=False)
        .reset_index(drop=True)
    )

    print("\n--- Resumen por generador ---")
    display(resumen_g)

    # --------------------------------------------------------
    # 13.6) ENS total y por bus
    # --------------------------------------------------------
    filas_s = []

    for i in B:
        for t in T:
            for y in Y:
                filas_s.append({
                    "bus": i,
                    "t": t,
                    "y": y,
                    "ENS": s[i, t, y].X,
                    "omega_t": float(omega_t[t]),
                    "ENS_ponderado_MWh": float(omega_t[t]) * s[i, t, y].X,
                    "demanda_corregida": d_fw[(i, t, y)],
                    "demanda_original": d_original[(i, t, y)]
                })

    df_s = pd.DataFrame(filas_s)

    total_ENS = df_s["ENS"].sum()
    print(f"\nENS total = {total_ENS:.6f}")

    df_s_pos = (
        df_s[df_s["ENS"] > 1e-6]
        .sort_values(["y", "t", "bus"])
        .reset_index(drop=True)
    )

    print("\n--- ENS positivo por bus, periodo y año ---")
    display(df_s_pos)

    # --------------------------------------------------------
    # 13.7) Inversiones IBR
    # --------------------------------------------------------
    filas_x = []

    for m in M:
        for y in Y:
            filas_x.append({
                "gen": m,
                "y": y,
                "x_my": x[m, y].X
            })

    df_x = pd.DataFrame(filas_x)

    print("\n--- Inversión x[m,y] positiva ---")
    display(
        df_x[df_x["x_my"] > 1e-6]
        .sort_values(["y", "gen"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.8) Inercia sintética h_syn
    # --------------------------------------------------------
    filas_h = []

    for m in M:
        for t in T:
            for y in Y:
                filas_h.append({
                    "gen": m,
                    "t": t,
                    "y": y,
                    "h_syn": h_syn[m, t, y].X
                })

    df_h = pd.DataFrame(filas_h)

    print("\n--- h_syn[m,t,y] positivo ---")
    display(
        df_h[df_h["h_syn"] > 1e-6]
        .sort_values(["y", "t", "gen"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.9) FFR r_ffr
    # --------------------------------------------------------
    filas_r = []

    for m in M:
        for t in T:
            for y in Y:
                filas_r.append({
                    "gen": m,
                    "t": t,
                    "y": y,
                    "r_ffr": r_ffr[m, t, y].X
                })

    df_r = pd.DataFrame(filas_r)

    print("\n--- r_ffr[m,t,y] positivo ---")
    display(
        df_r[df_r["r_ffr"] > 1e-6]
        .sort_values(["y", "t", "gen"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.10) Chequeo agregado de inercia y FFR por periodo/año
    # --------------------------------------------------------
    filas_req = []

    for t in T:
        for y in Y:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * u[g, t, y].X
                for g in GS
            )

            H_syn_val = sum(
                h_syn[m, t, y].X
                for m in M
            )

            R_val = sum(
                r_ffr[m, t, y].X
                for m in M
            )

            filas_req.append({
                "t": t,
                "y": y,
                "H_sync": H_sync_val,
                "H_syn": H_syn_val,
                "H_total": H_sync_val + H_syn_val,
                "Hreq": float(Hreq[y]),
                "margen_H": H_sync_val + H_syn_val - float(Hreq[y]),
                "R_total": R_val,
                "Rreq": float(Rreq[y]),
                "margen_R": R_val - float(Rreq[y])
            })

    df_req = pd.DataFrame(filas_req)

    print("\n--- Chequeo de requerimientos Hreq y Rreq ---")
    display(
        df_req
        .sort_values(["y", "t"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 13.11) Flujos de línea
    # --------------------------------------------------------
    filas_f = []

    for (i, j) in L_arcs:
        for t in T:
            for y in Y:
                val_f = f[i, j, t, y].X
                Fij = float(F0[(i, j)])

                filas_f.append({
                    "from_bus": i,
                    "to_bus": j,
                    "t": t,
                    "y": y,
                    "f_ijty": val_f,
                    "F_max": Fij,
                    "uso_linea_abs": abs(val_f) / Fij if Fij > 1e-9 else None
                })

    df_f = pd.DataFrame(filas_f)

    print("\n--- Líneas más cargadas ---")
    display(
        df_f
        .sort_values("uso_linea_abs", ascending=False)
        .head(20)
        .reset_index(drop=True)
    )

else:
    print("El modelo no quedó óptimo. Status =", fw_base.Status)

# ============================================================
# 14) IMPRIMIR PARÁMETRO X barra: xbar[m]
# ============================================================

print("\n==============================")
print("PARÁMETRO X barra por IBR")
print("==============================")
print("xbar[m] representa la capacidad máxima de inversión permitida para cada IBR m.")
print("Es decir, en el modelo se usa como: x[m,y] <= xbar[m]\n")

filas_xbar = []

for m in M:
    filas_xbar.append({
        "gen_IBR": m,
        "bus": g_bus[m] if m in g_bus else None,
        "xbar_m": float(xbar[m])
    })

df_xbar = pd.DataFrame(filas_xbar)

print("--- Valores de xbar[m] ---")
display(
    df_xbar
    .sort_values("gen_IBR")
    .reset_index(drop=True)
)


PESOS TEMPORALES FW IEEE39
Días representados       = 365
Horas por periodo t      = 1.0
Peso omega_t por periodo = 365.0
Horas equivalentes total = 1460.00

CORRECCIÓN DATASET IEEE39
Demanda pico original     = 2029.335000
Capacidad instalada Pmax  = 1315.700000
Capacidad disp. mínima    = 715.000000
Modo escalamiento demanda = auto_capacity
Factor demanda usado      = 0.63537366
Demanda pico corregida    = 1289.386000
Margen Pmax - peak demand = 26.314000
✅ Demanda corregida queda dentro de la capacidad instalada.
⚠️ La demanda pico corregida puede superar la capacidad disponible P_disp en algún periodo.
   Esto puede ser normal si P_disp modela renovables variables, pero conviene revisar.

PRE-CHECKS FORWARD IEEE39 - FORMULACIÓN ORIGINAL
✅ d_fw[(i,t,y)] completo para B,T,Y.
⚠️ Capacidad disponible 715.00 < demanda corregida 781.45 en (t=1, y=1).
⚠️ Capacidad disponible 1086.01 < demanda corregida 1172.17 en (t=13, y=1).
⚠️ Capacidad disponible 715.00 < demanda corregida 1133.10 en 

,t,y,demanda_original,demanda_corregida,omega_t,demanda_corregida_ponderada_MWh,factor,capacidad_Pmax_total,capacidad_disp_total
0,1,1,1229.90000,781.446061,365.0,285227.812121,0.635374,1315.7,715.000000
1,7,1,1291.39500,820.518364,365.0,299489.202727,0.635374,1315.7,843.351412
2,13,1,1844.85000,1172.169091,365.0,427841.718182,0.635374,1315.7,1086.010207
3,19,1,1783.35500,1133.096788,365.0,413580.327576,0.635374,1315.7,715.000000
4,1,2,1291.39500,820.518364,365.0,299489.202727,0.635374,1315.7,715.000000
5,7,2,1355.96475,861.544282,365.0,314463.662864,0.635374,1315.7,843.351412
6,13,2,1937.09250,1230.777545,365.0,449233.804091,0.635374,1315.7,1086.010207
7,19,2,1872.52275,1189.751627,365.0,434259.343955,0.635374,1315.7,715.000000
8,1,3,1352.89000,859.590667,365.0,313750.593333,0.635374,1315.7,715.000000
9,7,3,1420.53450,902.570200,365.0,329438.123000,0.635374,1315.7,843.351412



--- Tabla p[g,t,y] y e_g * p[g,t,y] (solo valores > 0) ---


,gen,t,y,bus,Es_IBR,emision_factor,p_gty,omega_t,generacion_ponderada_MWh,e_g_p_gty,e_g_p_gty_ponderado
0,223_CT_4,1,1,16,0,0.55,55.000000,365.0,20075.000000,30.250000,11041.250000
1,223_STEAM_1,1,1,39,0,0.95,155.000000,365.0,56575.000000,147.250000,53746.250000
2,223_STEAM_2,1,1,20,0,0.95,155.000000,365.0,56575.000000,147.250000,53746.250000
3,223_STEAM_3,1,1,8,0,0.95,285.795760,365.0,104315.452455,271.505972,99099.679832
4,223_CT_4,7,1,16,0,0.55,55.000000,365.0,20075.000000,30.250000,11041.250000
...,...,...,...,...,...,...,...,...,...,...,...
91,314_PV_4,13,3,25,1,0.00,18.765496,365.0,6849.406113,0.000000,0.000000
92,223_CT_4,19,3,16,0,0.55,55.000000,365.0,20075.000000,30.250000,11041.250000
93,223_STEAM_1,19,3,39,0,0.95,155.000000,365.0,56575.000000,147.250000,53746.250000
94,223_STEAM_2,19,3,20,0,0.95,155.000000,365.0,56575.000000,147.250000,53746.250000



--- Resumen generación/emisiones por (y,t) ---


...

,t,y,H_sync,H_syn,H_total,Hreq,margen_H,R_total,Rreq,margen_R
0,1,1,11.800000,4.8056,16.605600,8.334900,8.270700,48.056,48.056,0.0
1,7,1,11.255066,4.8056,16.060666,8.334900,7.725766,48.056,48.056,0.0
2,13,1,11.800000,4.8056,16.605600,8.334900,8.270700,48.056,48.056,0.0
3,19,1,11.800000,4.8056,16.605600,8.334900,8.270700,48.056,48.056,0.0
4,1,2,11.800000,4.8056,16.605600,8.751645,7.853955,48.056,48.056,0.0
5,7,2,11.390572,4.8056,16.196172,8.751645,7.444527,48.056,48.056,0.0
6,13,2,11.800000,4.8056,16.605600,8.751645,7.853955,48.056,48.056,0.0
7,19,2,11.800000,4.8056,16.605600,8.751645,7.853955,48.056,48.056,0.0
8,1,3,11.800000,4.8056,16.605600,9.168390,7.437210,48.056,48.056,0.0
9,7,3,11.485004,4.8056,16.290604,9.168390,7.122214,48.056,48.056,0.0



--- Líneas más cargadas ---


,from_bus,to_bus,t,y,f_ijty,F_max,uso_linea_abs
0,19,20,19,3,-100.0,100.0,1.0
1,17,27,19,1,100.0,100.0,1.0
2,17,27,19,2,100.0,100.0,1.0
3,16,17,19,3,100.0,100.0,1.0
4,16,17,19,2,100.0,100.0,1.0
5,15,16,13,3,100.0,100.0,1.0
6,16,17,19,1,100.0,100.0,1.0
7,16,19,19,3,-100.0,100.0,1.0
8,4,5,1,2,-100.0,100.0,1.0
9,4,5,1,1,-100.0,100.0,1.0



PARÁMETRO X barra por IBR
xbar[m] representa la capacidad máxima de inversión permitida para cada IBR m.
Es decir, en el modelo se usa como: x[m,y] <= xbar[m]

--- Valores de xbar[m] ---


,gen_IBR,bus,xbar_m
0,313_PV_1,3,190.2
1,313_PV_2,15,186.6
2,313_RTPV_1,26,203.4
3,313_RTPV_2,4,126.2
4,314_PV_1,24,103.2
5,314_PV_2,29,103.2
6,314_PV_3,27,185.4
7,314_PV_4,25,103.2


### Solo A (Pmax sincronos)

#### SOLO Dualidad

In [30]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd

# ============================================================
# FINAL IEEE39: WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO
#
# Caso experimental: SOLO Pmax de generadores síncronos mutable.
#
# Mutable:
#   - Pmax_sync_ce[g] para g in GS.
#
# Fijos:
#   - Hreq[y] conserva su valor original.
#   - xbar[m] conserva su valor original.
#   - Pmax de los IBR no se modifica.
#
# Rango:
#   0.85 * Pmax[g] <= Pmax_sync_ce[g] <= Pmax[g]
#
# Pmax_sync_ce entra bilinealmente en:
#   Pmax_sync_ce[g] * u[g,t,y] - p[g,t,y] >= 0
# ============================================================


# ============================================================
# 0) OPCIONES
# ============================================================

# Restricción D del IEEE39
epsilon_CO2_wcep = 2366

# Pesos objetivo J
alpha_Psync = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

# Opciones WCEP
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1

# Para Pmax síncrono:
#   0.85 Pmax_o <= Pmax_sync_ce <= Pmax_o
PSYNC_LB_FACTOR = 0.85
PSYNC_UB_FACTOR = 1.00

# Si las cotas duales te generan problemas, puedes poner esto en False
USE_DUAL_BOUNDS_FOR_MUTABLE_ROWS = True


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}


# ============================================================
# 2) DEMANDA CORREGIDA DEL FW IEEE39
# ============================================================
# Si ya existe d_fw desde la celda FW, se usa.
# Si no existe, se reconstruye con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}

    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0

    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)

    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )

    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}

    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")

                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")

else:
    print("✅ Usando d_fw existente desde el FW corregido.")


# Diagnóstico rápido de demanda usada por WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)

cap_total_pmax = sum(float(Pmax[g]) for g in G)

print(f"Demanda pico usada en WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema         = {cap_total_pmax:.6f}")


# ============================================================
# 3) BASES DE PARÁMETROS MUTABLES
# ============================================================

bus_slack = int(sorted(B)[0])

Hreq_base = {y: float(Hreq[y]) for y in Y}
Pmax_sync_base = {g: float(Pmax[g]) for g in GS}
xbar_base = {m: float(xbar[m]) for m in M}

print("\n" + "=" * 70)
print("BASES DE PARÁMETROS MUTABLES")
print("=" * 70)
print(f"|GS| = {len(GS)}")
print(f"|M|  = {len(M)}")
print(f"|Y|  = {len(Y)}")
print(f"Bus slack usado: {bus_slack}")


# ============================================================
# 4) COTA DUAL HEURÍSTICA PARA LA FILA MUTABLE
# ============================================================

UB_sync = 2.0 * max(max(float(cg[g]) for g in G), float(VOLL))

print("\n" + "=" * 70)
print("COTA DUAL IEEE39")
print("=" * 70)
print(f"UB_sync = {UB_sync:.6f}")
print(f"Usar cota dual en sync_ub = {USE_DUAL_BOUNDS_FOR_MUTABLE_ROWS}")


# ============================================================
# 5) CREAR MODELO WCEP
# ============================================================

print("\n" + "=" * 70)
print("CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE")
print("=" * 70)
print("Mutables:")
print("  - Pmax_sync_ce[g]  en [0.85, 1.00] Pmax[g]")

w = gp.Model("WCEP_IEEE39_DF_SOLO_Pmax_sync")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT


# ============================================================
# 6) VARIABLES PRIMALES LOWER-LEVEL
# ============================================================

p = w.addVars(G, T, Y, lb=0.0, name="p")

u = w.addVars(
    GS, T, Y,
    lb=0.0,
    ub=1.0,
    name="u"
)

s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")

h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")

r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")


# ============================================================
# 7) VARIABLES MUTABLES UPPER-LEVEL
# ============================================================

Pmax_sync_ce = w.addVars(GS, lb=0.0, name="Pmax_sync_ce")


# ------------------------------------------------------------
# 7.1) Cotas admisibles Pmax síncronos
# ------------------------------------------------------------

for g in GS:
    base = Pmax_sync_base[g]

    w.addConstr(
        Pmax_sync_ce[g] >= PSYNC_LB_FACTOR * base,
        name=f"adm_lb_Psync[{g}]"
    )

    w.addConstr(
        Pmax_sync_ce[g] <= PSYNC_UB_FACTOR * base,
        name=f"adm_ub_Psync[{g}]"
    )


# ============================================================
# 8) FACTIBILIDAD PRIMAL Ax >= b
# ============================================================

R_primal = {}
RHS_expr = {}


# ------------------------------------------------------------
# 8.1) Balance nodal: igualdad como dos desigualdades
# ------------------------------------------------------------

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val


# ------------------------------------------------------------
# 8.2) ENS upper bound: s <= d
# ------------------------------------------------------------

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])

            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val


# ------------------------------------------------------------
# 8.3) Flujo DC: igualdad como dos desigualdades
# ------------------------------------------------------------

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0


# ------------------------------------------------------------
# 8.4) Límites de flujo
# ------------------------------------------------------------

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij


# ------------------------------------------------------------
# 8.5) Slack bus
# ------------------------------------------------------------

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0


# ------------------------------------------------------------
# 8.6) Capacidad de generación general
#      Ojo: acá NO se muta Pmax de IBR.
#      Se usa P_disp o Pmax original.
# ------------------------------------------------------------

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base

            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val


# ------------------------------------------------------------
# 8.7) Restricciones de generadores síncronos
#      p >= Pmin u
#      p <= Pmax_sync_ce u
# ------------------------------------------------------------

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            # Bilineal porque Pmax_sync_ce[g] es mutable y u[g,t,y] es variable primal
            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addQConstr(
                Pmax_sync_ce[g] * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0


# ------------------------------------------------------------
# 8.8) Capacidad máxima de inversión IBR
#      xcap[m,y] <= xbar[m] (FIJO)
#      En Ax >= b:
#          -xcap[m,y] >= -xbar[m]
# ------------------------------------------------------------

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(
            -xcap[m, y] >= -float(xbar_base[m]),
            name=name
        )
        RHS_expr[name] = -float(xbar_base[m])


# ------------------------------------------------------------
# 8.9) Capacidad de inercia sintética y FFR
# ------------------------------------------------------------

for m in M:
    for t in T:
        for y in Y:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0


# ------------------------------------------------------------
# 8.10) Requerimiento de inercia
# ------------------------------------------------------------

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= float(Hreq_base[y]), name=name)
        RHS_expr[name] = float(Hreq_base[y])


# ------------------------------------------------------------
# 8.11) Requerimiento de FFR
# ------------------------------------------------------------

for t in T:
    for y in Y:
        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])


# ============================================================
# 9) RESTRICCIÓN D NO DUALIZADA
# ============================================================

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= epsilon_CO2_wcep,
            name=f"D_CO2_cap_{y}"
        )

w.update()


# ============================================================
# 10) VARIABLES DUALES
# ============================================================

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if USE_DUAL_BOUNDS_FOR_MUTABLE_ROWS:
        if name.startswith("sync_ub_"):
            ub_here = UB_sync

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()


# ============================================================
# 11) COSTOS DEL FORWARD
# ============================================================

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)


# ============================================================
# 12) FACTIBILIDAD DUAL A^T y <= c
# ============================================================

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()

            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2

                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]

        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )


# ============================================================
# 13) DUALIDAD FUERTE
# ============================================================

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]

    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)


# ============================================================
# 14) OBJETIVO L1: SOLO CAMBIO EN Pmax SÍNCRONO
# ============================================================

dPsync = w.addVars(GS, lb=0.0, name="dPsync")

for g in GS:
    base = Pmax_sync_base[g]
    diff = Pmax_sync_ce[g] - base

    w.addConstr(dPsync[g] >= diff, name=f"dPsync_pos[{g}]")
    w.addConstr(dPsync[g] >= -diff, name=f"dPsync_neg[{g}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for g in GS:
        base = abs(float(Pmax_sync_base[g]))
        if base > 1e-9:
            obj_wcep += alpha_Psync * dPsync[g] / base
        else:
            obj_wcep += alpha_Psync * dPsync[g]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_Psync * dPsync.sum(),
        GRB.MINIMIZE
    )


# ============================================================
# 15) OPTIMIZAR WCEP
# ============================================================

print("\n" + "=" * 70)
print("RESOLVIENDO WCEP FINAL IEEE39")
print("=" * 70)

w.optimize()


# ============================================================
# 16) RESULTADOS WCEP
# ============================================================

print("\n" + "=" * 70)
print("RESULTADOS WCEP FINAL IEEE39")
print("=" * 70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")

else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")


# ============================================================
# 17) REPORTE DETALLADO
# ============================================================

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "=" * 70)
    print("CAMBIOS ÓPTIMOS EN Pmax SÍNCRONO")
    print("=" * 70)

    # --------------------------------------------------------
    # 17.1) Pmax síncronos (único parámetro mutable)
    # --------------------------------------------------------

    print("\n--- Pmax_sync_ce síncronos ---")
    filas_psync = []

    for g in GS:
        val = float(Pmax_sync_ce[g].X)
        base = float(Pmax_sync_base[g])
        delta = val - base
        pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")

        filas_psync.append({
            "gen": g,
            "Pmax_original": base,
            "Pmax_sync_ce": val,
            "delta": delta,
            "pct": pct
        })

    df_psync = pd.DataFrame(filas_psync)
    df_psync_changes = df_psync[abs(df_psync["delta"]) > tol].copy()

    if len(df_psync_changes) == 0:
        print("Sin cambios relevantes.")
    else:
        display(
            df_psync_changes
            .sort_values("pct")
            .reset_index(drop=True)
        )

    # --------------------------------------------------------
    # 17.2) Revisión dualidad fuerte
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("=" * 70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    # --------------------------------------------------------
    # 17.5) Emisiones anuales
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("=" * 70)

    filas_emis = []

    for y in Y:
        emis_y = sum(
            float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = epsilon_CO2_wcep - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {epsilon_CO2_wcep:.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            holgura = None
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

        filas_emis.append({
            "y": y,
            "emisiones": emis_y,
            "limite_D": epsilon_CO2_wcep if y >= y_star else None,
            "holgura_D": holgura
        })

    df_emis_wcep = pd.DataFrame(filas_emis)

    # --------------------------------------------------------
    # 17.6) Chequeo H y R
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("CHEQUEO H Y R")
    print("=" * 70)

    filas_req = []

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_base[y])

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            filas_req.append({
                "y": y,
                "t": t,
                "H_sync": H_sync_val,
                "H_syn": H_syn_val,
                "H_total": H_total,
                "Hreq_fijo": H_lim,
                "margen_H": H_total - H_lim,
                "R_total": R_total,
                "Rreq": R_lim,
                "margen_R": R_total - R_lim
            })

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_fijo={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    df_req_wcep = pd.DataFrame(filas_req)

    # --------------------------------------------------------
    # 17.7) Inversión xcap y uso de xbar fijo
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("INVERSIÓN IBR Y USO DE xbar FIJO")
    print("=" * 70)

    filas_xcap = []

    for m in M:
        for y in Y:
            x_val = float(xcap[m, y].X)
            xb_val = float(xbar_base[m])
            xb_base = float(xbar_base[m])

            uso_xbar = x_val / xb_val if xb_val > 1e-9 else None

            filas_xcap.append({
                "gen": m,
                "y": y,
                "xcap": x_val,
                "xbar_original": xb_base,
                "xbar_fijo": xb_val,
                "uso_xbar_fijo": uso_xbar
            })

    df_xcap_wcep = pd.DataFrame(filas_xcap)

    display(
        df_xcap_wcep[
            (df_xcap_wcep["xcap"] > 1e-6)
            | (df_xcap_wcep["uso_xbar_fijo"].fillna(0.0) > 0.85)
        ]
        .sort_values(["y", "uso_xbar_fijo"], ascending=[True, False])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 17.8) Generación y emisiones por generador
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("GENERACIÓN Y EMISIONES POR GENERADOR")
    print("=" * 70)

    filas_p = []

    for g in G:
        for t in T:
            for y in Y:
                p_val = float(p[g, t, y].X)
                emis_val = float(eg.get(g, 0.0)) * p_val

                filas_p.append({
                    "gen": g,
                    "t": t,
                    "y": y,
                    "bus": g_bus[g],
                    "Es_IBR": int(g in M),
                    "Es_sync": int(g in GS),
                    "emision_factor": float(eg.get(g, 0.0)),
                    "p_gty": p_val,
                    "emisiones": emis_val
                })

    df_p_wcep = pd.DataFrame(filas_p)

    resumen_g_wcep = (
        df_p_wcep
        .groupby("gen", as_index=False)
        .agg(
            bus=("bus", "first"),
            Es_IBR=("Es_IBR", "first"),
            Es_sync=("Es_sync", "first"),
            emision_factor=("emision_factor", "first"),
            generacion_total=("p_gty", "sum"),
            emisiones_totales=("emisiones", "sum")
        )
        .sort_values("generacion_total", ascending=False)
        .reset_index(drop=True)
    )

    display(resumen_g_wcep)

    # --------------------------------------------------------
    # 17.9) Revisión de la dual acotada asociada a Pmax
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("=" * 70)

    hits_sync = 0

    for name, var in ydual.items():
        if name.startswith("sync_ub_"):
            if abs(var.X - UB_sync) <= 1e-6 * max(1.0, UB_sync):
                hits_sync += 1

    print(f"Duales sync_ub pegadas a cota: {hits_sync}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en WCEP = 1289.386000
Pmax total sistema         = 1315.700000

BASES DE PARÁMETROS MUTABLES
|GS| = 4
|M|  = 8
|Y|  = 3
Bus slack usado: 1

COTA DUAL IEEE39
UB_sync = 10000.000000
Usar cota dual en sync_ub = True

CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE
Mutables:
  - Pmax_sync_ce[g]  en [0.85, 1.00] Pmax[g]
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.005
Set parameter OutputFlag to value 1

RESOLVIENDO WCEP FINAL IEEE39
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12650H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
MIPGap  0.005
NonConvex  2

Optimize a model with 4133 rows, 7088 columns and 15964 nonzeros (Min)
Model fingerprint: 0x75c48fa3
Model has 4 linear objective coefficients
Model has 2965 quadratic constraints

,gen,Pmax_original,Pmax_sync_ce,delta,pct
0,223_STEAM_1,155.0,131.750000,-23.250000,-15.000000
1,223_STEAM_2,155.0,131.750000,-23.250000,-15.000000
2,223_CT_4,55.0,46.750000,-8.250000,-15.000000
3,223_STEAM_3,350.0,342.607917,-7.392083,-2.112024



REVISIÓN DUALIDAD FUERTE
c^T x  = 2353045752.92811918
b^T y  = 2353045752.92812061
gap    = -1.430511e-06

EMISIONES ANUALES DE LA SOLUCIÓN WCEP
Año 1: emisiones = 2331.131842 | sin restricción D
Año 2: emisiones = 2349.868848 | sin restricción D
Año 3: emisiones = 2366.000000 | límite D = 2366.000000 | holgura = 0.000000

CHEQUEO H Y R
(y=1, t=1) | H_total=16.220186, Hreq_fijo=8.334900, margen_H=7.885286e+00 | R_total=48.056000, Rreq=48.056000, margen_R=-1.847411e-13
(y=1, t=7) | H_total=16.300383, Hreq_fijo=8.334900, margen_H=7.965483e+00 | R_total=48.056000, Rreq=48.056000, margen_R=-1.847411e-13
(y=1, t=13) | H_total=16.605600, Hreq_fijo=8.334900, margen_H=8.270700e+00 | R_total=48.056000, Rreq=48.056000, margen_R=-1.847411e-13
(y=1, t=19) | H_total=16.605600, Hreq_fijo=8.334900, margen_H=8.270700e+00 | R_total=48.056000, Rreq=48.056000, margen_R=-1.847411e-13
(y=2, t=1) | H_total=16.297215, Hreq_fijo=8.751645, margen_H=7.545570e+00 | R_total=48.056000, Rreq=48.056000, margen_R=-1

,gen,y,xcap,xbar_original,xbar_fijo,uso_xbar_fijo
0,314_PV_1,1,103.20,103.2,103.2,1.000000
1,314_PV_2,1,103.20,103.2,103.2,1.000000
2,313_PV_1,1,190.20,190.2,190.2,1.000000
3,313_RTPV_2,1,83.96,126.2,126.2,0.665293
4,314_PV_1,2,103.20,103.2,103.2,1.000000
5,313_PV_1,2,190.20,190.2,190.2,1.000000
6,313_RTPV_2,2,126.20,126.2,126.2,1.000000
7,313_PV_2,2,60.96,186.6,186.6,0.326688
8,314_PV_1,3,103.20,103.2,103.2,1.000000
9,314_PV_2,3,103.20,103.2,103.2,1.000000



GENERACIÓN Y EMISIONES POR GENERADOR


,gen,bus,Es_IBR,Es_sync,emision_factor,generacion_total,emisiones_totales
0,223_STEAM_3,8,0,1,0.95,3931.105989,3734.55069
1,223_STEAM_1,39,0,1,0.95,1581.000000,1501.95000
2,223_STEAM_2,20,0,1,0.95,1581.000000,1501.95000
3,223_CT_4,16,0,1,0.55,561.000000,308.55000
4,313_RTPV_1,26,1,0,0.00,288.943285,0.00000
5,313_PV_1,3,1,0,0.00,265.120492,0.00000
6,314_PV_3,27,1,0,0.00,263.373083,0.00000
7,313_PV_2,15,1,0,0.00,260.102438,0.00000
8,314_PV_1,24,1,0,0.00,146.602493,0.00000
9,313_RTPV_2,4,1,0,0.00,103.943520,0.00000



REVISIÓN DE DUALES ACOTADAS
Duales sync_ub pegadas a cota: 0


In [31]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import time

# ============================================================
# FINAL IEEE39: SOLO WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO + PESOS TEMPORALES
#
# Mutable único:
#   1) Pmax_sync_ce[g] para g in GS, entrando en A.
#
# Fijo:
#   - Hreq y xbar fijos.
#   - No hay PADM ni warm start.
# ============================================================

# Medición de tiempo total de celda/script
t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE39
# Si ya calculaste en el FW emisiones ponderadas, puedes poner epsilon_CO2_wcep manual.
# Si lo dejas como None, el código intenta usar el 90% de las emisiones FW guardadas en memoria.
epsilon_CO2_wcep = 930756.144528 * 0.90  # 90% de las emisiones ponderadas del FW corregido (valor de ejemplo, ajustar según tu caso)
EPSILON_CO2_FRACTION = 0.90

# Objetivo J
alpha_Psync = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = True

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE39
# ============================================================
# Este bloque hace que WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# Si T = [1, 7, 13, 19] representa un perfil diario, usamos omega_t
# para expandir costos y emisiones a base anual sin crear 365*|T| periodos.
# Por defecto cada t se repite 365 veces y representa 1 hora.
# Si cada t representa un bloque de 6 horas, define TIME_BLOCK_HOURS = 6.0 antes de correr.
TIME_BLOCK_HOURS = float(globals().get("TIME_BLOCK_HOURS", 1.0))
N_DAYS_EQUIV = int(globals().get("N_DAYS_EQUIV", 365))

if "omega_t" not in globals():
    omega_t = {t: float(N_DAYS_EQUIV) * TIME_BLOCK_HOURS for t in T}
else:
    omega_t = {t: float(omega_t[t]) for t in T}

def omega(t):
    return float(omega_t[t])

print("Pesos temporales omega_t usados:", omega_t)

bus_slack = int(sorted(B)[0])

Pmax_sync_base = {g: float(Pmax[g]) for g in GS}

UB_sync = 2.0 * max(max(float(cg[g]) for g in G), float(VOLL))

# ============================================================
# Límite de emisiones D
# ============================================================
def _get_fw_emissions_by_year():
    candidates = [
        "emisiones_por_anio", "df_emis_fw", "df_emis", "df_emis_wcep"
    ]
    value_cols = [
        "emisiones_anuales_equivalentes",
        "emisiones_totales_ponderadas",
        "emisiones_totales_anuales",
        "emisiones",
    ]
    for name in candidates:
        if name in globals():
            df = globals()[name]
            if hasattr(df, "columns") and "y" in df.columns:
                for col in value_cols:
                    if col in df.columns:
                        return {int(row["y"]): float(row[col]) for _, row in df.iterrows()}
    return None

_fw_emis = _get_fw_emissions_by_year()

if epsilon_CO2_wcep is None:
    if _fw_emis is not None:
        epsilon_CO2_wcep_by_y = {y: EPSILON_CO2_FRACTION * float(_fw_emis[y]) for y in Y if y in _fw_emis}
        print("epsilon_CO2_wcep calculado como 90% de emisiones FW:", epsilon_CO2_wcep_by_y)
    else:
        raise ValueError(
            "epsilon_CO2_wcep=None, pero no encontré emisiones del FW. "
            "Define epsilon_CO2_wcep manualmente o ejecuta antes el FW con tabla de emisiones."
        )
elif isinstance(epsilon_CO2_wcep, dict):
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep[y]) for y in epsilon_CO2_wcep}
else:
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep) for y in Y}

def eps_CO2(y):
    return float(epsilon_CO2_wcep_by_y[int(y)])


print("\n" + "="*70)
print("COTAS DUALES IEEE39")
print("="*70)
print(f"UB_sync = {UB_sync:.6f}")
print(f"Bus slack usado: {bus_slack}")




# ============================================================
# 2) WCEP FINAL CON DUALIDAD FUERTE
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE39_DF_solo_dataset_corregido")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Pmax_sync_ce = w.addVars(GS, lb=0.0, name="Pmax_sync_ce")

for g in GS:
    base = Pmax_sync_base[g]
    w.addConstr(Pmax_sync_ce[g] >= 0.85 * base, name=f"adm_lb_Psync[{g}]")
    w.addConstr(Pmax_sync_ce[g] <= 1.00 * base, name=f"adm_ub_Psync[{g}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            # Bilineal porque Pmax_sync_ce[g] es mutable y u[g,t,y] es variable primal.
            R_primal[name] = w.addQConstr(
                Pmax_sync_ce[g] * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -float(xbar[m]), name=name)
        RHS_expr[name] = -float(xbar[m])

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= float(Hreq[y]), name=name)
        RHS_expr[name] = float(Hreq[y])

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(omega(t) * float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= eps_CO2(y),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sync_ub_"):
        ub_here = UB_sync

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dPsync = w.addVars(GS, lb=0.0, name="dPsync")

for g in GS:
    base = Pmax_sync_base[g]
    diff = Pmax_sync_ce[g] - base

    w.addConstr(dPsync[g] >= diff, name=f"dPsync_pos[{g}]")
    w.addConstr(dPsync[g] >= -diff, name=f"dPsync_neg[{g}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for g in GS:
        base = abs(float(Pmax_sync_base[g]))
        if base > 1e-9:
            obj_wcep += alpha_Psync * dPsync[g] / base
        else:
            obj_wcep += alpha_Psync * dPsync[g]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(alpha_Psync * dPsync.sum(), GRB.MINIMIZE)


# ============================================================
# 3) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE39")
print("="*70)

w.optimize()


# ============================================================
# 4) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE39")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Pmax_sync_ce síncronos ---")
    filas_psync = []

    for g in GS:
        val = float(Pmax_sync_ce[g].X)
        base = float(Pmax_sync_base[g])
        delta = val - base
        pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")

        filas_psync.append({
            "gen": g,
            "Pmax_original": base,
            "Pmax_sync_ce": val,
            "delta": delta,
            "pct": pct
        })

    df_psync = pd.DataFrame(filas_psync)
    df_psync_changes = df_psync[abs(df_psync["delta"]) > tol].copy()

    if len(df_psync_changes) == 0:
        print("Sin cambios relevantes.")
    else:
        display(df_psync_changes.sort_values("pct").reset_index(drop=True))

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            omega(t) * float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = eps_CO2(y) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {eps_CO2(y):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq[y])

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_sync = 0

    for name, var in ydual.items():
        if name.startswith("sync_ub_"):
            if abs(var.X - UB_sync) <= 1e-6 * max(1.0, UB_sync):
                hits_sync += 1

    print(f"Duales sync_ub pegadas a cota : {hits_sync}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# REGISTRO DE RESULTADOS WCEP
# ============================================================

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo total  = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }

else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }
# Tiempo total real de toda la celda/script
t_total_script = time.time() - t0_total_script
print("\n" + "="*70)
print("TIEMPO TOTAL DE CELDA / SCRIPT")
print("="*70)
print(f"Tiempo Gurobi optimize() = {float(w.Runtime):.2f} s")
print(f"Tiempo total script      = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")

# Guardar registro en DataFrame si pandas está disponible
df_registro_wcep = pd.DataFrame([registro_wcep])
display(df_registro_wcep)


✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en WCEP = 1289.386000
Pmax total sistema                 = 1315.700000
Pesos temporales omega_t usados: {1: 365.0, 7: 365.0, 13: 365.0, 19: 365.0}

COTAS DUALES IEEE39
UB_sync = 10000.000000
Bus slack usado: 1

CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.005
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 18000

RESOLVIENDO WCEP FINAL IEEE39
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12650H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  18000
MIPGap  0.005
NonConvex  2

Optimize a model with 4133 rows, 7088 columns and 15964 nonzeros (Min)
Model fingerprint: 0xb701b19e
Model has 4 linear objective coefficients
Model has 2965 quadratic constraints
C

,status,runtime_seconds,runtime_minutes,runtime_hours,sol_count,obj_val,obj_bound,mip_gap,mip_gap_percent
0,3,0.235,0.003917,0.000065,0,None,None,None,None


#### Dualidad + PADM

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import copy
import math
import time
import pandas as pd

t0_total_script = time.time()

# ============================================================
# FINAL IEEE39 CORREGIDO: PADM-W + WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO
# VERSIÓN SOLO Pmax_sync_ce MUTABLE, CON PADM-W COMO PUNTO DE PARTIDA
#
# Caso experimental:
#   - ÚNICO parámetro mutable: Pmax_sync_ce[g], g in GS.
#   - Hreq[y] queda fijo.
#   - xbar[m] queda fijo.
#
# Es decir, solo se modifica A mediante:
#   Pmax_sync_ce[g] * u[g,t,y] - p[g,t,y] >= 0
#
# Flujo:
#   1) PADM-W etapa 1: busca factibilidad.
#   2) PADM-W etapa 2: parte de factibilidad y reduce J(theta).
#   3) El WCEP completo con dualidad fuerte SIEMPRE se ejecuta después.
#      Si PADM-W certifica un punto factible, ese punto se carga completo
#      como Start del WCEP final. La etapa 2 solo intenta mejorarlo.
# ============================================================

# ============================================================
# 0) OPCIONES
# ============================================================

PADM_OUTPUT = 1
RUN_PADM_W = True

# Si la etapa 1 encuentra factibilidad, la etapa 2 intenta reducir J(theta)
# partiendo exactamente desde ese punto. Aunque la etapa 2 falle, se conserva
# el punto factible de la etapa 1 para iniciar el WCEP final.
RUN_PADM_STAGE_2 = True

# Si PADM-W no certifica factibilidad, permite usar su último punto como Start
# no certificado. Esto no reemplaza las restricciones del WCEP final.
USE_LAST_PADM_POINT_IF_NO_FEASIBLE = True

MAX_OUTER = 20   # antes 100; versión tranquila para depuración
MAX_INNER = 20   # antes 100; versión tranquila para depuración

FEAS_TOL = 1e-5
INNER_TOL = 1e-5

RHO0 = 500.0
MU0 = 500.0

PENALTY_FACTOR = 2.0
MAX_PENALTY = 1e9

TIME_LIMIT_SUBPROBLEM = 5 * 60 * 60  # 5 horas por subproblema PADM

# Restricción D del IEEE39
epsilon_CO2_wcep = 2366

# Objetivo J
# Solo se penaliza la modificación de Pmax síncrono.
alpha_Psync = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1
WCEP_TIME_LIMIT = 5 * 60 * 60  # 5 horas para WCEP final

# Rango de mutabilidad para Pmax síncrono:
#   0.85 Pmax_o <= Pmax_sync_ce <= Pmax_o
PSYNC_LB_FACTOR = 0.85
PSYNC_UB_FACTOR = 1.00

# Cotas duales heurísticas para filas mutables
USE_DUAL_BOUNDS_FOR_MUTABLE_ROWS = True

# En el caso solo Pmax_sync_ce mutable, theta solo modifica A, no b.
# Por eso NO conviene imponer c^T z <= b^T lambda como restricción dura
# dentro del subproblema theta: theta no puede corregir esa desigualdad.
# La dualidad fuerte se mantiene dura en el subproblema Y y en el WCEP final.
ENFORCE_THETA_STRONG_DUALITY = False
SD_THETA_PENALTY = 1e3


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE39
# ============================================================
# Este bloque hace que PADM-W y WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por PADM-W/WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en PADM-W/WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

bus_slack = int(sorted(B)[0])

Hreq_base = {y: float(Hreq[y]) for y in Y}
Pmax_sync_base = {g: float(Pmax[g]) for g in GS}
xbar_base = {m: float(xbar[m]) for m in M}

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_sync = 2.0 * max(max(float(cg[g]) for g in G), float(VOLL))
UB_xbar = 2.0 * max([float(cinv[m]) for m in M], default=1.0)

print("\n" + "="*70)
print("COTAS DUALES IEEE39")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_sync = {UB_sync:.6f}")
print(f"UB_xbar = {UB_xbar:.6f}")
print(f"Bus slack usado: {bus_slack}")


# ============================================================
# 2) REPRESENTACIÓN SIMBÓLICA PARA PADM-W
# ============================================================

def key_p(g,t,y):       return ("p", g, t, y)
def key_u(g,t,y):       return ("u", g, t, y)
def key_s(i,t,y):       return ("s", i, t, y)
def key_xcap(m,y):      return ("xcap", m, y)
def key_hsyn(m,t,y):    return ("h_syn", m, t, y)
def key_rffr(m,t,y):    return ("r_ffr", m, t, y)
def key_fpos(i,j,t,y):  return ("f_pos", i, j, t, y)
def key_fneg(i,j,t,y):  return ("f_neg", i, j, t, y)
def key_thp(i,t,y):     return ("theta_p", i, t, y)
def key_thm(i,t,y):     return ("theta_m", i, t, y)

def param_coeff(family, idx, mult=1.0):
    return ("PARAM", family, idx, float(mult))

def is_param_coeff(c):
    return isinstance(c, tuple) and len(c) == 4 and c[0] == "PARAM"

def coeff_value(c, theta_vals):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * float(theta_vals[fam][idx])
    return float(c)

def coeff_expr_in_theta_model(c, theta_vars):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * theta_vars[fam][idx]
    return float(c)

def rhs_const(val):
    return [("CONST", None, None, float(val))]

def rhs_param(family, idx, mult=1.0):
    return [("PARAM", family, idx, float(mult))]

def rhs_has_param(rhs_terms):
    return any(typ == "PARAM" for typ, fam, idx, mult in rhs_terms)

def rhs_eval(rhs_terms, theta_vals=None):
    out = 0.0

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            out += float(mult)
        elif typ == "PARAM":
            if theta_vals is None:
                raise ValueError("rhs_eval requiere theta_vals para RHS con parámetros.")
            out += float(mult) * float(theta_vals[fam][idx])
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return out

def rhs_expr_in_theta_model(rhs_terms, theta_vars):
    expr = gp.LinExpr()

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            expr += float(mult)
        elif typ == "PARAM":
            expr += float(mult) * theta_vars[fam][idx]
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return expr

def add_to_terms(terms, key, coeff):
    if key not in terms:
        terms[key] = []
    terms[key].append(coeff)

def neg_coeff(c):
    if is_param_coeff(c):
        return ("PARAM", c[1], c[2], -c[3])
    return -float(c)

def terms_has_param(terms):
    for k, coeffs in terms.items():
        for c in coeffs:
            if is_param_coeff(c):
                return True
    return False

def terms_expr_z_model(terms, z_vars, theta_vals):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        for c in coeffs:
            expr += coeff_value(c, theta_vals) * z_vars[k]

    return expr

def terms_expr_theta_model_with_fixed_z(terms, z_vals, theta_vars):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            expr += zval * coeff_expr_in_theta_model(c, theta_vars)

    return expr

def terms_eval(terms, z_vals, theta_vals):
    out = 0.0

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            out += coeff_value(c, theta_vals) * zval

    return out


# ============================================================
# 3) CREAR FILAS PRIMALES Ax >= b
# ============================================================

rows = {}
row_order = []

def add_row(name, terms, rhs_terms):
    if name in rows:
        raise ValueError(f"Fila repetida: {name}")

    rows[name] = {
        "terms": terms,
        "rhs": rhs_terms
    }

    row_order.append(name)

# ------------------------------------------------------------
# A.1 Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}

            for g in G_en_bus.get(i, []):
                add_to_terms(terms, key_p(g,t,y), 1.0)

            add_to_terms(terms, key_s(i,t,y), 1.0)

            for (k,j) in lineas_entran[i]:
                add_to_terms(terms, key_fpos(k,j,t,y), 1.0)
                add_to_terms(terms, key_fneg(k,j,t,y), -1.0)

            for (i2,j) in lineas_salen[i]:
                add_to_terms(terms, key_fpos(i2,j,t,y), -1.0)
                add_to_terms(terms, key_fneg(i2,j,t,y), 1.0)

            rhs_val = float(d_fw[(i,t,y)])

            add_row(f"bal_p_{i}_{t}_{y}", terms, rhs_const(rhs_val))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"bal_n_{i}_{t}_{y}", terms_n, rhs_const(-rhs_val))

# ------------------------------------------------------------
# A.2 ENS <= demanda
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}
            add_to_terms(terms, key_s(i,t,y), -1.0)
            add_row(f"ens_ub_{i}_{t}_{y}", terms, rhs_const(-float(d_fw[(i,t,y)])))

# ------------------------------------------------------------
# A.3 Flujo DC
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i,j)])

            terms = {}
            add_to_terms(terms, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms, key_fneg(i,j,t,y), -1.0)

            add_to_terms(terms, key_thp(i,t,y), -Bij)
            add_to_terms(terms, key_thm(i,t,y),  Bij)
            add_to_terms(terms, key_thp(j,t,y),  Bij)
            add_to_terms(terms, key_thm(j,t,y), -Bij)

            add_row(f"dc_p_{i}_{j}_{t}_{y}", terms, rhs_const(0.0))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"dc_n_{i}_{j}_{t}_{y}", terms_n, rhs_const(0.0))

# ------------------------------------------------------------
# A.4 Límites térmicos fijos
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i,j)])

            terms_lb = {}
            add_to_terms(terms_lb, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms_lb, key_fneg(i,j,t,y), -1.0)
            add_row(f"flb_{i}_{j}_{t}_{y}", terms_lb, rhs_const(-Fij))

            terms_ub = {}
            add_to_terms(terms_ub, key_fpos(i,j,t,y), -1.0)
            add_to_terms(terms_ub, key_fneg(i,j,t,y), 1.0)
            add_row(f"fub_{i}_{j}_{t}_{y}", terms_ub, rhs_const(-Fij))

# ------------------------------------------------------------
# A.5 Slack bus
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_p = {}
        add_to_terms(terms_p, key_thp(bus_slack,t,y), 1.0)
        add_to_terms(terms_p, key_thm(bus_slack,t,y), -1.0)
        add_row(f"slack_p_{t}_{y}", terms_p, rhs_const(0.0))

        terms_n = {}
        add_to_terms(terms_n, key_thp(bus_slack,t,y), -1.0)
        add_to_terms(terms_n, key_thm(bus_slack,t,y), 1.0)
        add_row(f"slack_n_{t}_{y}", terms_n, rhs_const(0.0))

print(f"Bus slack usado: {bus_slack}")

# ------------------------------------------------------------
# A.6 Límites generación fijos p <= P_disp
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_base = float(P_disp.get((g,t), Pmax[g]))

        for y in Y:
            terms = {}
            add_to_terms(terms, key_p(g,t,y), -1.0)
            add_row(f"p_cap_{g}_{t}_{y}", terms, rhs_const(-cap_base))

# ------------------------------------------------------------
# A.7 Síncronos
# ------------------------------------------------------------
for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            terms_lb = {}
            add_to_terms(terms_lb, key_p(g,t,y), 1.0)
            add_to_terms(terms_lb, key_u(g,t,y), -pmin_g)
            add_row(f"sync_lb_{g}_{t}_{y}", terms_lb, rhs_const(0.0))

            terms_ub = {}
            add_to_terms(terms_ub, key_u(g,t,y), param_coeff("Pmax_sync_ce", g, 1.0))
            add_to_terms(terms_ub, key_p(g,t,y), -1.0)
            add_row(f"sync_ub_{g}_{t}_{y}", terms_ub, rhs_const(0.0))

            terms_u = {}
            add_to_terms(terms_u, key_u(g,t,y), -1.0)
            add_row(f"u_ub_{g}_{t}_{y}", terms_u, rhs_const(-1.0))

# ------------------------------------------------------------
# A.8-A.10 Inversión, h_syn, FFR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        terms_x = {}
        add_to_terms(terms_x, key_xcap(m,y), -1.0)
        add_row(f"x_ub_{m}_{y}", terms_x, rhs_const(-float(xbar[m])))

        for t in T:
            terms_h = {}
            add_to_terms(terms_h, key_xcap(m,y), float(gamma_base[m]))
            add_to_terms(terms_h, key_hsyn(m,t,y), -1.0)
            add_row(f"hsyn_lim_{m}_{t}_{y}", terms_h, rhs_const(0.0))

            terms_r = {}
            add_to_terms(terms_r, key_xcap(m,y), float(kappa_base[m]))
            add_to_terms(terms_r, key_rffr(m,t,y), -1.0)
            add_row(f"ffr_lim_{m}_{t}_{y}", terms_r, rhs_const(0.0))

# ------------------------------------------------------------
# A.11 Seguridad: Hreq fijo y Rreq fijo
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_H = {}

        for g in GS:
            add_to_terms(terms_H, key_u(g,t,y), float(Hg.get(g, 0.0)))

        for m in M:
            add_to_terms(terms_H, key_hsyn(m,t,y), 1.0)

        add_row(f"sys_H_{t}_{y}", terms_H, rhs_const(float(Hreq[y])))

        terms_R = {}

        for m in M:
            add_to_terms(terms_R, key_rffr(m,t,y), 1.0)

        add_row(f"sys_R_{t}_{y}", terms_R, rhs_const(float(Rreq[y])))


# ============================================================
# 4) VARIABLES PRIMALES Y COSTOS c
# ============================================================

primal_keys = []

for g in G:
    for t in T:
        for y in Y:
            primal_keys.append(key_p(g,t,y))

for g in GS:
    for t in T:
        for y in Y:
            primal_keys.append(key_u(g,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_s(i,t,y))

for m in M:
    for y in Y:
        primal_keys.append(key_xcap(m,y))

for m in M:
    for t in T:
        for y in Y:
            primal_keys.append(key_hsyn(m,t,y))
            primal_keys.append(key_rffr(m,t,y))

for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            primal_keys.append(key_fpos(i,j,t,y))
            primal_keys.append(key_fneg(i,j,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_thp(i,t,y))
            primal_keys.append(key_thm(i,t,y))

c_map = {}

for g in G:
    for t in T:
        for y in Y:
            c_map[key_p(g,t,y)] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map[key_s(i,t,y)] = float(VOLL)

for m in M:
    for y in Y:
        c_map[key_xcap(m,y)] = float(cinv[m])

for k in primal_keys:
    c_map.setdefault(k, 0.0)


# ============================================================
# 5) CLASIFICACIÓN ACOPLADAS VS SEPARABLES
# ============================================================

coupled_primal_rows = [
    name for name in row_order
    if terms_has_param(rows[name]["terms"]) or rhs_has_param(rows[name]["rhs"])
]

separable_primal_rows = [
    name for name in row_order
    if name not in set(coupled_primal_rows)
]

coupled_dual_cols = set()

for name in row_order:
    terms = rows[name]["terms"]

    for k, coeffs in terms.items():
        if any(is_param_coeff(c) for c in coeffs):
            coupled_dual_cols.add(k)

coupled_dual_cols = sorted(list(coupled_dual_cols), key=str)

separable_dual_cols = [
    k for k in primal_keys
    if k not in set(coupled_dual_cols)
]

print("\n" + "="*70)
print("CLASIFICACIÓN PADM-W IEEE39 - FORMULACIÓN ORIGINAL")
print("="*70)
print(f"N° filas primales acopladas   : {len(coupled_primal_rows)}")
print(f"N° filas primales separables  : {len(separable_primal_rows)}")
print(f"N° columnas duales acopladas  : {len(coupled_dual_cols)}")
print(f"N° columnas duales separables : {len(separable_dual_cols)}")

bad_rows = [
    name for name in coupled_primal_rows
    if not name.startswith("sync_ub_")
]

if bad_rows:
    print("\n⚠️ Hay filas acopladas no esperadas:")
    print(bad_rows[:30])
else:
    print("\n✅ Filas acopladas esperadas: solo sync_ub_.")


# ============================================================
# 6) FUNCIONES PADM-W
# ============================================================

def initial_theta_vals():
    return {
        "Pmax_sync_ce": {g: float(Pmax_sync_base[g]) for g in GS},
    }

def compute_J(theta_vals):
    J = 0.0

    for g in GS:
        base = float(Pmax_sync_base[g])
        val = float(theta_vals["Pmax_sync_ce"][g])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_Psync * abs(val - base) / abs(base)
        else:
            J += alpha_Psync * abs(val - base)

    return J

def compute_ctx(z_vals):
    return sum(
        float(c_map[k]) * float(z_vals.get(k, 0.0))
        for k in primal_keys
    )

def compute_bty(theta_vals, ydual_vals):
    out = 0.0

    for name in row_order:
        out += rhs_eval(rows[name]["rhs"], theta_vals) * float(ydual_vals.get(name, 0.0))

    return out

def compute_primal_residuals(theta_vals, z_vals):
    res = {}

    for name in row_order:
        lhs = terms_eval(rows[name]["terms"], z_vals, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)
        res[name] = max(0.0, rhs - lhs)

    return res

def compute_dual_residuals(theta_vals, ydual_vals):
    res = {}

    for k in primal_keys:
        lhs = 0.0

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            yd = float(ydual_vals.get(name, 0.0))

            if abs(yd) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd

        rhs = float(c_map.get(k, 0.0))
        res[k] = max(0.0, lhs - rhs)

    return res

def compute_sd_residual(theta_vals, z_vals, ydual_vals):
    return max(0.0, compute_ctx(z_vals) - compute_bty(theta_vals, ydual_vals))

def max_residuals(theta_vals, z_vals, ydual_vals):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)
    sd = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return {
        "max_primal_all": max(pr.values()) if pr else 0.0,
        "max_dual_all": max(dr.values()) if dr else 0.0,
        "sd": sd,
        "max_primal_coupled": max([pr[name] for name in coupled_primal_rows], default=0.0),
        "max_dual_coupled": max([dr[k] for k in coupled_dual_cols], default=0.0),
    }

def compute_phi(theta_vals, z_vals, ydual_vals, rho, mu, use_true_objective=True):
    Jval = compute_J(theta_vals) if use_true_objective else 0.0

    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    pen_p = sum(float(rho[name]) * pr[name] for name in coupled_primal_rows)
    pen_d = sum(float(mu[k]) * dr[k] for k in coupled_dual_cols)

    sd_penalty = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return Jval + pen_p + pen_d + 1e3 * sd_penalty

def diff_theta(theta_a, theta_b):
    vals = []

    for g in GS:
        vals.append(abs(theta_a["Pmax_sync_ce"][g] - theta_b["Pmax_sync_ce"][g]))

    return max(vals) if vals else 0.0

def diff_dict(a, b):
    keys = set(a.keys()) | set(b.keys())
    return max(
        [abs(float(a.get(k,0.0)) - float(b.get(k,0.0))) for k in keys],
        default=0.0
    )


# ============================================================
# 7) SUBPROBLEMAS PADM-W
# ============================================================

def add_z_variables(model):
    z = {}

    for g in G:
        for t in T:
            for y in Y:
                z[key_p(g,t,y)] = model.addVar(lb=0.0, name=f"p[{g},{t},{y}]")

    for g in GS:
        for t in T:
            for y in Y:
                z[key_u(g,t,y)] = model.addVar(lb=0.0, ub=1.0, name=f"u[{g},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_s(i,t,y)] = model.addVar(lb=0.0, name=f"s[{i},{t},{y}]")

    for m in M:
        for y in Y:
            z[key_xcap(m,y)] = model.addVar(lb=0.0, name=f"xcap[{m},{y}]")

    for m in M:
        for t in T:
            for y in Y:
                z[key_hsyn(m,t,y)] = model.addVar(lb=0.0, name=f"h_syn[{m},{t},{y}]")
                z[key_rffr(m,t,y)] = model.addVar(lb=0.0, name=f"r_ffr[{m},{t},{y}]")

    for (i,j) in L_arcs:
        for t in T:
            for y in Y:
                z[key_fpos(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_pos[{i},{j},{t},{y}]")
                z[key_fneg(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_neg[{i},{j},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_thp(i,t,y)] = model.addVar(lb=0.0, name=f"theta_p[{i},{t},{y}]")
                z[key_thm(i,t,y)] = model.addVar(lb=0.0, name=f"theta_m[{i},{t},{y}]")

    return z

def add_dual_variables(model):
    yd = {}

    for name in row_order:
        ub_here = GRB.INFINITY

        if USE_DUAL_BOUNDS_FOR_MUTABLE_ROWS:
            if name.startswith("sync_ub_"):
                ub_here = UB_sync

        yd[name] = model.addVar(lb=0.0, ub=ub_here, name=f"ydual[{name}]")

    return yd

def add_theta_variables(model, warm_theta=None):
    theta = {
        "Pmax_sync_ce": {},
    }

    for g in GS:
        theta["Pmax_sync_ce"][g] = model.addVar(
            lb=PSYNC_LB_FACTOR * float(Pmax_sync_base[g]),
            ub=PSYNC_UB_FACTOR * float(Pmax_sync_base[g]),
            name=f"Pmax_sync_ce[{g}]"
        )

    if warm_theta is not None:
        for g in GS:
            theta["Pmax_sync_ce"][g].Start = float(warm_theta["Pmax_sync_ce"][g])

    return theta

def extract_theta(theta_vars):
    return {
        "Pmax_sync_ce": {g: float(theta_vars["Pmax_sync_ce"][g].X) for g in GS},
    }

def add_separable_primal_constraints_y(model, z_vars):
    theta_base = initial_theta_vals()

    for name in separable_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z_vars, theta_base)
        rhs = rhs_eval(rows[name]["rhs"], theta_base)
        model.addConstr(lhs >= rhs, name=f"Y_primal[{name}]")

def add_D_constraint_y(model, z_vars):
    for y in Y:
        if y >= y_star:
            model.addConstr(
                gp.quicksum(
                    float(eg[g]) * z_vars[key_p(g,t,y)]
                    for g in GE
                    for t in T
                )
                <= float(epsilon_CO2_wcep),
                name=f"D_CO2_cap_{y}"
            )

def add_separable_dual_constraints_y(model, ydual_vars):
    theta_base = initial_theta_vals()

    for k in separable_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_base) * ydual_vars[name]

        model.addConstr(lhs <= float(c_map.get(k, 0.0)), name=f"Y_dual[{k}]")

def add_strong_duality_y(model, theta_vals, z_vars, ydual_vars):
    ctx = gp.quicksum(float(c_map[k]) * z_vars[k] for k in primal_keys)

    bty = gp.LinExpr()

    for name in row_order:
        bty += rhs_eval(rows[name]["rhs"], theta_vals) * ydual_vars[name]

    model.addConstr(ctx <= bty, name="Y_StrongDuality")

def solve_theta_subproblem(
    z_vals,
    ydual_vals,
    rho,
    mu,
    warm_theta=None,
    log=False,
    use_true_objective=True
):
    m = gp.Model("PADM_theta_subproblem_IEEE39_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    theta = add_theta_variables(m, warm_theta=warm_theta)

    obj = gp.LinExpr()

    if use_true_objective:
        for g in GS:
            dvar = m.addVar(lb=0.0, name=f"dPsync[{g}]")
            diff = theta["Pmax_sync_ce"][g] - float(Pmax_sync_base[g])

            m.addConstr(dvar >= diff, name=f"dPsync_pos[{g}]")
            m.addConstr(dvar >= -diff, name=f"dPsync_neg[{g}]")

            base = abs(float(Pmax_sync_base[g]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_Psync) * dvar / base
            else:
                obj += float(alpha_Psync) * dvar

    for name in coupled_primal_rows:
        lhs = terms_expr_theta_model_with_fixed_z(rows[name]["terms"], z_vals, theta)
        rhs = rhs_expr_in_theta_model(rows[name]["rhs"], theta)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            ydval = float(ydual_vals.get(name, 0.0))

            if abs(ydval) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += ydval * coeff_expr_in_theta_model(c, theta)

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    # ------------------------------------------------------------
    # Dualidad fuerte en el subproblema theta
    # ------------------------------------------------------------
    # Importante para el caso SOLO Pmax_sync_ce mutable:
    # theta modifica A mediante sync_ub_, pero NO modifica b.
    # Entonces c^T z <= b^T lambda no depende realmente de theta.
    # Si se impone dura aquí, el subproblema theta puede quedar infactible
    # por una violación pequeña que theta no puede reparar.
    #
    # Por eso, por defecto NO se impone en theta. La dualidad fuerte
    # permanece dura en el subproblema Y y en el WCEP final.
    ctx_fixed = sum(float(c_map[k]) * float(z_vals.get(k, 0.0)) for k in primal_keys)

    bty_theta = gp.LinExpr()

    for name in row_order:
        ydval = float(ydual_vals.get(name, 0.0))

        if abs(ydval) <= 1e-12:
            continue

        bty_theta += ydval * rhs_expr_in_theta_model(rows[name]["rhs"], theta)

    if ENFORCE_THETA_STRONG_DUALITY:
        m.addConstr(ctx_fixed <= bty_theta, name="Theta_StrongDuality")
    else:
        # Versión robusta: se registra la posible violación como slack penalizado.
        # Esto evita infactibilidad artificial del subproblema theta.
        v_sd_theta = m.addVar(lb=0.0, name="viol_strong_duality_theta")
        m.addConstr(
            v_sd_theta >= ctx_fixed - bty_theta,
            name="def_viol_strong_duality_theta"
        )
        obj += float(SD_THETA_PENALTY) * v_sd_theta

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema THETA falló.")
        print(f"Status = {m.Status}")
        return None, m.Status, None

    theta_vals = extract_theta(theta)
    return theta_vals, m.Status, float(m.ObjVal)

def solve_y_subproblem(theta_vals, rho, mu, warm_z=None, warm_ydual=None, log=False):
    m = gp.Model("PADM_y_subproblem_IEEE39_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    z = add_z_variables(m)
    yd = add_dual_variables(m)

    m.update()

    if warm_z is not None:
        for k, v in z.items():
            if k in warm_z:
                v.Start = float(warm_z[k])

    if warm_ydual is not None:
        for name, v in yd.items():
            if name in warm_ydual:
                v.Start = float(warm_ydual[name])

    add_separable_primal_constraints_y(m, z)
    add_D_constraint_y(m, z)
    add_separable_dual_constraints_y(m, yd)
    add_strong_duality_y(m, theta_vals, z, yd)

    obj = gp.LinExpr()

    for name in coupled_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd[name]

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    if Y_OBJECTIVE_MODE == "ctx":
        obj += TIEBREAKER_CTX_WEIGHT * gp.quicksum(
            float(c_map[k]) * z[k]
            for k in primal_keys
        )

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema Y falló.")
        print(f"Status = {m.Status}")
        return None, None, m.Status, None

    z_vals = {k: float(v.X) for k, v in z.items()}
    yd_vals = {name: float(v.X) for name, v in yd.items()}

    return z_vals, yd_vals, m.Status, float(m.ObjVal)

def initialize_penalties():
    rho = {name: float(RHO0) for name in coupled_primal_rows}
    mu = {k: float(MU0) for k in coupled_dual_cols}
    return rho, mu

def update_penalties(theta_vals, z_vals, ydual_vals, rho, mu):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    n_rho = 0
    n_mu = 0

    for name in coupled_primal_rows:
        if pr[name] > FEAS_TOL:
            rho[name] = min(MAX_PENALTY, PENALTY_FACTOR * rho[name])
            n_rho += 1

    for k in coupled_dual_cols:
        if dr[k] > FEAS_TOL:
            mu[k] = min(MAX_PENALTY, PENALTY_FACTOR * mu[k])
            n_mu += 1

    return n_rho, n_mu


# ============================================================
# 8) LOOP PADM-W
# ============================================================

def run_padm(
    use_true_objective=True,
    theta_init=None,
    z_init=None,
    ydual_init=None,
    label="PADM"
):
    print("\n" + "="*70)
    print(f"INICIO {label}")
    print("="*70)
    print(f"use_true_objective = {use_true_objective}")

    rho, mu = initialize_penalties()

    theta_vals = copy.deepcopy(theta_init) if theta_init is not None else initial_theta_vals()

    if z_init is None or ydual_init is None:
        print("\nInicializando Y=(z,lambda)...")

        z_vals, ydual_vals, st_y, obj_y = solve_y_subproblem(
            theta_vals,
            rho,
            mu,
            warm_z=None,
            warm_ydual=None,
            log=False
        )

        if z_vals is None:
            print("❌ No se pudo inicializar Y.")

            return {
                "best_feasible": None,
                "theta": theta_vals,
                "z": None,
                "ydual": None,
                "history": [],
                "last_residuals": None,
                "rho": rho,
                "mu": mu,
            }

    else:
        z_vals = z_init.copy()
        ydual_vals = ydual_init.copy()

    history = []
    best_feasible = None

    res0 = max_residuals(theta_vals, z_vals, ydual_vals)

    print("\nChequeo inicial:")
    print(f"J(theta)       = {compute_J(theta_vals):.8f}")
    print(f"Max primal     = {res0['max_primal_all']:.3e}")
    print(f"Max dual       = {res0['max_dual_all']:.3e}")
    print(f"Strong duality = {res0['sd']:.3e}")

    for k_outer in range(MAX_OUTER):

        print("\n" + "#"*70)
        print(f"{label} | OUTER k = {k_outer}")
        print("#"*70)

        phi_prev = compute_phi(
            theta_vals,
            z_vals,
            ydual_vals,
            rho,
            mu,
            use_true_objective=use_true_objective
        )

        for l_inner in range(MAX_INNER):

            theta_old = copy.deepcopy(theta_vals)
            z_old = z_vals.copy()
            yd_old = ydual_vals.copy()

            theta_new, st_theta, obj_theta = solve_theta_subproblem(
                z_vals,
                ydual_vals,
                rho,
                mu,
                warm_theta=theta_vals,
                log=False,
                use_true_objective=use_true_objective
            )

            if theta_new is None:
                print("⚠️ Falló subproblema theta.")
                break

            theta_vals = theta_new

            z_new, yd_new, st_y, obj_y = solve_y_subproblem(
                theta_vals,
                rho,
                mu,
                warm_z=z_vals,
                warm_ydual=ydual_vals,
                log=False
            )

            if z_new is None:
                print("⚠️ Falló subproblema Y.")
                break

            z_vals = z_new
            ydual_vals = yd_new

            phi_now = compute_phi(
                theta_vals,
                z_vals,
                ydual_vals,
                rho,
                mu,
                use_true_objective=use_true_objective
            )

            res = max_residuals(theta_vals, z_vals, ydual_vals)

            dtheta = diff_theta(theta_vals, theta_old)
            dz = diff_dict(z_vals, z_old)
            dyd = diff_dict(ydual_vals, yd_old)
            dphi = abs(phi_now - phi_prev) / max(1.0, abs(phi_prev))

            if PADM_OUTPUT:
                print(
                    f"k={k_outer:02d}, l={l_inner:02d} | "
                    f"Phi={phi_now:.6e}, J={compute_J(theta_vals):.6e}, "
                    f"p_all={res['max_primal_all']:.2e}, "
                    f"d_all={res['max_dual_all']:.2e}, "
                    f"sd={res['sd']:.2e}, "
                    f"p_coup={res['max_primal_coupled']:.2e}, "
                    f"d_coup={res['max_dual_coupled']:.2e}, "
                    f"dtheta={dtheta:.2e}, dz={dz:.2e}, dyd={dyd:.2e}, dphi={dphi:.2e}"
                )

            history.append({
                "outer": k_outer,
                "inner": l_inner,
                "phi": phi_now,
                "J": compute_J(theta_vals),
                "max_primal_all": res["max_primal_all"],
                "max_dual_all": res["max_dual_all"],
                "sd": res["sd"],
                "max_primal_coupled": res["max_primal_coupled"],
                "max_dual_coupled": res["max_dual_coupled"],
                "dtheta": dtheta,
                "dz": dz,
                "dyd": dyd,
                "dphi": dphi,
                "rho_max": max(rho.values()) if rho else 0.0,
                "mu_max": max(mu.values()) if mu else 0.0,
            })

            if dphi <= INNER_TOL and dtheta <= INNER_TOL and dz <= INNER_TOL and dyd <= INNER_TOL:
                print(f"Inner loop converge en l={l_inner}.")
                break

            phi_prev = phi_now

        res = max_residuals(theta_vals, z_vals, ydual_vals)

        feasible = (
            res["max_primal_all"] <= FEAS_TOL
            and res["max_dual_all"] <= FEAS_TOL
            and res["sd"] <= FEAS_TOL
        )

        print("\n" + "-"*70)
        print(f"{label} | Chequeo fin outer k={k_outer}")
        print("-"*70)
        print(f"J(theta)             = {compute_J(theta_vals):.8f}")
        print(f"Max primal all       = {res['max_primal_all']:.3e}")
        print(f"Max dual all         = {res['max_dual_all']:.3e}")
        print(f"Strong duality       = {res['sd']:.3e}")
        print(f"Max primal coupled   = {res['max_primal_coupled']:.3e}")
        print(f"Max dual coupled     = {res['max_dual_coupled']:.3e}")
        print(f"rho_max              = {max(rho.values()) if rho else 0.0:.3e}")
        print(f"mu_max               = {max(mu.values()) if mu else 0.0:.3e}")

        if feasible:
            print(f"\n✅ {label} encontró solución factible primal-dual.")

            best_feasible = {
                "theta": copy.deepcopy(theta_vals),
                "z": z_vals.copy(),
                "ydual": ydual_vals.copy(),
                "J": compute_J(theta_vals),
                "residuals": res,
                "outer": k_outer,
                "history": history,
            }

            break

        n_rho, n_mu = update_penalties(theta_vals, z_vals, ydual_vals, rho, mu)

        print("\nActualización penalizaciones:")
        print(f"rho actualizadas: {n_rho}")
        print(f"mu actualizadas : {n_mu}")

    if best_feasible is None:
        print(f"\n⚠️ {label} terminó sin certificar factibilidad.")

    return {
        "best_feasible": best_feasible,
        "theta": theta_vals,
        "z": z_vals,
        "ydual": ydual_vals,
        "history": history,
        "last_residuals": max_residuals(theta_vals, z_vals, ydual_vals) if z_vals is not None else None,
        "rho": rho,
        "mu": mu,
    }


# ============================================================
# 9) EJECUTAR PADM-W Y SELECCIONAR EL START DEL WCEP FINAL
# ============================================================

PADM_WARMSTART = None
PADM_FEASIBLE_STAGE1 = None
PADM_FEASIBLE_STAGE2 = None

if RUN_PADM_W:

    # --------------------------------------------------------
    # Etapa 1: buscar cualquier punto primal-dual factible
    # --------------------------------------------------------
    res_ws = run_padm(
        use_true_objective=False,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM-W ETAPA 1: FACTIBILIDAD"
    )

    PADM_FEASIBLE_STAGE1 = res_ws.get("best_feasible")

    if PADM_FEASIBLE_STAGE1 is not None:

        theta_start = PADM_FEASIBLE_STAGE1["theta"]
        z_start = PADM_FEASIBLE_STAGE1["z"]
        yd_start = PADM_FEASIBLE_STAGE1["ydual"]

        print("\n" + "="*70)
        print("PUNTO FACTIBLE PADM-W ENCONTRADO")
        print("Este punto queda guardado desde ahora para iniciar el WCEP final.")
        print("="*70)

        # ----------------------------------------------------
        # Etapa 2 opcional: intentar mejorar J sin perder el
        # respaldo factible encontrado en la etapa 1.
        # ----------------------------------------------------
        if RUN_PADM_STAGE_2:
            res_final = run_padm(
                use_true_objective=True,
                theta_init=theta_start,
                z_init=z_start,
                ydual_init=yd_start,
                label="PADM-W ETAPA 2: OBJETIVO REAL"
            )

            PADM_FEASIBLE_STAGE2 = res_final.get("best_feasible")
        else:
            print("\nEtapa 2 desactivada: se usará directamente el punto factible de etapa 1.")
            res_final = res_ws
            PADM_FEASIBLE_STAGE2 = None

        # Preferir el punto mejorado de etapa 2 solo si también está certificado.
        # Si no, conservar SIEMPRE el factible de etapa 1.
        if PADM_FEASIBLE_STAGE2 is not None:
            final_solution = PADM_FEASIBLE_STAGE2
            final_label = "PADM-W ETAPA 2 FACTIBLE"
        else:
            if RUN_PADM_STAGE_2:
                print("\n⚠️ Etapa 2 no entregó otro punto certificado.")
                print("✅ Se conserva el punto factible de la etapa 1 como Start del WCEP final.")
            final_solution = PADM_FEASIBLE_STAGE1
            final_label = "PADM-W ETAPA 1 FACTIBLE"

    else:
        print("\n⚠️ Etapa 1 no encontró un punto certificado como factible.")
        final_solution = None
        final_label = "PADM-W SIN PUNTO FACTIBLE"
        res_final = res_ws

else:
    # Mantiene la posibilidad de ejecutar una sola corrida PADM sin etapa warm-start.
    res_final = run_padm(
        use_true_objective=True,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM"
    )

    final_solution = res_final.get("best_feasible")
    final_label = "PADM FACTIBLE" if final_solution is not None else "PADM SIN PUNTO FACTIBLE"

print("\n" + "="*70)
print("RESULTADOS FINALES PADM-W IEEE39")
print("="*70)

if final_solution is not None:
    sol_theta = final_solution["theta"]
    sol_z = final_solution["z"]
    sol_yd = final_solution["ydual"]

    print("✅ SOLUCIÓN FACTIBLE CERTIFICADA")
    print(f"Método final     : {final_label}")
    print(f"Outer encontrado : {final_solution['outer']}")
    print(f"J                : {final_solution['J']:.8f}")
    print("✅ Esta solución será cargada como Start del WCEP con dualidad fuerte.")

    PADM_WARMSTART = {
        "theta": copy.deepcopy(sol_theta),
        "z": sol_z.copy(),
        "ydual": sol_yd.copy(),
        "label": final_label,
        "J": final_solution["J"],
        "residuals": final_solution["residuals"],
        "certified": True,
    }

else:
    print("⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.")

    sol_theta = res_final.get("theta")
    sol_z = res_final.get("z")
    sol_yd = res_final.get("ydual")

    if USE_LAST_PADM_POINT_IF_NO_FEASIBLE and sol_z is not None and sol_yd is not None:
        res = max_residuals(sol_theta, sol_z, sol_yd)

        PADM_WARMSTART = {
            "theta": copy.deepcopy(sol_theta),
            "z": sol_z.copy(),
            "ydual": sol_yd.copy(),
            "label": "PADM_LAST_POINT_NOT_CERTIFIED",
            "J": compute_J(sol_theta),
            "residuals": res,
            "certified": False,
        }

        print("⚠️ Se usará el último punto PADM-W como Start NO certificado.")
        print("   El WCEP final seguirá imponiendo factibilidad primal, dual y dualidad fuerte.")
    else:
        PADM_WARMSTART = None
        print("❌ No hay punto PADM-W disponible para iniciar el WCEP final.")

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("CHECK PADM_WARMSTART IEEE39")
    print("="*70)

    sol_theta = PADM_WARMSTART["theta"]
    sol_z = PADM_WARMSTART["z"]
    sol_yd = PADM_WARMSTART["ydual"]

    res_check = max_residuals(sol_theta, sol_z, sol_yd)

    print(f"Origen          = {PADM_WARMSTART['label']}")
    print(f"Certificado     = {PADM_WARMSTART['certified']}")
    print(f"J(theta)        = {PADM_WARMSTART['J']:.8f}")
    print(f"Max primal      = {res_check['max_primal_all']:.6e}")
    print(f"Max dual        = {res_check['max_dual_all']:.6e}")
    print(f"Strong duality  = {res_check['sd']:.6e}")

    if PADM_WARMSTART["certified"]:
        print("✅ El punto cumple la tolerancia PADM-W y se usará en el WCEP final.")
    else:
        print("⚠️ El punto solo se usará como sugerencia inicial no certificada.")


# ============================================================
# 10) WCEP FINAL CON DUALIDAD FUERTE Y WARM START PADM-W
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE39_DF_warmstart_PADM_SOLO_Psync")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Pmax_sync_ce = w.addVars(GS, lb=0.0, name="Pmax_sync_ce")

for g in GS:
    base = Pmax_sync_base[g]
    w.addConstr(Pmax_sync_ce[g] >= PSYNC_LB_FACTOR * base, name=f"adm_lb_Psync[{g}]")
    w.addConstr(Pmax_sync_ce[g] <= PSYNC_UB_FACTOR * base, name=f"adm_ub_Psync[{g}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addQConstr(
                Pmax_sync_ce[g] * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -float(xbar[m]), name=name)
        RHS_expr[name] = -float(xbar[m])

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= float(Hreq[y]), name=name)
        RHS_expr[name] = float(Hreq[y])

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= epsilon_CO2_wcep,
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if USE_DUAL_BOUNDS_FOR_MUTABLE_ROWS:
        if name.startswith("sync_ub_"):
            ub_here = UB_sync

    ydual[name] = w.addVar(lb=0.0, ub=ub_here, name=f"ydual[{name}]")

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(lhs <= c_map_w.get(v, 0.0), name=f"df_{v.VarName}")

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(ctx - bty <= 0.0, name="StrongDuality")

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dPsync = w.addVars(GS, lb=0.0, name="dPsync")

for g in GS:
    base = Pmax_sync_base[g]
    diff = Pmax_sync_ce[g] - base

    w.addConstr(dPsync[g] >= diff, name=f"dPsync_pos[{g}]")
    w.addConstr(dPsync[g] >= -diff, name=f"dPsync_neg[{g}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for g in GS:
        base = abs(float(Pmax_sync_base[g]))
        if base > 1e-9:
            obj_wcep += alpha_Psync * dPsync[g] / base
        else:
            obj_wcep += alpha_Psync * dPsync[g]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_Psync * dPsync.sum(),
        GRB.MINIMIZE
    )

# ============================================================
# 11) APLICAR WARM START DEL PADM-W
# ============================================================

def safe_start(var, value):
    """Asigna un valor Start válido y devuelve True si fue asignado."""
    try:
        if value is not None and np.isfinite(float(value)):
            var.Start = float(value)
            return True
    except Exception:
        pass
    return False

# Asegura que todas las variables y restricciones ya estén incorporadas
# antes de cargar el punto inicial.
w.update()

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("APLICANDO PADM_WARMSTART AL WCEP FINAL IEEE39")
    print("="*70)
    print("Origen:", PADM_WARMSTART.get("label", "PADM"))

    ws_theta = PADM_WARMSTART["theta"]
    ws_z = PADM_WARMSTART["z"]
    ws_yd = PADM_WARMSTART["ydual"]

    n_start_values = 0

    for g in GS:
        n_start_values += int(safe_start(Pmax_sync_ce[g], ws_theta["Pmax_sync_ce"][g]))
        n_start_values += int(safe_start(dPsync[g], abs(ws_theta["Pmax_sync_ce"][g] - Pmax_sync_base[g])))


    for g in G:
        for t in T:
            for y in Y:
                n_start_values += int(safe_start(p[g, t, y], ws_z.get(("p", g, t, y), None)))

    for g in GS:
        for t in T:
            for y in Y:
                n_start_values += int(safe_start(u[g, t, y], ws_z.get(("u", g, t, y), None)))

    for i in B:
        for t in T:
            for y in Y:
                n_start_values += int(safe_start(s[i, t, y], ws_z.get(("s", i, t, y), None)))

    for m in M:
        for y in Y:
            n_start_values += int(safe_start(xcap[m, y], ws_z.get(("xcap", m, y), None)))

    for m in M:
        for t in T:
            for y in Y:
                n_start_values += int(safe_start(h_syn[m, t, y], ws_z.get(("h_syn", m, t, y), None)))
                n_start_values += int(safe_start(r_ffr[m, t, y], ws_z.get(("r_ffr", m, t, y), None)))

    for (i, j) in L_arcs:
        for t in T:
            for y in Y:
                n_start_values += int(safe_start(f_pos[i, j, t, y], ws_z.get(("f_pos", i, j, t, y), None)))
                n_start_values += int(safe_start(f_neg[i, j, t, y], ws_z.get(("f_neg", i, j, t, y), None)))

    for i in B:
        for t in T:
            for y in Y:
                n_start_values += int(safe_start(theta_p[i, t, y], ws_z.get(("theta_p", i, t, y), None)))
                n_start_values += int(safe_start(theta_m[i, t, y], ws_z.get(("theta_m", i, t, y), None)))

    for name, var in ydual.items():
        n_start_values += int(safe_start(var, ws_yd.get(name, None)))

    w.update()
    print(f"✅ Warm start aplicado al WCEP IEEE39: {n_start_values} valores Start cargados.")
    print(f"   Punto certificado: {PADM_WARMSTART.get('certified', False)}")
else:
    print("\n⚠️ No existe PADM_WARMSTART. El WCEP se resolverá sin Start.")

# ============================================================
# 12) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE39")
print("="*70)

w.optimize()

# ============================================================
# 13) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE39")
print("="*70)
print(f"Status = {w.Status}")
print(f"Tiempo WCEP optimize() = {float(w.Runtime):.2f} s = {float(w.Runtime)/60:.2f} min = {float(w.Runtime)/3600:.2f} h")
print(f"Soluciones encontradas = {int(w.SolCount)}")

if w.SolCount > 0:
    print(f"Mejor objetivo encontrado = {float(w.ObjVal):.8f}")
    print(f"Mejor bound               = {float(w.ObjBound):.8f}")
    print(f"MIPGap                    = {float(w.MIPGap):.6e}")
    print(f"MIPGap (%)                = {100.0*float(w.MIPGap):.4f}%")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")
elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")
elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:
    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Pmax_sync_ce síncronos ---")
    hubo_cambio = False
    for g in GS:
        val = float(Pmax_sync_ce[g].X)
        base = float(Pmax_sync_base[g])
        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{g}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")
    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- Hreq fijo ---")
    print("Hreq no es parámetro mutable en este experimento.")

    print("\n--- xbar fijo ---")
    print("xbar no es parámetro mutable en este experimento.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)
    ctx_val = ctx.getValue()
    bty_val = bty.getValue()
    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)
    for y in Y:
        emis_y = sum(float(eg[g]) * float(p[g, t, y].X) for g in GE for t in T)
        if y >= y_star:
            holgura = epsilon_CO2_wcep - emis_y
            print(f"Año {y}: emisiones = {emis_y:.6f} | límite D = {epsilon_CO2_wcep:.6f} | holgura = {holgura:.6f}")
        else:
            print(f"Año {y}: emisiones = {emis_y:.6f} | sin restricción D")

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)
    for y in Y:
        for t in T:
            H_sync_val = sum(float(Hg.get(g, 0.0)) * float(u[g, t, y].X) for g in GS)
            H_syn_val = sum(float(h_syn[m, t, y].X) for m in M)
            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq[y])
            R_total = sum(float(r_ffr[m, t, y].X) for m in M)
            R_lim = float(Rreq[y])
            print(
                f"(y={y}, t={t}) | H_total={H_total:.6f}, Hreq={H_lim:.6f}, "
                f"margen_H={H_total - H_lim:.6e} | R_total={R_total:.6f}, "
                f"Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("INVERSIÓN IBR Y USO DE xbar FIJO")
    print("="*70)
    for m in M:
        for y in Y:
            xv = float(xcap[m, y].X)
            xb = float(xbar[m])
            uso = xv / xb if xb > 1e-9 else 0.0
            if xv > 1e-6 or uso > 0.85:
                print(f"{m}, y={y}: xcap={xv:.6f}, xbar={xb:.6f}, uso={100*uso:.2f}%")

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)
    hits_sync = 0
    for name, var in ydual.items():
        if name.startswith("sync_ub_"):
            if abs(var.X - UB_sync) <= 1e-6 * max(1.0, UB_sync):
                hits_sync += 1
    print(f"Duales sync_ub pegadas a cota : {hits_sync}")
else:
    print("\nNo hay solución WCEP disponible para reportar.")


# ============================================================
# TIEMPO TOTAL DEL SCRIPT/CELDA
# ============================================================
t_total_script = time.time() - t0_total_script
print("\n" + "="*70)
print("TIEMPO TOTAL SCRIPT/CELDA")
print("="*70)
print(f"Tiempo total real = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")


✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en PADM-W/WCEP = 1289.386000
Pmax total sistema                 = 1315.700000

COTAS DUALES IEEE39
UB_H    = 324200000.000000
UB_sync = 10000.000000
UB_xbar = 3242000.000000
Bus slack usado: 1
Bus slack usado: 1

CLASIFICACIÓN PADM-W IEEE39 - FORMULACIÓN ORIGINAL
N° filas primales acopladas   : 48
N° filas primales separables  : 4116
N° columnas duales acopladas  : 48
N° columnas duales separables : 2868

✅ Filas acopladas esperadas: solo sync_ub_.

INICIO PADM-W ETAPA 1: FACTIBILIDAD
use_true_objective = False

Inicializando Y=(z,lambda)...

Chequeo inicial:
J(theta)       = 0.00000000
Max primal     = 1.595e-12
Max dual       = 2.653e+05
Strong duality = 9.537e-07

######################################################################
PADM-W ETAPA 1: FACTIBILIDAD | OUTER k = 0
######################################################################
k=00, l=00 | Phi=2.484096e+08, J=8.250000e+00, p_all=8.25e+00, d_all=2.29

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import copy
import math
import time

# ============================================================
# FINAL IEEE39: PADM-W + WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO:
#
# Mutable único:
#   - Pmax_sync_ce[g] para g in GS, entrando en A.
#
# Es decir:
#   - Hreq y xbar quedan fijos.
#   - Pmax_sync_ce aparece en la restricción sync_ub.
#   - La fila sync_ub es acoplada y genera bilinealidades.
#
# Flujo:
#   1) PADM-W etapa 1: busca factibilidad.
#   2) PADM-W etapa 2: parte de factibilidad y reduce J(theta).
#   3) WCEP completo con dualidad fuerte usando PADM_WARMSTART.
# ============================================================

# Medición de tiempo total de celda/script
t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

PADM_OUTPUT = 1
RUN_PADM_W = True

MAX_OUTER = 100
MAX_INNER = 100

FEAS_TOL = 1e-5
INNER_TOL = 1e-5

RHO0 = 500.0
MU0 = 500.0

PENALTY_FACTOR = 2.0
MAX_PENALTY = 1e9

TIME_LIMIT_SUBPROBLEM = 5 * 60 * 60  # 5 horas por subproblema PADM
WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE39
# Si ya calculaste en el FW emisiones ponderadas, puedes poner epsilon_CO2_wcep manual.
# Si lo dejas como None, el código intenta usar el 90% de las emisiones FW guardadas en memoria.
epsilon_CO2_wcep = None
EPSILON_CO2_FRACTION = 0.90

# Objetivo J
alpha_Psync = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE39
# ============================================================
# Este bloque hace que PADM-W y WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por PADM-W/WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en PADM-W/WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# Si T = [1, 7, 13, 19] representa un perfil diario, usamos omega_t
# para expandir costos y emisiones a base anual sin crear 365*|T| periodos.
# Por defecto cada t se repite 365 veces y representa 1 hora.
# Si cada t representa un bloque de 6 horas, define TIME_BLOCK_HOURS = 6.0 antes de correr.
TIME_BLOCK_HOURS = float(globals().get("TIME_BLOCK_HOURS", 1.0))
N_DAYS_EQUIV = int(globals().get("N_DAYS_EQUIV", 365))

if "omega_t" not in globals():
    omega_t = {t: float(N_DAYS_EQUIV) * TIME_BLOCK_HOURS for t in T}
else:
    omega_t = {t: float(omega_t[t]) for t in T}

def omega(t):
    return float(omega_t[t])

print("Pesos temporales omega_t usados:", omega_t)

bus_slack = int(sorted(B)[0])

Pmax_sync_base = {g: float(Pmax[g]) for g in GS}

UB_sync = 2.0 * max(max(float(cg[g]) for g in G), float(VOLL))

# ============================================================
# Límite de emisiones D
# ============================================================
def _get_fw_emissions_by_year():
    candidates = [
        "emisiones_por_anio", "df_emis_fw", "df_emis", "df_emis_wcep"
    ]
    value_cols = [
        "emisiones_anuales_equivalentes",
        "emisiones_totales_ponderadas",
        "emisiones_totales_anuales",
        "emisiones",
    ]
    for name in candidates:
        if name in globals():
            df = globals()[name]
            if hasattr(df, "columns") and "y" in df.columns:
                for col in value_cols:
                    if col in df.columns:
                        return {int(row["y"]): float(row[col]) for _, row in df.iterrows()}
    return None

_fw_emis = _get_fw_emissions_by_year()

if epsilon_CO2_wcep is None:
    if _fw_emis is not None:
        epsilon_CO2_wcep_by_y = {y: EPSILON_CO2_FRACTION * float(_fw_emis[y]) for y in Y if y in _fw_emis}
        print("epsilon_CO2_wcep calculado como 90% de emisiones FW:", epsilon_CO2_wcep_by_y)
    else:
        raise ValueError(
            "epsilon_CO2_wcep=None, pero no encontré emisiones del FW. "
            "Define epsilon_CO2_wcep manualmente o ejecuta antes el FW con tabla de emisiones."
        )
elif isinstance(epsilon_CO2_wcep, dict):
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep[y]) for y in epsilon_CO2_wcep}
else:
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep) for y in Y}

def eps_CO2(y):
    return float(epsilon_CO2_wcep_by_y[int(y)])


print("\n" + "="*70)
print("COTAS DUALES IEEE39")
print("="*70)
print(f"UB_sync = {UB_sync:.6f}")
print(f"Bus slack usado: {bus_slack}")


# ============================================================
# 2) REPRESENTACIÓN SIMBÓLICA PARA PADM-W
# ============================================================

def key_p(g,t,y):       return ("p", g, t, y)
def key_u(g,t,y):       return ("u", g, t, y)
def key_s(i,t,y):       return ("s", i, t, y)
def key_xcap(m,y):      return ("xcap", m, y)
def key_hsyn(m,t,y):    return ("h_syn", m, t, y)
def key_rffr(m,t,y):    return ("r_ffr", m, t, y)
def key_fpos(i,j,t,y):  return ("f_pos", i, j, t, y)
def key_fneg(i,j,t,y):  return ("f_neg", i, j, t, y)
def key_thp(i,t,y):     return ("theta_p", i, t, y)
def key_thm(i,t,y):     return ("theta_m", i, t, y)

def param_coeff(family, idx, mult=1.0):
    return ("PARAM", family, idx, float(mult))

def is_param_coeff(c):
    return isinstance(c, tuple) and len(c) == 4 and c[0] == "PARAM"

def coeff_value(c, theta_vals):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * float(theta_vals[fam][idx])
    return float(c)

def coeff_expr_in_theta_model(c, theta_vars):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * theta_vars[fam][idx]
    return float(c)

def rhs_const(val):
    return [("CONST", None, None, float(val))]

def rhs_param(family, idx, mult=1.0):
    return [("PARAM", family, idx, float(mult))]

def rhs_has_param(rhs_terms):
    return any(typ == "PARAM" for typ, fam, idx, mult in rhs_terms)

def rhs_eval(rhs_terms, theta_vals=None):
    out = 0.0

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            out += float(mult)
        elif typ == "PARAM":
            if theta_vals is None:
                raise ValueError("rhs_eval requiere theta_vals para RHS con parámetros.")
            out += float(mult) * float(theta_vals[fam][idx])
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return out

def rhs_expr_in_theta_model(rhs_terms, theta_vars):
    expr = gp.LinExpr()

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            expr += float(mult)
        elif typ == "PARAM":
            expr += float(mult) * theta_vars[fam][idx]
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return expr

def add_to_terms(terms, key, coeff):
    if key not in terms:
        terms[key] = []
    terms[key].append(coeff)

def neg_coeff(c):
    if is_param_coeff(c):
        return ("PARAM", c[1], c[2], -c[3])
    return -float(c)

def terms_has_param(terms):
    for k, coeffs in terms.items():
        for c in coeffs:
            if is_param_coeff(c):
                return True
    return False

def terms_expr_z_model(terms, z_vars, theta_vals):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        for c in coeffs:
            expr += coeff_value(c, theta_vals) * z_vars[k]

    return expr

def terms_expr_theta_model_with_fixed_z(terms, z_vals, theta_vars):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            expr += zval * coeff_expr_in_theta_model(c, theta_vars)

    return expr

def terms_eval(terms, z_vals, theta_vals):
    out = 0.0

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            out += coeff_value(c, theta_vals) * zval

    return out


# ============================================================
# 3) CREAR FILAS PRIMALES Ax >= b
# ============================================================

rows = {}
row_order = []

def add_row(name, terms, rhs_terms):
    if name in rows:
        raise ValueError(f"Fila repetida: {name}")

    rows[name] = {
        "terms": terms,
        "rhs": rhs_terms
    }

    row_order.append(name)

# ------------------------------------------------------------
# A.1 Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}

            for g in G_en_bus.get(i, []):
                add_to_terms(terms, key_p(g,t,y), 1.0)

            add_to_terms(terms, key_s(i,t,y), 1.0)

            for (k,j) in lineas_entran[i]:
                add_to_terms(terms, key_fpos(k,j,t,y), 1.0)
                add_to_terms(terms, key_fneg(k,j,t,y), -1.0)

            for (i2,j) in lineas_salen[i]:
                add_to_terms(terms, key_fpos(i2,j,t,y), -1.0)
                add_to_terms(terms, key_fneg(i2,j,t,y), 1.0)

            rhs_val = float(d_fw[(i,t,y)])

            add_row(f"bal_p_{i}_{t}_{y}", terms, rhs_const(rhs_val))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"bal_n_{i}_{t}_{y}", terms_n, rhs_const(-rhs_val))

# ------------------------------------------------------------
# A.2 ENS <= demanda
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}
            add_to_terms(terms, key_s(i,t,y), -1.0)
            add_row(f"ens_ub_{i}_{t}_{y}", terms, rhs_const(-float(d_fw[(i,t,y)])))

# ------------------------------------------------------------
# A.3 Flujo DC
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i,j)])

            terms = {}
            add_to_terms(terms, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms, key_fneg(i,j,t,y), -1.0)

            add_to_terms(terms, key_thp(i,t,y), -Bij)
            add_to_terms(terms, key_thm(i,t,y),  Bij)
            add_to_terms(terms, key_thp(j,t,y),  Bij)
            add_to_terms(terms, key_thm(j,t,y), -Bij)

            add_row(f"dc_p_{i}_{j}_{t}_{y}", terms, rhs_const(0.0))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"dc_n_{i}_{j}_{t}_{y}", terms_n, rhs_const(0.0))

# ------------------------------------------------------------
# A.4 Límites térmicos fijos
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i,j)])

            terms_lb = {}
            add_to_terms(terms_lb, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms_lb, key_fneg(i,j,t,y), -1.0)
            add_row(f"flb_{i}_{j}_{t}_{y}", terms_lb, rhs_const(-Fij))

            terms_ub = {}
            add_to_terms(terms_ub, key_fpos(i,j,t,y), -1.0)
            add_to_terms(terms_ub, key_fneg(i,j,t,y), 1.0)
            add_row(f"fub_{i}_{j}_{t}_{y}", terms_ub, rhs_const(-Fij))

# ------------------------------------------------------------
# A.5 Slack bus
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_p = {}
        add_to_terms(terms_p, key_thp(bus_slack,t,y), 1.0)
        add_to_terms(terms_p, key_thm(bus_slack,t,y), -1.0)
        add_row(f"slack_p_{t}_{y}", terms_p, rhs_const(0.0))

        terms_n = {}
        add_to_terms(terms_n, key_thp(bus_slack,t,y), -1.0)
        add_to_terms(terms_n, key_thm(bus_slack,t,y), 1.0)
        add_row(f"slack_n_{t}_{y}", terms_n, rhs_const(0.0))

print(f"Bus slack usado: {bus_slack}")

# ------------------------------------------------------------
# A.6 Límites generación fijos p <= P_disp
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_base = float(P_disp.get((g,t), Pmax[g]))

        for y in Y:
            terms = {}
            add_to_terms(terms, key_p(g,t,y), -1.0)
            add_row(f"p_cap_{g}_{t}_{y}", terms, rhs_const(-cap_base))

# ------------------------------------------------------------
# A.7 Síncronos
# ------------------------------------------------------------
for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            terms_lb = {}
            add_to_terms(terms_lb, key_p(g,t,y), 1.0)
            add_to_terms(terms_lb, key_u(g,t,y), -pmin_g)
            add_row(f"sync_lb_{g}_{t}_{y}", terms_lb, rhs_const(0.0))

            terms_ub = {}
            add_to_terms(terms_ub, key_u(g,t,y), param_coeff("Pmax_sync_ce", g, 1.0))
            add_to_terms(terms_ub, key_p(g,t,y), -1.0)
            add_row(f"sync_ub_{g}_{t}_{y}", terms_ub, rhs_const(0.0))

            terms_u = {}
            add_to_terms(terms_u, key_u(g,t,y), -1.0)
            add_row(f"u_ub_{g}_{t}_{y}", terms_u, rhs_const(-1.0))

# ------------------------------------------------------------
# A.8-A.10 Inversión, h_syn, FFR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        terms_x = {}
        add_to_terms(terms_x, key_xcap(m,y), -1.0)
        add_row(f"x_ub_{m}_{y}", terms_x, rhs_const(-float(xbar[m])))

        for t in T:
            terms_h = {}
            add_to_terms(terms_h, key_xcap(m,y), float(gamma_base[m]))
            add_to_terms(terms_h, key_hsyn(m,t,y), -1.0)
            add_row(f"hsyn_lim_{m}_{t}_{y}", terms_h, rhs_const(0.0))

            terms_r = {}
            add_to_terms(terms_r, key_xcap(m,y), float(kappa_base[m]))
            add_to_terms(terms_r, key_rffr(m,t,y), -1.0)
            add_row(f"ffr_lim_{m}_{t}_{y}", terms_r, rhs_const(0.0))

# ------------------------------------------------------------
# A.11 Seguridad: Hreq fijo, Rreq fijo
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_H = {}

        for g in GS:
            add_to_terms(terms_H, key_u(g,t,y), float(Hg.get(g, 0.0)))

        for m in M:
            add_to_terms(terms_H, key_hsyn(m,t,y), 1.0)

        add_row(f"sys_H_{t}_{y}", terms_H, rhs_const(float(Hreq[y])))

        terms_R = {}

        for m in M:
            add_to_terms(terms_R, key_rffr(m,t,y), 1.0)

        add_row(f"sys_R_{t}_{y}", terms_R, rhs_const(float(Rreq[y])))


# ============================================================
# 4) VARIABLES PRIMALES Y COSTOS c
# ============================================================

primal_keys = []

for g in G:
    for t in T:
        for y in Y:
            primal_keys.append(key_p(g,t,y))

for g in GS:
    for t in T:
        for y in Y:
            primal_keys.append(key_u(g,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_s(i,t,y))

for m in M:
    for y in Y:
        primal_keys.append(key_xcap(m,y))

for m in M:
    for t in T:
        for y in Y:
            primal_keys.append(key_hsyn(m,t,y))
            primal_keys.append(key_rffr(m,t,y))

for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            primal_keys.append(key_fpos(i,j,t,y))
            primal_keys.append(key_fneg(i,j,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_thp(i,t,y))
            primal_keys.append(key_thm(i,t,y))

c_map = {}

for g in G:
    for t in T:
        for y in Y:
            c_map[key_p(g,t,y)] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map[key_s(i,t,y)] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map[key_xcap(m,y)] = float(cinv[m])

for k in primal_keys:
    c_map.setdefault(k, 0.0)


# ============================================================
# 5) CLASIFICACIÓN ACOPLADAS VS SEPARABLES
# ============================================================

coupled_primal_rows = [
    name for name in row_order
    if terms_has_param(rows[name]["terms"]) or rhs_has_param(rows[name]["rhs"])
]

separable_primal_rows = [
    name for name in row_order
    if name not in set(coupled_primal_rows)
]

coupled_dual_cols = set()

for name in row_order:
    terms = rows[name]["terms"]

    for k, coeffs in terms.items():
        if any(is_param_coeff(c) for c in coeffs):
            coupled_dual_cols.add(k)

coupled_dual_cols = sorted(list(coupled_dual_cols), key=str)

separable_dual_cols = [
    k for k in primal_keys
    if k not in set(coupled_dual_cols)
]

print("\n" + "="*70)
print("CLASIFICACIÓN PADM-W IEEE39 - FORMULACIÓN ORIGINAL")
print("="*70)
print(f"N° filas primales acopladas   : {len(coupled_primal_rows)}")
print(f"N° filas primales separables  : {len(separable_primal_rows)}")
print(f"N° columnas duales acopladas  : {len(coupled_dual_cols)}")
print(f"N° columnas duales separables : {len(separable_dual_cols)}")

bad_rows = [
    name for name in coupled_primal_rows
    if not name.startswith("sync_ub_")
]

if bad_rows:
    print("\n⚠️ Hay filas acopladas no esperadas:")
    print(bad_rows[:30])
else:
    print("\n✅ Filas acopladas esperadas: sync_ub_.")


# ============================================================
# 6) FUNCIONES PADM-W
# ============================================================

def initial_theta_vals():
    return {
        "Pmax_sync_ce": {g: float(Pmax_sync_base[g]) for g in GS},
    }

def compute_J(theta_vals):
    J = 0.0

    for g in GS:
        base = float(Pmax_sync_base[g])
        val = float(theta_vals["Pmax_sync_ce"][g])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_Psync * abs(val - base) / abs(base)
        else:
            J += alpha_Psync * abs(val - base)

    return J

def compute_ctx(z_vals):
    return sum(
        float(c_map[k]) * float(z_vals.get(k, 0.0))
        for k in primal_keys
    )

def compute_bty(theta_vals, ydual_vals):
    out = 0.0

    for name in row_order:
        out += rhs_eval(rows[name]["rhs"], theta_vals) * float(ydual_vals.get(name, 0.0))

    return out

def compute_primal_residuals(theta_vals, z_vals):
    res = {}

    for name in row_order:
        lhs = terms_eval(rows[name]["terms"], z_vals, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)
        res[name] = max(0.0, rhs - lhs)

    return res

def compute_dual_residuals(theta_vals, ydual_vals):
    res = {}

    for k in primal_keys:
        lhs = 0.0

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            yd = float(ydual_vals.get(name, 0.0))

            if abs(yd) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd

        rhs = float(c_map.get(k, 0.0))
        res[k] = max(0.0, lhs - rhs)

    return res

def compute_sd_residual(theta_vals, z_vals, ydual_vals):
    return max(0.0, compute_ctx(z_vals) - compute_bty(theta_vals, ydual_vals))

def max_residuals(theta_vals, z_vals, ydual_vals):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)
    sd = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return {
        "max_primal_all": max(pr.values()) if pr else 0.0,
        "max_dual_all": max(dr.values()) if dr else 0.0,
        "sd": sd,
        "max_primal_coupled": max([pr[name] for name in coupled_primal_rows], default=0.0),
        "max_dual_coupled": max([dr[k] for k in coupled_dual_cols], default=0.0),
    }

def compute_phi(theta_vals, z_vals, ydual_vals, rho, mu, use_true_objective=True):
    Jval = compute_J(theta_vals) if use_true_objective else 0.0

    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    pen_p = sum(float(rho[name]) * pr[name] for name in coupled_primal_rows)
    pen_d = sum(float(mu[k]) * dr[k] for k in coupled_dual_cols)

    sd_penalty = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return Jval + pen_p + pen_d + 1e3 * sd_penalty

def diff_theta(theta_a, theta_b):
    vals = []

    for g in GS:
        vals.append(abs(theta_a["Pmax_sync_ce"][g] - theta_b["Pmax_sync_ce"][g]))

    return max(vals) if vals else 0.0

def diff_dict(a, b):
    keys = set(a.keys()) | set(b.keys())
    return max(
        [abs(float(a.get(k,0.0)) - float(b.get(k,0.0))) for k in keys],
        default=0.0
    )


# ============================================================
# 7) SUBPROBLEMAS PADM-W
# ============================================================

def add_z_variables(model):
    z = {}

    for g in G:
        for t in T:
            for y in Y:
                z[key_p(g,t,y)] = model.addVar(lb=0.0, name=f"p[{g},{t},{y}]")

    for g in GS:
        for t in T:
            for y in Y:
                z[key_u(g,t,y)] = model.addVar(lb=0.0, ub=1.0, name=f"u[{g},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_s(i,t,y)] = model.addVar(lb=0.0, name=f"s[{i},{t},{y}]")

    for m in M:
        for y in Y:
            z[key_xcap(m,y)] = model.addVar(lb=0.0, name=f"xcap[{m},{y}]")

    for m in M:
        for t in T:
            for y in Y:
                z[key_hsyn(m,t,y)] = model.addVar(lb=0.0, name=f"h_syn[{m},{t},{y}]")
                z[key_rffr(m,t,y)] = model.addVar(lb=0.0, name=f"r_ffr[{m},{t},{y}]")

    for (i,j) in L_arcs:
        for t in T:
            for y in Y:
                z[key_fpos(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_pos[{i},{j},{t},{y}]")
                z[key_fneg(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_neg[{i},{j},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_thp(i,t,y)] = model.addVar(lb=0.0, name=f"theta_p[{i},{t},{y}]")
                z[key_thm(i,t,y)] = model.addVar(lb=0.0, name=f"theta_m[{i},{t},{y}]")

    return z

def add_dual_variables(model):
    yd = {}

    for name in row_order:
        ub_here = GRB.INFINITY

        if name.startswith("sync_ub_"):
            ub_here = UB_sync

        yd[name] = model.addVar(lb=0.0, ub=ub_here, name=f"ydual[{name}]")

    return yd

def add_theta_variables(model, warm_theta=None):
    theta = {
        "Pmax_sync_ce": {},
    }

    for g in GS:
        theta["Pmax_sync_ce"][g] = model.addVar(
            lb=0.85 * float(Pmax_sync_base[g]),
            ub=1.00 * float(Pmax_sync_base[g]),
            name=f"Pmax_sync_ce[{g}]"
        )

    if warm_theta is not None:
        for g in GS:
            theta["Pmax_sync_ce"][g].Start = float(warm_theta["Pmax_sync_ce"][g])

    return theta

def extract_theta(theta_vars):
    return {
        "Pmax_sync_ce": {g: float(theta_vars["Pmax_sync_ce"][g].X) for g in GS},
    }

def add_separable_primal_constraints_y(model, z_vars):
    theta_base = initial_theta_vals()

    for name in separable_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z_vars, theta_base)
        rhs = rhs_eval(rows[name]["rhs"], theta_base)
        model.addConstr(lhs >= rhs, name=f"Y_primal[{name}]")

def add_D_constraint_y(model, z_vars):
    for y in Y:
        if y >= y_star:
            model.addConstr(
                gp.quicksum(
                    omega(t) * float(eg[g]) * z_vars[key_p(g,t,y)]
                    for g in GE
                    for t in T
                )
                <= float(eps_CO2(y)),
                name=f"D_CO2_cap_{y}"
            )

def add_separable_dual_constraints_y(model, ydual_vars):
    theta_base = initial_theta_vals()

    for k in separable_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_base) * ydual_vars[name]

        model.addConstr(lhs <= float(c_map.get(k, 0.0)), name=f"Y_dual[{k}]")

def add_strong_duality_y(model, theta_vals, z_vars, ydual_vars):
    ctx = gp.quicksum(float(c_map[k]) * z_vars[k] for k in primal_keys)

    bty = gp.LinExpr()

    for name in row_order:
        bty += rhs_eval(rows[name]["rhs"], theta_vals) * ydual_vars[name]

    model.addConstr(ctx <= bty, name="Y_StrongDuality")

def solve_theta_subproblem(
    z_vals,
    ydual_vals,
    rho,
    mu,
    warm_theta=None,
    log=False,
    use_true_objective=True
):
    m = gp.Model("PADM_theta_subproblem_IEEE39_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    theta = add_theta_variables(m, warm_theta=warm_theta)

    obj = gp.LinExpr()

    if use_true_objective:
        for g in GS:
            dvar = m.addVar(lb=0.0, name=f"dPsync[{g}]")
            diff = theta["Pmax_sync_ce"][g] - float(Pmax_sync_base[g])

            m.addConstr(dvar >= diff, name=f"dPsync_pos[{g}]")
            m.addConstr(dvar >= -diff, name=f"dPsync_neg[{g}]")

            base = abs(float(Pmax_sync_base[g]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_Psync) * dvar / base
            else:
                obj += float(alpha_Psync) * dvar

    for name in coupled_primal_rows:
        lhs = terms_expr_theta_model_with_fixed_z(rows[name]["terms"], z_vals, theta)
        rhs = rhs_expr_in_theta_model(rows[name]["rhs"], theta)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            ydval = float(ydual_vals.get(name, 0.0))

            if abs(ydval) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += ydval * coeff_expr_in_theta_model(c, theta)

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    # Dualidad fuerte dura en theta:
    #   c^T z fijo <= b(theta)^T y fijo
    ctx_fixed = sum(float(c_map[k]) * float(z_vals.get(k, 0.0)) for k in primal_keys)

    bty_theta = gp.LinExpr()

    for name in row_order:
        ydval = float(ydual_vals.get(name, 0.0))

        if abs(ydval) <= 1e-12:
            continue

        bty_theta += ydval * rhs_expr_in_theta_model(rows[name]["rhs"], theta)

    m.addConstr(ctx_fixed <= bty_theta, name="Theta_StrongDuality")

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema THETA falló.")
        print(f"Status = {m.Status}")
        return None, m.Status, None

    theta_vals = extract_theta(theta)
    return theta_vals, m.Status, float(m.ObjVal)

def solve_y_subproblem(theta_vals, rho, mu, warm_z=None, warm_ydual=None, log=False):
    m = gp.Model("PADM_y_subproblem_IEEE39_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    z = add_z_variables(m)
    yd = add_dual_variables(m)

    m.update()

    if warm_z is not None:
        for k, v in z.items():
            if k in warm_z:
                v.Start = float(warm_z[k])

    if warm_ydual is not None:
        for name, v in yd.items():
            if name in warm_ydual:
                v.Start = float(warm_ydual[name])

    add_separable_primal_constraints_y(m, z)
    add_D_constraint_y(m, z)
    add_separable_dual_constraints_y(m, yd)
    add_strong_duality_y(m, theta_vals, z, yd)

    obj = gp.LinExpr()

    for name in coupled_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd[name]

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    if Y_OBJECTIVE_MODE == "ctx":
        obj += TIEBREAKER_CTX_WEIGHT * gp.quicksum(
            float(c_map[k]) * z[k]
            for k in primal_keys
        )

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema Y falló.")
        print(f"Status = {m.Status}")
        return None, None, m.Status, None

    z_vals = {k: float(v.X) for k, v in z.items()}
    yd_vals = {name: float(v.X) for name, v in yd.items()}

    return z_vals, yd_vals, m.Status, float(m.ObjVal)

def initialize_penalties():
    rho = {name: float(RHO0) for name in coupled_primal_rows}
    mu = {k: float(MU0) for k in coupled_dual_cols}
    return rho, mu

def update_penalties(theta_vals, z_vals, ydual_vals, rho, mu):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    n_rho = 0
    n_mu = 0

    for name in coupled_primal_rows:
        if pr[name] > FEAS_TOL:
            rho[name] = min(MAX_PENALTY, PENALTY_FACTOR * rho[name])
            n_rho += 1

    for k in coupled_dual_cols:
        if dr[k] > FEAS_TOL:
            mu[k] = min(MAX_PENALTY, PENALTY_FACTOR * mu[k])
            n_mu += 1

    return n_rho, n_mu


# ============================================================
# 8) LOOP PADM-W
# ============================================================

def run_padm(
    use_true_objective=True,
    theta_init=None,
    z_init=None,
    ydual_init=None,
    label="PADM"
):
    print("\n" + "="*70)
    print(f"INICIO {label}")
    print("="*70)
    print(f"use_true_objective = {use_true_objective}")

    rho, mu = initialize_penalties()

    theta_vals = copy.deepcopy(theta_init) if theta_init is not None else initial_theta_vals()

    if z_init is None or ydual_init is None:
        print("\nInicializando Y=(z,lambda)...")

        z_vals, ydual_vals, st_y, obj_y = solve_y_subproblem(
            theta_vals,
            rho,
            mu,
            warm_z=None,
            warm_ydual=None,
            log=False
        )

        if z_vals is None:
            print("❌ No se pudo inicializar Y.")

            return {
                "best_feasible": None,
                "theta": theta_vals,
                "z": None,
                "ydual": None,
                "history": [],
                "last_residuals": None,
                "rho": rho,
                "mu": mu,
            }

    else:
        z_vals = z_init.copy()
        ydual_vals = ydual_init.copy()

    history = []
    best_feasible = None

    res0 = max_residuals(theta_vals, z_vals, ydual_vals)

    print("\nChequeo inicial:")
    print(f"J(theta)       = {compute_J(theta_vals):.8f}")
    print(f"Max primal     = {res0['max_primal_all']:.3e}")
    print(f"Max dual       = {res0['max_dual_all']:.3e}")
    print(f"Strong duality = {res0['sd']:.3e}")

    for k_outer in range(MAX_OUTER):

        print("\n" + "#"*70)
        print(f"{label} | OUTER k = {k_outer}")
        print("#"*70)

        phi_prev = compute_phi(
            theta_vals,
            z_vals,
            ydual_vals,
            rho,
            mu,
            use_true_objective=use_true_objective
        )

        for l_inner in range(MAX_INNER):

            theta_old = copy.deepcopy(theta_vals)
            z_old = z_vals.copy()
            yd_old = ydual_vals.copy()

            theta_new, st_theta, obj_theta = solve_theta_subproblem(
                z_vals,
                ydual_vals,
                rho,
                mu,
                warm_theta=theta_vals,
                log=False,
                use_true_objective=use_true_objective
            )

            if theta_new is None:
                print("⚠️ Falló subproblema theta.")
                break

            theta_vals = theta_new

            z_new, yd_new, st_y, obj_y = solve_y_subproblem(
                theta_vals,
                rho,
                mu,
                warm_z=z_vals,
                warm_ydual=ydual_vals,
                log=False
            )

            if z_new is None:
                print("⚠️ Falló subproblema Y.")
                break

            z_vals = z_new
            ydual_vals = yd_new

            phi_now = compute_phi(
                theta_vals,
                z_vals,
                ydual_vals,
                rho,
                mu,
                use_true_objective=use_true_objective
            )

            res = max_residuals(theta_vals, z_vals, ydual_vals)

            dtheta = diff_theta(theta_vals, theta_old)
            dz = diff_dict(z_vals, z_old)
            dyd = diff_dict(ydual_vals, yd_old)
            dphi = abs(phi_now - phi_prev) / max(1.0, abs(phi_prev))

            if PADM_OUTPUT:
                print(
                    f"k={k_outer:02d}, l={l_inner:02d} | "
                    f"Phi={phi_now:.6e}, J={compute_J(theta_vals):.6e}, "
                    f"p_all={res['max_primal_all']:.2e}, "
                    f"d_all={res['max_dual_all']:.2e}, "
                    f"sd={res['sd']:.2e}, "
                    f"p_coup={res['max_primal_coupled']:.2e}, "
                    f"d_coup={res['max_dual_coupled']:.2e}, "
                    f"dtheta={dtheta:.2e}, dz={dz:.2e}, dyd={dyd:.2e}, dphi={dphi:.2e}"
                )

            history.append({
                "outer": k_outer,
                "inner": l_inner,
                "phi": phi_now,
                "J": compute_J(theta_vals),
                "max_primal_all": res["max_primal_all"],
                "max_dual_all": res["max_dual_all"],
                "sd": res["sd"],
                "max_primal_coupled": res["max_primal_coupled"],
                "max_dual_coupled": res["max_dual_coupled"],
                "dtheta": dtheta,
                "dz": dz,
                "dyd": dyd,
                "dphi": dphi,
                "rho_max": max(rho.values()) if rho else 0.0,
                "mu_max": max(mu.values()) if mu else 0.0,
            })

            if dphi <= INNER_TOL and dtheta <= INNER_TOL and dz <= INNER_TOL and dyd <= INNER_TOL:
                print(f"Inner loop converge en l={l_inner}.")
                break

            phi_prev = phi_now

        res = max_residuals(theta_vals, z_vals, ydual_vals)

        feasible = (
            res["max_primal_all"] <= FEAS_TOL
            and res["max_dual_all"] <= FEAS_TOL
            and res["sd"] <= FEAS_TOL
        )

        print("\n" + "-"*70)
        print(f"{label} | Chequeo fin outer k={k_outer}")
        print("-"*70)
        print(f"J(theta)             = {compute_J(theta_vals):.8f}")
        print(f"Max primal all       = {res['max_primal_all']:.3e}")
        print(f"Max dual all         = {res['max_dual_all']:.3e}")
        print(f"Strong duality       = {res['sd']:.3e}")
        print(f"Max primal coupled   = {res['max_primal_coupled']:.3e}")
        print(f"Max dual coupled     = {res['max_dual_coupled']:.3e}")
        print(f"rho_max              = {max(rho.values()) if rho else 0.0:.3e}")
        print(f"mu_max               = {max(mu.values()) if mu else 0.0:.3e}")

        if feasible:
            print(f"\n✅ {label} encontró solución factible primal-dual.")

            best_feasible = {
                "theta": copy.deepcopy(theta_vals),
                "z": z_vals.copy(),
                "ydual": ydual_vals.copy(),
                "J": compute_J(theta_vals),
                "residuals": res,
                "outer": k_outer,
                "history": history,
            }

            break

        n_rho, n_mu = update_penalties(theta_vals, z_vals, ydual_vals, rho, mu)

        print("\nActualización penalizaciones:")
        print(f"rho actualizadas: {n_rho}")
        print(f"mu actualizadas : {n_mu}")

    if best_feasible is None:
        print(f"\n⚠️ {label} terminó sin certificar factibilidad.")

    return {
        "best_feasible": best_feasible,
        "theta": theta_vals,
        "z": z_vals,
        "ydual": ydual_vals,
        "history": history,
        "last_residuals": max_residuals(theta_vals, z_vals, ydual_vals) if z_vals is not None else None,
        "rho": rho,
        "mu": mu,
    }


# ============================================================
# 9) EJECUTAR PADM-W
# ============================================================

PADM_WARMSTART = None

if RUN_PADM_W:

    res_ws = run_padm(
        use_true_objective=False,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM-W ETAPA 1: FACTIBILIDAD"
    )

    if res_ws["best_feasible"] is not None:

        theta_start = res_ws["best_feasible"]["theta"]
        z_start = res_ws["best_feasible"]["z"]
        yd_start = res_ws["best_feasible"]["ydual"]

        print("\n" + "="*70)
        print("WARMSTART FACTIBLE ENCONTRADO. INICIANDO ETAPA 2 CON J REAL.")
        print("="*70)

        res_final = run_padm(
            use_true_objective=True,
            theta_init=theta_start,
            z_init=z_start,
            ydual_init=yd_start,
            label="PADM-W ETAPA 2: OBJETIVO REAL"
        )

        if res_final["best_feasible"] is not None:
            final_solution = res_final["best_feasible"]
            final_label = "PADM-W ETAPA 2"
        else:
            print("\n⚠️ Etapa 2 no certificó factibilidad. Se conserva warmstart factible de etapa 1.")
            final_solution = res_ws["best_feasible"]
            final_label = "PADM-W ETAPA 1"

    else:
        print("\n⚠️ Etapa 1 no encontró punto factible.")
        final_solution = None
        final_label = "PADM-W FAILED"
        res_final = res_ws

else:
    res_final = run_padm(
        use_true_objective=True,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM"
    )

    if res_final["best_feasible"] is not None:
        final_solution = res_final["best_feasible"]
        final_label = "PADM"
    else:
        final_solution = None
        final_label = "PADM FAILED"

print("\n" + "="*70)
print("RESULTADOS FINALES PADM-W IEEE39")
print("="*70)

if final_solution is not None:
    sol_theta = final_solution["theta"]
    sol_z = final_solution["z"]
    sol_yd = final_solution["ydual"]

    print("✅ SOLUCIÓN FACTIBLE CERTIFICADA")
    print(f"Método final     : {final_label}")
    print(f"Outer encontrado : {final_solution['outer']}")
    print(f"J                : {final_solution['J']:.8f}")

    PADM_WARMSTART = {
        "theta": copy.deepcopy(sol_theta),
        "z": sol_z.copy(),
        "ydual": sol_yd.copy(),
        "label": final_label,
        "J": final_solution["J"],
        "residuals": final_solution["residuals"],
    }

else:
    print("⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.")

    sol_theta = res_final["theta"]
    sol_z = res_final["z"]
    sol_yd = res_final["ydual"]

    if sol_z is not None and sol_yd is not None:
        res = max_residuals(sol_theta, sol_z, sol_yd)

        PADM_WARMSTART = {
            "theta": copy.deepcopy(sol_theta),
            "z": sol_z.copy(),
            "ydual": sol_yd.copy(),
            "label": "PADM_LAST_POINT_NOT_CERTIFIED",
            "J": compute_J(sol_theta),
            "residuals": res,
        }

        print("⚠️ Se guardó el último punto de PADM-W como Start no certificado.")
    else:
        PADM_WARMSTART = None
        print("❌ No hay punto PADM-W disponible para warm start.")

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("CHECK PADM_WARMSTART IEEE39")
    print("="*70)

    sol_theta = PADM_WARMSTART["theta"]
    sol_z = PADM_WARMSTART["z"]
    sol_yd = PADM_WARMSTART["ydual"]

    res_check = max_residuals(sol_theta, sol_z, sol_yd)

    print(f"Origen          = {PADM_WARMSTART['label']}")
    print(f"J(theta)        = {PADM_WARMSTART['J']:.8f}")
    print(f"Max primal      = {res_check['max_primal_all']:.6e}")
    print(f"Max dual        = {res_check['max_dual_all']:.6e}")
    print(f"Strong duality  = {res_check['sd']:.6e}")


# ============================================================
# 10) WCEP FINAL CON DUALIDAD FUERTE Y WARM START PADM-W
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE39_DF_warmstart_PADM_Psync_dataset_corregido")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Pmax_sync_ce = w.addVars(GS, lb=0.0, name="Pmax_sync_ce")

for g in GS:
    base = Pmax_sync_base[g]
    w.addConstr(Pmax_sync_ce[g] >= 0.85 * base, name=f"adm_lb_Psync[{g}]")
    w.addConstr(Pmax_sync_ce[g] <= 1.00 * base, name=f"adm_ub_Psync[{g}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            # Bilineal porque Pmax_sync_ce[g] es mutable y u[g,t,y] es variable primal.
            R_primal[name] = w.addQConstr(
                Pmax_sync_ce[g] * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -float(xbar[m]), name=name)
        RHS_expr[name] = -float(xbar[m])

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= float(Hreq[y]), name=name)
        RHS_expr[name] = float(Hreq[y])

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(omega(t) * float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= eps_CO2(y),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sync_ub_"):
        ub_here = UB_sync

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dPsync = w.addVars(GS, lb=0.0, name="dPsync")

for g in GS:
    base = Pmax_sync_base[g]
    diff = Pmax_sync_ce[g] - base

    w.addConstr(dPsync[g] >= diff, name=f"dPsync_pos[{g}]")
    w.addConstr(dPsync[g] >= -diff, name=f"dPsync_neg[{g}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for g in GS:
        base = abs(float(Pmax_sync_base[g]))
        if base > 1e-9:
            obj_wcep += alpha_Psync * dPsync[g] / base
        else:
            obj_wcep += alpha_Psync * dPsync[g]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(alpha_Psync * dPsync.sum(), GRB.MINIMIZE)


# ============================================================
# 11) APLICAR WARM START DEL PADM-W
# ============================================================

def safe_start(var, value):
    try:
        if value is not None and np.isfinite(float(value)):
            var.Start = float(value)
    except Exception:
        pass

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("APLICANDO PADM_WARMSTART AL WCEP FINAL IEEE39")
    print("="*70)
    print("Origen:", PADM_WARMSTART.get("label", "PADM"))

    ws_theta = PADM_WARMSTART["theta"]
    ws_z = PADM_WARMSTART["z"]
    ws_yd = PADM_WARMSTART["ydual"]

    for g in GS:
        safe_start(Pmax_sync_ce[g], ws_theta["Pmax_sync_ce"][g])
        safe_start(dPsync[g], abs(ws_theta["Pmax_sync_ce"][g] - Pmax_sync_base[g]))

    for g in G:
        for t in T:
            for y in Y:
                safe_start(p[g, t, y], ws_z.get(("p", g, t, y), None))

    for g in GS:
        for t in T:
            for y in Y:
                safe_start(u[g, t, y], ws_z.get(("u", g, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(s[i, t, y], ws_z.get(("s", i, t, y), None))

    for m in M:
        for y in Y:
            safe_start(xcap[m, y], ws_z.get(("xcap", m, y), None))

    for m in M:
        for t in T:
            for y in Y:
                safe_start(h_syn[m, t, y], ws_z.get(("h_syn", m, t, y), None))
                safe_start(r_ffr[m, t, y], ws_z.get(("r_ffr", m, t, y), None))

    for (i, j) in L_arcs:
        for t in T:
            for y in Y:
                safe_start(f_pos[i, j, t, y], ws_z.get(("f_pos", i, j, t, y), None))
                safe_start(f_neg[i, j, t, y], ws_z.get(("f_neg", i, j, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(theta_p[i, t, y], ws_z.get(("theta_p", i, t, y), None))
                safe_start(theta_m[i, t, y], ws_z.get(("theta_m", i, t, y), None))

    for name, var in ydual.items():
        safe_start(var, ws_yd.get(name, None))

    print("✅ Warm start aplicado al WCEP IEEE39.")
else:
    print("\n⚠️ No existe PADM_WARMSTART. El WCEP se resolverá sin Start.")


# ============================================================
# 12) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE39")
print("="*70)

w.optimize()


# ============================================================
# 13) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE39")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Pmax_sync_ce síncronos ---")
    hubo_cambio = False

    for g in GS:
        val = float(Pmax_sync_ce[g].X)
        base = float(Pmax_sync_base[g])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{g}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            omega(t) * float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = eps_CO2(y) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {eps_CO2(y):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq[y])

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_sync = 0

    for name, var in ydual.items():
        if name.startswith("sync_ub_"):
            if abs(var.X - UB_sync) <= 1e-6 * max(1.0, UB_sync):
                hits_sync += 1

    print(f"Duales sync_ub pegadas a cota : {hits_sync}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# REGISTRO DE RESULTADOS WCEP
# ============================================================

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP FINAL")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo Gurobi optimize() = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds_gurobi": runtime,
        "runtime_minutes_gurobi": runtime / 60,
        "runtime_hours_gurobi": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }
else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds_gurobi": runtime,
        "runtime_minutes_gurobi": runtime / 60,
        "runtime_hours_gurobi": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }

t_total_script = time.time() - t0_total_script
registro_wcep["runtime_seconds_total_script"] = t_total_script
registro_wcep["runtime_minutes_total_script"] = t_total_script / 60
registro_wcep["runtime_hours_total_script"] = t_total_script / 3600

print(f"Tiempo total script = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")

df_registro_wcep = pd.DataFrame([registro_wcep])
display(df_registro_wcep)


### Solo b (Hreq y X_barra)

#### Solo Dualidad

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import time

# ============================================================
# FINAL IEEE39: SOLO WCEP CON DUALIDAD FUERTE
# DERIVADO DEL CÓDIGO ESTRELLA + DATASET CORREGIDO
#
# Caso experimental: SOLO parámetros en b mutables.
#
# Mutables:
#   1) Hreq_ce[y]  en b
#   2) xbar_ce[m]  en b
#
# Fijo:
#   - Pmax fijo; NO se muta Pmax.
#   - No hay PADM ni warm start.
#   - No se usan pesos temporales ni 365 días: mantiene el caso original de 4 periodos.
# ============================================================

# Medición de tiempo total de celda/script
t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE39: caso original de 4 periodos, sin ponderación anual
epsilon_CO2_wcep = 2366

# Objetivo J
alpha_H = 2.0
alpha_Xbar = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1

# Rangos de mutabilidad para Hreq y xbar
MUT_LB_FACTOR = 0.85
MUT_UB_FACTOR = 1.15

# Cotas duales heurísticas en las filas donde aparecen parámetros mutables
USE_DUAL_BOUNDS_FOR_MUTABLE_ROWS = True


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE39
# ============================================================
# Este bloque hace que WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

bus_slack = int(sorted(B)[0])

Hreq_base = {y: float(Hreq[y]) for y in Y}
xbar_base = {m: float(xbar[m]) for m in M}

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_xbar = 2.0 * max(float(cinv[m]) for m in M) if len(M) > 0 else 1e8

print("\n" + "="*70)
print("COTAS DUALES IEEE39")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_xbar = {UB_xbar:.6f}")
print(f"Bus slack usado: {bus_slack}")
print("Parámetros mutables:")
print(f"  Hreq_ce[y] en [{MUT_LB_FACTOR:.2f}, {MUT_UB_FACTOR:.2f}] * Hreq[y]")
print(f"  xbar_ce[m] en [{MUT_LB_FACTOR:.2f}, {MUT_UB_FACTOR:.2f}] * xbar[m]")
print("Pmax queda fijo en su valor original.")




# ============================================================
# 2) WCEP FINAL CON DUALIDAD FUERTE
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE39 SOLO HREQ + XBAR CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE39_SOLO_Hreq_Xbar_DF")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Hreq_ce = w.addVars(Y, lb=0.0, name="Hreq_ce")
xbar_ce = w.addVars(M, lb=0.0, name="xbar_ce")

for y in Y:
    base = Hreq_base[y]
    w.addConstr(Hreq_ce[y] >= MUT_LB_FACTOR * base, name=f"adm_lb_Hreq[{y}]")
    w.addConstr(Hreq_ce[y] <= MUT_UB_FACTOR * base, name=f"adm_ub_Hreq[{y}]")

for m in M:
    base = xbar_base[m]
    w.addConstr(xbar_ce[m] >= MUT_LB_FACTOR * base, name=f"adm_lb_xbar[{m}]")
    w.addConstr(xbar_ce[m] <= MUT_UB_FACTOR * base, name=f"adm_ub_xbar[{m}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(Pmax[g]) * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -xbar_ce[m], name=name)
        RHS_expr[name] = -xbar_ce[m]

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= Hreq_ce[y], name=name)
        RHS_expr[name] = Hreq_ce[y]

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= float(epsilon_CO2_wcep),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if USE_DUAL_BOUNDS_FOR_MUTABLE_ROWS:
        if name.startswith("sys_H_"):
            ub_here = UB_H
        elif name.startswith("x_ub_"):
            ub_here = UB_xbar

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dHreq = w.addVars(Y, lb=0.0, name="dHreq")
dXbar = w.addVars(M, lb=0.0, name="dXbar")

for y in Y:
    base = Hreq_base[y]
    diff = Hreq_ce[y] - base

    w.addConstr(dHreq[y] >= diff, name=f"dHreq_pos[{y}]")
    w.addConstr(dHreq[y] >= -diff, name=f"dHreq_neg[{y}]")

for m in M:
    base = xbar_base[m]
    diff = xbar_ce[m] - base

    w.addConstr(dXbar[m] >= diff, name=f"dXbar_pos[{m}]")
    w.addConstr(dXbar[m] >= -diff, name=f"dXbar_neg[{m}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for y in Y:
        base = abs(float(Hreq_base[y]))
        if base > 1e-9:
            obj_wcep += alpha_H * dHreq[y] / base
        else:
            obj_wcep += alpha_H * dHreq[y]

    for m in M:
        base = abs(float(xbar_base[m]))
        if base > 1e-9:
            obj_wcep += alpha_Xbar * dXbar[m] / base
        else:
            obj_wcep += alpha_Xbar * dXbar[m]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_H * dHreq.sum()
        + alpha_Xbar * dXbar.sum(),
        GRB.MINIMIZE
    )




print("\n⚠️ WCEP correrá sin PADM y sin warm start.")

# ============================================================
# 3) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE39")
print("="*70)

w.optimize()


# ============================================================
# 4) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE39")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Hreq_ce ---")
    hubo_cambio = False

    for y in Y:
        val = float(Hreq_ce[y].X)
        base = float(Hreq_base[y])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{y}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- xbar_ce ---")
    hubo_cambio = False

    for m in M:
        val = float(xbar_ce[m].X)
        base = float(xbar_base[m])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{m}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = float(epsilon_CO2_wcep) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {float(epsilon_CO2_wcep):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_ce[y].X)

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_ce={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_xbar = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("x_ub_"):
            if abs(var.X - UB_xbar) <= 1e-6 * max(1.0, UB_xbar):
                hits_xbar += 1

    print(f"Duales sys_H pegadas a cota   : {hits_H}")
    print(f"Duales x_ub pegadas a cota    : {hits_xbar}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# REGISTRO DE RESULTADOS WCEP
# ============================================================

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo total  = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }

else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }
# Tiempo total real de toda la celda/script
t_total_script = time.time() - t0_total_script
print("\n" + "="*70)
print("TIEMPO TOTAL DE CELDA / SCRIPT")
print("="*70)
print(f"Tiempo Gurobi optimize() = {float(w.Runtime):.2f} s")
print(f"Tiempo total script      = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")

# Guardar registro en DataFrame si pandas está disponible
df_registro_wcep = pd.DataFrame([registro_wcep])
display(df_registro_wcep)


✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en WCEP = 1289.386000
Pmax total sistema                 = 1315.700000

COTAS DUALES IEEE39
UB_H    = 324200000.000000
UB_xbar = 3242000.000000
Bus slack usado: 1
Parámetros mutables:
  Hreq_ce[y] en [0.85, 1.15] * Hreq[y]
  xbar_ce[m] en [0.85, 1.15] * xbar[m]
Pmax queda fijo en su valor original.

CONSTRUYENDO WCEP FINAL IEEE39 SOLO HREQ + XBAR CON DUALIDAD FUERTE
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.005
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 18000

⚠️ WCEP correrá sin PADM y sin warm start.

RESOLVIENDO WCEP FINAL IEEE39
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12650H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  18000
MIPGap  0.005
NonConvex  2

Optimize a model with 4209 row

,status,runtime_seconds,runtime_minutes,runtime_hours,sol_count,obj_val,obj_bound,mip_gap,mip_gap_percent
0,3,0.238,0.003967,0.000066,0,None,None,None,None


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import time

# ============================================================
# FINAL IEEE39: SOLO WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO + PESOS TEMPORALES
#
# Mutables:
#   1) Hreq_ce[y]  en b
#   2) xbar_ce[m]  en b
#
# Fijo:
#   - Pmax fijo; NO se muta Pmax.
#   - No hay PADM ni warm start.
# ============================================================

# Medición de tiempo total de celda/script
t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE39
# Si ya calculaste en el FW emisiones ponderadas, puedes poner epsilon_CO2_wcep manual.
# Si lo dejas como None, el código intenta usar el 90% de las emisiones FW guardadas en memoria.
epsilon_CO2_wcep = 930756.144528 * 0.9  # Puedes poner aquí el valor manual de epsilon CO2 para IEEE39
EPSILON_CO2_FRACTION = 0.90

# Objetivo J
alpha_H = 2.0
alpha_Xbar = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = True

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE39
# ============================================================
# Este bloque hace que WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# Si T = [1, 7, 13, 19] representa un perfil diario, usamos omega_t
# para expandir costos y emisiones a base anual sin crear 365*|T| periodos.
# Por defecto cada t se repite 365 veces y representa 1 hora.
# Si cada t representa un bloque de 6 horas, define TIME_BLOCK_HOURS = 6.0 antes de correr.
TIME_BLOCK_HOURS = float(globals().get("TIME_BLOCK_HOURS", 1.0))
N_DAYS_EQUIV = int(globals().get("N_DAYS_EQUIV", 365))

if "omega_t" not in globals():
    omega_t = {t: float(N_DAYS_EQUIV) * TIME_BLOCK_HOURS for t in T}
else:
    omega_t = {t: float(omega_t[t]) for t in T}

def omega(t):
    return float(omega_t[t])

print("Pesos temporales omega_t usados:", omega_t)

bus_slack = int(sorted(B)[0])

Hreq_base = {y: float(Hreq[y]) for y in Y}
xbar_base = {m: float(xbar[m]) for m in M}

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_xbar = 2.0 * max(float(cinv[m]) for m in M) if len(M) > 0 else 1e8

# ============================================================
# Límite de emisiones D
# ============================================================
def _get_fw_emissions_by_year():
    candidates = [
        "emisiones_por_anio", "df_emis_fw", "df_emis", "df_emis_wcep"
    ]
    value_cols = [
        "emisiones_anuales_equivalentes",
        "emisiones_totales_ponderadas",
        "emisiones_totales_anuales",
        "emisiones",
    ]
    for name in candidates:
        if name in globals():
            df = globals()[name]
            if hasattr(df, "columns") and "y" in df.columns:
                for col in value_cols:
                    if col in df.columns:
                        return {int(row["y"]): float(row[col]) for _, row in df.iterrows()}
    return None

_fw_emis = _get_fw_emissions_by_year()

if epsilon_CO2_wcep is None:
    if _fw_emis is not None:
        epsilon_CO2_wcep_by_y = {y: EPSILON_CO2_FRACTION * float(_fw_emis[y]) for y in Y if y in _fw_emis}
        print("epsilon_CO2_wcep calculado como 90% de emisiones FW:", epsilon_CO2_wcep_by_y)
    else:
        raise ValueError(
            "epsilon_CO2_wcep=None, pero no encontré emisiones del FW. "
            "Define epsilon_CO2_wcep manualmente o ejecuta antes el FW con tabla de emisiones."
        )
elif isinstance(epsilon_CO2_wcep, dict):
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep[y]) for y in epsilon_CO2_wcep}
else:
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep) for y in Y}

def eps_CO2(y):
    return float(epsilon_CO2_wcep_by_y[int(y)])


print("\n" + "="*70)
print("COTAS DUALES IEEE39")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_xbar = {UB_xbar:.6f}")
print(f"Bus slack usado: {bus_slack}")




# ============================================================
# 2) WCEP FINAL CON DUALIDAD FUERTE
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE39_DF_solo_dataset_corregido")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Hreq_ce = w.addVars(Y, lb=0.0, name="Hreq_ce")
xbar_ce = w.addVars(M, lb=0.0, name="xbar_ce")

for y in Y:
    base = Hreq_base[y]
    w.addConstr(Hreq_ce[y] >= 0.85 * base, name=f"adm_lb_Hreq[{y}]")
    w.addConstr(Hreq_ce[y] <= 1.15 * base, name=f"adm_ub_Hreq[{y}]")

for m in M:
    base = xbar_base[m]
    w.addConstr(xbar_ce[m] >= 0.85 * base, name=f"adm_lb_xbar[{m}]")
    w.addConstr(xbar_ce[m] <= 1.15 * base, name=f"adm_ub_xbar[{m}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(Pmax[g]) * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -xbar_ce[m], name=name)
        RHS_expr[name] = -xbar_ce[m]

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= Hreq_ce[y], name=name)
        RHS_expr[name] = Hreq_ce[y]

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(omega(t) * float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= eps_CO2(y),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sys_H_"):
        ub_here = UB_H
    elif name.startswith("x_ub_"):
        ub_here = UB_xbar

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dHreq = w.addVars(Y, lb=0.0, name="dHreq")
dXbar = w.addVars(M, lb=0.0, name="dXbar")

for y in Y:
    base = Hreq_base[y]
    diff = Hreq_ce[y] - base

    w.addConstr(dHreq[y] >= diff, name=f"dHreq_pos[{y}]")
    w.addConstr(dHreq[y] >= -diff, name=f"dHreq_neg[{y}]")

for m in M:
    base = xbar_base[m]
    diff = xbar_ce[m] - base

    w.addConstr(dXbar[m] >= diff, name=f"dXbar_pos[{m}]")
    w.addConstr(dXbar[m] >= -diff, name=f"dXbar_neg[{m}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for y in Y:
        base = abs(float(Hreq_base[y]))
        if base > 1e-9:
            obj_wcep += alpha_H * dHreq[y] / base
        else:
            obj_wcep += alpha_H * dHreq[y]

    for m in M:
        base = abs(float(xbar_base[m]))
        if base > 1e-9:
            obj_wcep += alpha_Xbar * dXbar[m] / base
        else:
            obj_wcep += alpha_Xbar * dXbar[m]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_H * dHreq.sum()
        + alpha_Xbar * dXbar.sum(),
        GRB.MINIMIZE
    )




print("\n⚠️ WCEP correrá sin PADM y sin warm start.")

# ============================================================
# 3) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE39")
print("="*70)

w.optimize()


# ============================================================
# 4) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE39")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Hreq_ce ---")
    hubo_cambio = False

    for y in Y:
        val = float(Hreq_ce[y].X)
        base = float(Hreq_base[y])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{y}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- xbar_ce ---")
    hubo_cambio = False

    for m in M:
        val = float(xbar_ce[m].X)
        base = float(xbar_base[m])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{m}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            omega(t) * float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = eps_CO2(y) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {eps_CO2(y):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_ce[y].X)

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_ce={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_xbar = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("x_ub_"):
            if abs(var.X - UB_xbar) <= 1e-6 * max(1.0, UB_xbar):
                hits_xbar += 1

    print(f"Duales sys_H pegadas a cota   : {hits_H}")
    print(f"Duales x_ub pegadas a cota    : {hits_xbar}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# REGISTRO DE RESULTADOS WCEP
# ============================================================

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo total  = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }

else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }
# Tiempo total real de toda la celda/script
t_total_script = time.time() - t0_total_script
print("\n" + "="*70)
print("TIEMPO TOTAL DE CELDA / SCRIPT")
print("="*70)
print(f"Tiempo Gurobi optimize() = {float(w.Runtime):.2f} s")
print(f"Tiempo total script      = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")

# Guardar registro en DataFrame si pandas está disponible
df_registro_wcep = pd.DataFrame([registro_wcep])
display(df_registro_wcep)


✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en WCEP = 1289.386000
Pmax total sistema                 = 1315.700000
Pesos temporales omega_t usados: {1: 365.0, 7: 365.0, 13: 365.0, 19: 365.0}

COTAS DUALES IEEE39
UB_H    = 324200000.000000
UB_xbar = 3242000.000000
Bus slack usado: 1

CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.005
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 18000

⚠️ WCEP correrá sin PADM y sin warm start.

RESOLVIENDO WCEP FINAL IEEE39
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12650H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  18000
MIPGap  0.005
NonConvex  2

Optimize a model with 4209 rows, 7102 columns and 16138 nonzeros (Min)
Model fingerprint: 0x80ea0c36
Model h

,status,runtime_seconds,runtime_minutes,runtime_hours,sol_count,obj_val,obj_bound,mip_gap,mip_gap_percent
0,3,0.256,0.004267,0.000071,0,None,None,None,None


#### Dualidad + PADM

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import copy
import math
import time

t0_total_script = time.time()

# ============================================================
# FINAL IEEE39: PADM-W + WCEP CON DUALIDAD FUERTE - SOLO Hreq Y xbar MUTABLES
# DERIVADO DEL CÓDIGO ESTRELLA + DATASET CORREGIDO:
#
#   sum_g Hg[g] u[g,t,y] + sum_m h_syn[m,t,y] >= Hreq_ce[y]
#
# Es decir:
#   - NO se usa zH.
#   - Hreq_ce queda en b.
#   - Pmax fijo; NO se muta Pmax.
#   - xbar_ce queda en b.
#
# Flujo:
#   1) PADM-W etapa 1: busca factibilidad.
#   2) PADM-W etapa 2: parte de factibilidad y reduce J(theta).
#   3) WCEP completo con dualidad fuerte usando PADM_WARMSTART.
# ============================================================

# ============================================================
# 0) OPCIONES
# ============================================================

PADM_OUTPUT = 1
RUN_PADM_W = True

# La etapa 1 busca factibilidad. La etapa 2 intenta mejorar J desde ese punto.
RUN_PADM_STAGE_2 = True

# Si no se certifica factibilidad, permite usar el último punto como Start no certificado.
USE_LAST_PADM_POINT_IF_NO_FEASIBLE = True

MAX_OUTER = 20
MAX_INNER = 20

FEAS_TOL = 1e-5
INNER_TOL = 1e-5

RHO0 = 500.0
MU0 = 500.0

PENALTY_FACTOR = 2.0
MAX_PENALTY = 1e9

TIME_LIMIT_SUBPROBLEM = 5 * 60 * 60  # 5 horas por subproblema PADM
WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE39
# Caso original de 4 periodos: NO se usan pesos temporales de 365 días.
epsilon_CO2_wcep = 2366
EPSILON_CO2_FRACTION = 0.90

# Objetivo J
alpha_H = 2.0
alpha_Xbar = 1.0

# Rangos de mutabilidad para Hreq y xbar
MUT_LB_FACTOR = 0.85
MUT_UB_FACTOR = 1.15

# Parámetros mutables en este experimento:
#   - Hreq_ce[y]  (RHS/b)
#   - xbar_ce[m]  (RHS/b)
# Pmax queda fijo, por lo que no aparece Pmax_sync_ce.

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE39
# ============================================================
# Este bloque hace que PADM-W y WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por PADM-W/WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en PADM-W/WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

# ============================================================
# SIN PESOS TEMPORALES: CASO ORIGINAL DE 4 PERIODOS
# ============================================================
# En este experimento volvemos al modelo original: T se usa tal como viene
# y no se expande a 365 días. Por compatibilidad con funciones anteriores,
# definimos omega(t)=1.0, pero NO aparece multiplicando costos ni emisiones.
def omega(t):
    return 1.0

print("Caso original de 4 periodos: sin pesos temporales omega_t.")

bus_slack = int(sorted(B)[0])

Hreq_base = {y: float(Hreq[y]) for y in Y}
xbar_base = {m: float(xbar[m]) for m in M}

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_xbar = 2.0 * max(float(cinv[m]) for m in M) if len(M) > 0 else 1e8

# ============================================================
# Límite de emisiones D
# ============================================================
def _get_fw_emissions_by_year():
    candidates = [
        "emisiones_por_anio", "df_emis_fw", "df_emis", "df_emis_wcep"
    ]
    value_cols = [
        "emisiones_anuales_equivalentes",
        "emisiones_totales_ponderadas",
        "emisiones_totales_anuales",
        "emisiones",
    ]
    for name in candidates:
        if name in globals():
            df = globals()[name]
            if hasattr(df, "columns") and "y" in df.columns:
                for col in value_cols:
                    if col in df.columns:
                        return {int(row["y"]): float(row[col]) for _, row in df.iterrows()}
    return None

_fw_emis = _get_fw_emissions_by_year()

if epsilon_CO2_wcep is None:
    if _fw_emis is not None:
        epsilon_CO2_wcep_by_y = {y: EPSILON_CO2_FRACTION * float(_fw_emis[y]) for y in Y if y in _fw_emis}
        print("epsilon_CO2_wcep calculado como 90% de emisiones FW:", epsilon_CO2_wcep_by_y)
    else:
        raise ValueError(
            "epsilon_CO2_wcep=None, pero no encontré emisiones del FW. "
            "Define epsilon_CO2_wcep manualmente o ejecuta antes el FW con tabla de emisiones."
        )
elif isinstance(epsilon_CO2_wcep, dict):
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep[y]) for y in epsilon_CO2_wcep}
else:
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep) for y in Y}

def eps_CO2(y):
    return float(epsilon_CO2_wcep_by_y[int(y)])


print("\n" + "="*70)
print("COTAS DUALES IEEE39")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_xbar = {UB_xbar:.6f}")
print(f"Bus slack usado: {bus_slack}")
print("Parámetros mutables:")
print(f"  Hreq_ce[y] en [{MUT_LB_FACTOR:.2f}, {MUT_UB_FACTOR:.2f}] * Hreq[y]")
print(f"  xbar_ce[m] en [{MUT_LB_FACTOR:.2f}, {MUT_UB_FACTOR:.2f}] * xbar[m]")
print("Pmax queda fijo en su valor original.")


# ============================================================
# 2) REPRESENTACIÓN SIMBÓLICA PARA PADM-W
# ============================================================

def key_p(g,t,y):       return ("p", g, t, y)
def key_u(g,t,y):       return ("u", g, t, y)
def key_s(i,t,y):       return ("s", i, t, y)
def key_xcap(m,y):      return ("xcap", m, y)
def key_hsyn(m,t,y):    return ("h_syn", m, t, y)
def key_rffr(m,t,y):    return ("r_ffr", m, t, y)
def key_fpos(i,j,t,y):  return ("f_pos", i, j, t, y)
def key_fneg(i,j,t,y):  return ("f_neg", i, j, t, y)
def key_thp(i,t,y):     return ("theta_p", i, t, y)
def key_thm(i,t,y):     return ("theta_m", i, t, y)

def param_coeff(family, idx, mult=1.0):
    return ("PARAM", family, idx, float(mult))

def is_param_coeff(c):
    return isinstance(c, tuple) and len(c) == 4 and c[0] == "PARAM"

def coeff_value(c, theta_vals):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * float(theta_vals[fam][idx])
    return float(c)

def coeff_expr_in_theta_model(c, theta_vars):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * theta_vars[fam][idx]
    return float(c)

def rhs_const(val):
    return [("CONST", None, None, float(val))]

def rhs_param(family, idx, mult=1.0):
    return [("PARAM", family, idx, float(mult))]

def rhs_has_param(rhs_terms):
    return any(typ == "PARAM" for typ, fam, idx, mult in rhs_terms)

def rhs_eval(rhs_terms, theta_vals=None):
    out = 0.0

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            out += float(mult)
        elif typ == "PARAM":
            if theta_vals is None:
                raise ValueError("rhs_eval requiere theta_vals para RHS con parámetros.")
            out += float(mult) * float(theta_vals[fam][idx])
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return out

def rhs_expr_in_theta_model(rhs_terms, theta_vars):
    expr = gp.LinExpr()

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            expr += float(mult)
        elif typ == "PARAM":
            expr += float(mult) * theta_vars[fam][idx]
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return expr

def add_to_terms(terms, key, coeff):
    if key not in terms:
        terms[key] = []
    terms[key].append(coeff)

def neg_coeff(c):
    if is_param_coeff(c):
        return ("PARAM", c[1], c[2], -c[3])
    return -float(c)

def terms_has_param(terms):
    for k, coeffs in terms.items():
        for c in coeffs:
            if is_param_coeff(c):
                return True
    return False

def terms_expr_z_model(terms, z_vars, theta_vals):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        for c in coeffs:
            expr += coeff_value(c, theta_vals) * z_vars[k]

    return expr

def terms_expr_theta_model_with_fixed_z(terms, z_vals, theta_vars):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            expr += zval * coeff_expr_in_theta_model(c, theta_vars)

    return expr

def terms_eval(terms, z_vals, theta_vals):
    out = 0.0

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            out += coeff_value(c, theta_vals) * zval

    return out


# ============================================================
# 3) CREAR FILAS PRIMALES Ax >= b
# ============================================================

rows = {}
row_order = []

def add_row(name, terms, rhs_terms):
    if name in rows:
        raise ValueError(f"Fila repetida: {name}")

    rows[name] = {
        "terms": terms,
        "rhs": rhs_terms
    }

    row_order.append(name)

# ------------------------------------------------------------
# A.1 Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}

            for g in G_en_bus.get(i, []):
                add_to_terms(terms, key_p(g,t,y), 1.0)

            add_to_terms(terms, key_s(i,t,y), 1.0)

            for (k,j) in lineas_entran[i]:
                add_to_terms(terms, key_fpos(k,j,t,y), 1.0)
                add_to_terms(terms, key_fneg(k,j,t,y), -1.0)

            for (i2,j) in lineas_salen[i]:
                add_to_terms(terms, key_fpos(i2,j,t,y), -1.0)
                add_to_terms(terms, key_fneg(i2,j,t,y), 1.0)

            rhs_val = float(d_fw[(i,t,y)])

            add_row(f"bal_p_{i}_{t}_{y}", terms, rhs_const(rhs_val))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"bal_n_{i}_{t}_{y}", terms_n, rhs_const(-rhs_val))

# ------------------------------------------------------------
# A.2 ENS <= demanda
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}
            add_to_terms(terms, key_s(i,t,y), -1.0)
            add_row(f"ens_ub_{i}_{t}_{y}", terms, rhs_const(-float(d_fw[(i,t,y)])))

# ------------------------------------------------------------
# A.3 Flujo DC
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i,j)])

            terms = {}
            add_to_terms(terms, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms, key_fneg(i,j,t,y), -1.0)

            add_to_terms(terms, key_thp(i,t,y), -Bij)
            add_to_terms(terms, key_thm(i,t,y),  Bij)
            add_to_terms(terms, key_thp(j,t,y),  Bij)
            add_to_terms(terms, key_thm(j,t,y), -Bij)

            add_row(f"dc_p_{i}_{j}_{t}_{y}", terms, rhs_const(0.0))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"dc_n_{i}_{j}_{t}_{y}", terms_n, rhs_const(0.0))

# ------------------------------------------------------------
# A.4 Límites térmicos fijos
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i,j)])

            terms_lb = {}
            add_to_terms(terms_lb, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms_lb, key_fneg(i,j,t,y), -1.0)
            add_row(f"flb_{i}_{j}_{t}_{y}", terms_lb, rhs_const(-Fij))

            terms_ub = {}
            add_to_terms(terms_ub, key_fpos(i,j,t,y), -1.0)
            add_to_terms(terms_ub, key_fneg(i,j,t,y), 1.0)
            add_row(f"fub_{i}_{j}_{t}_{y}", terms_ub, rhs_const(-Fij))

# ------------------------------------------------------------
# A.5 Slack bus
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_p = {}
        add_to_terms(terms_p, key_thp(bus_slack,t,y), 1.0)
        add_to_terms(terms_p, key_thm(bus_slack,t,y), -1.0)
        add_row(f"slack_p_{t}_{y}", terms_p, rhs_const(0.0))

        terms_n = {}
        add_to_terms(terms_n, key_thp(bus_slack,t,y), -1.0)
        add_to_terms(terms_n, key_thm(bus_slack,t,y), 1.0)
        add_row(f"slack_n_{t}_{y}", terms_n, rhs_const(0.0))

print(f"Bus slack usado: {bus_slack}")

# ------------------------------------------------------------
# A.6 Límites generación fijos p <= P_disp
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_base = float(P_disp.get((g,t), Pmax[g]))

        for y in Y:
            terms = {}
            add_to_terms(terms, key_p(g,t,y), -1.0)
            add_row(f"p_cap_{g}_{t}_{y}", terms, rhs_const(-cap_base))

# ------------------------------------------------------------
# A.7 Síncronos
# ------------------------------------------------------------
for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            terms_lb = {}
            add_to_terms(terms_lb, key_p(g,t,y), 1.0)
            add_to_terms(terms_lb, key_u(g,t,y), -pmin_g)
            add_row(f"sync_lb_{g}_{t}_{y}", terms_lb, rhs_const(0.0))

            terms_ub = {}
            add_to_terms(terms_ub, key_u(g,t,y), float(Pmax[g]))
            add_to_terms(terms_ub, key_p(g,t,y), -1.0)
            add_row(f"sync_ub_{g}_{t}_{y}", terms_ub, rhs_const(0.0))

            terms_u = {}
            add_to_terms(terms_u, key_u(g,t,y), -1.0)
            add_row(f"u_ub_{g}_{t}_{y}", terms_u, rhs_const(-1.0))

# ------------------------------------------------------------
# A.8-A.10 Inversión, h_syn, FFR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        terms_x = {}
        add_to_terms(terms_x, key_xcap(m,y), -1.0)
        add_row(f"x_ub_{m}_{y}", terms_x, rhs_param("xbar_ce", m, -1.0))

        for t in T:
            terms_h = {}
            add_to_terms(terms_h, key_xcap(m,y), float(gamma_base[m]))
            add_to_terms(terms_h, key_hsyn(m,t,y), -1.0)
            add_row(f"hsyn_lim_{m}_{t}_{y}", terms_h, rhs_const(0.0))

            terms_r = {}
            add_to_terms(terms_r, key_xcap(m,y), float(kappa_base[m]))
            add_to_terms(terms_r, key_rffr(m,t,y), -1.0)
            add_row(f"ffr_lim_{m}_{t}_{y}", terms_r, rhs_const(0.0))

# ------------------------------------------------------------
# A.11 Seguridad: Hreq_ce en b, Rreq fijo
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_H = {}

        for g in GS:
            add_to_terms(terms_H, key_u(g,t,y), float(Hg.get(g, 0.0)))

        for m in M:
            add_to_terms(terms_H, key_hsyn(m,t,y), 1.0)

        add_row(f"sys_H_{t}_{y}", terms_H, rhs_param("Hreq_ce", y, 1.0))

        terms_R = {}

        for m in M:
            add_to_terms(terms_R, key_rffr(m,t,y), 1.0)

        add_row(f"sys_R_{t}_{y}", terms_R, rhs_const(float(Rreq[y])))


# ============================================================
# 4) VARIABLES PRIMALES Y COSTOS c
# ============================================================

primal_keys = []

for g in G:
    for t in T:
        for y in Y:
            primal_keys.append(key_p(g,t,y))

for g in GS:
    for t in T:
        for y in Y:
            primal_keys.append(key_u(g,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_s(i,t,y))

for m in M:
    for y in Y:
        primal_keys.append(key_xcap(m,y))

for m in M:
    for t in T:
        for y in Y:
            primal_keys.append(key_hsyn(m,t,y))
            primal_keys.append(key_rffr(m,t,y))

for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            primal_keys.append(key_fpos(i,j,t,y))
            primal_keys.append(key_fneg(i,j,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_thp(i,t,y))
            primal_keys.append(key_thm(i,t,y))

c_map = {}

for g in G:
    for t in T:
        for y in Y:
            c_map[key_p(g,t,y)] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map[key_s(i,t,y)] = float(VOLL)

for m in M:
    for y in Y:
        c_map[key_xcap(m,y)] = float(cinv[m])

for k in primal_keys:
    c_map.setdefault(k, 0.0)


# ============================================================
# 5) CLASIFICACIÓN ACOPLADAS VS SEPARABLES
# ============================================================

coupled_primal_rows = [
    name for name in row_order
    if terms_has_param(rows[name]["terms"]) or rhs_has_param(rows[name]["rhs"])
]

separable_primal_rows = [
    name for name in row_order
    if name not in set(coupled_primal_rows)
]

coupled_dual_cols = set()

for name in row_order:
    terms = rows[name]["terms"]

    for k, coeffs in terms.items():
        if any(is_param_coeff(c) for c in coeffs):
            coupled_dual_cols.add(k)

coupled_dual_cols = sorted(list(coupled_dual_cols), key=str)

separable_dual_cols = [
    k for k in primal_keys
    if k not in set(coupled_dual_cols)
]

print("\n" + "="*70)
print("CLASIFICACIÓN PADM-W IEEE39 - FORMULACIÓN ORIGINAL")
print("="*70)
print(f"N° filas primales acopladas   : {len(coupled_primal_rows)}")
print(f"N° filas primales separables  : {len(separable_primal_rows)}")
print(f"N° columnas duales acopladas  : {len(coupled_dual_cols)}")
print(f"N° columnas duales separables : {len(separable_dual_cols)}")

bad_rows = [
    name for name in coupled_primal_rows
    if not (name.startswith("sys_H_") or name.startswith("x_ub_"))
]

if bad_rows:
    print("\n⚠️ Hay filas acopladas no esperadas:")
    print(bad_rows[:30])
else:
    print("\n✅ Filas acopladas esperadas: sys_H_ y x_ub_.")


# ============================================================
# 6) FUNCIONES PADM-W
# ============================================================

def initial_theta_vals():
    return {
        "Hreq_ce": {y: float(Hreq_base[y]) for y in Y},
        "xbar_ce": {m: float(xbar_base[m]) for m in M},
    }

def compute_J(theta_vals):
    J = 0.0

    for y in Y:
        base = float(Hreq_base[y])
        val = float(theta_vals["Hreq_ce"][y])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_H * abs(val - base) / abs(base)
        else:
            J += alpha_H * abs(val - base)

    for m in M:
        base = float(xbar_base[m])
        val = float(theta_vals["xbar_ce"][m])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_Xbar * abs(val - base) / abs(base)
        else:
            J += alpha_Xbar * abs(val - base)

    return J

def compute_ctx(z_vals):
    return sum(
        float(c_map[k]) * float(z_vals.get(k, 0.0))
        for k in primal_keys
    )

def compute_bty(theta_vals, ydual_vals):
    out = 0.0

    for name in row_order:
        out += rhs_eval(rows[name]["rhs"], theta_vals) * float(ydual_vals.get(name, 0.0))

    return out

def compute_primal_residuals(theta_vals, z_vals):
    res = {}

    for name in row_order:
        lhs = terms_eval(rows[name]["terms"], z_vals, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)
        res[name] = max(0.0, rhs - lhs)

    return res

def compute_dual_residuals(theta_vals, ydual_vals):
    res = {}

    for k in primal_keys:
        lhs = 0.0

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            yd = float(ydual_vals.get(name, 0.0))

            if abs(yd) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd

        rhs = float(c_map.get(k, 0.0))
        res[k] = max(0.0, lhs - rhs)

    return res

def compute_sd_residual(theta_vals, z_vals, ydual_vals):
    return max(0.0, compute_ctx(z_vals) - compute_bty(theta_vals, ydual_vals))

def max_residuals(theta_vals, z_vals, ydual_vals):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)
    sd = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return {
        "max_primal_all": max(pr.values()) if pr else 0.0,
        "max_dual_all": max(dr.values()) if dr else 0.0,
        "sd": sd,
        "max_primal_coupled": max([pr[name] for name in coupled_primal_rows], default=0.0),
        "max_dual_coupled": max([dr[k] for k in coupled_dual_cols], default=0.0),
    }

def compute_phi(theta_vals, z_vals, ydual_vals, rho, mu, use_true_objective=True):
    Jval = compute_J(theta_vals) if use_true_objective else 0.0

    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    pen_p = sum(float(rho[name]) * pr[name] for name in coupled_primal_rows)
    pen_d = sum(float(mu[k]) * dr[k] for k in coupled_dual_cols)

    sd_penalty = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return Jval + pen_p + pen_d + 1e3 * sd_penalty

def diff_theta(theta_a, theta_b):
    vals = []

    for y in Y:
        vals.append(abs(theta_a["Hreq_ce"][y] - theta_b["Hreq_ce"][y]))

    for m in M:
        vals.append(abs(theta_a["xbar_ce"][m] - theta_b["xbar_ce"][m]))

    return max(vals) if vals else 0.0

def diff_dict(a, b):
    keys = set(a.keys()) | set(b.keys())
    return max(
        [abs(float(a.get(k,0.0)) - float(b.get(k,0.0))) for k in keys],
        default=0.0
    )


# ============================================================
# 7) SUBPROBLEMAS PADM-W
# ============================================================

def add_z_variables(model):
    z = {}

    for g in G:
        for t in T:
            for y in Y:
                z[key_p(g,t,y)] = model.addVar(lb=0.0, name=f"p[{g},{t},{y}]")

    for g in GS:
        for t in T:
            for y in Y:
                z[key_u(g,t,y)] = model.addVar(lb=0.0, ub=1.0, name=f"u[{g},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_s(i,t,y)] = model.addVar(lb=0.0, name=f"s[{i},{t},{y}]")

    for m in M:
        for y in Y:
            z[key_xcap(m,y)] = model.addVar(lb=0.0, name=f"xcap[{m},{y}]")

    for m in M:
        for t in T:
            for y in Y:
                z[key_hsyn(m,t,y)] = model.addVar(lb=0.0, name=f"h_syn[{m},{t},{y}]")
                z[key_rffr(m,t,y)] = model.addVar(lb=0.0, name=f"r_ffr[{m},{t},{y}]")

    for (i,j) in L_arcs:
        for t in T:
            for y in Y:
                z[key_fpos(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_pos[{i},{j},{t},{y}]")
                z[key_fneg(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_neg[{i},{j},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_thp(i,t,y)] = model.addVar(lb=0.0, name=f"theta_p[{i},{t},{y}]")
                z[key_thm(i,t,y)] = model.addVar(lb=0.0, name=f"theta_m[{i},{t},{y}]")

    return z

def add_dual_variables(model):
    yd = {}

    for name in row_order:
        ub_here = GRB.INFINITY

        if name.startswith("sys_H_"):
            ub_here = UB_H
        elif name.startswith("x_ub_"):
            ub_here = UB_xbar

        yd[name] = model.addVar(lb=0.0, ub=ub_here, name=f"ydual[{name}]")

    return yd

def add_theta_variables(model, warm_theta=None):
    theta = {
        "Hreq_ce": {},
        "xbar_ce": {},
    }

    for y in Y:
        theta["Hreq_ce"][y] = model.addVar(
            lb=MUT_LB_FACTOR * float(Hreq_base[y]),
            ub=MUT_UB_FACTOR * float(Hreq_base[y]),
            name=f"Hreq_ce[{y}]"
        )

    for m in M:
        theta["xbar_ce"][m] = model.addVar(
            lb=MUT_LB_FACTOR * float(xbar_base[m]),
            ub=MUT_UB_FACTOR * float(xbar_base[m]),
            name=f"xbar_ce[{m}]"
        )

    if warm_theta is not None:
        for y in Y:
            theta["Hreq_ce"][y].Start = float(warm_theta["Hreq_ce"][y])
        for m in M:
            theta["xbar_ce"][m].Start = float(warm_theta["xbar_ce"][m])

    return theta

def extract_theta(theta_vars):
    return {
        "Hreq_ce": {y: float(theta_vars["Hreq_ce"][y].X) for y in Y},
        "xbar_ce": {m: float(theta_vars["xbar_ce"][m].X) for m in M},
    }

def add_separable_primal_constraints_y(model, z_vars):
    theta_base = initial_theta_vals()

    for name in separable_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z_vars, theta_base)
        rhs = rhs_eval(rows[name]["rhs"], theta_base)
        model.addConstr(lhs >= rhs, name=f"Y_primal[{name}]")

def add_D_constraint_y(model, z_vars):
    for y in Y:
        if y >= y_star:
            model.addConstr(
                gp.quicksum(
                    float(eg[g]) * z_vars[key_p(g,t,y)]
                    for g in GE
                    for t in T
                )
                <= float(eps_CO2(y)),
                name=f"D_CO2_cap_{y}"
            )

def add_separable_dual_constraints_y(model, ydual_vars):
    theta_base = initial_theta_vals()

    for k in separable_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_base) * ydual_vars[name]

        model.addConstr(lhs <= float(c_map.get(k, 0.0)), name=f"Y_dual[{k}]")

def add_strong_duality_y(model, theta_vals, z_vars, ydual_vars):
    ctx = gp.quicksum(float(c_map[k]) * z_vars[k] for k in primal_keys)

    bty = gp.LinExpr()

    for name in row_order:
        bty += rhs_eval(rows[name]["rhs"], theta_vals) * ydual_vars[name]

    model.addConstr(ctx <= bty, name="Y_StrongDuality")

def solve_theta_subproblem(
    z_vals,
    ydual_vals,
    rho,
    mu,
    warm_theta=None,
    log=False,
    use_true_objective=True
):
    m = gp.Model("PADM_theta_subproblem_IEEE39_Hreq_Xbar")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    theta = add_theta_variables(m, warm_theta=warm_theta)

    obj = gp.LinExpr()

    if use_true_objective:
        for y in Y:
            dvar = m.addVar(lb=0.0, name=f"dHreq[{y}]")
            diff = theta["Hreq_ce"][y] - float(Hreq_base[y])

            m.addConstr(dvar >= diff, name=f"dHreq_pos[{y}]")
            m.addConstr(dvar >= -diff, name=f"dHreq_neg[{y}]")

            base = abs(float(Hreq_base[y]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_H) * dvar / base
            else:
                obj += float(alpha_H) * dvar

        for mm in M:
            dvar = m.addVar(lb=0.0, name=f"dXbar[{mm}]")
            diff = theta["xbar_ce"][mm] - float(xbar_base[mm])

            m.addConstr(dvar >= diff, name=f"dXbar_pos[{mm}]")
            m.addConstr(dvar >= -diff, name=f"dXbar_neg[{mm}]")

            base = abs(float(xbar_base[mm]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_Xbar) * dvar / base
            else:
                obj += float(alpha_Xbar) * dvar

    for name in coupled_primal_rows:
        lhs = terms_expr_theta_model_with_fixed_z(rows[name]["terms"], z_vals, theta)
        rhs = rhs_expr_in_theta_model(rows[name]["rhs"], theta)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            ydval = float(ydual_vals.get(name, 0.0))

            if abs(ydval) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += ydval * coeff_expr_in_theta_model(c, theta)

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    # Dualidad fuerte dura en theta:
    #   c^T z fijo <= b(theta)^T y fijo
    ctx_fixed = sum(float(c_map[k]) * float(z_vals.get(k, 0.0)) for k in primal_keys)

    bty_theta = gp.LinExpr()

    for name in row_order:
        ydval = float(ydual_vals.get(name, 0.0))

        if abs(ydval) <= 1e-12:
            continue

        bty_theta += ydval * rhs_expr_in_theta_model(rows[name]["rhs"], theta)

    # En este caso theta = (Hreq_ce, xbar_ce) modifica el RHS b.
    # Por eso b(theta)^T lambda sí depende de theta, y la dualidad fuerte
    # puede mantenerse como restricción dura en el subproblema theta.
    m.addConstr(ctx_fixed <= bty_theta, name="Theta_StrongDuality")

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema THETA falló.")
        print(f"Status = {m.Status}")
        return None, m.Status, None

    theta_vals = extract_theta(theta)
    return theta_vals, m.Status, float(m.ObjVal)

def solve_y_subproblem(theta_vals, rho, mu, warm_z=None, warm_ydual=None, log=False):
    m = gp.Model("PADM_y_subproblem_IEEE39_Hreq_Xbar")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    z = add_z_variables(m)
    yd = add_dual_variables(m)

    m.update()

    if warm_z is not None:
        for k, v in z.items():
            if k in warm_z:
                v.Start = float(warm_z[k])

    if warm_ydual is not None:
        for name, v in yd.items():
            if name in warm_ydual:
                v.Start = float(warm_ydual[name])

    add_separable_primal_constraints_y(m, z)
    add_D_constraint_y(m, z)
    add_separable_dual_constraints_y(m, yd)
    add_strong_duality_y(m, theta_vals, z, yd)

    obj = gp.LinExpr()

    for name in coupled_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd[name]

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    if Y_OBJECTIVE_MODE == "ctx":
        obj += TIEBREAKER_CTX_WEIGHT * gp.quicksum(
            float(c_map[k]) * z[k]
            for k in primal_keys
        )

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema Y falló.")
        print(f"Status = {m.Status}")
        return None, None, m.Status, None

    z_vals = {k: float(v.X) for k, v in z.items()}
    yd_vals = {name: float(v.X) for name, v in yd.items()}

    return z_vals, yd_vals, m.Status, float(m.ObjVal)

def initialize_penalties():
    rho = {name: float(RHO0) for name in coupled_primal_rows}
    mu = {k: float(MU0) for k in coupled_dual_cols}
    return rho, mu

def update_penalties(theta_vals, z_vals, ydual_vals, rho, mu):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    n_rho = 0
    n_mu = 0

    for name in coupled_primal_rows:
        if pr[name] > FEAS_TOL:
            rho[name] = min(MAX_PENALTY, PENALTY_FACTOR * rho[name])
            n_rho += 1

    for k in coupled_dual_cols:
        if dr[k] > FEAS_TOL:
            mu[k] = min(MAX_PENALTY, PENALTY_FACTOR * mu[k])
            n_mu += 1

    return n_rho, n_mu


# ============================================================
# 8) LOOP PADM-W
# ============================================================

def run_padm(
    use_true_objective=True,
    theta_init=None,
    z_init=None,
    ydual_init=None,
    label="PADM"
):
    print("\n" + "="*70)
    print(f"INICIO {label}")
    print("="*70)
    print(f"use_true_objective = {use_true_objective}")

    rho, mu = initialize_penalties()

    theta_vals = copy.deepcopy(theta_init) if theta_init is not None else initial_theta_vals()

    if z_init is None or ydual_init is None:
        print("\nInicializando Y=(z,lambda)...")

        z_vals, ydual_vals, st_y, obj_y = solve_y_subproblem(
            theta_vals,
            rho,
            mu,
            warm_z=None,
            warm_ydual=None,
            log=False
        )

        if z_vals is None:
            print("❌ No se pudo inicializar Y.")

            return {
                "best_feasible": None,
                "theta": theta_vals,
                "z": None,
                "ydual": None,
                "history": [],
                "last_residuals": None,
                "rho": rho,
                "mu": mu,
            }

    else:
        z_vals = z_init.copy()
        ydual_vals = ydual_init.copy()

    history = []
    best_feasible = None

    res0 = max_residuals(theta_vals, z_vals, ydual_vals)

    print("\nChequeo inicial:")
    print(f"J(theta)       = {compute_J(theta_vals):.8f}")
    print(f"Max primal     = {res0['max_primal_all']:.3e}")
    print(f"Max dual       = {res0['max_dual_all']:.3e}")
    print(f"Strong duality = {res0['sd']:.3e}")

    for k_outer in range(MAX_OUTER):

        print("\n" + "#"*70)
        print(f"{label} | OUTER k = {k_outer}")
        print("#"*70)

        phi_prev = compute_phi(
            theta_vals,
            z_vals,
            ydual_vals,
            rho,
            mu,
            use_true_objective=use_true_objective
        )

        for l_inner in range(MAX_INNER):

            theta_old = copy.deepcopy(theta_vals)
            z_old = z_vals.copy()
            yd_old = ydual_vals.copy()

            theta_new, st_theta, obj_theta = solve_theta_subproblem(
                z_vals,
                ydual_vals,
                rho,
                mu,
                warm_theta=theta_vals,
                log=False,
                use_true_objective=use_true_objective
            )

            if theta_new is None:
                print("⚠️ Falló subproblema theta.")
                break

            theta_vals = theta_new

            z_new, yd_new, st_y, obj_y = solve_y_subproblem(
                theta_vals,
                rho,
                mu,
                warm_z=z_vals,
                warm_ydual=ydual_vals,
                log=False
            )

            if z_new is None:
                print("⚠️ Falló subproblema Y.")
                break

            z_vals = z_new
            ydual_vals = yd_new

            phi_now = compute_phi(
                theta_vals,
                z_vals,
                ydual_vals,
                rho,
                mu,
                use_true_objective=use_true_objective
            )

            res = max_residuals(theta_vals, z_vals, ydual_vals)

            dtheta = diff_theta(theta_vals, theta_old)
            dz = diff_dict(z_vals, z_old)
            dyd = diff_dict(ydual_vals, yd_old)
            dphi = abs(phi_now - phi_prev) / max(1.0, abs(phi_prev))

            if PADM_OUTPUT:
                print(
                    f"k={k_outer:02d}, l={l_inner:02d} | "
                    f"Phi={phi_now:.6e}, J={compute_J(theta_vals):.6e}, "
                    f"p_all={res['max_primal_all']:.2e}, "
                    f"d_all={res['max_dual_all']:.2e}, "
                    f"sd={res['sd']:.2e}, "
                    f"p_coup={res['max_primal_coupled']:.2e}, "
                    f"d_coup={res['max_dual_coupled']:.2e}, "
                    f"dtheta={dtheta:.2e}, dz={dz:.2e}, dyd={dyd:.2e}, dphi={dphi:.2e}"
                )

            history.append({
                "outer": k_outer,
                "inner": l_inner,
                "phi": phi_now,
                "J": compute_J(theta_vals),
                "max_primal_all": res["max_primal_all"],
                "max_dual_all": res["max_dual_all"],
                "sd": res["sd"],
                "max_primal_coupled": res["max_primal_coupled"],
                "max_dual_coupled": res["max_dual_coupled"],
                "dtheta": dtheta,
                "dz": dz,
                "dyd": dyd,
                "dphi": dphi,
                "rho_max": max(rho.values()) if rho else 0.0,
                "mu_max": max(mu.values()) if mu else 0.0,
            })

            if dphi <= INNER_TOL and dtheta <= INNER_TOL and dz <= INNER_TOL and dyd <= INNER_TOL:
                print(f"Inner loop converge en l={l_inner}.")
                break

            phi_prev = phi_now

        res = max_residuals(theta_vals, z_vals, ydual_vals)

        feasible = (
            res["max_primal_all"] <= FEAS_TOL
            and res["max_dual_all"] <= FEAS_TOL
            and res["sd"] <= FEAS_TOL
        )

        print("\n" + "-"*70)
        print(f"{label} | Chequeo fin outer k={k_outer}")
        print("-"*70)
        print(f"J(theta)             = {compute_J(theta_vals):.8f}")
        print(f"Max primal all       = {res['max_primal_all']:.3e}")
        print(f"Max dual all         = {res['max_dual_all']:.3e}")
        print(f"Strong duality       = {res['sd']:.3e}")
        print(f"Max primal coupled   = {res['max_primal_coupled']:.3e}")
        print(f"Max dual coupled     = {res['max_dual_coupled']:.3e}")
        print(f"rho_max              = {max(rho.values()) if rho else 0.0:.3e}")
        print(f"mu_max               = {max(mu.values()) if mu else 0.0:.3e}")

        if feasible:
            print(f"\n✅ {label} encontró solución factible primal-dual.")

            best_feasible = {
                "theta": copy.deepcopy(theta_vals),
                "z": z_vals.copy(),
                "ydual": ydual_vals.copy(),
                "J": compute_J(theta_vals),
                "residuals": res,
                "outer": k_outer,
                "history": history,
            }

            break

        n_rho, n_mu = update_penalties(theta_vals, z_vals, ydual_vals, rho, mu)

        print("\nActualización penalizaciones:")
        print(f"rho actualizadas: {n_rho}")
        print(f"mu actualizadas : {n_mu}")

    if best_feasible is None:
        print(f"\n⚠️ {label} terminó sin certificar factibilidad.")

    return {
        "best_feasible": best_feasible,
        "theta": theta_vals,
        "z": z_vals,
        "ydual": ydual_vals,
        "history": history,
        "last_residuals": max_residuals(theta_vals, z_vals, ydual_vals) if z_vals is not None else None,
        "rho": rho,
        "mu": mu,
    }


# ============================================================
# 9) EJECUTAR PADM-W
# ============================================================

PADM_WARMSTART = None

if RUN_PADM_W:

    res_ws = run_padm(
        use_true_objective=False,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM-W ETAPA 1: FACTIBILIDAD"
    )

    if res_ws["best_feasible"] is not None:

        theta_start = res_ws["best_feasible"]["theta"]
        z_start = res_ws["best_feasible"]["z"]
        yd_start = res_ws["best_feasible"]["ydual"]

        if RUN_PADM_STAGE_2:
            print("\n" + "="*70)
            print("WARMSTART FACTIBLE ENCONTRADO. INICIANDO ETAPA 2 CON J REAL.")
            print("="*70)

            res_final = run_padm(
                use_true_objective=True,
                theta_init=theta_start,
                z_init=z_start,
                ydual_init=yd_start,
                label="PADM-W ETAPA 2: OBJETIVO REAL"
            )

            if res_final["best_feasible"] is not None:
                final_solution = res_final["best_feasible"]
                final_label = "PADM-W ETAPA 2"
            else:
                print("\n⚠️ Etapa 2 no certificó factibilidad. Se conserva el punto factible de etapa 1.")
                final_solution = res_ws["best_feasible"]
                final_label = "PADM-W ETAPA 1"
        else:
            print("\n✅ Etapa 2 desactivada. Se usará directamente el punto factible de etapa 1.")
            res_final = res_ws
            final_solution = res_ws["best_feasible"]
            final_label = "PADM-W ETAPA 1"

    else:
        print("\n⚠️ Etapa 1 no encontró punto factible.")
        final_solution = None
        final_label = "PADM-W FAILED"
        res_final = res_ws

else:
    res_final = run_padm(
        use_true_objective=True,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM"
    )

    if res_final["best_feasible"] is not None:
        final_solution = res_final["best_feasible"]
        final_label = "PADM"
    else:
        final_solution = None
        final_label = "PADM FAILED"

print("\n" + "="*70)
print("RESULTADOS FINALES PADM-W IEEE39")
print("="*70)

if final_solution is not None:
    sol_theta = final_solution["theta"]
    sol_z = final_solution["z"]
    sol_yd = final_solution["ydual"]

    print("✅ SOLUCIÓN FACTIBLE CERTIFICADA")
    print(f"Método final     : {final_label}")
    print(f"Outer encontrado : {final_solution['outer']}")
    print(f"J                : {final_solution['J']:.8f}")

    PADM_WARMSTART = {
        "theta": copy.deepcopy(sol_theta),
        "z": sol_z.copy(),
        "ydual": sol_yd.copy(),
        "label": final_label,
        "J": final_solution["J"],
        "residuals": final_solution["residuals"],
    }

else:
    print("⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.")

    sol_theta = res_final["theta"]
    sol_z = res_final["z"]
    sol_yd = res_final["ydual"]

    if USE_LAST_PADM_POINT_IF_NO_FEASIBLE and sol_z is not None and sol_yd is not None:
        res = max_residuals(sol_theta, sol_z, sol_yd)

        PADM_WARMSTART = {
            "theta": copy.deepcopy(sol_theta),
            "z": sol_z.copy(),
            "ydual": sol_yd.copy(),
            "label": "PADM_LAST_POINT_NOT_CERTIFIED",
            "J": compute_J(sol_theta),
            "residuals": res,
        }

        print("⚠️ Se guardó el último punto de PADM-W como Start no certificado.")
    else:
        PADM_WARMSTART = None
        print("❌ No hay punto PADM-W disponible para warm start.")

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("CHECK PADM_WARMSTART IEEE39")
    print("="*70)

    sol_theta = PADM_WARMSTART["theta"]
    sol_z = PADM_WARMSTART["z"]
    sol_yd = PADM_WARMSTART["ydual"]

    res_check = max_residuals(sol_theta, sol_z, sol_yd)

    print(f"Origen          = {PADM_WARMSTART['label']}")
    print(f"J(theta)        = {PADM_WARMSTART['J']:.8f}")
    print(f"Max primal      = {res_check['max_primal_all']:.6e}")
    print(f"Max dual        = {res_check['max_dual_all']:.6e}")
    print(f"Strong duality  = {res_check['sd']:.6e}")


# ============================================================
# 10) WCEP FINAL CON DUALIDAD FUERTE Y WARM START PADM-W
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE39_SOLO_Hreq_Xbar_PADMW_START_DF")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Hreq_ce = w.addVars(Y, lb=0.0, name="Hreq_ce")
xbar_ce = w.addVars(M, lb=0.0, name="xbar_ce")

for y in Y:
    base = Hreq_base[y]
    w.addConstr(Hreq_ce[y] >= MUT_LB_FACTOR * base, name=f"adm_lb_Hreq[{y}]")
    w.addConstr(Hreq_ce[y] <= MUT_UB_FACTOR * base, name=f"adm_ub_Hreq[{y}]")

for m in M:
    base = xbar_base[m]
    w.addConstr(xbar_ce[m] >= MUT_LB_FACTOR * base, name=f"adm_lb_xbar[{m}]")
    w.addConstr(xbar_ce[m] <= MUT_UB_FACTOR * base, name=f"adm_ub_xbar[{m}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(Pmax[g]) * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -xbar_ce[m], name=name)
        RHS_expr[name] = -xbar_ce[m]

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= Hreq_ce[y], name=name)
        RHS_expr[name] = Hreq_ce[y]

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= eps_CO2(y),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sys_H_"):
        ub_here = UB_H
    elif name.startswith("x_ub_"):
        ub_here = UB_xbar

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dHreq = w.addVars(Y, lb=0.0, name="dHreq")
dXbar = w.addVars(M, lb=0.0, name="dXbar")

for y in Y:
    base = Hreq_base[y]
    diff = Hreq_ce[y] - base

    w.addConstr(dHreq[y] >= diff, name=f"dHreq_pos[{y}]")
    w.addConstr(dHreq[y] >= -diff, name=f"dHreq_neg[{y}]")

for m in M:
    base = xbar_base[m]
    diff = xbar_ce[m] - base

    w.addConstr(dXbar[m] >= diff, name=f"dXbar_pos[{m}]")
    w.addConstr(dXbar[m] >= -diff, name=f"dXbar_neg[{m}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for y in Y:
        base = abs(float(Hreq_base[y]))
        if base > 1e-9:
            obj_wcep += alpha_H * dHreq[y] / base
        else:
            obj_wcep += alpha_H * dHreq[y]

    for m in M:
        base = abs(float(xbar_base[m]))
        if base > 1e-9:
            obj_wcep += alpha_Xbar * dXbar[m] / base
        else:
            obj_wcep += alpha_Xbar * dXbar[m]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_H * dHreq.sum()
        + alpha_Xbar * dXbar.sum(),
        GRB.MINIMIZE
    )


# ============================================================
# 11) APLICAR WARM START DEL PADM-W
# ============================================================

def safe_start(var, value):
    try:
        if value is not None and np.isfinite(float(value)):
            var.Start = float(value)
    except Exception:
        pass

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("APLICANDO PADM_WARMSTART AL WCEP FINAL IEEE39")
    print("="*70)
    print("Origen:", PADM_WARMSTART.get("label", "PADM"))

    ws_theta = PADM_WARMSTART["theta"]
    ws_z = PADM_WARMSTART["z"]
    ws_yd = PADM_WARMSTART["ydual"]

    for y in Y:
        safe_start(Hreq_ce[y], ws_theta["Hreq_ce"][y])
        safe_start(dHreq[y], abs(ws_theta["Hreq_ce"][y] - Hreq_base[y]))

    for m in M:
        safe_start(xbar_ce[m], ws_theta["xbar_ce"][m])
        safe_start(dXbar[m], abs(ws_theta["xbar_ce"][m] - xbar_base[m]))

    for g in G:
        for t in T:
            for y in Y:
                safe_start(p[g, t, y], ws_z.get(("p", g, t, y), None))

    for g in GS:
        for t in T:
            for y in Y:
                safe_start(u[g, t, y], ws_z.get(("u", g, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(s[i, t, y], ws_z.get(("s", i, t, y), None))

    for m in M:
        for y in Y:
            safe_start(xcap[m, y], ws_z.get(("xcap", m, y), None))

    for m in M:
        for t in T:
            for y in Y:
                safe_start(h_syn[m, t, y], ws_z.get(("h_syn", m, t, y), None))
                safe_start(r_ffr[m, t, y], ws_z.get(("r_ffr", m, t, y), None))

    for (i, j) in L_arcs:
        for t in T:
            for y in Y:
                safe_start(f_pos[i, j, t, y], ws_z.get(("f_pos", i, j, t, y), None))
                safe_start(f_neg[i, j, t, y], ws_z.get(("f_neg", i, j, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(theta_p[i, t, y], ws_z.get(("theta_p", i, t, y), None))
                safe_start(theta_m[i, t, y], ws_z.get(("theta_m", i, t, y), None))

    for name, var in ydual.items():
        safe_start(var, ws_yd.get(name, None))

    print("✅ Warm start aplicado al WCEP IEEE39.")
else:
    print("\n⚠️ No existe PADM_WARMSTART. El WCEP se resolverá sin Start.")


# ============================================================
# 12) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE39")
print("="*70)

w.optimize()


# ============================================================
# 13) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE39")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

# ============================================================
# REGISTRO DE OPTIMIZACIÓN WCEP FINAL
# ============================================================
print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP FINAL")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo WCEP optimize() = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }
else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }

df_registro_wcep = pd.DataFrame([registro_wcep])
display(df_registro_wcep)

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Hreq_ce ---")
    hubo_cambio = False

    for y in Y:
        val = float(Hreq_ce[y].X)
        base = float(Hreq_base[y])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{y}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- xbar_ce ---")
    hubo_cambio = False

    for m in M:
        val = float(xbar_ce[m].X)
        base = float(xbar_base[m])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{m}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = eps_CO2(y) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {eps_CO2(y):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_ce[y].X)

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_ce={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_xbar = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("x_ub_"):
            if abs(var.X - UB_xbar) <= 1e-6 * max(1.0, UB_xbar):
                hits_xbar += 1

    print(f"Duales sys_H pegadas a cota   : {hits_H}")
    print(f"Duales x_ub pegadas a cota    : {hits_xbar}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# TIEMPO TOTAL SCRIPT
# ============================================================
t_total_script = time.time() - t0_total_script
print("\n" + "="*70)
print("TIEMPO TOTAL DE CELDA / SCRIPT")
print("="*70)
print(f"Tiempo total script = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")


✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en PADM-W/WCEP = 1289.386000
Pmax total sistema                 = 1315.700000
Caso original de 4 periodos: sin pesos temporales omega_t.

COTAS DUALES IEEE39
UB_H    = 324200000.000000
UB_xbar = 3242000.000000
Bus slack usado: 1
Parámetros mutables:
  Hreq_ce[y] en [0.85, 1.15] * Hreq[y]
  xbar_ce[m] en [0.85, 1.15] * xbar[m]
Pmax queda fijo en su valor original.
Bus slack usado: 1

CLASIFICACIÓN PADM-W IEEE39 - FORMULACIÓN ORIGINAL
N° filas primales acopladas   : 36
N° filas primales separables  : 4128
N° columnas duales acopladas  : 0
N° columnas duales separables : 2916

✅ Filas acopladas esperadas: sys_H_ y x_ub_.

INICIO PADM-W ETAPA 1: FACTIBILIDAD
use_true_objective = False

Inicializando Y=(z,lambda)...

⚠️ Subproblema Y falló.
Status = 3
❌ No se pudo inicializar Y.

⚠️ Etapa 1 no encontró punto factible.

RESULTADOS FINALES PADM-W IEEE39
⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.
❌ No hay punto PADM-W disponible

,status,runtime_seconds,runtime_minutes,runtime_hours,sol_count,obj_val,obj_bound,mip_gap,mip_gap_percent
0,3,0.244,0.004067,0.000068,0,None,None,None,None



No hay solución WCEP disponible para reportar.

TIEMPO TOTAL DE CELDA / SCRIPT
Tiempo total script = 17.02 s = 0.28 min = 0.00 h


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import copy
import math
import time

t0_total_script = time.time()

# ============================================================
# FINAL IEEE39: PADM-W + WCEP CON DUALIDAD FUERTE
# FORMULACIÓN ORIGINAL + DATASET CORREGIDO:
#
#   sum_g Hg[g] u[g,t,y] + sum_m h_syn[m,t,y] >= Hreq_ce[y]
#
# Es decir:
#   - NO se usa zH.
#   - Hreq_ce queda en b.
#   - Pmax fijo; NO se muta Pmax.
#   - xbar_ce queda en b.
#
# Flujo:
#   1) PADM-W etapa 1: busca factibilidad.
#   2) PADM-W etapa 2: parte de factibilidad y reduce J(theta).
#   3) WCEP completo con dualidad fuerte usando PADM_WARMSTART.
# ============================================================

# ============================================================
# 0) OPCIONES
# ============================================================

PADM_OUTPUT = 1
RUN_PADM_W = True

MAX_OUTER = 100
MAX_INNER = 100

FEAS_TOL = 1e-5
INNER_TOL = 1e-5

RHO0 = 500.0
MU0 = 500.0

PENALTY_FACTOR = 2.0
MAX_PENALTY = 1e9

TIME_LIMIT_SUBPROBLEM = 5 * 60 * 60  # 5 horas por subproblema PADM
WCEP_TIME_LIMIT = 5 * 60 * 60        # 5 horas para WCEP final

# Restricción D del IEEE39
# Si ya calculaste en el FW emisiones ponderadas, puedes poner epsilon_CO2_wcep manual.
# Si lo dejas como None, el código intenta usar el 90% de las emisiones FW guardadas en memoria.
epsilon_CO2_wcep = 930756.144528 * 0.95
EPSILON_CO2_FRACTION = 0.90

# Objetivo J
alpha_H = 2.0
alpha_Xbar = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1


# ============================================================
# 1) PREPARACIÓN GENERAL
# ============================================================

L_arcs = gp.tuplelist(L)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [arc for arc in L_arcs if arc[1] == bus] for bus in B}
lineas_salen  = {bus: [arc for arc in L_arcs if arc[0] == bus] for bus in B}

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE39
# ============================================================
# Este bloque hace que PADM-W y WCEP usen la misma demanda corregida
# del Forward nuevo. Si ya existe d_fw desde la celda FW, se respeta.
# Si no existe, se reconstruye automáticamente con el mismo criterio:
# demanda escalada para que la demanda pico quede dentro de Pmax total.

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 0.713))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

# Diagnóstico rápido de demanda usada por PADM-W/WCEP
peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)
cap_total_pmax = sum(float(Pmax[g]) for g in G)
print(f"Demanda pico usada en PADM-W/WCEP = {peak_dem_fw:.6f}")
print(f"Pmax total sistema                 = {cap_total_pmax:.6f}")

# ============================================================
# PESOS TEMPORALES PARA REPRESENTAR EL AÑO
# ============================================================
# Si T = [1, 7, 13, 19] representa un perfil diario, usamos omega_t
# para expandir costos y emisiones a base anual sin crear 365*|T| periodos.
# Por defecto cada t se repite 365 veces y representa 1 hora.
# Si cada t representa un bloque de 6 horas, define TIME_BLOCK_HOURS = 6.0 antes de correr.
TIME_BLOCK_HOURS = float(globals().get("TIME_BLOCK_HOURS", 1.0))
N_DAYS_EQUIV = int(globals().get("N_DAYS_EQUIV", 365))

if "omega_t" not in globals():
    omega_t = {t: float(N_DAYS_EQUIV) * TIME_BLOCK_HOURS for t in T}
else:
    omega_t = {t: float(omega_t[t]) for t in T}

def omega(t):
    return float(omega_t[t])

print("Pesos temporales omega_t usados:", omega_t)

bus_slack = int(sorted(B)[0])

Hreq_base = {y: float(Hreq[y]) for y in Y}
xbar_base = {m: float(xbar[m]) for m in M}

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_xbar = 2.0 * max(float(cinv[m]) for m in M) if len(M) > 0 else 1e8

# ============================================================
# Límite de emisiones D
# ============================================================
def _get_fw_emissions_by_year():
    candidates = [
        "emisiones_por_anio", "df_emis_fw", "df_emis", "df_emis_wcep"
    ]
    value_cols = [
        "emisiones_anuales_equivalentes",
        "emisiones_totales_ponderadas",
        "emisiones_totales_anuales",
        "emisiones",
    ]
    for name in candidates:
        if name in globals():
            df = globals()[name]
            if hasattr(df, "columns") and "y" in df.columns:
                for col in value_cols:
                    if col in df.columns:
                        return {int(row["y"]): float(row[col]) for _, row in df.iterrows()}
    return None

_fw_emis = _get_fw_emissions_by_year()

if epsilon_CO2_wcep is None:
    if _fw_emis is not None:
        epsilon_CO2_wcep_by_y = {y: EPSILON_CO2_FRACTION * float(_fw_emis[y]) for y in Y if y in _fw_emis}
        print("epsilon_CO2_wcep calculado como 90% de emisiones FW:", epsilon_CO2_wcep_by_y)
    else:
        raise ValueError(
            "epsilon_CO2_wcep=None, pero no encontré emisiones del FW. "
            "Define epsilon_CO2_wcep manualmente o ejecuta antes el FW con tabla de emisiones."
        )
elif isinstance(epsilon_CO2_wcep, dict):
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep[y]) for y in epsilon_CO2_wcep}
else:
    epsilon_CO2_wcep_by_y = {y: float(epsilon_CO2_wcep) for y in Y}

def eps_CO2(y):
    return float(epsilon_CO2_wcep_by_y[int(y)])


print("\n" + "="*70)
print("COTAS DUALES IEEE39")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_xbar = {UB_xbar:.6f}")
print(f"Bus slack usado: {bus_slack}")


# ============================================================
# 2) REPRESENTACIÓN SIMBÓLICA PARA PADM-W
# ============================================================

def key_p(g,t,y):       return ("p", g, t, y)
def key_u(g,t,y):       return ("u", g, t, y)
def key_s(i,t,y):       return ("s", i, t, y)
def key_xcap(m,y):      return ("xcap", m, y)
def key_hsyn(m,t,y):    return ("h_syn", m, t, y)
def key_rffr(m,t,y):    return ("r_ffr", m, t, y)
def key_fpos(i,j,t,y):  return ("f_pos", i, j, t, y)
def key_fneg(i,j,t,y):  return ("f_neg", i, j, t, y)
def key_thp(i,t,y):     return ("theta_p", i, t, y)
def key_thm(i,t,y):     return ("theta_m", i, t, y)

def param_coeff(family, idx, mult=1.0):
    return ("PARAM", family, idx, float(mult))

def is_param_coeff(c):
    return isinstance(c, tuple) and len(c) == 4 and c[0] == "PARAM"

def coeff_value(c, theta_vals):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * float(theta_vals[fam][idx])
    return float(c)

def coeff_expr_in_theta_model(c, theta_vars):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * theta_vars[fam][idx]
    return float(c)

def rhs_const(val):
    return [("CONST", None, None, float(val))]

def rhs_param(family, idx, mult=1.0):
    return [("PARAM", family, idx, float(mult))]

def rhs_has_param(rhs_terms):
    return any(typ == "PARAM" for typ, fam, idx, mult in rhs_terms)

def rhs_eval(rhs_terms, theta_vals=None):
    out = 0.0

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            out += float(mult)
        elif typ == "PARAM":
            if theta_vals is None:
                raise ValueError("rhs_eval requiere theta_vals para RHS con parámetros.")
            out += float(mult) * float(theta_vals[fam][idx])
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return out

def rhs_expr_in_theta_model(rhs_terms, theta_vars):
    expr = gp.LinExpr()

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            expr += float(mult)
        elif typ == "PARAM":
            expr += float(mult) * theta_vars[fam][idx]
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return expr

def add_to_terms(terms, key, coeff):
    if key not in terms:
        terms[key] = []
    terms[key].append(coeff)

def neg_coeff(c):
    if is_param_coeff(c):
        return ("PARAM", c[1], c[2], -c[3])
    return -float(c)

def terms_has_param(terms):
    for k, coeffs in terms.items():
        for c in coeffs:
            if is_param_coeff(c):
                return True
    return False

def terms_expr_z_model(terms, z_vars, theta_vals):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        for c in coeffs:
            expr += coeff_value(c, theta_vals) * z_vars[k]

    return expr

def terms_expr_theta_model_with_fixed_z(terms, z_vals, theta_vars):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            expr += zval * coeff_expr_in_theta_model(c, theta_vars)

    return expr

def terms_eval(terms, z_vals, theta_vals):
    out = 0.0

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            out += coeff_value(c, theta_vals) * zval

    return out


# ============================================================
# 3) CREAR FILAS PRIMALES Ax >= b
# ============================================================

rows = {}
row_order = []

def add_row(name, terms, rhs_terms):
    if name in rows:
        raise ValueError(f"Fila repetida: {name}")

    rows[name] = {
        "terms": terms,
        "rhs": rhs_terms
    }

    row_order.append(name)

# ------------------------------------------------------------
# A.1 Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}

            for g in G_en_bus.get(i, []):
                add_to_terms(terms, key_p(g,t,y), 1.0)

            add_to_terms(terms, key_s(i,t,y), 1.0)

            for (k,j) in lineas_entran[i]:
                add_to_terms(terms, key_fpos(k,j,t,y), 1.0)
                add_to_terms(terms, key_fneg(k,j,t,y), -1.0)

            for (i2,j) in lineas_salen[i]:
                add_to_terms(terms, key_fpos(i2,j,t,y), -1.0)
                add_to_terms(terms, key_fneg(i2,j,t,y), 1.0)

            rhs_val = float(d_fw[(i,t,y)])

            add_row(f"bal_p_{i}_{t}_{y}", terms, rhs_const(rhs_val))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"bal_n_{i}_{t}_{y}", terms_n, rhs_const(-rhs_val))

# ------------------------------------------------------------
# A.2 ENS <= demanda
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}
            add_to_terms(terms, key_s(i,t,y), -1.0)
            add_row(f"ens_ub_{i}_{t}_{y}", terms, rhs_const(-float(d_fw[(i,t,y)])))

# ------------------------------------------------------------
# A.3 Flujo DC
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i,j)])

            terms = {}
            add_to_terms(terms, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms, key_fneg(i,j,t,y), -1.0)

            add_to_terms(terms, key_thp(i,t,y), -Bij)
            add_to_terms(terms, key_thm(i,t,y),  Bij)
            add_to_terms(terms, key_thp(j,t,y),  Bij)
            add_to_terms(terms, key_thm(j,t,y), -Bij)

            add_row(f"dc_p_{i}_{j}_{t}_{y}", terms, rhs_const(0.0))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"dc_n_{i}_{j}_{t}_{y}", terms_n, rhs_const(0.0))

# ------------------------------------------------------------
# A.4 Límites térmicos fijos
# ------------------------------------------------------------
for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i,j)])

            terms_lb = {}
            add_to_terms(terms_lb, key_fpos(i,j,t,y), 1.0)
            add_to_terms(terms_lb, key_fneg(i,j,t,y), -1.0)
            add_row(f"flb_{i}_{j}_{t}_{y}", terms_lb, rhs_const(-Fij))

            terms_ub = {}
            add_to_terms(terms_ub, key_fpos(i,j,t,y), -1.0)
            add_to_terms(terms_ub, key_fneg(i,j,t,y), 1.0)
            add_row(f"fub_{i}_{j}_{t}_{y}", terms_ub, rhs_const(-Fij))

# ------------------------------------------------------------
# A.5 Slack bus
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_p = {}
        add_to_terms(terms_p, key_thp(bus_slack,t,y), 1.0)
        add_to_terms(terms_p, key_thm(bus_slack,t,y), -1.0)
        add_row(f"slack_p_{t}_{y}", terms_p, rhs_const(0.0))

        terms_n = {}
        add_to_terms(terms_n, key_thp(bus_slack,t,y), -1.0)
        add_to_terms(terms_n, key_thm(bus_slack,t,y), 1.0)
        add_row(f"slack_n_{t}_{y}", terms_n, rhs_const(0.0))

print(f"Bus slack usado: {bus_slack}")

# ------------------------------------------------------------
# A.6 Límites generación fijos p <= P_disp
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_base = float(P_disp.get((g,t), Pmax[g]))

        for y in Y:
            terms = {}
            add_to_terms(terms, key_p(g,t,y), -1.0)
            add_row(f"p_cap_{g}_{t}_{y}", terms, rhs_const(-cap_base))

# ------------------------------------------------------------
# A.7 Síncronos
# ------------------------------------------------------------
for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            terms_lb = {}
            add_to_terms(terms_lb, key_p(g,t,y), 1.0)
            add_to_terms(terms_lb, key_u(g,t,y), -pmin_g)
            add_row(f"sync_lb_{g}_{t}_{y}", terms_lb, rhs_const(0.0))

            terms_ub = {}
            add_to_terms(terms_ub, key_u(g,t,y), float(Pmax[g]))
            add_to_terms(terms_ub, key_p(g,t,y), -1.0)
            add_row(f"sync_ub_{g}_{t}_{y}", terms_ub, rhs_const(0.0))

            terms_u = {}
            add_to_terms(terms_u, key_u(g,t,y), -1.0)
            add_row(f"u_ub_{g}_{t}_{y}", terms_u, rhs_const(-1.0))

# ------------------------------------------------------------
# A.8-A.10 Inversión, h_syn, FFR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        terms_x = {}
        add_to_terms(terms_x, key_xcap(m,y), -1.0)
        add_row(f"x_ub_{m}_{y}", terms_x, rhs_param("xbar_ce", m, -1.0))

        for t in T:
            terms_h = {}
            add_to_terms(terms_h, key_xcap(m,y), float(gamma_base[m]))
            add_to_terms(terms_h, key_hsyn(m,t,y), -1.0)
            add_row(f"hsyn_lim_{m}_{t}_{y}", terms_h, rhs_const(0.0))

            terms_r = {}
            add_to_terms(terms_r, key_xcap(m,y), float(kappa_base[m]))
            add_to_terms(terms_r, key_rffr(m,t,y), -1.0)
            add_row(f"ffr_lim_{m}_{t}_{y}", terms_r, rhs_const(0.0))

# ------------------------------------------------------------
# A.11 Seguridad: Hreq_ce en b, Rreq fijo
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_H = {}

        for g in GS:
            add_to_terms(terms_H, key_u(g,t,y), float(Hg.get(g, 0.0)))

        for m in M:
            add_to_terms(terms_H, key_hsyn(m,t,y), 1.0)

        add_row(f"sys_H_{t}_{y}", terms_H, rhs_param("Hreq_ce", y, 1.0))

        terms_R = {}

        for m in M:
            add_to_terms(terms_R, key_rffr(m,t,y), 1.0)

        add_row(f"sys_R_{t}_{y}", terms_R, rhs_const(float(Rreq[y])))


# ============================================================
# 4) VARIABLES PRIMALES Y COSTOS c
# ============================================================

primal_keys = []

for g in G:
    for t in T:
        for y in Y:
            primal_keys.append(key_p(g,t,y))

for g in GS:
    for t in T:
        for y in Y:
            primal_keys.append(key_u(g,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_s(i,t,y))

for m in M:
    for y in Y:
        primal_keys.append(key_xcap(m,y))

for m in M:
    for t in T:
        for y in Y:
            primal_keys.append(key_hsyn(m,t,y))
            primal_keys.append(key_rffr(m,t,y))

for (i,j) in L_arcs:
    for t in T:
        for y in Y:
            primal_keys.append(key_fpos(i,j,t,y))
            primal_keys.append(key_fneg(i,j,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_thp(i,t,y))
            primal_keys.append(key_thm(i,t,y))

c_map = {}

for g in G:
    for t in T:
        for y in Y:
            c_map[key_p(g,t,y)] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map[key_s(i,t,y)] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map[key_xcap(m,y)] = float(cinv[m])

for k in primal_keys:
    c_map.setdefault(k, 0.0)


# ============================================================
# 5) CLASIFICACIÓN ACOPLADAS VS SEPARABLES
# ============================================================

coupled_primal_rows = [
    name for name in row_order
    if terms_has_param(rows[name]["terms"]) or rhs_has_param(rows[name]["rhs"])
]

separable_primal_rows = [
    name for name in row_order
    if name not in set(coupled_primal_rows)
]

coupled_dual_cols = set()

for name in row_order:
    terms = rows[name]["terms"]

    for k, coeffs in terms.items():
        if any(is_param_coeff(c) for c in coeffs):
            coupled_dual_cols.add(k)

coupled_dual_cols = sorted(list(coupled_dual_cols), key=str)

separable_dual_cols = [
    k for k in primal_keys
    if k not in set(coupled_dual_cols)
]

print("\n" + "="*70)
print("CLASIFICACIÓN PADM-W IEEE39 - FORMULACIÓN ORIGINAL")
print("="*70)
print(f"N° filas primales acopladas   : {len(coupled_primal_rows)}")
print(f"N° filas primales separables  : {len(separable_primal_rows)}")
print(f"N° columnas duales acopladas  : {len(coupled_dual_cols)}")
print(f"N° columnas duales separables : {len(separable_dual_cols)}")

bad_rows = [
    name for name in coupled_primal_rows
    if not (name.startswith("sys_H_") or name.startswith("x_ub_"))
]

if bad_rows:
    print("\n⚠️ Hay filas acopladas no esperadas:")
    print(bad_rows[:30])
else:
    print("\n✅ Filas acopladas esperadas: sys_H_ y x_ub_.")


# ============================================================
# 6) FUNCIONES PADM-W
# ============================================================

def initial_theta_vals():
    return {
        "Hreq_ce": {y: float(Hreq_base[y]) for y in Y},
        "xbar_ce": {m: float(xbar_base[m]) for m in M},
    }

def compute_J(theta_vals):
    J = 0.0

    for y in Y:
        base = float(Hreq_base[y])
        val = float(theta_vals["Hreq_ce"][y])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_H * abs(val - base) / abs(base)
        else:
            J += alpha_H * abs(val - base)

    for m in M:
        base = float(xbar_base[m])
        val = float(theta_vals["xbar_ce"][m])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_Xbar * abs(val - base) / abs(base)
        else:
            J += alpha_Xbar * abs(val - base)

    return J

def compute_ctx(z_vals):
    return sum(
        float(c_map[k]) * float(z_vals.get(k, 0.0))
        for k in primal_keys
    )

def compute_bty(theta_vals, ydual_vals):
    out = 0.0

    for name in row_order:
        out += rhs_eval(rows[name]["rhs"], theta_vals) * float(ydual_vals.get(name, 0.0))

    return out

def compute_primal_residuals(theta_vals, z_vals):
    res = {}

    for name in row_order:
        lhs = terms_eval(rows[name]["terms"], z_vals, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)
        res[name] = max(0.0, rhs - lhs)

    return res

def compute_dual_residuals(theta_vals, ydual_vals):
    res = {}

    for k in primal_keys:
        lhs = 0.0

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            yd = float(ydual_vals.get(name, 0.0))

            if abs(yd) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd

        rhs = float(c_map.get(k, 0.0))
        res[k] = max(0.0, lhs - rhs)

    return res

def compute_sd_residual(theta_vals, z_vals, ydual_vals):
    return max(0.0, compute_ctx(z_vals) - compute_bty(theta_vals, ydual_vals))

def max_residuals(theta_vals, z_vals, ydual_vals):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)
    sd = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return {
        "max_primal_all": max(pr.values()) if pr else 0.0,
        "max_dual_all": max(dr.values()) if dr else 0.0,
        "sd": sd,
        "max_primal_coupled": max([pr[name] for name in coupled_primal_rows], default=0.0),
        "max_dual_coupled": max([dr[k] for k in coupled_dual_cols], default=0.0),
    }

def compute_phi(theta_vals, z_vals, ydual_vals, rho, mu, use_true_objective=True):
    Jval = compute_J(theta_vals) if use_true_objective else 0.0

    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    pen_p = sum(float(rho[name]) * pr[name] for name in coupled_primal_rows)
    pen_d = sum(float(mu[k]) * dr[k] for k in coupled_dual_cols)

    sd_penalty = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return Jval + pen_p + pen_d + 1e3 * sd_penalty

def diff_theta(theta_a, theta_b):
    vals = []

    for y in Y:
        vals.append(abs(theta_a["Hreq_ce"][y] - theta_b["Hreq_ce"][y]))

    for m in M:
        vals.append(abs(theta_a["xbar_ce"][m] - theta_b["xbar_ce"][m]))

    return max(vals) if vals else 0.0

def diff_dict(a, b):
    keys = set(a.keys()) | set(b.keys())
    return max(
        [abs(float(a.get(k,0.0)) - float(b.get(k,0.0))) for k in keys],
        default=0.0
    )


# ============================================================
# 7) SUBPROBLEMAS PADM-W
# ============================================================

def add_z_variables(model):
    z = {}

    for g in G:
        for t in T:
            for y in Y:
                z[key_p(g,t,y)] = model.addVar(lb=0.0, name=f"p[{g},{t},{y}]")

    for g in GS:
        for t in T:
            for y in Y:
                z[key_u(g,t,y)] = model.addVar(lb=0.0, ub=1.0, name=f"u[{g},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_s(i,t,y)] = model.addVar(lb=0.0, name=f"s[{i},{t},{y}]")

    for m in M:
        for y in Y:
            z[key_xcap(m,y)] = model.addVar(lb=0.0, name=f"xcap[{m},{y}]")

    for m in M:
        for t in T:
            for y in Y:
                z[key_hsyn(m,t,y)] = model.addVar(lb=0.0, name=f"h_syn[{m},{t},{y}]")
                z[key_rffr(m,t,y)] = model.addVar(lb=0.0, name=f"r_ffr[{m},{t},{y}]")

    for (i,j) in L_arcs:
        for t in T:
            for y in Y:
                z[key_fpos(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_pos[{i},{j},{t},{y}]")
                z[key_fneg(i,j,t,y)] = model.addVar(lb=0.0, name=f"f_neg[{i},{j},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_thp(i,t,y)] = model.addVar(lb=0.0, name=f"theta_p[{i},{t},{y}]")
                z[key_thm(i,t,y)] = model.addVar(lb=0.0, name=f"theta_m[{i},{t},{y}]")

    return z

def add_dual_variables(model):
    yd = {}

    for name in row_order:
        ub_here = GRB.INFINITY

        if name.startswith("sys_H_"):
            ub_here = UB_H
        elif name.startswith("x_ub_"):
            ub_here = UB_xbar

        yd[name] = model.addVar(lb=0.0, ub=ub_here, name=f"ydual[{name}]")

    return yd

def add_theta_variables(model, warm_theta=None):
    theta = {
        "Hreq_ce": {},
        "xbar_ce": {},
    }

    for y in Y:
        theta["Hreq_ce"][y] = model.addVar(
            lb=0.85 * float(Hreq_base[y]),
            ub=1.15 * float(Hreq_base[y]),
            name=f"Hreq_ce[{y}]"
        )

    for m in M:
        theta["xbar_ce"][m] = model.addVar(
            lb=0.85 * float(xbar_base[m]),
            ub=1.15 * float(xbar_base[m]),
            name=f"xbar_ce[{m}]"
        )

    if warm_theta is not None:
        for y in Y:
            theta["Hreq_ce"][y].Start = float(warm_theta["Hreq_ce"][y])
        for m in M:
            theta["xbar_ce"][m].Start = float(warm_theta["xbar_ce"][m])

    return theta

def extract_theta(theta_vars):
    return {
        "Hreq_ce": {y: float(theta_vars["Hreq_ce"][y].X) for y in Y},
        "xbar_ce": {m: float(theta_vars["xbar_ce"][m].X) for m in M},
    }

def add_separable_primal_constraints_y(model, z_vars):
    theta_base = initial_theta_vals()

    for name in separable_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z_vars, theta_base)
        rhs = rhs_eval(rows[name]["rhs"], theta_base)
        model.addConstr(lhs >= rhs, name=f"Y_primal[{name}]")

def add_D_constraint_y(model, z_vars):
    for y in Y:
        if y >= y_star:
            model.addConstr(
                gp.quicksum(
                    omega(t) * float(eg[g]) * z_vars[key_p(g,t,y)]
                    for g in GE
                    for t in T
                )
                <= float(eps_CO2(y)),
                name=f"D_CO2_cap_{y}"
            )

def add_separable_dual_constraints_y(model, ydual_vars):
    theta_base = initial_theta_vals()

    for k in separable_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_base) * ydual_vars[name]

        model.addConstr(lhs <= float(c_map.get(k, 0.0)), name=f"Y_dual[{k}]")

def add_strong_duality_y(model, theta_vals, z_vars, ydual_vars):
    ctx = gp.quicksum(float(c_map[k]) * z_vars[k] for k in primal_keys)

    bty = gp.LinExpr()

    for name in row_order:
        bty += rhs_eval(rows[name]["rhs"], theta_vals) * ydual_vars[name]

    model.addConstr(ctx <= bty, name="Y_StrongDuality")

def solve_theta_subproblem(
    z_vals,
    ydual_vals,
    rho,
    mu,
    warm_theta=None,
    log=False,
    use_true_objective=True
):
    m = gp.Model("PADM_theta_subproblem_IEEE39_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    theta = add_theta_variables(m, warm_theta=warm_theta)

    obj = gp.LinExpr()

    if use_true_objective:
        for y in Y:
            dvar = m.addVar(lb=0.0, name=f"dHreq[{y}]")
            diff = theta["Hreq_ce"][y] - float(Hreq_base[y])

            m.addConstr(dvar >= diff, name=f"dHreq_pos[{y}]")
            m.addConstr(dvar >= -diff, name=f"dHreq_neg[{y}]")

            base = abs(float(Hreq_base[y]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_H) * dvar / base
            else:
                obj += float(alpha_H) * dvar

        for mm in M:
            dvar = m.addVar(lb=0.0, name=f"dXbar[{mm}]")
            diff = theta["xbar_ce"][mm] - float(xbar_base[mm])

            m.addConstr(dvar >= diff, name=f"dXbar_pos[{mm}]")
            m.addConstr(dvar >= -diff, name=f"dXbar_neg[{mm}]")

            base = abs(float(xbar_base[mm]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_Xbar) * dvar / base
            else:
                obj += float(alpha_Xbar) * dvar

    for name in coupled_primal_rows:
        lhs = terms_expr_theta_model_with_fixed_z(rows[name]["terms"], z_vals, theta)
        rhs = rhs_expr_in_theta_model(rows[name]["rhs"], theta)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            ydval = float(ydual_vals.get(name, 0.0))

            if abs(ydval) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += ydval * coeff_expr_in_theta_model(c, theta)

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    # Dualidad fuerte dura en theta:
    #   c^T z fijo <= b(theta)^T y fijo
    ctx_fixed = sum(float(c_map[k]) * float(z_vals.get(k, 0.0)) for k in primal_keys)

    bty_theta = gp.LinExpr()

    for name in row_order:
        ydval = float(ydual_vals.get(name, 0.0))

        if abs(ydval) <= 1e-12:
            continue

        bty_theta += ydval * rhs_expr_in_theta_model(rows[name]["rhs"], theta)

    m.addConstr(ctx_fixed <= bty_theta, name="Theta_StrongDuality")

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema THETA falló.")
        print(f"Status = {m.Status}")
        return None, m.Status, None

    theta_vals = extract_theta(theta)
    return theta_vals, m.Status, float(m.ObjVal)

def solve_y_subproblem(theta_vals, rho, mu, warm_z=None, warm_ydual=None, log=False):
    m = gp.Model("PADM_y_subproblem_IEEE39_dataset_corregido")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    z = add_z_variables(m)
    yd = add_dual_variables(m)

    m.update()

    if warm_z is not None:
        for k, v in z.items():
            if k in warm_z:
                v.Start = float(warm_z[k])

    if warm_ydual is not None:
        for name, v in yd.items():
            if name in warm_ydual:
                v.Start = float(warm_ydual[name])

    add_separable_primal_constraints_y(m, z)
    add_D_constraint_y(m, z)
    add_separable_dual_constraints_y(m, yd)
    add_strong_duality_y(m, theta_vals, z, yd)

    obj = gp.LinExpr()

    for name in coupled_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd[name]

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    if Y_OBJECTIVE_MODE == "ctx":
        obj += TIEBREAKER_CTX_WEIGHT * gp.quicksum(
            float(c_map[k]) * z[k]
            for k in primal_keys
        )

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema Y falló.")
        print(f"Status = {m.Status}")
        return None, None, m.Status, None

    z_vals = {k: float(v.X) for k, v in z.items()}
    yd_vals = {name: float(v.X) for name, v in yd.items()}

    return z_vals, yd_vals, m.Status, float(m.ObjVal)

def initialize_penalties():
    rho = {name: float(RHO0) for name in coupled_primal_rows}
    mu = {k: float(MU0) for k in coupled_dual_cols}
    return rho, mu

def update_penalties(theta_vals, z_vals, ydual_vals, rho, mu):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    n_rho = 0
    n_mu = 0

    for name in coupled_primal_rows:
        if pr[name] > FEAS_TOL:
            rho[name] = min(MAX_PENALTY, PENALTY_FACTOR * rho[name])
            n_rho += 1

    for k in coupled_dual_cols:
        if dr[k] > FEAS_TOL:
            mu[k] = min(MAX_PENALTY, PENALTY_FACTOR * mu[k])
            n_mu += 1

    return n_rho, n_mu


# ============================================================
# 8) LOOP PADM-W
# ============================================================

def run_padm(
    use_true_objective=True,
    theta_init=None,
    z_init=None,
    ydual_init=None,
    label="PADM"
):
    print("\n" + "="*70)
    print(f"INICIO {label}")
    print("="*70)
    print(f"use_true_objective = {use_true_objective}")

    rho, mu = initialize_penalties()

    theta_vals = copy.deepcopy(theta_init) if theta_init is not None else initial_theta_vals()

    if z_init is None or ydual_init is None:
        print("\nInicializando Y=(z,lambda)...")

        z_vals, ydual_vals, st_y, obj_y = solve_y_subproblem(
            theta_vals,
            rho,
            mu,
            warm_z=None,
            warm_ydual=None,
            log=False
        )

        if z_vals is None:
            print("❌ No se pudo inicializar Y.")

            return {
                "best_feasible": None,
                "theta": theta_vals,
                "z": None,
                "ydual": None,
                "history": [],
                "last_residuals": None,
                "rho": rho,
                "mu": mu,
            }

    else:
        z_vals = z_init.copy()
        ydual_vals = ydual_init.copy()

    history = []
    best_feasible = None

    res0 = max_residuals(theta_vals, z_vals, ydual_vals)

    print("\nChequeo inicial:")
    print(f"J(theta)       = {compute_J(theta_vals):.8f}")
    print(f"Max primal     = {res0['max_primal_all']:.3e}")
    print(f"Max dual       = {res0['max_dual_all']:.3e}")
    print(f"Strong duality = {res0['sd']:.3e}")

    for k_outer in range(MAX_OUTER):

        print("\n" + "#"*70)
        print(f"{label} | OUTER k = {k_outer}")
        print("#"*70)

        phi_prev = compute_phi(
            theta_vals,
            z_vals,
            ydual_vals,
            rho,
            mu,
            use_true_objective=use_true_objective
        )

        for l_inner in range(MAX_INNER):

            theta_old = copy.deepcopy(theta_vals)
            z_old = z_vals.copy()
            yd_old = ydual_vals.copy()

            theta_new, st_theta, obj_theta = solve_theta_subproblem(
                z_vals,
                ydual_vals,
                rho,
                mu,
                warm_theta=theta_vals,
                log=False,
                use_true_objective=use_true_objective
            )

            if theta_new is None:
                print("⚠️ Falló subproblema theta.")
                break

            theta_vals = theta_new

            z_new, yd_new, st_y, obj_y = solve_y_subproblem(
                theta_vals,
                rho,
                mu,
                warm_z=z_vals,
                warm_ydual=ydual_vals,
                log=False
            )

            if z_new is None:
                print("⚠️ Falló subproblema Y.")
                break

            z_vals = z_new
            ydual_vals = yd_new

            phi_now = compute_phi(
                theta_vals,
                z_vals,
                ydual_vals,
                rho,
                mu,
                use_true_objective=use_true_objective
            )

            res = max_residuals(theta_vals, z_vals, ydual_vals)

            dtheta = diff_theta(theta_vals, theta_old)
            dz = diff_dict(z_vals, z_old)
            dyd = diff_dict(ydual_vals, yd_old)
            dphi = abs(phi_now - phi_prev) / max(1.0, abs(phi_prev))

            if PADM_OUTPUT:
                print(
                    f"k={k_outer:02d}, l={l_inner:02d} | "
                    f"Phi={phi_now:.6e}, J={compute_J(theta_vals):.6e}, "
                    f"p_all={res['max_primal_all']:.2e}, "
                    f"d_all={res['max_dual_all']:.2e}, "
                    f"sd={res['sd']:.2e}, "
                    f"p_coup={res['max_primal_coupled']:.2e}, "
                    f"d_coup={res['max_dual_coupled']:.2e}, "
                    f"dtheta={dtheta:.2e}, dz={dz:.2e}, dyd={dyd:.2e}, dphi={dphi:.2e}"
                )

            history.append({
                "outer": k_outer,
                "inner": l_inner,
                "phi": phi_now,
                "J": compute_J(theta_vals),
                "max_primal_all": res["max_primal_all"],
                "max_dual_all": res["max_dual_all"],
                "sd": res["sd"],
                "max_primal_coupled": res["max_primal_coupled"],
                "max_dual_coupled": res["max_dual_coupled"],
                "dtheta": dtheta,
                "dz": dz,
                "dyd": dyd,
                "dphi": dphi,
                "rho_max": max(rho.values()) if rho else 0.0,
                "mu_max": max(mu.values()) if mu else 0.0,
            })

            if dphi <= INNER_TOL and dtheta <= INNER_TOL and dz <= INNER_TOL and dyd <= INNER_TOL:
                print(f"Inner loop converge en l={l_inner}.")
                break

            phi_prev = phi_now

        res = max_residuals(theta_vals, z_vals, ydual_vals)

        feasible = (
            res["max_primal_all"] <= FEAS_TOL
            and res["max_dual_all"] <= FEAS_TOL
            and res["sd"] <= FEAS_TOL
        )

        print("\n" + "-"*70)
        print(f"{label} | Chequeo fin outer k={k_outer}")
        print("-"*70)
        print(f"J(theta)             = {compute_J(theta_vals):.8f}")
        print(f"Max primal all       = {res['max_primal_all']:.3e}")
        print(f"Max dual all         = {res['max_dual_all']:.3e}")
        print(f"Strong duality       = {res['sd']:.3e}")
        print(f"Max primal coupled   = {res['max_primal_coupled']:.3e}")
        print(f"Max dual coupled     = {res['max_dual_coupled']:.3e}")
        print(f"rho_max              = {max(rho.values()) if rho else 0.0:.3e}")
        print(f"mu_max               = {max(mu.values()) if mu else 0.0:.3e}")

        if feasible:
            print(f"\n✅ {label} encontró solución factible primal-dual.")

            best_feasible = {
                "theta": copy.deepcopy(theta_vals),
                "z": z_vals.copy(),
                "ydual": ydual_vals.copy(),
                "J": compute_J(theta_vals),
                "residuals": res,
                "outer": k_outer,
                "history": history,
            }

            break

        n_rho, n_mu = update_penalties(theta_vals, z_vals, ydual_vals, rho, mu)

        print("\nActualización penalizaciones:")
        print(f"rho actualizadas: {n_rho}")
        print(f"mu actualizadas : {n_mu}")

    if best_feasible is None:
        print(f"\n⚠️ {label} terminó sin certificar factibilidad.")

    return {
        "best_feasible": best_feasible,
        "theta": theta_vals,
        "z": z_vals,
        "ydual": ydual_vals,
        "history": history,
        "last_residuals": max_residuals(theta_vals, z_vals, ydual_vals) if z_vals is not None else None,
        "rho": rho,
        "mu": mu,
    }


# ============================================================
# 9) EJECUTAR PADM-W
# ============================================================

PADM_WARMSTART = None

if RUN_PADM_W:

    res_ws = run_padm(
        use_true_objective=False,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM-W ETAPA 1: FACTIBILIDAD"
    )

    if res_ws["best_feasible"] is not None:

        theta_start = res_ws["best_feasible"]["theta"]
        z_start = res_ws["best_feasible"]["z"]
        yd_start = res_ws["best_feasible"]["ydual"]

        print("\n" + "="*70)
        print("WARMSTART FACTIBLE ENCONTRADO. INICIANDO ETAPA 2 CON J REAL.")
        print("="*70)

        res_final = run_padm(
            use_true_objective=True,
            theta_init=theta_start,
            z_init=z_start,
            ydual_init=yd_start,
            label="PADM-W ETAPA 2: OBJETIVO REAL"
        )

        if res_final["best_feasible"] is not None:
            final_solution = res_final["best_feasible"]
            final_label = "PADM-W ETAPA 2"
        else:
            print("\n⚠️ Etapa 2 no certificó factibilidad. Se conserva warmstart factible de etapa 1.")
            final_solution = res_ws["best_feasible"]
            final_label = "PADM-W ETAPA 1"

    else:
        print("\n⚠️ Etapa 1 no encontró punto factible.")
        final_solution = None
        final_label = "PADM-W FAILED"
        res_final = res_ws

else:
    res_final = run_padm(
        use_true_objective=True,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM"
    )

    if res_final["best_feasible"] is not None:
        final_solution = res_final["best_feasible"]
        final_label = "PADM"
    else:
        final_solution = None
        final_label = "PADM FAILED"

print("\n" + "="*70)
print("RESULTADOS FINALES PADM-W IEEE39")
print("="*70)

if final_solution is not None:
    sol_theta = final_solution["theta"]
    sol_z = final_solution["z"]
    sol_yd = final_solution["ydual"]

    print("✅ SOLUCIÓN FACTIBLE CERTIFICADA")
    print(f"Método final     : {final_label}")
    print(f"Outer encontrado : {final_solution['outer']}")
    print(f"J                : {final_solution['J']:.8f}")

    PADM_WARMSTART = {
        "theta": copy.deepcopy(sol_theta),
        "z": sol_z.copy(),
        "ydual": sol_yd.copy(),
        "label": final_label,
        "J": final_solution["J"],
        "residuals": final_solution["residuals"],
    }

else:
    print("⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.")

    sol_theta = res_final["theta"]
    sol_z = res_final["z"]
    sol_yd = res_final["ydual"]

    if sol_z is not None and sol_yd is not None:
        res = max_residuals(sol_theta, sol_z, sol_yd)

        PADM_WARMSTART = {
            "theta": copy.deepcopy(sol_theta),
            "z": sol_z.copy(),
            "ydual": sol_yd.copy(),
            "label": "PADM_LAST_POINT_NOT_CERTIFIED",
            "J": compute_J(sol_theta),
            "residuals": res,
        }

        print("⚠️ Se guardó el último punto de PADM-W como Start no certificado.")
    else:
        PADM_WARMSTART = None
        print("❌ No hay punto PADM-W disponible para warm start.")

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("CHECK PADM_WARMSTART IEEE39")
    print("="*70)

    sol_theta = PADM_WARMSTART["theta"]
    sol_z = PADM_WARMSTART["z"]
    sol_yd = PADM_WARMSTART["ydual"]

    res_check = max_residuals(sol_theta, sol_z, sol_yd)

    print(f"Origen          = {PADM_WARMSTART['label']}")
    print(f"J(theta)        = {PADM_WARMSTART['J']:.8f}")
    print(f"Max primal      = {res_check['max_primal_all']:.6e}")
    print(f"Max dual        = {res_check['max_dual_all']:.6e}")
    print(f"Strong duality  = {res_check['sd']:.6e}")


# ============================================================
# 10) WCEP FINAL CON DUALIDAD FUERTE Y WARM START PADM-W
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE39_DF_warmstart_PADM_dataset_corregido")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Hreq_ce = w.addVars(Y, lb=0.0, name="Hreq_ce")
xbar_ce = w.addVars(M, lb=0.0, name="xbar_ce")

for y in Y:
    base = Hreq_base[y]
    w.addConstr(Hreq_ce[y] >= 0.85 * base, name=f"adm_lb_Hreq[{y}]")
    w.addConstr(Hreq_ce[y] <= 1.15 * base, name=f"adm_ub_Hreq[{y}]")

for m in M:
    base = xbar_base[m]
    w.addConstr(xbar_ce[m] >= 0.85 * base, name=f"adm_lb_xbar[{m}]")
    w.addConstr(xbar_ce[m] <= 1.15 * base, name=f"adm_ub_xbar[{m}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[k, j, t, y] - f_neg[k, j, t, y]
                    for (k, j) in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[i2, j, t, y] - f_neg[i2, j, t, y]
                    for (i2, j) in lineas_salen[i]
                )
            )

            rhs_val = float(d_fw[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_fw[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Bij = float(Bline[(i, j)])

            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for (i, j) in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[(i, j)])
            flow = f_pos[i, j, t, y] - f_neg[i, j, t, y]

            name = f"flb_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{i}_{j}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(Pmax[g]) * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -xbar_ce[m], name=name)
        RHS_expr[name] = -xbar_ce[m]

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= Hreq_ce[y], name=name)
        RHS_expr[name] = Hreq_ce[y]

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(omega(t) * float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= eps_CO2(y),
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sys_H_"):
        ub_here = UB_H
    elif name.startswith("x_ub_"):
        ub_here = UB_xbar

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = omega(t) * float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = omega(t) * float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dHreq = w.addVars(Y, lb=0.0, name="dHreq")
dXbar = w.addVars(M, lb=0.0, name="dXbar")

for y in Y:
    base = Hreq_base[y]
    diff = Hreq_ce[y] - base

    w.addConstr(dHreq[y] >= diff, name=f"dHreq_pos[{y}]")
    w.addConstr(dHreq[y] >= -diff, name=f"dHreq_neg[{y}]")

for m in M:
    base = xbar_base[m]
    diff = xbar_ce[m] - base

    w.addConstr(dXbar[m] >= diff, name=f"dXbar_pos[{m}]")
    w.addConstr(dXbar[m] >= -diff, name=f"dXbar_neg[{m}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for y in Y:
        base = abs(float(Hreq_base[y]))
        if base > 1e-9:
            obj_wcep += alpha_H * dHreq[y] / base
        else:
            obj_wcep += alpha_H * dHreq[y]

    for m in M:
        base = abs(float(xbar_base[m]))
        if base > 1e-9:
            obj_wcep += alpha_Xbar * dXbar[m] / base
        else:
            obj_wcep += alpha_Xbar * dXbar[m]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_H * dHreq.sum()
        + alpha_Xbar * dXbar.sum(),
        GRB.MINIMIZE
    )


# ============================================================
# 11) APLICAR WARM START DEL PADM-W
# ============================================================

def safe_start(var, value):
    try:
        if value is not None and np.isfinite(float(value)):
            var.Start = float(value)
    except Exception:
        pass

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("APLICANDO PADM_WARMSTART AL WCEP FINAL IEEE39")
    print("="*70)
    print("Origen:", PADM_WARMSTART.get("label", "PADM"))

    ws_theta = PADM_WARMSTART["theta"]
    ws_z = PADM_WARMSTART["z"]
    ws_yd = PADM_WARMSTART["ydual"]

    for y in Y:
        safe_start(Hreq_ce[y], ws_theta["Hreq_ce"][y])
        safe_start(dHreq[y], abs(ws_theta["Hreq_ce"][y] - Hreq_base[y]))

    for m in M:
        safe_start(xbar_ce[m], ws_theta["xbar_ce"][m])
        safe_start(dXbar[m], abs(ws_theta["xbar_ce"][m] - xbar_base[m]))

    for g in G:
        for t in T:
            for y in Y:
                safe_start(p[g, t, y], ws_z.get(("p", g, t, y), None))

    for g in GS:
        for t in T:
            for y in Y:
                safe_start(u[g, t, y], ws_z.get(("u", g, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(s[i, t, y], ws_z.get(("s", i, t, y), None))

    for m in M:
        for y in Y:
            safe_start(xcap[m, y], ws_z.get(("xcap", m, y), None))

    for m in M:
        for t in T:
            for y in Y:
                safe_start(h_syn[m, t, y], ws_z.get(("h_syn", m, t, y), None))
                safe_start(r_ffr[m, t, y], ws_z.get(("r_ffr", m, t, y), None))

    for (i, j) in L_arcs:
        for t in T:
            for y in Y:
                safe_start(f_pos[i, j, t, y], ws_z.get(("f_pos", i, j, t, y), None))
                safe_start(f_neg[i, j, t, y], ws_z.get(("f_neg", i, j, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(theta_p[i, t, y], ws_z.get(("theta_p", i, t, y), None))
                safe_start(theta_m[i, t, y], ws_z.get(("theta_m", i, t, y), None))

    for name, var in ydual.items():
        safe_start(var, ws_yd.get(name, None))

    print("✅ Warm start aplicado al WCEP IEEE39.")
else:
    print("\n⚠️ No existe PADM_WARMSTART. El WCEP se resolverá sin Start.")


# ============================================================
# 12) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE39")
print("="*70)

w.optimize()


# ============================================================
# 13) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE39")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.TIME_LIMIT:
    print("⏱️ WCEP terminó por límite de tiempo.")
    if w.SolCount > 0:
        print(f"Mejor Obj = {w.ObjVal:.8f}")
        print(f"Best bound = {w.ObjBound:.8f}")
        print(f"MIPGap = {w.MIPGap:.6%}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

# ============================================================
# REGISTRO DE OPTIMIZACIÓN WCEP FINAL
# ============================================================
print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP FINAL")
print("="*70)

runtime = float(w.Runtime)
status = int(w.Status)
sol_count = int(w.SolCount)

print(f"Status Gurobi = {status}")
print(f"Tiempo WCEP optimize() = {runtime:.2f} s = {runtime/60:.2f} min = {runtime/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count}")

if sol_count > 0:
    obj_val = float(w.ObjVal)
    obj_bound = float(w.ObjBound)
    mip_gap = float(w.MIPGap)

    print(f"Mejor objetivo encontrado = {obj_val:.8f}")
    print(f"Mejor bound               = {obj_bound:.8f}")
    print(f"MIPGap                    = {mip_gap:.6e}")
    print(f"MIPGap (%)                = {100*mip_gap:.4f}%")

    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": obj_val,
        "obj_bound": obj_bound,
        "mip_gap": mip_gap,
        "mip_gap_percent": 100 * mip_gap,
    }
else:
    print("No se encontró solución factible.")
    registro_wcep = {
        "status": status,
        "runtime_seconds": runtime,
        "runtime_minutes": runtime / 60,
        "runtime_hours": runtime / 3600,
        "sol_count": sol_count,
        "obj_val": None,
        "obj_bound": None,
        "mip_gap": None,
        "mip_gap_percent": None,
    }

df_registro_wcep = pd.DataFrame([registro_wcep])
display(df_registro_wcep)

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Hreq_ce ---")
    hubo_cambio = False

    for y in Y:
        val = float(Hreq_ce[y].X)
        base = float(Hreq_base[y])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{y}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- xbar_ce ---")
    hubo_cambio = False

    for m in M:
        val = float(xbar_ce[m].X)
        base = float(xbar_base[m])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{m}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            omega(t) * float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = eps_CO2(y) - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {eps_CO2(y):.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_ce[y].X)

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_ce={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_xbar = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("x_ub_"):
            if abs(var.X - UB_xbar) <= 1e-6 * max(1.0, UB_xbar):
                hits_xbar += 1

    print(f"Duales sys_H pegadas a cota   : {hits_H}")
    print(f"Duales x_ub pegadas a cota    : {hits_xbar}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

# ============================================================
# TIEMPO TOTAL SCRIPT
# ============================================================
t_total_script = time.time() - t0_total_script
print("\n" + "="*70)
print("TIEMPO TOTAL DE CELDA / SCRIPT")
print("="*70)
print(f"Tiempo total script = {t_total_script:.2f} s = {t_total_script/60:.2f} min = {t_total_script/3600:.2f} h")


✅ Usando d_fw existente desde el FW corregido.
Demanda pico usada en PADM-W/WCEP = 1289.386000
Pmax total sistema                 = 1315.700000
Pesos temporales omega_t usados: {1: 365.0, 7: 365.0, 13: 365.0, 19: 365.0}

COTAS DUALES IEEE39
UB_H    = 324200000.000000
UB_xbar = 3242000.000000
Bus slack usado: 1
Bus slack usado: 1

CLASIFICACIÓN PADM-W IEEE39 - FORMULACIÓN ORIGINAL
N° filas primales acopladas   : 36
N° filas primales separables  : 4128
N° columnas duales acopladas  : 0
N° columnas duales separables : 2916

✅ Filas acopladas esperadas: sys_H_ y x_ub_.

INICIO PADM-W ETAPA 1: FACTIBILIDAD
use_true_objective = False

Inicializando Y=(z,lambda)...

⚠️ Subproblema Y falló.
Status = 3
❌ No se pudo inicializar Y.

⚠️ Etapa 1 no encontró punto factible.

RESULTADOS FINALES PADM-W IEEE39
⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.
❌ No hay punto PADM-W disponible para warm start.

CONSTRUYENDO WCEP FINAL IEEE39 CON DUALIDAD FUERTE
Set parameter NonConvex to value 2
Set parameter M

,status,runtime_seconds,runtime_minutes,runtime_hours,sol_count,obj_val,obj_bound,mip_gap,mip_gap_percent
0,3,0.539,0.008983,0.00015,0,None,None,None,None



No hay solución WCEP disponible para reportar.

TIEMPO TOTAL DE CELDA / SCRIPT
Tiempo total script = 24.50 s = 0.41 min = 0.01 h


# Sistema IEEE 57-bus


# Dataset

## Demanda Nodal

In [ ]:
import pandas as pd

bus_data = pd.read_csv("ieee57_bus_data.csv")

print("Dimensión:", bus_data.shape)
print("Columnas:")
print(bus_data.columns.tolist())

bus_data.head()

Dimensión: (57, 19)
Columnas:
['bus_id', 'bus_name', 'base_kv_label', 'area_number', 'loss_zone', 'bus_type', 'voltage_pu', 'voltage_angle_deg', 'load_mw', 'load_mvar', 'generation_mw', 'generation_mvar', 'base_kv', 'desired_voltage_pu', 'max_reactive_mvar', 'min_reactive_mvar', 'shunt_conductance_pu', 'shunt_susceptance_pu', 'remote_controlled_bus']


,bus_id,bus_name,base_kv_label,area_number,loss_zone,bus_type,voltage_pu,voltage_angle_deg,load_mw,load_mvar,generation_mw,generation_mvar,base_kv,desired_voltage_pu,max_reactive_mvar,min_reactive_mvar,shunt_conductance_pu,shunt_susceptance_pu,remote_controlled_bus
0,1,Kanawha,V1,1,1,3,1.040,0.00,55.0,17.0,128.9,-16.1,0.0,1.040,0.0,0.0,0.0,0.0,0
1,2,Turner,V1,1,1,2,1.010,-1.18,3.0,88.0,0.0,-0.8,0.0,1.010,50.0,-17.0,0.0,0.0,0
2,3,Logan,V1,1,1,2,0.985,-5.97,41.0,21.0,40.0,-1.0,0.0,0.985,60.0,-10.0,0.0,0.0,0
3,4,Sprigg,V1,1,1,0,0.981,-7.32,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0
4,5,Bus 5,V1,1,1,0,0.976,-8.52,13.0,4.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0


In [ ]:
# Crear demanda nodal base a partir del IEEE 57
demanda = bus_data[["bus_id", "load_mw"]].copy()

# Renombrar columnas
demanda.columns = ["bus", "demanda"]

# Asegurar formato correcto
demanda["bus"] = demanda["bus"].astype(int)
demanda["demanda"] = demanda["demanda"].fillna(0).astype(float)

# Guardar
demanda.to_csv("demanda_nodal_ieee57.csv", index=False)

print("Archivo creado: demanda_nodal_ieee57.csv")
demanda.head(10)

Archivo creado: demanda_nodal_ieee57.csv


,bus,demanda
0,1,55.0
1,2,3.0
2,3,41.0
3,4,0.0
4,5,13.0
5,6,75.0
6,7,0.0
7,8,150.0
8,9,121.0
9,10,5.0


In [ ]:
# Cargar demanda base
df = pd.read_csv("demanda_nodal_ieee57.csv")

# Definir periodos y proporciones
periodos = {
    1: 0.20,
    7: 0.21,
    13: 0.30,
    19: 0.29
}

# Expandir
filas = []

for _, row in df.iterrows():
    bus = int(row["bus"])
    demanda_total = float(row["demanda"])

    for t, alpha in periodos.items():
        filas.append({
            "bus": bus,
            "periodo": int(t),
            "demanda_mw": demanda_total * alpha
        })

df_final = pd.DataFrame(filas)

# Guardar
df_final.to_csv("demanda_nodal_ieee57_final.csv", index=False)

print("Archivo creado: demanda_nodal_ieee57_final.csv")
print(df_final.head(12))

Archivo creado: demanda_nodal_ieee57_final.csv
    bus  periodo  demanda_mw
0     1        1       11.00
1     1        7       11.55
2     1       13       16.50
3     1       19       15.95
4     2        1        0.60
5     2        7        0.63
6     2       13        0.90
7     2       19        0.87
8     3        1        8.20
9     3        7        8.61
10    3       13       12.30
11    3       19       11.89


In [ ]:
# Verificar que la suma por bus conserve la demanda original
check = df_final.groupby("bus", as_index=False)["demanda_mw"].sum()
check = check.merge(df, on="bus", how="left")
check["error"] = check["demanda_mw"] - check["demanda"]

print("Chequeo conservación demanda:")
print(check.head(20))

print("\nError máximo absoluto:")
print(check["error"].abs().max())

Chequeo conservación demanda:
    bus  demanda_mw  demanda  error
0     1        55.0     55.0    0.0
1     2         3.0      3.0    0.0
2     3        41.0     41.0    0.0
3     4         0.0      0.0    0.0
4     5        13.0     13.0    0.0
5     6        75.0     75.0    0.0
6     7         0.0      0.0    0.0
7     8       150.0    150.0    0.0
8     9       121.0    121.0    0.0
9    10         5.0      5.0    0.0
10   11         0.0      0.0    0.0
11   12       377.0    377.0    0.0
12   13        18.0     18.0    0.0
13   14        10.5     10.5    0.0
14   15        22.0     22.0    0.0
15   16        43.0     43.0    0.0
16   17        42.0     42.0    0.0
17   18        27.2     27.2    0.0
18   19         3.3      3.3    0.0
19   20         2.3      2.3    0.0

Error máximo absoluto:
8.881784197001252e-16


## Generadores Procesados

In [ ]:
import pandas as pd

# --------------------------------------------------
# 1) Cargar archivo base desde reduced
# --------------------------------------------------
df = pd.read_csv(r"base_reduced_generadores_procesados.csv")

# Si estás usando ruta absoluta, reemplaza la línea de arriba por esa ruta
# df = pd.read_csv(r"C:\...\CODIGO_PAPER_FINAL\reduced\generadores_procesados.csv")

# --------------------------------------------------
# 2) Definir columnas esperadas
# --------------------------------------------------
columnas_objetivo = [
    "id_gen", "bus_id", "tipo_tech", "fuel_orig", "Pmax", "Pmin",
    "Costo_Var", "Inercia_H", "Emisiones_tCO2_MWh", "Es_IBR",
    "Gamma", "Kappa", "Costo_Inv", "Gamma_max", "Kappa_max"
]

df = df[columnas_objetivo].copy()

# --------------------------------------------------
# 3) Reasignación de generadores a buses para IEEE57
# --------------------------------------------------
mapa_buses = {
    "223_STEAM_1": 8,
    "223_STEAM_2": 6,
    "223_STEAM_3": 12,
    "223_CT_4": 1,
    "223_CT_5": 3,
    "314_PV_1": 12,
    "314_PV_2": 18,
    "313_PV_1": 16,
    "314_PV_3": 3,
    "314_PV_4": 8,
    "313_PV_2": 17,
    "212_CSP_1": 9,
    "313_RTPV_1": 1,
    "313_RTPV_2": 4
}

# Aplicar reasignación
df57 = df.copy()
df57["bus_id"] = df57.apply(
    lambda row: mapa_buses[row["id_gen"]] if row["id_gen"] in mapa_buses else row["bus_id"],
    axis=1
)

# --------------------------------------------------
# 4) Guardar archivo final IEEE57
# --------------------------------------------------
df57 = df57[columnas_objetivo].copy()
df57.to_csv("generadores_procesados_ieee57_final.csv", index=False)

print("Archivo creado: generadores_procesados_ieee57_final.csv")
print(f"Cantidad de generadores: {len(df57)}")

print("\nGeneradores reasignados:")
print(
    df57[df57["id_gen"].isin(mapa_buses.keys())]
    .sort_values("id_gen")[["id_gen", "bus_id", "tipo_tech", "fuel_orig", "Pmax"]]
)

Archivo creado: generadores_procesados_ieee57_final.csv
Cantidad de generadores: 14

Generadores reasignados:
         id_gen  bus_id tipo_tech fuel_orig   Pmax
11    212_CSP_1       9        PV     Solar  200.0
3      223_CT_4       1        CT        NG   55.0
4      223_CT_5       3        CT        NG   55.0
0   223_STEAM_1       8      COAL      Coal  155.0
1   223_STEAM_2       6      COAL      Coal  155.0
2   223_STEAM_3      12      COAL      Coal  350.0
7      313_PV_1      16        PV     Solar   95.1
10     313_PV_2      17        PV     Solar   93.3
12   313_RTPV_1       1        PV     Solar  101.7
13   313_RTPV_2       4        PV     Solar   63.1
5      314_PV_1      12        PV     Solar   51.6
6      314_PV_2      18        PV     Solar   51.6
8      314_PV_3       3        PV     Solar   92.7
9      314_PV_4       8        PV     Solar   51.6


In [ ]:
ids_requeridos = set(mapa_buses.keys())
ids_presentes = set(df["id_gen"].unique())

print("Faltan estos generadores en el archivo base:")
print(sorted(ids_requeridos - ids_presentes))

Faltan estos generadores en el archivo base:
[]


## Líneas procesadas

In [ ]:
import pandas as pd

# Cargar branch data
branch_data = pd.read_csv("ieee57_branch_data.csv")

# Crear lineas procesadas estilo IEEE14 / IEEE30
lineas_proc = pd.DataFrame()

lineas_proc["linea"] = range(1, len(branch_data) + 1)
lineas_proc["desde"] = branch_data["from_bus"].astype(int)
lineas_proc["hacia"] = branch_data["to_bus"].astype(int)
lineas_proc["x"] = branch_data["reactance_pu"].astype(float)
lineas_proc["fmax"] = 100.0

# Guardar
lineas_proc.to_csv("lineas_procesadas_ieee57.csv", index=False)

print("Archivo creado: lineas_procesadas_ieee57.csv")
lineas_proc.head(10)

Archivo creado: lineas_procesadas_ieee57.csv


,linea,desde,hacia,x,fmax
0,1,1,2,0.0280,100.0
1,2,2,3,0.0850,100.0
2,3,3,4,0.0366,100.0
3,4,4,5,0.1320,100.0
4,5,4,6,0.1480,100.0
5,6,6,7,0.1020,100.0
6,7,6,8,0.1730,100.0
7,8,8,9,0.0505,100.0
8,9,9,10,0.1679,100.0
9,10,9,11,0.0848,100.0


In [ ]:
print("Cantidad de líneas:", len(lineas_proc))
print(lineas_proc.describe())

Cantidad de líneas: 80
         linea      desde      hacia          x   fmax
count  80.0000  80.000000  80.000000  80.000000   80.0
mean   40.5000  23.312500  29.600000   0.243045  100.0
std    23.2379  16.621318  16.428287   0.303566    0.0
min     1.0000   1.000000   2.000000   0.015200  100.0
25%    20.7500   9.000000  15.000000   0.064625  100.0
50%    40.5000  21.500000  29.000000   0.122350  100.0
75%    60.2500  37.000000  44.250000   0.254500  100.0
max    80.0000  57.000000  57.000000   1.355000  100.0


## Perfil Solar

In [ ]:
import pandas as pd

# --------------------------------------------------
# 1) Cargar perfil solar base desde reduced
# --------------------------------------------------
df = pd.read_csv(r"base_reduced_perfil_solar_final.csv")

# Si estás usando ruta absoluta, reemplaza la línea de arriba por esa ruta
# df = pd.read_csv(r"C:\...\CODIGO_PAPER_FINAL\reduced\perfil_solar_final.csv")

df = df[["Gen_ID", "Period", "P_Available_MW"]].copy()

# --------------------------------------------------
# 2) Cargar generadores base y generadores IEEE57
#    para obtener los Pmax
# --------------------------------------------------
gen_base = pd.read_csv(r"base_reduced_generadores_procesados.csv")
gen57 = pd.read_csv(r"generadores_procesados_ieee57_final.csv")

gen_base = gen_base[["id_gen", "Pmax"]].copy()
gen57 = gen57[["id_gen", "Pmax"]].copy()

pmax_dict_base = dict(zip(gen_base["id_gen"], gen_base["Pmax"]))
pmax_dict_57 = dict(zip(gen57["id_gen"], gen57["Pmax"]))

# --------------------------------------------------
# 3) Definir qué generadores faltan y qué perfil usar
# --------------------------------------------------
# nuevo_generador : generador_base_cuya_forma_se_copia
mapa_perfiles = {
    "313_RTPV_1": "313_PV_1",
    "313_RTPV_2": "313_PV_2",
    "212_CSP_1":  "314_PV_3"
}

# --------------------------------------------------
# 4) Crear perfiles faltantes escalando por Pmax
# --------------------------------------------------
perfiles_nuevos = []

for gen_nuevo, gen_ref in mapa_perfiles.items():
    if gen_ref not in pmax_dict_base:
        raise ValueError(f"No se encontró Pmax del generador base {gen_ref} en generadores_procesados.csv")

    if gen_nuevo not in pmax_dict_57:
        raise ValueError(f"No se encontró Pmax del generador nuevo {gen_nuevo} en generadores_procesados_ieee57_final.csv")

    perfil_ref = df[df["Gen_ID"] == gen_ref].copy()

    if perfil_ref.empty:
        raise ValueError(f"No se encontró perfil solar base para {gen_ref}")

    pmax_ref = float(pmax_dict_base[gen_ref])
    pmax_nuevo = float(pmax_dict_57[gen_nuevo])

    perfil_nuevo = perfil_ref.copy()
    perfil_nuevo["Gen_ID"] = gen_nuevo
    perfil_nuevo["P_Available_MW"] = perfil_nuevo["P_Available_MW"] * (pmax_nuevo / pmax_ref)

    perfiles_nuevos.append(perfil_nuevo)

# Unir todos los perfiles nuevos
if perfiles_nuevos:
    df_nuevos = pd.concat(perfiles_nuevos, ignore_index=True)
else:
    df_nuevos = pd.DataFrame(columns=["Gen_ID", "Period", "P_Available_MW"])

# --------------------------------------------------
# 5) Eliminar posibles duplicados y concatenar
# --------------------------------------------------
gens_nuevos = list(mapa_perfiles.keys())

df57 = df[~df["Gen_ID"].isin(gens_nuevos)].copy()
df57 = pd.concat([df57, df_nuevos], ignore_index=True)

# Ordenar
df57 = df57.sort_values(["Gen_ID", "Period"]).reset_index(drop=True)

# --------------------------------------------------
# 6) Guardar archivo final IEEE57
# --------------------------------------------------
df57.to_csv("perfil_solar_ieee57_final.csv", index=False)

print("Archivo creado: perfil_solar_ieee57_final.csv")

print("\nGeneradores con perfil agregado:")
for g in gens_nuevos:
    print(f"\nPerfil de {g}:")
    print(df57[df57["Gen_ID"] == g])

Archivo creado: perfil_solar_ieee57_final.csv

Generadores con perfil agregado:

Perfil de 313_RTPV_1:
        Gen_ID  Period  P_Available_MW
24  313_RTPV_1       1        0.000000
25  313_RTPV_1       4        0.000000
26  313_RTPV_1       7       22.775896
27  313_RTPV_1      10       70.510609
28  313_RTPV_1      13       73.538532
29  313_RTPV_1      16       48.448231
30  313_RTPV_1      19        0.000000
31  313_RTPV_1      22        0.000000

Perfil de 313_RTPV_2:
        Gen_ID  Period  P_Available_MW
32  313_RTPV_2       1        0.000000
33  313_RTPV_2       4        0.000000
34  313_RTPV_2       7       14.325085
35  313_RTPV_2      10       44.567546
36  313_RTPV_2      13       46.265351
37  313_RTPV_2      16       31.377596
38  313_RTPV_2      19        0.000000
39  313_RTPV_2      22        0.000000

Perfil de 212_CSP_1:
      Gen_ID  Period  P_Available_MW
0  212_CSP_1       1        0.000000
1  212_CSP_1       4        0.000000
2  212_CSP_1       7       46.201685
3 

## Datos extra

In [ ]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0) CONFIG: usar CSV IEEE57 finales
# ============================================================
PATH = "."

PATH_GEN   = os.path.join(PATH, "generadores_procesados_ieee57_final.csv")
PATH_LINE  = os.path.join(PATH, "lineas_procesadas_ieee57.csv")
PATH_LOAD  = os.path.join(PATH, "demanda_nodal_ieee57_final.csv")
PATH_SOLAR = os.path.join(PATH, "perfil_solar_ieee57_final.csv")

df_gen   = pd.read_csv(PATH_GEN)
df_lines = pd.read_csv(PATH_LINE)
df_load  = pd.read_csv(PATH_LOAD)

df_wind = pd.DataFrame(columns=["Gen_ID", "Period", "P_Available_MW"])
df_solar = pd.read_csv(PATH_SOLAR) if os.path.exists(PATH_SOLAR) else pd.DataFrame(
    columns=["Gen_ID", "Period", "P_Available_MW"]
)

# ============================================================
# 1) LIMPIAR NOMBRES DE COLUMNAS
# ============================================================
df_gen.columns   = df_gen.columns.str.strip()
df_lines.columns = df_lines.columns.str.strip()
df_load.columns  = df_load.columns.str.strip()
df_solar.columns = df_solar.columns.str.strip()

# ============================================================
# 2) RENOMBRAR COLUMNAS PARA DEJARLAS COMPATIBLES
# ============================================================

# ----- DEMANDA -----
df_load = df_load.rename(columns={
    "bus": "Bus ID",
    "periodo": "Period",
    "demanda_mw": "Nodal_Load_MW"
})

# ----- LINEAS -----
df_lines = df_lines.rename(columns={
    "desde": "from_bus",
    "hacia": "to_bus",
    "x": "Reactancia_X",
    "fmax": "F_max"
})

# ----- PERFIL SOLAR -----
df_solar = df_solar.rename(columns={
    "id_gen": "Gen_ID",
    "periodo": "Period",
    "disponibilidad_mw": "P_Available_MW"
})

# ============================================================
# 3) CHEQUEO DE COLUMNAS OBLIGATORIAS
# ============================================================
cols_gen_req = [
    "id_gen", "bus_id", "Pmax", "Pmin", "Costo_Var", "Inercia_H",
    "Emisiones_tCO2_MWh", "Es_IBR", "Gamma", "Kappa", "Costo_Inv"
]
cols_line_req = ["from_bus", "to_bus", "Reactancia_X", "F_max"]
cols_load_req = ["Bus ID", "Period", "Nodal_Load_MW"]

for c in cols_gen_req:
    if c not in df_gen.columns:
        raise ValueError(f"Falta columna en generadores: {c}")

for c in cols_line_req:
    if c not in df_lines.columns:
        raise ValueError(f"Falta columna en líneas: {c}")

for c in cols_load_req:
    if c not in df_load.columns:
        raise ValueError(f"Falta columna en demanda: {c}")

# ============================================================
# 4) SANITIZAR TIPOS
# ============================================================
df_gen["id_gen"] = df_gen["id_gen"].astype(str)
df_gen["bus_id"] = df_gen["bus_id"].astype(int)

df_load["Bus ID"] = df_load["Bus ID"].astype(int)
df_load["Period"] = df_load["Period"].astype(int)
df_load["Nodal_Load_MW"] = df_load["Nodal_Load_MW"].astype(float)

df_lines["from_bus"] = df_lines["from_bus"].astype(int)
df_lines["to_bus"]   = df_lines["to_bus"].astype(int)
df_lines["F_max"] = df_lines["F_max"].astype(float)
df_lines["Reactancia_X"] = df_lines["Reactancia_X"].astype(float)

if not df_solar.empty:
    df_solar["Gen_ID"] = df_solar["Gen_ID"].astype(str)
    df_solar["Period"] = df_solar["Period"].astype(int)
    df_solar["P_Available_MW"] = df_solar["P_Available_MW"].astype(float)

# ============================================================
# 5) DEFINICIÓN DE SETS
# ============================================================
Y = [1, 2, 3]
year_growth = {1: 1.00, 2: 1.05, 3: 1.10}

B = sorted(df_load["Bus ID"].unique().tolist())
T = sorted(df_load["Period"].unique().tolist())

G  = df_gen["id_gen"].tolist()
GS = df_gen[df_gen["Es_IBR"] == 0]["id_gen"].tolist()
M  = df_gen[df_gen["Es_IBR"] == 1]["id_gen"].tolist()
GE = df_gen[df_gen["Emisiones_tCO2_MWh"] > 0]["id_gen"].tolist()

print(f"Sets definidos: |B|={len(B)}, |T|={len(T)}, |G|={len(G)} (|GS|={len(GS)}, |M|={len(M)}, |GE|={len(GE)})")

# ============================================================
# 6) PARÁMETROS DE GENERADORES
# ============================================================
Pmax = df_gen.set_index("id_gen")["Pmax"].to_dict()
Pmin = df_gen.set_index("id_gen")["Pmin"].to_dict()
cg   = df_gen.set_index("id_gen")["Costo_Var"].to_dict()
eg   = df_gen.set_index("id_gen")["Emisiones_tCO2_MWh"].to_dict()
cinv = df_gen.set_index("id_gen")["Costo_Inv"].to_dict()

Hg = {g: float(df_gen.set_index("id_gen")["Inercia_H"].to_dict().get(g, 0.0)) for g in GS}

g_bus = df_gen.set_index("id_gen")["bus_id"].to_dict()

G_en_bus = {i: [] for i in B}
for g in G:
    bi = int(g_bus[g])
    if bi in G_en_bus:
        G_en_bus[bi].append(g)

# ============================================================
# 7) PARÁMETROS IBR
# ============================================================
gamma_base = df_gen.set_index("id_gen")["Gamma"].to_dict()
kappa_base = df_gen.set_index("id_gen")["Kappa"].to_dict()

gamma_max = df_gen.set_index("id_gen")["Gamma_max"].to_dict() if "Gamma_max" in df_gen.columns else {m: 0.5 for m in M}
kappa_max = df_gen.set_index("id_gen")["Kappa_max"].to_dict() if "Kappa_max" in df_gen.columns else {m: 0.5 for m in M}

xbar_mult = 2.0
xbar = {m: float(Pmax[m]) * xbar_mult for m in M}

# ============================================================
# 8) RED
# ============================================================
L_id = []
from_bus_line = {}
to_bus_line = {}
F0 = {}
Bline = {}

for idx, row in df_lines.iterrows():
    ell = int(idx + 1)   # o usar row["linea"] si existe
    i = int(row["from_bus"])
    j = int(row["to_bus"])

    L_id.append(ell)
    from_bus_line[ell] = i
    to_bus_line[ell] = j
    F0[ell] = float(row["F_max"])

    x_val = float(row["Reactancia_X"])
    if abs(x_val) < 1e-6:
        x_val = 1e-4
    Bline[ell] = 1.0 / x_val

lineas_entran = {i: [] for i in B}
lineas_salen  = {i: [] for i in B}

for ell in L_id:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    if i in lineas_salen:
        lineas_salen[i].append(ell)
    if j in lineas_entran:
        lineas_entran[j].append(ell)

print(f"Red: |L|={len(L_id)}")

# ============================================================
# 9) DEMANDA
# ============================================================
raw_demand = df_load.set_index(["Bus ID", "Period"])["Nodal_Load_MW"].to_dict()

d = {}
for y in Y:
    factor = year_growth[y]
    for i in B:
        for t in T:
            d[(i, t, y)] = float(raw_demand.get((i, t), 0.0)) * factor

# ============================================================
# 10) PERFIL RENOVABLE
# ============================================================
P_disp = {(g, t): float(Pmax[g]) for g in G for t in T}

if not df_solar.empty:
    for _, row in df_solar.iterrows():
        g_id = str(row["Gen_ID"])
        t = int(row["Period"])
        val = float(row["P_Available_MW"])
        if (g_id in G) and (t in T):
            P_disp[(g_id, t)] = val

# ============================================================
# 11) ESCALARES
# ============================================================
VOLL = 5000.0
y_star = 3
epsilon_CO2 = 0

# ============================================================
# 12) Hreq y Rreq
# ============================================================
D_peak = {y: max(sum(d[(i, t, y)] for i in B) for t in T) for y in Y}

max_hsyn = sum(float(gamma_base.get(m, 0.0)) * float(xbar[m]) for m in M) if len(M) > 0 else 0.0
max_sync = sum(float(Hg.get(g, 0.0)) for g in GS) if len(GS) > 0 else 0.0
cap_total = max_hsyn + max_sync

alpha_H = 0.35
H_floor = 0.05

k_H = (alpha_H * cap_total / D_peak[1]) if D_peak[1] > 1e-9 else 0.0
Hreq = {}
for y in Y:
    base = k_H * D_peak[y]
    floor = H_floor * cap_total
    Hreq[y] = max(base, floor)

MaxR = sum(float(kappa_base.get(m, 0.0)) * float(xbar[m]) for m in M) if len(M) > 0 else 0.0
beta_R = 0.40
R_floor = 0.05

Rreq = {}
for y in Y:
    base = beta_R * MaxR
    floor = R_floor * MaxR
    Rreq[y] = max(base, floor)

print("cap_total =", round(cap_total, 4), " MaxR =", round(MaxR, 4))
print("D_peak =", {y: round(D_peak[y], 2) for y in Y})
print("Hreq =", {y: round(Hreq[y], 4) for y in Y})
print("Rreq =", {y: round(Rreq[y], 4) for y in Y})

Sets definidos: |B|=57, |T|=4, |G|=14 (|GS|=5, |M|=9, |GE|=5)
Red: |L|=80
cap_total = 30.614  MaxR = 160.14
D_peak = {1: 375.24, 2: 394.0, 3: 412.76}
Hreq = {1: 10.7149, 2: 11.2506, 3: 11.7864}
Rreq = {1: 64.056, 2: 64.056, 3: 64.056}


# Resolución WCEP

### FW

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import time

t0_total_fw = time.time()

# ============================================================
# FINAL IEEE57
# PASO 1) FORWARD PROBLEM - FORMULACIÓN ORIGINAL + WELL-POSED HREQ
#
# Hreq queda en su posición original y se ajusta para well-posedness:
#   sum_g Hg[g] u[g,t,y] + sum_m h_syn[m,t,y] >= Hreq[y]
#
# No se usa zH.
# No se mueve Hreq hacia A.
# ============================================================


# ============================================================
# 0) OPCIONES WELL-POSED IEEE57
# ============================================================
# En IEEE57 el Hreq original estaba demasiado ajustado: el FW operaba
# exactamente en el límite de inercia en algunos periodos solares.
# Para dar margen operativo, se escala Hreq usando el criterio discutido:
# alpha_H original ≈ 0.35 -> alpha_H well-posed = 0.28.
#
# Importante:
#   - NO se introduce zH.
#   - Hreq sigue estando en el RHS de la restricción de inercia.
#   - Se construye Hreq_fw y NO se sobreescribe Hreq.
#
# Si quieres 0.25 en vez de 0.28, cambia solo HREQ_ALPHA_WELLPOSED.
# ============================================================

USE_WELLPOSED_HREQ = True
HREQ_ALPHA_ORIGINAL = 0.35
HREQ_ALPHA_WELLPOSED = 0.28
HREQ_MIN_RELATIVE_MARGIN = 1e-4

# ============================================================
# 0.1) DEMANDA CORREGIDA POR CAPACIDAD
# ============================================================
# Tal como en IEEE39, el Forward usa d_fw:
#   - si ya existe d_fw en memoria, lo respeta;
#   - si no existe, lo reconstruye desde d;
#   - en modo auto_capacity, escala la demanda para que la demanda pico
#     no supere DEMAND_CAP_MARGIN * sum_g Pmax[g].
#
# Esto evita que el dataset parta artificialmente con demanda mayor
# que la capacidad máxima total del sistema.
DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 1.0))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

FW_TIME_LIMIT = None  # puedes poner 5*60*60 si quieres límite de 5 horas


# ============================================================
# 1) ROBUSTECER LÍNEAS
# ============================================================
# En IEEE57 usamos IDs únicos de línea
L_arcs = L_id

# ============================================================
# 2) FIX: reconstruir B si fue pisado
# ============================================================
if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))


# ============================================================
# 3) TOPOLOGÍA ROBUSTA
# ============================================================
lineas_entran = {bus: [] for bus in B}
lineas_salen  = {bus: [] for bus in B}

for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    if j in lineas_entran:
        lineas_entran[j].append(ell)

    if i in lineas_salen:
        lineas_salen[i].append(ell)


# ============================================================
# 4) PRE-CHECKS
# ============================================================
print("\n==============================")
print("PRE-CHECKS FORWARD IEEE57 - FORMULACIÓN ORIGINAL")
print("==============================")

missing_d = [
    (i, t, y)
    for i in B
    for t in T
    for y in Y
    if (i, t, y) not in d
]

if missing_d:
    print(f"⚠️ Faltan {len(missing_d)} entradas en d[(i,t,y)]. Ejemplo:", missing_d[:5])
else:
    print("✅ d[(i,t,y)] completo para B,T,Y.")


# ============================================================
# 4.0) CONSTRUIR DEMANDA CORREGIDA d_fw
# ============================================================
# d_fw será la demanda que realmente usa el Forward.
# No sobreescribimos d, para mantener trazabilidad del dataset original.
if "d_fw" not in globals():
    print("\n⚠️ No existe d_fw. Se construye demanda corregida para IEEE57.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0

    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)

    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )

    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

else:
    print("\n✅ Usando d_fw existente desde una celda anterior.")
    d_original = {k: float(v) for k, v in d.items()}
    demanda_original_ty = {
        (t, y): sum(
            d_original[(i, t, y)]
            for i in B
            if (i, t, y) in d_original
        )
        for t in T
        for y in Y
    }
    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    # Si d_fw ya existe, inferimos el factor aproximado solo para reporte.
    peak_dem_fw_tmp = max(
        sum(float(d_fw[(i, t, y)]) for i in B)
        for t in T
        for y in Y
    )
    demand_scale_factor = (
        peak_dem_fw_tmp / peak_dem_original
        if peak_dem_original > 1e-9
        else 1.0
    )

peak_dem_fw = max(
    sum(float(d_fw[(i, t, y)]) for i in B)
    for t in T
    for y in Y
)

print("\n==============================")
print("DEMANDA CORREGIDA IEEE57")
print("==============================")
print(f"DEMAND_SCALE_MODE        = {DEMAND_SCALE_MODE}")
print(f"DEMAND_CAP_MARGIN        = {DEMAND_CAP_MARGIN:.4f}")
print(f"Demanda pico original    = {peak_dem_original:.6f}")
print(f"Pmax total sistema       = {cap_total_pmax:.6f}")
print(f"Factor demanda usado     = {demand_scale_factor:.8f}")
print(f"Demanda pico usada d_fw  = {peak_dem_fw:.6f}")
print(f"Margen cap/demanda d_fw  = {cap_total_pmax - peak_dem_fw:.6f}")

for y in Y:
    for t in T:
        dem_tot = sum(d_fw[(i, t, y)] for i in B)
        cap_tot = sum(Pmax[g] for g in G)

        if cap_tot + 1e-6 < dem_tot:
            print(
                f"⚠️ Capacidad total {cap_tot:.2f} < demanda {dem_tot:.2f} "
                f"en (t={t}, y={y})."
            )
            break

max_sync_H = sum(Hg.get(g, 0.0) for g in GS)

gamma_fwd_check = gamma_base.copy()
max_hsyn = (
    sum(gamma_fwd_check[m] * xbar[m] for m in M)
    if len(M) > 0
    else 0.0
)

max_H_total = max_sync_H + max_hsyn

kappa_fwd_check = kappa_base.copy()
max_R_total = (
    sum(kappa_fwd_check[m] * xbar[m] for m in M)
    if len(M) > 0
    else 0.0
)

# ============================================================
# 4.1) AJUSTE WELL-POSED DE HREQ
# ============================================================
Hreq_original = {y: float(Hreq[y]) for y in Y}

if USE_WELLPOSED_HREQ:
    if abs(HREQ_ALPHA_ORIGINAL) <= 1e-12:
        raise ValueError("HREQ_ALPHA_ORIGINAL no puede ser cero.")

    hreq_scale_factor = float(HREQ_ALPHA_WELLPOSED) / float(HREQ_ALPHA_ORIGINAL)

    Hreq_fw = {
        y: hreq_scale_factor * Hreq_original[y]
        for y in Y
    }

    # Protección adicional: aunque usemos alpha_H=0.28, no permitimos
    # que Hreq_fw quede pegado al máximo teórico del sistema.
    for y in Y:
        Hreq_fw[y] = min(
            float(Hreq_fw[y]),
            (1.0 - HREQ_MIN_RELATIVE_MARGIN) * float(max_H_total)
        )

else:
    hreq_scale_factor = 1.0
    Hreq_fw = Hreq_original.copy()

print("\n==============================")
print("AJUSTE WELL-POSED HREQ IEEE57")
print("==============================")
print(f"USE_WELLPOSED_HREQ    = {USE_WELLPOSED_HREQ}")
print(f"alpha_H original ref. = {HREQ_ALPHA_ORIGINAL:.4f}")
print(f"alpha_H well-posed    = {HREQ_ALPHA_WELLPOSED:.4f}")
print(f"factor Hreq usado     = {hreq_scale_factor:.8f}")

for y in Y:
    print(
        f"Hreq año {y}: original={Hreq_original[y]:.6f} -> "
        f"Hreq_fw={Hreq_fw[y]:.6f}"
    )

print(
    f"Max H posible (sync + hsyn) = {max_H_total:.3f} "
    f"(sync={max_sync_H:.3f}, hsyn={max_hsyn:.3f})"
)

print(f"Max R posible (FFR)         = {max_R_total:.3f}")

for y in Y:
    if Hreq_fw[y] > max_H_total + 1e-6:
        print(
            f"❌ ALERTA: Hreq[{y}]={Hreq_fw[y]:.3f} > MaxH={max_H_total:.3f} "
            f"=> INFEASIBLE por inercia."
        )

    if Rreq[y] > max_R_total + 1e-6:
        print(
            f"❌ ALERTA: Rreq[{y}]={Rreq[y]:.3f} > MaxR={max_R_total:.3f} "
            f"=> INFEASIBLE por FFR."
        )


# ============================================================
# 5) MODELO FORWARD IEEE57 - FORMULACIÓN ORIGINAL
# ============================================================
fw_base = gp.Model("FW_base_IEEE57_original")
fw_base.Params.OutputFlag = 1
if FW_TIME_LIMIT is not None:
    fw_base.Params.TimeLimit = FW_TIME_LIMIT


# ============================================================
# 6) VARIABLES
# ============================================================
p = fw_base.addVars(
    G, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="p"
)

u = fw_base.addVars(
    GS, T, Y,
    lb=0.0,
    ub=1.0,
    vtype=GRB.CONTINUOUS,
    name="u"
)

s = fw_base.addVars(
    B, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="s"
)

theta = fw_base.addVars(
    B, T, Y,
    lb=-GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="theta"
)

f = fw_base.addVars(
    L_arcs, T, Y,
    lb=-GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="f"
)

x = fw_base.addVars(
    M, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="x"
)

h_syn = fw_base.addVars(
    M, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="h_syn"
)

r_ffr = fw_base.addVars(
    M, T, Y,
    lb=0.0,
    vtype=GRB.CONTINUOUS,
    name="r_ffr"
)


# ============================================================
# 7) RESTRICCIONES
# ============================================================

# ------------------------------------------------------------
# 7.1) Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            gen = gp.quicksum(
                p[g, t, y]
                for g in G_en_bus.get(i, [])
            )

            inflow = gp.quicksum(
                f[ell, t, y]
                for ell in lineas_entran[i]
            )

            outflow = gp.quicksum(
                f[ell, t, y]
                for ell in lineas_salen[i]
            )

            fw_base.addConstr(
                gen - d_fw[(i, t, y)] + s[i, t, y] + inflow - outflow == 0,
                name=f"balance[{i},{t},{y}]"
            )


# ------------------------------------------------------------
# 7.2) ENS <= demanda
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                s[i, t, y] <= d_fw[(i, t, y)],
                name=f"ens_ub[{i},{t},{y}]"
            )


# ------------------------------------------------------------
# 7.3) Flujo DC
# ------------------------------------------------------------
for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    for t in T:
        for y in Y:
            fw_base.addConstr(
                f[ell, t, y]
                == Bline[ell] * (theta[i, t, y] - theta[j, t, y]),
                name=f"dcflow[{ell},{t},{y}]"
            )


# ------------------------------------------------------------
# 7.4) Límites de línea
# ------------------------------------------------------------
for ell in L_arcs:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                f[ell, t, y] <= F0[ell],
                name=f"f_ub[{ell},{t},{y}]"
            )

            fw_base.addConstr(
                f[ell, t, y] >= -F0[ell],
                name=f"f_lb[{ell},{t},{y}]"
            )


# ------------------------------------------------------------
# 7.5) Bus slack
# ------------------------------------------------------------
bus_slack = int(sorted(B)[0])

for t in T:
    for y in Y:
        fw_base.addConstr(
            theta[bus_slack, t, y] == 0.0,
            name=f"slack[{t},{y}]"
        )

print(f"Bus slack usado: {bus_slack}")


# ------------------------------------------------------------
# 7.6) Disponibilidad de generación
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_gt = P_disp.get((g, t), Pmax[g])

        for y in Y:
            fw_base.addConstr(
                p[g, t, y] <= cap_gt,
                name=f"pdisp[{g},{t},{y}]"
            )


# ------------------------------------------------------------
# 7.7) Síncronos: Pmin*u <= p <= Pmax*u
# ------------------------------------------------------------
for g in GS:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                p[g, t, y] >= Pmin.get(g, 0.0) * u[g, t, y],
                name=f"sync_min[{g},{t},{y}]"
            )

            fw_base.addConstr(
                p[g, t, y] <= Pmax[g] * u[g, t, y],
                name=f"sync_max[{g},{t},{y}]"
            )


# ------------------------------------------------------------
# 7.8) Límite inversión
# ------------------------------------------------------------
for m in M:
    for y in Y:
        fw_base.addConstr(
            x[m, y] <= xbar[m],
            name=f"x_ub[{m},{y}]"
        )


# ------------------------------------------------------------
# 7.9) h_syn <= gamma * x
# ------------------------------------------------------------
for m in M:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                h_syn[m, t, y] <= gamma_base[m] * x[m, y],
                name=f"hsyn_cap[{m},{t},{y}]"
            )


# ------------------------------------------------------------
# 7.10) r_ffr <= kappa * x
# ------------------------------------------------------------
for m in M:
    for t in T:
        for y in Y:
            fw_base.addConstr(
                r_ffr[m, t, y] <= kappa_base[m] * x[m, y],
                name=f"ffr_cap[{m},{t},{y}]"
            )


# ------------------------------------------------------------
# 7.11) Inercia requerida - FORMULACIÓN ORIGINAL
#
#   sum_g Hg[g] u[g,t,y] + sum_m h_syn[m,t,y] >= Hreq_fw[y]
#
# Aquí Hreq queda en el lado derecho.
# No se usa zH.
# No se mueve Hreq hacia A.
# ------------------------------------------------------------
for t in T:
    for y in Y:
        fw_base.addConstr(
            gp.quicksum(
                Hg.get(g, 0.0) * u[g, t, y]
                for g in GS
            )
            + gp.quicksum(
                h_syn[m, t, y]
                for m in M
            )
            >= Hreq_fw[y],
            name=f"inertia_req[{t},{y}]"
        )


# ------------------------------------------------------------
# 7.12) FFR requerida
# ------------------------------------------------------------
for t in T:
    for y in Y:
        fw_base.addConstr(
            gp.quicksum(
                r_ffr[m, t, y]
                for m in M
            ) >= Rreq[y],
            name=f"ffr_req[{t},{y}]"
        )


# ============================================================
# 8) ACTUALIZAR MODELO
# ============================================================
fw_base.update()

print(f"\nVariables: {fw_base.NumVars} | Restricciones: {fw_base.NumConstrs}")


# ============================================================
# 9) OBJETIVO
# ============================================================
obj_oper = gp.quicksum(
    cg[g] * p[g, t, y]
    for g in G
    for t in T
    for y in Y
)

obj_ens = VOLL * gp.quicksum(
    s[i, t, y]
    for i in B
    for t in T
    for y in Y
)

obj_inv = gp.quicksum(
    cinv[m] * x[m, y]
    for m in M
    for y in Y
)

fw_base.setObjective(
    obj_oper + obj_ens + obj_inv,
    GRB.MINIMIZE
)


# ============================================================
# 10) RESOLVER
# ============================================================
fw_base.optimize()

fw_runtime_optimize = float(fw_base.Runtime)
fw_runtime_total = time.time() - t0_total_fw

print("\n==============================")
print("TIEMPOS FORWARD IEEE57")
print("==============================")
print(f"Tiempo Gurobi optimize() = {fw_runtime_optimize:.2f} s = {fw_runtime_optimize/60:.2f} min")
print(f"Tiempo total script      = {fw_runtime_total:.2f} s = {fw_runtime_total/60:.2f} min")


# ============================================================
# 11) DIAGNÓSTICO BÁSICO
# ============================================================
if fw_base.Status == GRB.INFEASIBLE:
    print("\n❌ Modelo INFEASIBLE.")
    print("No se escribió archivo IIS/ILP.")

elif fw_base.Status == GRB.OPTIMAL:
    print("\n✅ Modelo resuelto óptimamente.")
    print(f"Valor objetivo = {fw_base.ObjVal:,.4f}")

    ens_total = sum(
        s[i, t, y].X
        for i in B
        for t in T
        for y in Y
    )

    print(f"ENS total = {ens_total:.6f}")

else:
    print(f"\n⚠️ Modelo terminó con status Gurobi = {fw_base.Status}")


# ============================================================
# 12) RESULTADOS FORWARD IEEE57 - FORMULACIÓN ORIGINAL
# ============================================================

if fw_base.Status == GRB.OPTIMAL:

    print("\n==============================")
    print("RESULTADOS FW_base IEEE57 - FORMULACIÓN ORIGINAL")
    print("==============================")
    print(f"Valor objetivo total = {fw_base.ObjVal:,.4f}")

    # --------------------------------------------------------
    # 12.1) Tabla de generación p[g,t,y]
    #       y emisiones e_g * p[g,t,y]
    # --------------------------------------------------------
    filas_p = []

    for g in G:
        for t in T:
            for y in Y:
                val_p = p[g, t, y].X
                val_ep = eg.get(g, 0.0) * val_p

                filas_p.append({
                    "gen": g,
                    "t": t,
                    "y": y,
                    "bus": g_bus[g],
                    "Es_IBR": int(g in M),
                    "emision_factor": eg.get(g, 0.0),
                    "p_gty": val_p,
                    "e_g_p_gty": val_ep
                })

    df_p = pd.DataFrame(filas_p)


    # --------------------------------------------------------
    # 12.2) Mostrar solo generación positiva
    # --------------------------------------------------------
    df_p_pos = df_p[df_p["p_gty"] > 1e-6].copy()
    df_p_pos = df_p_pos.sort_values(["y", "t", "gen"]).reset_index(drop=True)

    print("\n--- Tabla p[g,t,y] y e_g * p[g,t,y] (solo valores > 0) ---")
    display(df_p_pos)


    # --------------------------------------------------------
    # 12.3) Resumen por periodo y año
    # --------------------------------------------------------
    resumen_ty = (
        df_p
        .groupby(["y", "t"], as_index=False)
        .agg(
            generacion_total=("p_gty", "sum"),
            emisiones_totales_intervalo=("e_g_p_gty", "sum")
        )
    )

    print("\n--- Resumen por (y,t) ---")
    display(resumen_ty)


    # --------------------------------------------------------
    # 12.4) Emisiones totales por año
    # --------------------------------------------------------
    emisiones_por_anio = (
        df_p
        .groupby("y", as_index=False)
        .agg(
            emisiones_totales_anuales=("e_g_p_gty", "sum"),
            generacion_total_anual=("p_gty", "sum")
        )
    )

    emisiones_por_anio["epsilon_90pct"] = 0.90 * emisiones_por_anio["emisiones_totales_anuales"]

    print("\n--- Emisiones totales por año ---")
    display(emisiones_por_anio)

    print("\n--- Valores sugeridos para epsilon_CO2_wcep = 90% emisiones FW ---")
    for _, row in emisiones_por_anio.iterrows():
        print(
            f"y={int(row['y'])}: "
            f"E_FW={row['emisiones_totales_anuales']:.6f} | "
            f"0.90*E_FW={row['epsilon_90pct']:.6f}"
        )


    # --------------------------------------------------------
    # 12.5) Resumen por generador
    # --------------------------------------------------------
    resumen_g = (
        df_p
        .groupby("gen", as_index=False)
        .agg(
            bus=("bus", "first"),
            Es_IBR=("Es_IBR", "first"),
            emision_factor=("emision_factor", "first"),
            generacion_total=("p_gty", "sum"),
            emisiones_totales=("e_g_p_gty", "sum")
        )
        .sort_values("generacion_total", ascending=False)
        .reset_index(drop=True)
    )

    print("\n--- Resumen por generador ---")
    display(resumen_g)


    # --------------------------------------------------------
    # 12.6) ENS total y por bus
    # --------------------------------------------------------
    filas_s = []

    for i in B:
        for t in T:
            for y in Y:
                filas_s.append({
                    "bus": i,
                    "t": t,
                    "y": y,
                    "ENS": s[i, t, y].X
                })

    df_s = pd.DataFrame(filas_s)

    total_ENS = df_s["ENS"].sum()
    print(f"\nENS total = {total_ENS:.6f}")

    df_s_pos = (
        df_s[df_s["ENS"] > 1e-6]
        .sort_values(["y", "t", "bus"])
        .reset_index(drop=True)
    )

    print("\n--- ENS positivo por bus, periodo y año ---")
    display(df_s_pos)


    # --------------------------------------------------------
    # 12.7) Inversiones IBR
    # --------------------------------------------------------
    filas_x = []

    for m in M:
        for y in Y:
            filas_x.append({
                "gen": m,
                "y": y,
                "x_my": x[m, y].X
            })

    df_x = pd.DataFrame(filas_x)

    print("\n--- Inversión x[m,y] positiva ---")
    display(
        df_x[df_x["x_my"] > 1e-6]
        .sort_values(["y", "gen"])
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 12.8) Inercia sintética h_syn
    # --------------------------------------------------------
    filas_h = []

    for m in M:
        for t in T:
            for y in Y:
                filas_h.append({
                    "gen": m,
                    "t": t,
                    "y": y,
                    "h_syn": h_syn[m, t, y].X
                })

    df_h = pd.DataFrame(filas_h)

    print("\n--- h_syn[m,t,y] positivo ---")
    display(
        df_h[df_h["h_syn"] > 1e-6]
        .sort_values(["y", "t", "gen"])
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 12.9) FFR r_ffr
    # --------------------------------------------------------
    filas_r = []

    for m in M:
        for t in T:
            for y in Y:
                filas_r.append({
                    "gen": m,
                    "t": t,
                    "y": y,
                    "r_ffr": r_ffr[m, t, y].X
                })

    df_r = pd.DataFrame(filas_r)

    print("\n--- r_ffr[m,t,y] positivo ---")
    display(
        df_r[df_r["r_ffr"] > 1e-6]
        .sort_values(["y", "t", "gen"])
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 12.10) Chequeo agregado de inercia y FFR por periodo/año
    # --------------------------------------------------------
    filas_req = []

    for t in T:
        for y in Y:
            H_sync_val = sum(
                Hg.get(g, 0.0) * u[g, t, y].X
                for g in GS
            )

            H_syn_val = sum(
                h_syn[m, t, y].X
                for m in M
            )

            R_val = sum(
                r_ffr[m, t, y].X
                for m in M
            )

            filas_req.append({
                "t": t,
                "y": y,
                "H_sync": H_sync_val,
                "H_syn": H_syn_val,
                "H_total": H_sync_val + H_syn_val,
                "Hreq_original": Hreq_original[y],
                "Hreq_fw": Hreq_fw[y],
                "margen_H": H_sync_val + H_syn_val - Hreq_fw[y],
                "R_total": R_val,
                "Rreq": Rreq[y],
                "margen_R": R_val - Rreq[y]
            })

    df_req = pd.DataFrame(filas_req)

    print("\n--- Chequeo de requerimientos Hreq y Rreq ---")
    display(
        df_req
        .sort_values(["y", "t"])
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 12.11) Flujos de línea
    # --------------------------------------------------------
    filas_f = []

    for ell in L_arcs:
        i = from_bus_line[ell]
        j = to_bus_line[ell]

        for t in T:
            for y in Y:
                val_f = f[ell, t, y].X
                Fij = F0[ell]

                filas_f.append({
                    "linea": ell,
                    "from_bus": i,
                    "to_bus": j,
                    "t": t,
                    "y": y,
                    "f_ellty": val_f,
                    "F_max": Fij,
                    "uso_linea_abs": abs(val_f) / Fij if Fij > 1e-9 else None
                })

    df_f = pd.DataFrame(filas_f)

    print("\n--- Líneas más cargadas ---")
    display(
        df_f
        .sort_values("uso_linea_abs", ascending=False)
        .head(20)
        .reset_index(drop=True)
    )

else:
    print("El modelo no quedó óptimo. Status =", fw_base.Status)


PRE-CHECKS FORWARD IEEE57 - FORMULACIÓN ORIGINAL
✅ d[(i,t,y)] completo para B,T,Y.

⚠️ No existe d_fw. Se construye demanda corregida para IEEE57.

DEMANDA CORREGIDA IEEE57
DEMAND_SCALE_MODE        = auto_capacity
DEMAND_CAP_MARGIN        = 0.9800
Demanda pico original    = 412.764000
Pmax total sistema       = 1570.700000
Factor demanda usado     = 1.00000000
Demanda pico usada d_fw  = 412.764000
Margen cap/demanda d_fw  = 1157.936000

AJUSTE WELL-POSED HREQ IEEE57
USE_WELLPOSED_HREQ    = True
alpha_H original ref. = 0.3500
alpha_H well-posed    = 0.2800
factor Hreq usado     = 0.80000000
Hreq año 1: original=10.714900 -> Hreq_fw=8.571920
Hreq año 2: original=11.250645 -> Hreq_fw=9.000516
Hreq año 3: original=11.786390 -> Hreq_fw=9.429112
Max H posible (sync + hsyn) = 30.614 (sync=14.600, hsyn=16.014)
Max R posible (FFR)         = 160.140
Set parameter Username
Set parameter LicenseID to value 2748035
Academic license - for non-commercial use only - expires 2026-12-01
Set parameter O

,gen,t,y,bus,Es_IBR,emision_factor,p_gty,e_g_p_gty
0,223_STEAM_2,1,1,6,0,0.95,155.000000,147.250000
1,223_STEAM_3,1,1,12,0,0.95,95.160000,90.402000
2,212_CSP_1,7,1,9,1,0.00,46.201685,0.000000
3,223_STEAM_2,7,1,6,0,0.95,62.377356,59.258489
4,313_PV_1,7,1,16,1,0.00,21.297814,0.000000
...,...,...,...,...,...,...,...,...
62,314_PV_3,13,3,3,1,0.00,66.959016,0.000000
63,314_PV_4,13,3,8,1,0.00,35.977596,0.000000
64,223_STEAM_1,19,3,8,0,0.95,128.414925,121.994178
65,223_STEAM_2,19,3,6,0,0.95,155.000000,147.250000



--- Resumen por (y,t) ---


,y,t,generacion_total,emisiones_totales_intervalo
0,1,1,250.1600,237.652000
1,1,7,262.6680,59.258489
2,1,13,375.2400,42.532083
3,1,19,362.7320,344.595400
4,2,1,262.6680,249.534600
5,2,7,275.8014,71.735219
6,2,13,394.0020,50.946851
7,2,19,380.8686,361.825170
8,3,1,275.1760,261.417200
9,3,7,288.9348,84.211949



--- Emisiones totales por año ---


...

,gen,t,y,r_ffr
0,212_CSP_1,1,1,24.696
1,313_PV_1,1,1,19.020
2,313_RTPV_1,1,1,20.340
3,212_CSP_1,7,1,24.696
4,313_PV_1,7,1,19.020
5,313_RTPV_1,7,1,20.340
6,212_CSP_1,13,1,24.696
7,313_PV_1,13,1,19.020
8,313_RTPV_1,13,1,20.340
9,212_CSP_1,19,1,24.696



--- Chequeo de requerimientos Hreq y Rreq ---


,t,y,H_sync,H_syn,H_total,Hreq_original,Hreq_fw,margen_H,R_total,Rreq,margen_R
0,1,1,3.815657,6.4056,10.221257,10.714900,8.571920,1.649337,64.056,64.056,0.0
1,7,1,3.000000,6.4056,9.405600,10.714900,8.571920,0.833680,64.056,64.056,0.0
2,13,1,2.166320,6.4056,8.571920,10.714900,8.571920,0.000000,64.056,64.056,0.0
3,19,1,6.744971,6.4056,13.150571,10.714900,8.571920,4.578651,64.056,64.056,0.0
4,1,2,6.066895,6.4056,12.472495,11.250645,9.000516,3.471979,64.056,64.056,0.0
5,7,2,3.000000,6.4056,9.405600,11.250645,9.000516,0.405084,64.056,64.056,0.0
6,13,2,2.594916,6.4056,9.000516,11.250645,9.000516,0.000000,64.056,64.056,0.0
7,19,2,6.867873,6.4056,13.273473,11.250645,9.000516,4.272957,64.056,64.056,0.0
8,1,3,3.023512,6.4056,9.429112,11.786390,9.429112,0.000000,64.056,64.056,0.0
9,7,3,4.289233,6.4056,10.694833,11.786390,9.429112,1.265721,64.056,64.056,0.0



--- Líneas más cargadas ---


,linea,from_bus,to_bus,t,y,f_ellty,F_max,uso_linea_abs
0,8,8,9,1,2,100.000000,100.0,1.000000
1,8,8,9,19,2,100.000000,100.0,1.000000
2,8,8,9,19,3,100.000000,100.0,1.000000
3,8,8,9,19,1,100.000000,100.0,1.000000
4,25,12,13,1,3,84.458886,100.0,0.844589
5,3,3,4,1,2,-70.775992,100.0,0.707760
6,3,3,4,19,1,-66.932099,100.0,0.669321
7,3,3,4,19,2,-66.235393,100.0,0.662354
8,3,3,4,19,3,-65.538688,100.0,0.655387
9,3,3,4,13,1,-59.008783,100.0,0.590088


### (variando A) Dualidad fuerte

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import time

# ============================================================
# FINAL IEEE57: WCEP CON DUALIDAD FUERTE
# CASO: SOLO Pmax SÍNCRONO MUTABLE
#
#   - Sin PADM.
#   - Solo se modifica A mediante Pmax_sync_ce[g] para g in GS.
#   - Hreq queda fijo en b, usando Hreq_fw si existe desde el FW.
#   - xbar queda fijo.
#   - Se usa d_fw si existe; si no, se reconstruye con escala por capacidad.
# ============================================================

t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

# Restricción D del IEEE57
epsilon_CO2_wcep = 766

# Objetivo J
alpha_Psync = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

# Opciones WCEP
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1
WCEP_TIME_LIMIT = 5 * 60 * 60  # 5 horas

# Pmax_sync_ce solo puede reducirse hasta -15%; no se inventa capacidad.
P_SYNC_LB_FACTOR = 0.85
P_SYNC_UB_FACTOR = 1.00

# ============================================================
# 1) PREPARACIÓN GENERAL IEEE57
# ============================================================

L_arcs = list(L_id)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [] for bus in B}
lineas_salen  = {bus: [] for bus in B}

for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    if j in lineas_entran:
        lineas_entran[j].append(ell)

    if i in lineas_salen:
        lineas_salen[i].append(ell)

bus_slack = int(sorted(B)[0])

# ============================================================
# REFERENCIAS WELL-POSED DEL FW
# ============================================================
# Este bloque hace que el PADM-W + WCEP sea consistente con el FW
# well-posed. Si el FW anterior ya creó Hreq_fw, se usa directamente.
# Si no existe, se reconstruye con el criterio usado en el FW:
#   HREQ_ALPHA_WELLPOSED / HREQ_ALPHA_ORIGINAL = 0.28 / 0.35.
#
# Importante:
#   - NO se usa zH.
#   - Hreq_ce sigue estando en el RHS/b.
#   - Hreq_base ahora corresponde al Hreq well-posed, no al Hreq original.
# ============================================================

if "Hreq_fw" in globals():
    Hreq_ref = {y: float(Hreq_fw[y]) for y in Y}
    HREQ_REFERENCE_LABEL = "Hreq_fw existente del FW well-posed"
else:
    HREQ_ALPHA_ORIGINAL = 0.35
    HREQ_ALPHA_WELLPOSED = 0.28
    HREQ_SCALE_FACTOR = HREQ_ALPHA_WELLPOSED / HREQ_ALPHA_ORIGINAL
    Hreq_ref = {y: HREQ_SCALE_FACTOR * float(Hreq[y]) for y in Y}
    Hreq_fw = Hreq_ref.copy()
    HREQ_REFERENCE_LABEL = (
        f"Hreq_fw reconstruido automáticamente: "
        f"{HREQ_ALPHA_WELLPOSED}/{HREQ_ALPHA_ORIGINAL} = {HREQ_SCALE_FACTOR:.6f}"
    )

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE57
# ============================================================
# Si existe d_fw desde el FW corregido, se usa. Si no existe, se
# reconstruye con el mismo criterio usado en el FW:
#   demand_scale_factor = min(1, 0.98 * sum(Pmax) / demanda_pico_original)

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 1.0))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

d_ref = d_fw
DEMAND_REFERENCE_LABEL = "d_fw"

Hreq_base = {y: float(Hreq_ref[y]) for y in Y}
Pmax_sync_base = {g: float(Pmax[g]) for g in GS}

print("\n" + "="*70)
print("REFERENCIAS USADAS POR WCEP IEEE57")
print("="*70)
print("Referencia demanda:", DEMAND_REFERENCE_LABEL)
print("Referencia Hreq   :", HREQ_REFERENCE_LABEL)
print("Cotas Pmax_sync_ce: [0.85, 1.00] alrededor de Pmax_base")
print("Hreq queda fijo en Hreq_base/well-posed")
for y in Y:
    print(f"Año {y}: Hreq original = {float(Hreq[y]):.6f} | Hreq_base/well-posed = {float(Hreq_base[y]):.6f}")

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_sync = 2.0 * max(max(float(cg[g]) for g in G), float(VOLL))

print("\n" + "="*70)
print("COTAS DUALES IEEE57")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_sync = {UB_sync:.6f}")
print(f"Bus slack usado: {bus_slack}")


print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE57 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE57_DF_SOLO_Pmax_sync")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Pmax_sync_ce = w.addVars(GS, lb=0.0, name="Pmax_sync_ce")

for g in GS:
    base = Pmax_sync_base[g]
    w.addConstr(Pmax_sync_ce[g] >= P_SYNC_LB_FACTOR * base, name=f"adm_lb_Psync[{g}]")
    w.addConstr(Pmax_sync_ce[g] <= P_SYNC_UB_FACTOR * base, name=f"adm_ub_Psync[{g}]")


# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[ell, t, y] - f_neg[ell, t, y]
                    for ell in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[ell, t, y] - f_neg[ell, t, y]
                    for ell in lineas_salen[i]
                )
            )

            rhs_val = float(d_ref[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_ref[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    for t in T:
        for y in Y:
            Bij = float(Bline[ell])

            flow = f_pos[ell, t, y] - f_neg[ell, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for ell in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[ell])
            flow = f_pos[ell, t, y] - f_neg[ell, t, y]

            name = f"flb_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addQConstr(
                Pmax_sync_ce[g] * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -float(xbar[m]), name=name)
        RHS_expr[name] = -float(xbar[m])

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= float(Hreq_base[y]), name=name)
        RHS_expr[name] = float(Hreq_base[y])

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= epsilon_CO2_wcep,
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sync_ub_"):
        ub_here = UB_sync

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dPsync = w.addVars(GS, lb=0.0, name="dPsync")

for g in GS:
    base = Pmax_sync_base[g]
    diff = Pmax_sync_ce[g] - base

    w.addConstr(dPsync[g] >= diff, name=f"dPsync_pos[{g}]")
    w.addConstr(dPsync[g] >= -diff, name=f"dPsync_neg[{g}]")


if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for g in GS:
        base = abs(float(Pmax_sync_base[g]))
        if base > 1e-9:
            obj_wcep += alpha_Psync * dPsync[g] / base
        else:
            obj_wcep += alpha_Psync * dPsync[g]


    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_Psync * dPsync.sum(),
        GRB.MINIMIZE
    )


# ============================================================
# 11) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE57")
print("="*70)

w.optimize()

# Registro de tiempos/GAP WCEP
runtime_wcep = float(w.Runtime)
status_wcep = int(w.Status)
sol_count_wcep = int(w.SolCount)

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP IEEE57")
print("="*70)
print(f"Status Gurobi = {status_wcep}")
print(f"Tiempo optimize() = {runtime_wcep:.2f} s = {runtime_wcep/60:.2f} min = {runtime_wcep/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count_wcep}")

if sol_count_wcep > 0:
    print(f"Mejor objetivo encontrado = {float(w.ObjVal):.8f}")
    print(f"Mejor bound               = {float(w.ObjBound):.8f}")
    print(f"MIPGap                    = {float(w.MIPGap):.6e}")
    print(f"MIPGap (%)                = {100*float(w.MIPGap):.4f}%")
else:
    print("No se encontró solución factible.")

total_script_time = time.time() - t0_total_script
print(f"Tiempo total script = {total_script_time:.2f} s = {total_script_time/60:.2f} min = {total_script_time/3600:.2f} h")


# ============================================================
# 12) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE57")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Pmax_sync_ce síncronos ---")
    print(f"Cotas usadas: {P_SYNC_LB_FACTOR:.2f}*Pmax_base <= Pmax_sync_ce <= {P_SYNC_UB_FACTOR:.2f}*Pmax_base")
    hubo_cambio = False

    for g in GS:
        val = float(Pmax_sync_ce[g].X)
        base = float(Pmax_sync_base[g])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{g}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- Hreq fijo ---")
    for y in Y:
        print(f"Año {y}: Hreq_base fijo = {float(Hreq_base[y]):.6f}")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = epsilon_CO2_wcep - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {epsilon_CO2_wcep:.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_base[y])

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_base={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_sync = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("sync_ub_"):
            if abs(var.X - UB_sync) <= 1e-6 * max(1.0, UB_sync):
                hits_sync += 1

    print(f"Duales sys_H pegadas a cota   : {hits_H}")
    print(f"Duales sync_ub pegadas a cota : {hits_sync}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

### (Variando A) Dualidad fuerte + PADM

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import copy
import math
import time

# ============================================================
# FINAL IEEE57 WELL-POSED: PADM-W + WCEP CON DUALIDAD FUERTE
# CASO: SOLO Pmax SÍNCRONO MUTABLE
#
#   sum_g Hg[g] u[g,t,y] + sum_m h_syn[m,t,y] >= Hreq_base[y]
#
# Es decir:
#   - NO se usa zH.
#   - Hreq queda fijo en b.
#   - Pmax_sync_ce queda en A y es el único parámetro mutable.
#
# Flujo:
#   1) PADM-W etapa 1: busca factibilidad.
#   2) PADM-W etapa 2: parte de factibilidad y reduce J(theta).
#   3) WCEP completo con dualidad fuerte usando PADM_WARMSTART.
# ============================================================

t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

PADM_OUTPUT = 1
RUN_PADM_W = True

MAX_OUTER = 20
MAX_INNER = 20

FEAS_TOL = 1e-5
INNER_TOL = 1e-5

RHO0 = 500.0
MU0 = 500.0

PENALTY_FACTOR = 2.0
MAX_PENALTY = 1e9

TIME_LIMIT_SUBPROBLEM = 5 * 60 * 60  # 5 horas por subproblema PADM

# Restricción D del IEEE57
epsilon_CO2_wcep = 766

# Objetivo J
alpha_Psync = 1.0
# alpha_H ya no se usa: Hreq queda fijo.

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1
WCEP_TIME_LIMIT = 5 * 60 * 60  # 5 horas WCEP final

# Para solo Pmax en A, no imponemos dualidad fuerte dura en theta
# porque b no depende de theta y theta no puede reparar c^Tz <= b^Tlambda.
ENFORCE_THETA_STRONG_DUALITY = False
SD_THETA_PENALTY = 1e3

# Cotas de mutabilidad
# Pmax_sync_ce solo puede reducirse hasta -15%; no se inventa capacidad.
P_SYNC_LB_FACTOR = 0.85
P_SYNC_UB_FACTOR = 1.00


# ============================================================
# 1) PREPARACIÓN GENERAL IEEE57
# ============================================================

L_arcs = list(L_id)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [] for bus in B}
lineas_salen  = {bus: [] for bus in B}

for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    if j in lineas_entran:
        lineas_entran[j].append(ell)

    if i in lineas_salen:
        lineas_salen[i].append(ell)

bus_slack = int(sorted(B)[0])

# ============================================================
# REFERENCIAS WELL-POSED DEL FW
# ============================================================
# Este bloque hace que el PADM-W + WCEP sea consistente con el FW
# well-posed. Si el FW anterior ya creó Hreq_fw, se usa directamente.
# Si no existe, se reconstruye con el criterio usado en el FW:
#   HREQ_ALPHA_WELLPOSED / HREQ_ALPHA_ORIGINAL = 0.28 / 0.35.
#
# Importante:
#   - NO se usa zH.
#   - Hreq_ce sigue estando en el RHS/b.
#   - Hreq_base ahora corresponde al Hreq well-posed, no al Hreq original.
# ============================================================

if "Hreq_fw" in globals():
    Hreq_ref = {y: float(Hreq_fw[y]) for y in Y}
    HREQ_REFERENCE_LABEL = "Hreq_fw existente del FW well-posed"
else:
    HREQ_ALPHA_ORIGINAL = 0.35
    HREQ_ALPHA_WELLPOSED = 0.28
    HREQ_SCALE_FACTOR = HREQ_ALPHA_WELLPOSED / HREQ_ALPHA_ORIGINAL
    Hreq_ref = {y: HREQ_SCALE_FACTOR * float(Hreq[y]) for y in Y}
    Hreq_fw = Hreq_ref.copy()
    HREQ_REFERENCE_LABEL = (
        f"Hreq_fw reconstruido automáticamente: "
        f"{HREQ_ALPHA_WELLPOSED}/{HREQ_ALPHA_ORIGINAL} = {HREQ_SCALE_FACTOR:.6f}"
    )

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE57
# ============================================================
# Si existe d_fw desde el FW corregido, se usa. Si no existe, se
# reconstruye con el mismo criterio usado en el FW:
#   demand_scale_factor = min(1, 0.98 * sum(Pmax) / demanda_pico_original)

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 1.0))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

d_ref = d_fw
DEMAND_REFERENCE_LABEL = "d_fw"

Hreq_base = {y: float(Hreq_ref[y]) for y in Y}
Pmax_sync_base = {g: float(Pmax[g]) for g in GS}

print("\n" + "="*70)
print("REFERENCIAS USADAS POR PADM-W + WCEP IEEE57")
print("="*70)
print("Referencia demanda:", DEMAND_REFERENCE_LABEL)
print("Referencia Hreq   :", HREQ_REFERENCE_LABEL)
print("Cotas Pmax_sync_ce: [0.85, 1.00] alrededor de Pmax_base")
print("Hreq queda fijo en Hreq_base/well-posed")
for y in Y:
    print(f"Año {y}: Hreq original = {float(Hreq[y]):.6f} | Hreq_base/well-posed = {float(Hreq_base[y]):.6f}")

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_sync = 2.0 * max(max(float(cg[g]) for g in G), float(VOLL))

print("\n" + "="*70)
print("COTAS DUALES IEEE57")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_sync = {UB_sync:.6f}")
print(f"Bus slack usado: {bus_slack}")


# ============================================================
# 2) REPRESENTACIÓN SIMBÓLICA
# ============================================================

def key_p(g,t,y):       return ("p", g, t, y)
def key_u(g,t,y):       return ("u", g, t, y)
def key_s(i,t,y):       return ("s", i, t, y)
def key_xcap(m,y):      return ("xcap", m, y)
def key_hsyn(m,t,y):    return ("h_syn", m, t, y)
def key_rffr(m,t,y):    return ("r_ffr", m, t, y)
def key_fpos(ell,t,y):  return ("f_pos", ell, t, y)
def key_fneg(ell,t,y):  return ("f_neg", ell, t, y)
def key_thp(i,t,y):     return ("theta_p", i, t, y)
def key_thm(i,t,y):     return ("theta_m", i, t, y)

def param_coeff(family, idx, mult=1.0):
    return ("PARAM", family, idx, float(mult))

def is_param_coeff(c):
    return isinstance(c, tuple) and len(c) == 4 and c[0] == "PARAM"

def coeff_value(c, theta_vals):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * float(theta_vals[fam][idx])
    return float(c)

def coeff_expr_in_theta_model(c, theta_vars):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * theta_vars[fam][idx]
    return float(c)

def rhs_const(val):
    return [("CONST", None, None, float(val))]

def rhs_param(family, idx, mult=1.0):
    return [("PARAM", family, idx, float(mult))]

def rhs_has_param(rhs_terms):
    return any(typ == "PARAM" for typ, fam, idx, mult in rhs_terms)

def rhs_eval(rhs_terms, theta_vals=None):
    out = 0.0

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            out += float(mult)
        elif typ == "PARAM":
            if theta_vals is None:
                raise ValueError("rhs_eval requiere theta_vals para RHS con parámetros.")
            out += float(mult) * float(theta_vals[fam][idx])
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return out

def rhs_expr_in_theta_model(rhs_terms, theta_vars):
    expr = gp.LinExpr()

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            expr += float(mult)
        elif typ == "PARAM":
            expr += float(mult) * theta_vars[fam][idx]
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return expr

def add_to_terms(terms, key, coeff):
    if key not in terms:
        terms[key] = []
    terms[key].append(coeff)

def neg_coeff(c):
    if is_param_coeff(c):
        return ("PARAM", c[1], c[2], -c[3])
    return -float(c)

def terms_has_param(terms):
    for k, coeffs in terms.items():
        for c in coeffs:
            if is_param_coeff(c):
                return True
    return False

def terms_expr_z_model(terms, z_vars, theta_vals):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        for c in coeffs:
            expr += coeff_value(c, theta_vals) * z_vars[k]

    return expr

def terms_expr_theta_model_with_fixed_z(terms, z_vals, theta_vars):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            expr += zval * coeff_expr_in_theta_model(c, theta_vars)

    return expr

def terms_eval(terms, z_vals, theta_vals):
    out = 0.0

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            out += coeff_value(c, theta_vals) * zval

    return out


# ============================================================
# 3) CREAR FILAS PRIMALES Ax >= b
# ============================================================

rows = {}
row_order = []

def add_row(name, terms, rhs_terms):
    if name in rows:
        raise ValueError(f"Fila repetida: {name}")

    rows[name] = {
        "terms": terms,
        "rhs": rhs_terms
    }

    row_order.append(name)

# ------------------------------------------------------------
# A.1 Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}

            for g in G_en_bus.get(i, []):
                add_to_terms(terms, key_p(g,t,y), 1.0)

            add_to_terms(terms, key_s(i,t,y), 1.0)

            for ell in lineas_entran[i]:
                add_to_terms(terms, key_fpos(ell,t,y), 1.0)
                add_to_terms(terms, key_fneg(ell,t,y), -1.0)

            for ell in lineas_salen[i]:
                add_to_terms(terms, key_fpos(ell,t,y), -1.0)
                add_to_terms(terms, key_fneg(ell,t,y), 1.0)

            rhs_val = float(d_ref[(i,t,y)])

            add_row(f"bal_p_{i}_{t}_{y}", terms, rhs_const(rhs_val))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"bal_n_{i}_{t}_{y}", terms_n, rhs_const(-rhs_val))

# ------------------------------------------------------------
# A.2 ENS <= demanda
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}
            add_to_terms(terms, key_s(i,t,y), -1.0)
            add_row(f"ens_ub_{i}_{t}_{y}", terms, rhs_const(-float(d_ref[(i,t,y)])))

# ------------------------------------------------------------
# A.3 Flujo DC
# ------------------------------------------------------------
for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    for t in T:
        for y in Y:
            Bij = float(Bline[ell])

            terms = {}
            add_to_terms(terms, key_fpos(ell,t,y), 1.0)
            add_to_terms(terms, key_fneg(ell,t,y), -1.0)

            add_to_terms(terms, key_thp(i,t,y), -Bij)
            add_to_terms(terms, key_thm(i,t,y),  Bij)
            add_to_terms(terms, key_thp(j,t,y),  Bij)
            add_to_terms(terms, key_thm(j,t,y), -Bij)

            add_row(f"dc_p_{ell}_{t}_{y}", terms, rhs_const(0.0))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"dc_n_{ell}_{t}_{y}", terms_n, rhs_const(0.0))

# ------------------------------------------------------------
# A.4 Límites térmicos fijos
# ------------------------------------------------------------
for ell in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[ell])

            terms_lb = {}
            add_to_terms(terms_lb, key_fpos(ell,t,y), 1.0)
            add_to_terms(terms_lb, key_fneg(ell,t,y), -1.0)
            add_row(f"flb_{ell}_{t}_{y}", terms_lb, rhs_const(-Fij))

            terms_ub = {}
            add_to_terms(terms_ub, key_fpos(ell,t,y), -1.0)
            add_to_terms(terms_ub, key_fneg(ell,t,y), 1.0)
            add_row(f"fub_{ell}_{t}_{y}", terms_ub, rhs_const(-Fij))

# ------------------------------------------------------------
# A.5 Slack bus
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_p = {}
        add_to_terms(terms_p, key_thp(bus_slack,t,y), 1.0)
        add_to_terms(terms_p, key_thm(bus_slack,t,y), -1.0)
        add_row(f"slack_p_{t}_{y}", terms_p, rhs_const(0.0))

        terms_n = {}
        add_to_terms(terms_n, key_thp(bus_slack,t,y), -1.0)
        add_to_terms(terms_n, key_thm(bus_slack,t,y), 1.0)
        add_row(f"slack_n_{t}_{y}", terms_n, rhs_const(0.0))

print(f"Bus slack usado: {bus_slack}")

# ------------------------------------------------------------
# A.6 Límites generación fijos p <= P_disp
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_base = float(P_disp.get((g,t), Pmax[g]))

        for y in Y:
            terms = {}
            add_to_terms(terms, key_p(g,t,y), -1.0)
            add_row(f"p_cap_{g}_{t}_{y}", terms, rhs_const(-cap_base))

# ------------------------------------------------------------
# A.7 Síncronos
# ------------------------------------------------------------
for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            terms_lb = {}
            add_to_terms(terms_lb, key_p(g,t,y), 1.0)
            add_to_terms(terms_lb, key_u(g,t,y), -pmin_g)
            add_row(f"sync_lb_{g}_{t}_{y}", terms_lb, rhs_const(0.0))

            terms_ub = {}
            add_to_terms(terms_ub, key_u(g,t,y), param_coeff("Pmax_sync_ce", g, 1.0))
            add_to_terms(terms_ub, key_p(g,t,y), -1.0)
            add_row(f"sync_ub_{g}_{t}_{y}", terms_ub, rhs_const(0.0))

            terms_u = {}
            add_to_terms(terms_u, key_u(g,t,y), -1.0)
            add_row(f"u_ub_{g}_{t}_{y}", terms_u, rhs_const(-1.0))

# ------------------------------------------------------------
# A.8-A.10 Inversión, h_syn, FFR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        terms_x = {}
        add_to_terms(terms_x, key_xcap(m,y), -1.0)
        add_row(f"x_ub_{m}_{y}", terms_x, rhs_const(-float(xbar[m])))

        for t in T:
            terms_h = {}
            add_to_terms(terms_h, key_xcap(m,y), float(gamma_base[m]))
            add_to_terms(terms_h, key_hsyn(m,t,y), -1.0)
            add_row(f"hsyn_lim_{m}_{t}_{y}", terms_h, rhs_const(0.0))

            terms_r = {}
            add_to_terms(terms_r, key_xcap(m,y), float(kappa_base[m]))
            add_to_terms(terms_r, key_rffr(m,t,y), -1.0)
            add_row(f"ffr_lim_{m}_{t}_{y}", terms_r, rhs_const(0.0))

# ------------------------------------------------------------
# A.11 Seguridad: Hreq fijo en b, Rreq fijo
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_H = {}

        for g in GS:
            add_to_terms(terms_H, key_u(g,t,y), float(Hg.get(g, 0.0)))

        for m in M:
            add_to_terms(terms_H, key_hsyn(m,t,y), 1.0)

        add_row(f"sys_H_{t}_{y}", terms_H, rhs_const(float(Hreq_base[y])))

        terms_R = {}

        for m in M:
            add_to_terms(terms_R, key_rffr(m,t,y), 1.0)

        add_row(f"sys_R_{t}_{y}", terms_R, rhs_const(float(Rreq[y])))


# ============================================================
# 4) VARIABLES PRIMALES Y COSTOS c
# ============================================================

primal_keys = []

for g in G:
    for t in T:
        for y in Y:
            primal_keys.append(key_p(g,t,y))

for g in GS:
    for t in T:
        for y in Y:
            primal_keys.append(key_u(g,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_s(i,t,y))

for m in M:
    for y in Y:
        primal_keys.append(key_xcap(m,y))

for m in M:
    for t in T:
        for y in Y:
            primal_keys.append(key_hsyn(m,t,y))
            primal_keys.append(key_rffr(m,t,y))

for ell in L_arcs:
    for t in T:
        for y in Y:
            primal_keys.append(key_fpos(ell,t,y))
            primal_keys.append(key_fneg(ell,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_thp(i,t,y))
            primal_keys.append(key_thm(i,t,y))

c_map = {}

for g in G:
    for t in T:
        for y in Y:
            c_map[key_p(g,t,y)] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map[key_s(i,t,y)] = float(VOLL)

for m in M:
    for y in Y:
        c_map[key_xcap(m,y)] = float(cinv[m])

for k in primal_keys:
    c_map.setdefault(k, 0.0)


# ============================================================
# 5) CLASIFICACIÓN ACOPLADAS VS SEPARABLES
# ============================================================

coupled_primal_rows = [
    name for name in row_order
    if terms_has_param(rows[name]["terms"]) or rhs_has_param(rows[name]["rhs"])
]

separable_primal_rows = [
    name for name in row_order
    if name not in set(coupled_primal_rows)
]

coupled_dual_cols = set()

for name in row_order:
    terms = rows[name]["terms"]

    for k, coeffs in terms.items():
        if any(is_param_coeff(c) for c in coeffs):
            coupled_dual_cols.add(k)

coupled_dual_cols = sorted(list(coupled_dual_cols), key=str)

separable_dual_cols = [
    k for k in primal_keys
    if k not in set(coupled_dual_cols)
]

print("\n" + "="*70)
print("CLASIFICACIÓN PADM-W IEEE57 - FORMULACIÓN ORIGINAL")
print("="*70)
print(f"N° filas primales acopladas   : {len(coupled_primal_rows)}")
print(f"N° filas primales separables  : {len(separable_primal_rows)}")
print(f"N° columnas duales acopladas  : {len(coupled_dual_cols)}")
print(f"N° columnas duales separables : {len(separable_dual_cols)}")

bad_rows = [
    name for name in coupled_primal_rows
    if not name.startswith("sync_ub_")
]

if bad_rows:
    print("\n⚠️ Hay filas acopladas no esperadas:")
    print(bad_rows[:30])
else:
    print("\n✅ Filas acopladas esperadas: solo sync_ub_.")


# ============================================================
# 6) FUNCIONES PADM-W
# ============================================================

def initial_theta_vals():
    return {
        "Pmax_sync_ce": {g: float(Pmax_sync_base[g]) for g in GS},
    }

def compute_J(theta_vals):
    J = 0.0

    for g in GS:
        base = float(Pmax_sync_base[g])
        val = float(theta_vals["Pmax_sync_ce"][g])

        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_Psync * abs(val - base) / abs(base)
        else:
            J += alpha_Psync * abs(val - base)


    return J

def compute_ctx(z_vals):
    return sum(
        float(c_map[k]) * float(z_vals.get(k, 0.0))
        for k in primal_keys
    )

def compute_bty(theta_vals, ydual_vals):
    out = 0.0

    for name in row_order:
        out += rhs_eval(rows[name]["rhs"], theta_vals) * float(ydual_vals.get(name, 0.0))

    return out

def compute_primal_residuals(theta_vals, z_vals):
    res = {}

    for name in row_order:
        lhs = terms_eval(rows[name]["terms"], z_vals, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)
        res[name] = max(0.0, rhs - lhs)

    return res

def compute_dual_residuals(theta_vals, ydual_vals):
    res = {}

    for k in primal_keys:
        lhs = 0.0

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            yd = float(ydual_vals.get(name, 0.0))

            if abs(yd) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd

        rhs = float(c_map.get(k, 0.0))
        res[k] = max(0.0, lhs - rhs)

    return res

def compute_sd_residual(theta_vals, z_vals, ydual_vals):
    return max(0.0, compute_ctx(z_vals) - compute_bty(theta_vals, ydual_vals))

def max_residuals(theta_vals, z_vals, ydual_vals):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)
    sd = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return {
        "max_primal_all": max(pr.values()) if pr else 0.0,
        "max_dual_all": max(dr.values()) if dr else 0.0,
        "sd": sd,
        "max_primal_coupled": max([pr[name] for name in coupled_primal_rows], default=0.0),
        "max_dual_coupled": max([dr[k] for k in coupled_dual_cols], default=0.0),
    }

def compute_phi(theta_vals, z_vals, ydual_vals, rho, mu, use_true_objective=True):
    Jval = compute_J(theta_vals) if use_true_objective else 0.0

    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    pen_p = sum(float(rho[name]) * pr[name] for name in coupled_primal_rows)
    pen_d = sum(float(mu[k]) * dr[k] for k in coupled_dual_cols)

    sd_penalty = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return Jval + pen_p + pen_d + 1e3 * sd_penalty

def diff_theta(theta_a, theta_b):
    vals = []

    for g in GS:
        vals.append(abs(theta_a["Pmax_sync_ce"][g] - theta_b["Pmax_sync_ce"][g]))


    return max(vals) if vals else 0.0

def diff_dict(a, b):
    keys = set(a.keys()) | set(b.keys())
    return max(
        [abs(float(a.get(k,0.0)) - float(b.get(k,0.0))) for k in keys],
        default=0.0
    )


# ============================================================
# 7) SUBPROBLEMAS PADM-W
# ============================================================

def add_z_variables(model):
    z = {}

    for g in G:
        for t in T:
            for y in Y:
                z[key_p(g,t,y)] = model.addVar(lb=0.0, name=f"p[{g},{t},{y}]")

    for g in GS:
        for t in T:
            for y in Y:
                z[key_u(g,t,y)] = model.addVar(lb=0.0, ub=1.0, name=f"u[{g},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_s(i,t,y)] = model.addVar(lb=0.0, name=f"s[{i},{t},{y}]")

    for m in M:
        for y in Y:
            z[key_xcap(m,y)] = model.addVar(lb=0.0, name=f"xcap[{m},{y}]")

    for m in M:
        for t in T:
            for y in Y:
                z[key_hsyn(m,t,y)] = model.addVar(lb=0.0, name=f"h_syn[{m},{t},{y}]")
                z[key_rffr(m,t,y)] = model.addVar(lb=0.0, name=f"r_ffr[{m},{t},{y}]")

    for ell in L_arcs:
        for t in T:
            for y in Y:
                z[key_fpos(ell,t,y)] = model.addVar(lb=0.0, name=f"f_pos[{ell},{t},{y}]")
                z[key_fneg(ell,t,y)] = model.addVar(lb=0.0, name=f"f_neg[{ell},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_thp(i,t,y)] = model.addVar(lb=0.0, name=f"theta_p[{i},{t},{y}]")
                z[key_thm(i,t,y)] = model.addVar(lb=0.0, name=f"theta_m[{i},{t},{y}]")

    return z

def add_dual_variables(model):
    yd = {}

    for name in row_order:
        ub_here = GRB.INFINITY

        if name.startswith("sync_ub_"):
            ub_here = UB_sync

        yd[name] = model.addVar(lb=0.0, ub=ub_here, name=f"ydual[{name}]")

    return yd

def add_theta_variables(model, warm_theta=None):
    theta = {
        "Pmax_sync_ce": {},
    }

    for g in GS:
        theta["Pmax_sync_ce"][g] = model.addVar(
            lb=P_SYNC_LB_FACTOR * float(Pmax_sync_base[g]),
            ub=P_SYNC_UB_FACTOR * float(Pmax_sync_base[g]),
            name=f"Pmax_sync_ce[{g}]"
        )


    if warm_theta is not None:
        for g in GS:
            theta["Pmax_sync_ce"][g].Start = float(warm_theta["Pmax_sync_ce"][g])

    return theta

def extract_theta(theta_vars):
    return {
        "Pmax_sync_ce": {g: float(theta_vars["Pmax_sync_ce"][g].X) for g in GS},
    }

def add_separable_primal_constraints_y(model, z_vars):
    theta_base = initial_theta_vals()

    for name in separable_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z_vars, theta_base)
        rhs = rhs_eval(rows[name]["rhs"], theta_base)
        model.addConstr(lhs >= rhs, name=f"Y_primal[{name}]")

def add_D_constraint_y(model, z_vars):
    for y in Y:
        if y >= y_star:
            model.addConstr(
                gp.quicksum(
                    float(eg[g]) * z_vars[key_p(g,t,y)]
                    for g in GE
                    for t in T
                )
                <= float(epsilon_CO2_wcep),
                name=f"D_CO2_cap_{y}"
            )

def add_separable_dual_constraints_y(model, ydual_vars):
    theta_base = initial_theta_vals()

    for k in separable_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_base) * ydual_vars[name]

        model.addConstr(lhs <= float(c_map.get(k, 0.0)), name=f"Y_dual[{k}]")

def add_strong_duality_y(model, theta_vals, z_vars, ydual_vars):
    ctx = gp.quicksum(float(c_map[k]) * z_vars[k] for k in primal_keys)

    bty = gp.LinExpr()

    for name in row_order:
        bty += rhs_eval(rows[name]["rhs"], theta_vals) * ydual_vars[name]

    model.addConstr(ctx <= bty, name="Y_StrongDuality")

def solve_theta_subproblem(
    z_vals,
    ydual_vals,
    rho,
    mu,
    warm_theta=None,
    log=False,
    use_true_objective=True
):
    m = gp.Model("PADM_theta_subproblem_IEEE57_wellposed_original")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    theta = add_theta_variables(m, warm_theta=warm_theta)

    obj = gp.LinExpr()

    if use_true_objective:
        for g in GS:
            dvar = m.addVar(lb=0.0, name=f"dPsync[{g}]")
            diff = theta["Pmax_sync_ce"][g] - float(Pmax_sync_base[g])

            m.addConstr(dvar >= diff, name=f"dPsync_pos[{g}]")
            m.addConstr(dvar >= -diff, name=f"dPsync_neg[{g}]")

            base = abs(float(Pmax_sync_base[g]))

            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_Psync) * dvar / base
            else:
                obj += float(alpha_Psync) * dvar


    for name in coupled_primal_rows:
        lhs = terms_expr_theta_model_with_fixed_z(rows[name]["terms"], z_vals, theta)
        rhs = rhs_expr_in_theta_model(rows[name]["rhs"], theta)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            ydval = float(ydual_vals.get(name, 0.0))

            if abs(ydval) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += ydval * coeff_expr_in_theta_model(c, theta)

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    # Dualidad fuerte dura en theta:
    #   c^T z fijo <= b(theta)^T y fijo
    ctx_fixed = sum(float(c_map[k]) * float(z_vals.get(k, 0.0)) for k in primal_keys)

    bty_theta = gp.LinExpr()

    for name in row_order:
        ydval = float(ydual_vals.get(name, 0.0))

        if abs(ydval) <= 1e-12:
            continue

        bty_theta += ydval * rhs_expr_in_theta_model(rows[name]["rhs"], theta)

    if ENFORCE_THETA_STRONG_DUALITY:
        m.addConstr(ctx_fixed <= bty_theta, name="Theta_StrongDuality")
    else:
        v_sd = m.addVar(lb=0.0, name="viol_strong_duality_theta")
        m.addConstr(v_sd >= ctx_fixed - bty_theta, name="def_viol_strong_duality_theta")
        obj += float(SD_THETA_PENALTY) * v_sd

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema THETA falló.")
        print(f"Status = {m.Status}")
        return None, m.Status, None

    theta_vals = extract_theta(theta)
    return theta_vals, m.Status, float(m.ObjVal)

def solve_y_subproblem(theta_vals, rho, mu, warm_z=None, warm_ydual=None, log=False):
    m = gp.Model("PADM_y_subproblem_IEEE57_wellposed_original")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    z = add_z_variables(m)
    yd = add_dual_variables(m)

    m.update()

    if warm_z is not None:
        for k, v in z.items():
            if k in warm_z:
                v.Start = float(warm_z[k])

    if warm_ydual is not None:
        for name, v in yd.items():
            if name in warm_ydual:
                v.Start = float(warm_ydual[name])

    add_separable_primal_constraints_y(m, z)
    add_D_constraint_y(m, z)
    add_separable_dual_constraints_y(m, yd)
    add_strong_duality_y(m, theta_vals, z, yd)

    obj = gp.LinExpr()

    for name in coupled_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd[name]

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    if Y_OBJECTIVE_MODE == "ctx":
        obj += TIEBREAKER_CTX_WEIGHT * gp.quicksum(
            float(c_map[k]) * z[k]
            for k in primal_keys
        )

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema Y falló.")
        print(f"Status = {m.Status}")
        return None, None, m.Status, None

    z_vals = {k: float(v.X) for k, v in z.items()}
    yd_vals = {name: float(v.X) for name, v in yd.items()}

    return z_vals, yd_vals, m.Status, float(m.ObjVal)

def initialize_penalties():
    rho = {name: float(RHO0) for name in coupled_primal_rows}
    mu = {k: float(MU0) for k in coupled_dual_cols}
    return rho, mu

def update_penalties(theta_vals, z_vals, ydual_vals, rho, mu):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    n_rho = 0
    n_mu = 0

    for name in coupled_primal_rows:
        if pr[name] > FEAS_TOL:
            rho[name] = min(MAX_PENALTY, PENALTY_FACTOR * rho[name])
            n_rho += 1

    for k in coupled_dual_cols:
        if dr[k] > FEAS_TOL:
            mu[k] = min(MAX_PENALTY, PENALTY_FACTOR * mu[k])
            n_mu += 1

    return n_rho, n_mu


# ============================================================
# 8) LOOP PADM-W
# ============================================================

def run_padm(
    use_true_objective=True,
    theta_init=None,
    z_init=None,
    ydual_init=None,
    label="PADM"
):
    print("\n" + "="*70)
    print(f"INICIO {label}")
    print("="*70)
    print(f"use_true_objective = {use_true_objective}")

    rho, mu = initialize_penalties()

    theta_vals = copy.deepcopy(theta_init) if theta_init is not None else initial_theta_vals()

    if z_init is None or ydual_init is None:
        print("\nInicializando Y=(z,lambda)...")

        z_vals, ydual_vals, st_y, obj_y = solve_y_subproblem(
            theta_vals,
            rho,
            mu,
            warm_z=None,
            warm_ydual=None,
            log=False
        )

        if z_vals is None:
            print("❌ No se pudo inicializar Y.")

            return {
                "best_feasible": None,
                "theta": theta_vals,
                "z": None,
                "ydual": None,
                "history": [],
                "last_residuals": None,
                "rho": rho,
                "mu": mu,
            }

    else:
        z_vals = z_init.copy()
        ydual_vals = ydual_init.copy()

    history = []
    best_feasible = None

    res0 = max_residuals(theta_vals, z_vals, ydual_vals)

    print("\nChequeo inicial:")
    print(f"J(theta)       = {compute_J(theta_vals):.8f}")
    print(f"Max primal     = {res0['max_primal_all']:.3e}")
    print(f"Max dual       = {res0['max_dual_all']:.3e}")
    print(f"Strong duality = {res0['sd']:.3e}")

    for k_outer in range(MAX_OUTER):

        print("\n" + "#"*70)
        print(f"{label} | OUTER k = {k_outer}")
        print("#"*70)

        phi_prev = compute_phi(
            theta_vals,
            z_vals,
            ydual_vals,
            rho,
            mu,
            use_true_objective=use_true_objective
        )

        for l_inner in range(MAX_INNER):

            theta_old = copy.deepcopy(theta_vals)
            z_old = z_vals.copy()
            yd_old = ydual_vals.copy()

            theta_new, st_theta, obj_theta = solve_theta_subproblem(
                z_vals,
                ydual_vals,
                rho,
                mu,
                warm_theta=theta_vals,
                log=False,
                use_true_objective=use_true_objective
            )

            if theta_new is None:
                print("⚠️ Falló subproblema theta.")
                break

            theta_vals = theta_new

            z_new, yd_new, st_y, obj_y = solve_y_subproblem(
                theta_vals,
                rho,
                mu,
                warm_z=z_vals,
                warm_ydual=ydual_vals,
                log=False
            )

            if z_new is None:
                print("⚠️ Falló subproblema Y.")
                break

            z_vals = z_new
            ydual_vals = yd_new

            phi_now = compute_phi(
                theta_vals,
                z_vals,
                ydual_vals,
                rho,
                mu,
                use_true_objective=use_true_objective
            )

            res = max_residuals(theta_vals, z_vals, ydual_vals)

            dtheta = diff_theta(theta_vals, theta_old)
            dz = diff_dict(z_vals, z_old)
            dyd = diff_dict(ydual_vals, yd_old)
            dphi = abs(phi_now - phi_prev) / max(1.0, abs(phi_prev))

            if PADM_OUTPUT:
                print(
                    f"k={k_outer:02d}, l={l_inner:02d} | "
                    f"Phi={phi_now:.6e}, J={compute_J(theta_vals):.6e}, "
                    f"p_all={res['max_primal_all']:.2e}, "
                    f"d_all={res['max_dual_all']:.2e}, "
                    f"sd={res['sd']:.2e}, "
                    f"p_coup={res['max_primal_coupled']:.2e}, "
                    f"d_coup={res['max_dual_coupled']:.2e}, "
                    f"dtheta={dtheta:.2e}, dz={dz:.2e}, dyd={dyd:.2e}, dphi={dphi:.2e}"
                )

            history.append({
                "outer": k_outer,
                "inner": l_inner,
                "phi": phi_now,
                "J": compute_J(theta_vals),
                "max_primal_all": res["max_primal_all"],
                "max_dual_all": res["max_dual_all"],
                "sd": res["sd"],
                "max_primal_coupled": res["max_primal_coupled"],
                "max_dual_coupled": res["max_dual_coupled"],
                "dtheta": dtheta,
                "dz": dz,
                "dyd": dyd,
                "dphi": dphi,
                "rho_max": max(rho.values()) if rho else 0.0,
                "mu_max": max(mu.values()) if mu else 0.0,
            })

            if dphi <= INNER_TOL and dtheta <= INNER_TOL and dz <= INNER_TOL and dyd <= INNER_TOL:
                print(f"Inner loop converge en l={l_inner}.")
                break

            phi_prev = phi_now

        res = max_residuals(theta_vals, z_vals, ydual_vals)

        feasible = (
            res["max_primal_all"] <= FEAS_TOL
            and res["max_dual_all"] <= FEAS_TOL
            and res["sd"] <= FEAS_TOL
        )

        print("\n" + "-"*70)
        print(f"{label} | Chequeo fin outer k={k_outer}")
        print("-"*70)
        print(f"J(theta)             = {compute_J(theta_vals):.8f}")
        print(f"Max primal all       = {res['max_primal_all']:.3e}")
        print(f"Max dual all         = {res['max_dual_all']:.3e}")
        print(f"Strong duality       = {res['sd']:.3e}")
        print(f"Max primal coupled   = {res['max_primal_coupled']:.3e}")
        print(f"Max dual coupled     = {res['max_dual_coupled']:.3e}")
        print(f"rho_max              = {max(rho.values()) if rho else 0.0:.3e}")
        print(f"mu_max               = {max(mu.values()) if mu else 0.0:.3e}")

        if feasible:
            print(f"\n✅ {label} encontró solución factible primal-dual.")

            best_feasible = {
                "theta": copy.deepcopy(theta_vals),
                "z": z_vals.copy(),
                "ydual": ydual_vals.copy(),
                "J": compute_J(theta_vals),
                "residuals": res,
                "outer": k_outer,
                "history": history,
            }

            break

        n_rho, n_mu = update_penalties(theta_vals, z_vals, ydual_vals, rho, mu)

        print("\nActualización penalizaciones:")
        print(f"rho actualizadas: {n_rho}")
        print(f"mu actualizadas : {n_mu}")

    if best_feasible is None:
        print(f"\n⚠️ {label} terminó sin certificar factibilidad.")

    return {
        "best_feasible": best_feasible,
        "theta": theta_vals,
        "z": z_vals,
        "ydual": ydual_vals,
        "history": history,
        "last_residuals": max_residuals(theta_vals, z_vals, ydual_vals) if z_vals is not None else None,
        "rho": rho,
        "mu": mu,
    }


# ============================================================
# 9) EJECUTAR PADM-W
# ============================================================

PADM_WARMSTART = None

if RUN_PADM_W:

    res_ws = run_padm(
        use_true_objective=False,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM-W ETAPA 1: FACTIBILIDAD"
    )

    if res_ws["best_feasible"] is not None:

        theta_start = res_ws["best_feasible"]["theta"]
        z_start = res_ws["best_feasible"]["z"]
        yd_start = res_ws["best_feasible"]["ydual"]

        print("\n" + "="*70)
        print("WARMSTART FACTIBLE ENCONTRADO. INICIANDO ETAPA 2 CON J REAL.")
        print("="*70)

        res_final = run_padm(
            use_true_objective=True,
            theta_init=theta_start,
            z_init=z_start,
            ydual_init=yd_start,
            label="PADM-W ETAPA 2: OBJETIVO REAL"
        )

        if res_final["best_feasible"] is not None:
            final_solution = res_final["best_feasible"]
            final_label = "PADM-W ETAPA 2"
        else:
            print("\n⚠️ Etapa 2 no certificó factibilidad. Se conserva warmstart factible de etapa 1.")
            final_solution = res_ws["best_feasible"]
            final_label = "PADM-W ETAPA 1"

    else:
        print("\n⚠️ Etapa 1 no encontró punto factible.")
        final_solution = None
        final_label = "PADM-W FAILED"
        res_final = res_ws

else:
    res_final = run_padm(
        use_true_objective=True,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM"
    )

    if res_final["best_feasible"] is not None:
        final_solution = res_final["best_feasible"]
        final_label = "PADM"
    else:
        final_solution = None
        final_label = "PADM FAILED"

print("\n" + "="*70)
print("RESULTADOS FINALES PADM-W IEEE57")
print("="*70)

if final_solution is not None:
    sol_theta = final_solution["theta"]
    sol_z = final_solution["z"]
    sol_yd = final_solution["ydual"]

    print("✅ SOLUCIÓN FACTIBLE CERTIFICADA")
    print(f"Método final     : {final_label}")
    print(f"Outer encontrado : {final_solution['outer']}")
    print(f"J                : {final_solution['J']:.8f}")

    PADM_WARMSTART = {
        "theta": copy.deepcopy(sol_theta),
        "z": sol_z.copy(),
        "ydual": sol_yd.copy(),
        "label": final_label,
        "J": final_solution["J"],
        "residuals": final_solution["residuals"],
    }

else:
    print("⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.")

    sol_theta = res_final["theta"]
    sol_z = res_final["z"]
    sol_yd = res_final["ydual"]

    if sol_z is not None and sol_yd is not None:
        res = max_residuals(sol_theta, sol_z, sol_yd)

        PADM_WARMSTART = {
            "theta": copy.deepcopy(sol_theta),
            "z": sol_z.copy(),
            "ydual": sol_yd.copy(),
            "label": "PADM_LAST_POINT_NOT_CERTIFIED",
            "J": compute_J(sol_theta),
            "residuals": res,
        }

        print("⚠️ Se guardó el último punto de PADM-W como Start no certificado.")
    else:
        PADM_WARMSTART = None
        print("❌ No hay punto PADM-W disponible para warm start.")

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("CHECK PADM_WARMSTART IEEE57")
    print("="*70)

    sol_theta = PADM_WARMSTART["theta"]
    sol_z = PADM_WARMSTART["z"]
    sol_yd = PADM_WARMSTART["ydual"]

    res_check = max_residuals(sol_theta, sol_z, sol_yd)

    print(f"Origen          = {PADM_WARMSTART['label']}")
    print(f"J(theta)        = {PADM_WARMSTART['J']:.8f}")
    print(f"Max primal      = {res_check['max_primal_all']:.6e}")
    print(f"Max dual        = {res_check['max_dual_all']:.6e}")
    print(f"Strong duality  = {res_check['sd']:.6e}")


# ============================================================
# 10) WCEP FINAL CON DUALIDAD FUERTE Y WARM START PADM-W
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE57 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE57_wellposed_DF_warmstart_PADM_original")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Pmax_sync_ce = w.addVars(GS, lb=0.0, name="Pmax_sync_ce")

for g in GS:
    base = Pmax_sync_base[g]
    w.addConstr(Pmax_sync_ce[g] >= P_SYNC_LB_FACTOR * base, name=f"adm_lb_Psync[{g}]")
    w.addConstr(Pmax_sync_ce[g] <= P_SYNC_UB_FACTOR * base, name=f"adm_ub_Psync[{g}]")


# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[ell, t, y] - f_neg[ell, t, y]
                    for ell in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[ell, t, y] - f_neg[ell, t, y]
                    for ell in lineas_salen[i]
                )
            )

            rhs_val = float(d_ref[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_ref[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    for t in T:
        for y in Y:
            Bij = float(Bline[ell])

            flow = f_pos[ell, t, y] - f_neg[ell, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for ell in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[ell])
            flow = f_pos[ell, t, y] - f_neg[ell, t, y]

            name = f"flb_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addQConstr(
                Pmax_sync_ce[g] * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -float(xbar[m]), name=name)
        RHS_expr[name] = -float(xbar[m])

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= float(Hreq_base[y]), name=name)
        RHS_expr[name] = float(Hreq_base[y])

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= epsilon_CO2_wcep,
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sync_ub_"):
        ub_here = UB_sync

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dPsync = w.addVars(GS, lb=0.0, name="dPsync")

for g in GS:
    base = Pmax_sync_base[g]
    diff = Pmax_sync_ce[g] - base

    w.addConstr(dPsync[g] >= diff, name=f"dPsync_pos[{g}]")
    w.addConstr(dPsync[g] >= -diff, name=f"dPsync_neg[{g}]")


if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for g in GS:
        base = abs(float(Pmax_sync_base[g]))
        if base > 1e-9:
            obj_wcep += alpha_Psync * dPsync[g] / base
        else:
            obj_wcep += alpha_Psync * dPsync[g]


    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_Psync * dPsync.sum(),
        GRB.MINIMIZE
    )


# ============================================================
# 11) APLICAR WARM START DEL PADM-W
# ============================================================

def safe_start(var, value):
    try:
        if value is not None and np.isfinite(float(value)):
            var.Start = float(value)
    except Exception:
        pass

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("APLICANDO PADM_WARMSTART AL WCEP FINAL IEEE57")
    print("="*70)
    print("Origen:", PADM_WARMSTART.get("label", "PADM"))

    ws_theta = PADM_WARMSTART["theta"]
    ws_z = PADM_WARMSTART["z"]
    ws_yd = PADM_WARMSTART["ydual"]

    for g in GS:
        safe_start(Pmax_sync_ce[g], ws_theta["Pmax_sync_ce"][g])
        safe_start(dPsync[g], abs(ws_theta["Pmax_sync_ce"][g] - Pmax_sync_base[g]))


    for g in G:
        for t in T:
            for y in Y:
                safe_start(p[g, t, y], ws_z.get(("p", g, t, y), None))

    for g in GS:
        for t in T:
            for y in Y:
                safe_start(u[g, t, y], ws_z.get(("u", g, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(s[i, t, y], ws_z.get(("s", i, t, y), None))

    for m in M:
        for y in Y:
            safe_start(xcap[m, y], ws_z.get(("xcap", m, y), None))

    for m in M:
        for t in T:
            for y in Y:
                safe_start(h_syn[m, t, y], ws_z.get(("h_syn", m, t, y), None))
                safe_start(r_ffr[m, t, y], ws_z.get(("r_ffr", m, t, y), None))

    for ell in L_arcs:
        for t in T:
            for y in Y:
                safe_start(f_pos[ell, t, y], ws_z.get(("f_pos", ell, t, y), None))
                safe_start(f_neg[ell, t, y], ws_z.get(("f_neg", ell, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(theta_p[i, t, y], ws_z.get(("theta_p", i, t, y), None))
                safe_start(theta_m[i, t, y], ws_z.get(("theta_m", i, t, y), None))

    for name, var in ydual.items():
        safe_start(var, ws_yd.get(name, None))

    print("✅ Warm start aplicado al WCEP IEEE57.")
else:
    print("\n⚠️ No existe PADM_WARMSTART. El WCEP se resolverá sin Start.")


# ============================================================
# 12) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE57")
print("="*70)

w.optimize()

# Registro de tiempos/GAP WCEP
runtime_wcep = float(w.Runtime)
status_wcep = int(w.Status)
sol_count_wcep = int(w.SolCount)

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP IEEE57")
print("="*70)
print(f"Status Gurobi = {status_wcep}")
print(f"Tiempo optimize() = {runtime_wcep:.2f} s = {runtime_wcep/60:.2f} min = {runtime_wcep/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count_wcep}")

if sol_count_wcep > 0:
    print(f"Mejor objetivo encontrado = {float(w.ObjVal):.8f}")
    print(f"Mejor bound               = {float(w.ObjBound):.8f}")
    print(f"MIPGap                    = {float(w.MIPGap):.6e}")
    print(f"MIPGap (%)                = {100*float(w.MIPGap):.4f}%")
else:
    print("No se encontró solución factible.")

total_script_time = time.time() - t0_total_script
print(f"Tiempo total script = {total_script_time:.2f} s = {total_script_time/60:.2f} min = {total_script_time/3600:.2f} h")


# ============================================================
# 13) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE57")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Pmax_sync_ce síncronos ---")
    print(f"Cotas usadas: {P_SYNC_LB_FACTOR:.2f}*Pmax_base <= Pmax_sync_ce <= {P_SYNC_UB_FACTOR:.2f}*Pmax_base")
    hubo_cambio = False

    for g in GS:
        val = float(Pmax_sync_ce[g].X)
        base = float(Pmax_sync_base[g])

        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{g}: {base:.4f} -> {val:.4f}   ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- Hreq fijo ---")
    for y in Y:
        print(f"Año {y}: Hreq_base fijo = {float(Hreq_base[y]):.6f}")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = epsilon_CO2_wcep - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {epsilon_CO2_wcep:.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_base[y])

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_base={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_sync = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("sync_ub_"):
            if abs(var.X - UB_sync) <= 1e-6 * max(1.0, UB_sync):
                hits_sync += 1

    print(f"Duales sys_H pegadas a cota   : {hits_H}")
    print(f"Duales sync_ub pegadas a cota : {hits_sync}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

✅ Usando d_fw existente desde el FW corregido.

REFERENCIAS USADAS POR PADM-W + WCEP IEEE57
Referencia demanda: d_fw
Referencia Hreq   : Hreq_fw existente del FW well-posed
Cotas Pmax_sync_ce: [0.85, 1.00] alrededor de Pmax_base
Hreq queda fijo en Hreq_base/well-posed
Año 1: Hreq original = 10.714900 | Hreq_base/well-posed = 8.571920
Año 2: Hreq original = 11.250645 | Hreq_base/well-posed = 9.000516
Año 3: Hreq original = 11.786390 | Hreq_base/well-posed = 9.429112

COTAS DUALES IEEE57
UB_H    = 324200000.000000
UB_sync = 10000.000000
Bus slack usado: 1
Bus slack usado: 1

CLASIFICACIÓN PADM-W IEEE57 - FORMULACIÓN ORIGINAL
N° filas primales acopladas   : 60
N° filas primales separables  : 6471
N° columnas duales acopladas  : 60
N° columnas duales separables : 4383

✅ Filas acopladas esperadas: solo sync_ub_.

INICIO PADM-W ETAPA 1: FACTIBILIDAD
use_true_objective = False

Inicializando Y=(z,lambda)...

Chequeo inicial:
J(theta)       = 0.00000000
Max primal     = 1.291e-11
Max dual    

### (Variando b) dualidad fuerte

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import time

# ============================================================
# FINAL IEEE57: WCEP CON DUALIDAD FUERTE
# CASO: SOLO Hreq Y xbar MUTABLES
#
#   - Sin PADM.
#   - Solo se modifica b mediante Hreq_ce[y] y xbar_ce[m].
#   - Pmax queda fijo; NO se muta Pmax.
#   - Se usa Hreq_fw si existe desde el FW well-posed.
#   - Se usa d_fw si existe; si no, se reconstruye con escala por capacidad.
# ============================================================

t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

# Restricción D del IEEE57
epsilon_CO2_wcep = 766

# Objetivo J
alpha_H = 2.0
alpha_Xbar = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

# Opciones WCEP
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1
WCEP_TIME_LIMIT = 5 * 60 * 60  # 5 horas

# Hreq_ce y xbar_ce varían ±15%
HREQ_LB_FACTOR = 0.85
HREQ_UB_FACTOR = 1.15
XBAR_LB_FACTOR = 0.85
XBAR_UB_FACTOR = 1.15

# ============================================================
# 1) PREPARACIÓN GENERAL IEEE57
# ============================================================

L_arcs = list(L_id)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [] for bus in B}
lineas_salen  = {bus: [] for bus in B}

for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    if j in lineas_entran:
        lineas_entran[j].append(ell)

    if i in lineas_salen:
        lineas_salen[i].append(ell)

bus_slack = int(sorted(B)[0])

# ============================================================
# REFERENCIAS WELL-POSED DEL FW
# ============================================================
# Este bloque hace que el PADM-W + WCEP sea consistente con el FW
# well-posed. Si el FW anterior ya creó Hreq_fw, se usa directamente.
# Si no existe, se reconstruye con el criterio usado en el FW:
#   HREQ_ALPHA_WELLPOSED / HREQ_ALPHA_ORIGINAL = 0.28 / 0.35.
#
# Importante:
#   - NO se usa zH.
#   - Hreq_ce sigue estando en el RHS/b.
#   - Hreq_base ahora corresponde al Hreq well-posed, no al Hreq original.
# ============================================================

if "Hreq_fw" in globals():
    Hreq_ref = {y: float(Hreq_fw[y]) for y in Y}
    HREQ_REFERENCE_LABEL = "Hreq_fw existente del FW well-posed"
else:
    HREQ_ALPHA_ORIGINAL = 0.35
    HREQ_ALPHA_WELLPOSED = 0.28
    HREQ_SCALE_FACTOR = HREQ_ALPHA_WELLPOSED / HREQ_ALPHA_ORIGINAL
    Hreq_ref = {y: HREQ_SCALE_FACTOR * float(Hreq[y]) for y in Y}
    Hreq_fw = Hreq_ref.copy()
    HREQ_REFERENCE_LABEL = (
        f"Hreq_fw reconstruido automáticamente: "
        f"{HREQ_ALPHA_WELLPOSED}/{HREQ_ALPHA_ORIGINAL} = {HREQ_SCALE_FACTOR:.6f}"
    )

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE57
# ============================================================
# Si existe d_fw desde el FW corregido, se usa. Si no existe, se
# reconstruye con el mismo criterio usado en el FW:
#   demand_scale_factor = min(1, 0.98 * sum(Pmax) / demanda_pico_original)

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 1.0))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

d_ref = d_fw
DEMAND_REFERENCE_LABEL = "d_fw"

Hreq_base = {y: float(Hreq_ref[y]) for y in Y}
xbar_base = {m: float(xbar[m]) for m in M}

print("\n" + "="*70)
print("REFERENCIAS USADAS POR WCEP IEEE57")
print("="*70)
print("Referencia demanda:", DEMAND_REFERENCE_LABEL)
print("Referencia Hreq   :", HREQ_REFERENCE_LABEL)
print("Pmax queda fijo; NO se muta Pmax")
print("Cotas Hreq_ce: ±15% alrededor de Hreq_base/well-posed")
print("Cotas xbar_ce: ±15% alrededor de xbar_base")
for y in Y:
    print(f"Año {y}: Hreq original = {float(Hreq[y]):.6f} | Hreq_base/well-posed = {float(Hreq_base[y]):.6f}")

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_xbar = 2.0 * max([float(cinv[m]) for m in M], default=1.0)

print("\n" + "="*70)
print("COTAS DUALES IEEE57")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_xbar = {UB_xbar:.6f}")
print(f"Bus slack usado: {bus_slack}")


print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE57 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE57_DF_SOLO_Hreq_Xbar")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Hreq_ce = w.addVars(Y, lb=0.0, name="Hreq_ce")
xbar_ce = w.addVars(M, lb=0.0, name="xbar_ce")

for y in Y:
    base = Hreq_base[y]
    w.addConstr(Hreq_ce[y] >= HREQ_LB_FACTOR * base, name=f"adm_lb_Hreq[{y}]")
    w.addConstr(Hreq_ce[y] <= HREQ_UB_FACTOR * base, name=f"adm_ub_Hreq[{y}]")

for m in M:
    base = xbar_base[m]
    w.addConstr(xbar_ce[m] >= XBAR_LB_FACTOR * base, name=f"adm_lb_xbar[{m}]")
    w.addConstr(xbar_ce[m] <= XBAR_UB_FACTOR * base, name=f"adm_ub_xbar[{m}]")


# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[ell, t, y] - f_neg[ell, t, y]
                    for ell in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[ell, t, y] - f_neg[ell, t, y]
                    for ell in lineas_salen[i]
                )
            )

            rhs_val = float(d_ref[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_ref[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    for t in T:
        for y in Y:
            Bij = float(Bline[ell])

            flow = f_pos[ell, t, y] - f_neg[ell, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for ell in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[ell])
            flow = f_pos[ell, t, y] - f_neg[ell, t, y]

            name = f"flb_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(Pmax[g]) * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -xbar_ce[m], name=name)
        RHS_expr[name] = -xbar_ce[m]

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= Hreq_ce[y], name=name)
        RHS_expr[name] = Hreq_ce[y]

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= epsilon_CO2_wcep,
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sys_H_"):
        ub_here = UB_H
    elif name.startswith("x_ub_"):
        ub_here = UB_xbar

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dHreq = w.addVars(Y, lb=0.0, name="dHreq")
dXbar = w.addVars(M, lb=0.0, name="dXbar")

for y in Y:
    base = Hreq_base[y]
    diff = Hreq_ce[y] - base

    w.addConstr(dHreq[y] >= diff, name=f"dHreq_pos[{y}]")
    w.addConstr(dHreq[y] >= -diff, name=f"dHreq_neg[{y}]")

for m in M:
    base = xbar_base[m]
    diff = xbar_ce[m] - base

    w.addConstr(dXbar[m] >= diff, name=f"dXbar_pos[{m}]")
    w.addConstr(dXbar[m] >= -diff, name=f"dXbar_neg[{m}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()

    for y in Y:
        base = abs(float(Hreq_base[y]))
        if base > 1e-9:
            obj_wcep += alpha_H * dHreq[y] / base
        else:
            obj_wcep += alpha_H * dHreq[y]

    for m in M:
        base = abs(float(xbar_base[m]))
        if base > 1e-9:
            obj_wcep += alpha_Xbar * dXbar[m] / base
        else:
            obj_wcep += alpha_Xbar * dXbar[m]

    w.setObjective(obj_wcep, GRB.MINIMIZE)

else:
    w.setObjective(
        alpha_H * dHreq.sum()
        + alpha_Xbar * dXbar.sum(),
        GRB.MINIMIZE
    )


# ============================================================
# 11) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE57")
print("="*70)

w.optimize()

# Registro de tiempos/GAP WCEP
runtime_wcep = float(w.Runtime)
status_wcep = int(w.Status)
sol_count_wcep = int(w.SolCount)

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP IEEE57")
print("="*70)
print(f"Status Gurobi = {status_wcep}")
print(f"Tiempo optimize() = {runtime_wcep:.2f} s = {runtime_wcep/60:.2f} min = {runtime_wcep/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count_wcep}")

if sol_count_wcep > 0:
    print(f"Mejor objetivo encontrado = {float(w.ObjVal):.8f}")
    print(f"Mejor bound               = {float(w.ObjBound):.8f}")
    print(f"MIPGap                    = {float(w.MIPGap):.6e}")
    print(f"MIPGap (%)                = {100*float(w.MIPGap):.4f}%")
else:
    print("No se encontró solución factible.")

total_script_time = time.time() - t0_total_script
print(f"Tiempo total script = {total_script_time:.2f} s = {total_script_time/60:.2f} min = {total_script_time/3600:.2f} h")


# ============================================================
# 12) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE57")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")
elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")
elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Hreq_ce ---")
    print(f"Cotas usadas: {HREQ_LB_FACTOR:.2f}*Hreq_base <= Hreq_ce <= {HREQ_UB_FACTOR:.2f}*Hreq_base")
    hubo_cambio = False

    for y in Y:
        val = float(Hreq_ce[y].X)
        base = float(Hreq_base[y])
        delta = val - base
        pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")

        if abs(delta) > tol:
            hubo_cambio = True
            print(f"Año {y}: {base:.6f} -> {val:.6f} | delta={delta:+.6f} ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- xbar_ce ---")
    print(f"Cotas usadas: {XBAR_LB_FACTOR:.2f}*xbar_base <= xbar_ce <= {XBAR_UB_FACTOR:.2f}*xbar_base")
    hubo_cambio = False

    for m in M:
        val = float(xbar_ce[m].X)
        base = float(xbar_base[m])
        delta = val - base
        pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")

        if abs(delta) > tol:
            hubo_cambio = True
            print(f"{m}: {base:.6f} -> {val:.6f} | delta={delta:+.6f} ({pct:+.2f}%)")

    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- Pmax fijo ---")
    print("Pmax no fue modificado en este caso.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = epsilon_CO2_wcep - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {epsilon_CO2_wcep:.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_ce[y].X)

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_ce={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("INVERSIÓN IBR Y USO DE xbar_ce")
    print("="*70)

    filas_xcap = []

    for m in M:
        for y in Y:
            x_val = float(xcap[m, y].X)
            xb_val = float(xbar_ce[m].X)
            xb_base = float(xbar_base[m])
            uso_xbar = x_val / xb_val if xb_val > 1e-9 else None

            filas_xcap.append({
                "gen": m,
                "y": y,
                "xcap": x_val,
                "xbar_original": xb_base,
                "xbar_ce": xb_val,
                "uso_xbar_ce": uso_xbar
            })

    df_xcap_wcep = pd.DataFrame(filas_xcap)

    display(
        df_xcap_wcep[
            (df_xcap_wcep["xcap"] > 1e-6)
            | (df_xcap_wcep["uso_xbar_ce"].fillna(0.0) > 0.85)
        ]
        .sort_values(["y", "uso_xbar_ce"], ascending=[True, False])
        .reset_index(drop=True)
    )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_xbar = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("x_ub_"):
            if abs(var.X - UB_xbar) <= 1e-6 * max(1.0, UB_xbar):
                hits_xbar += 1

    print(f"Duales sys_H pegadas a cota : {hits_H}")
    print(f"Duales x_ub pegadas a cota  : {hits_xbar}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")


✅ Usando d_fw existente desde el FW corregido.

REFERENCIAS USADAS POR WCEP IEEE57
Referencia demanda: d_fw
Referencia Hreq   : Hreq_fw existente del FW well-posed
Pmax queda fijo; NO se muta Pmax
Cotas Hreq_ce: ±15% alrededor de Hreq_base/well-posed
Cotas xbar_ce: ±15% alrededor de xbar_base
Año 1: Hreq original = 10.714900 | Hreq_base/well-posed = 8.571920
Año 2: Hreq original = 11.250645 | Hreq_base/well-posed = 9.000516
Año 3: Hreq original = 11.786390 | Hreq_base/well-posed = 9.429112

COTAS DUALES IEEE57
UB_H    = 324200000.000000
UB_xbar = 3242000.000000
Bus slack usado: 1

CONSTRUYENDO WCEP FINAL IEEE57 CON DUALIDAD FUERTE
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.005
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 18000

RESOLVIENDO WCEP FINAL IEEE57
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12650H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 ph

,gen,y,xcap,xbar_original,xbar_ce,uso_xbar_ce
0,314_PV_3,1,182.622269,185.4,185.400115,0.985017
1,313_RTPV_2,1,123.433698,126.2,126.200130,0.978079
2,314_PV_2,1,100.441673,103.2,103.200136,0.973271
3,314_PV_4,1,100.441673,103.2,103.200136,0.973271
4,314_PV_1,1,100.441673,103.2,103.200136,0.973271
5,313_RTPV_1,1,30.834583,203.4,203.400003,0.151596
6,313_PV_2,1,0.778263,186.6,186.600000,0.004171
7,313_PV_1,1,0.778589,190.2,190.200000,0.004094
8,212_CSP_1,1,0.787572,400.0,400.000000,0.001969
9,314_PV_3,2,182.623167,185.4,185.400115,0.985022



REVISIÓN DE DUALES ACOTADAS
Duales sys_H pegadas a cota : 0
Duales x_ub pegadas a cota  : 0


### (Variando b) dualidad fuerte + PADM

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import copy
import math
import time

# ============================================================
# FINAL IEEE57 WELL-POSED: PADM-W + WCEP CON DUALIDAD FUERTE
# CASO: SOLO Hreq Y xbar MUTABLES
#
#   sum_g Hg[g] u[g,t,y] + sum_m h_syn[m,t,y] >= Hreq_base[y]
#
# Es decir:
#   - NO se usa zH.
#   - Hreq_ce queda mutable en b.
#   - Pmax queda fijo; Hreq_ce y xbar_ce quedan en b.
#
# Flujo:
#   1) PADM-W etapa 1: busca factibilidad.
#   2) PADM-W etapa 2: parte de factibilidad y reduce J(theta).
#   3) WCEP completo con dualidad fuerte usando PADM_WARMSTART.
# ============================================================

t0_total_script = time.time()

# ============================================================
# 0) OPCIONES
# ============================================================

PADM_OUTPUT = 1
RUN_PADM_W = True

MAX_OUTER = 20
MAX_INNER = 20

FEAS_TOL = 1e-5
INNER_TOL = 1e-5

RHO0 = 500.0
MU0 = 500.0

PENALTY_FACTOR = 2.0
MAX_PENALTY = 1e9

TIME_LIMIT_SUBPROBLEM = 5 * 60 * 60  # 5 horas por subproblema PADM

# Restricción D del IEEE57
epsilon_CO2_wcep = 766

# Objetivo J
alpha_H = 2.0
alpha_Xbar = 1.0

# Si quieres objetivo porcentual, cambia esto a True
USE_PERCENT_OBJECTIVE = False

Y_OBJECTIVE_MODE = "penalty"
TIEBREAKER_CTX_WEIGHT = 1e-9

# Opciones WCEP final
WCEP_MIPGAP = 5e-3
WCEP_OUTPUT = 1
WCEP_TIME_LIMIT = 5 * 60 * 60  # 5 horas WCEP final

# Cotas de mutabilidad
# Hreq_ce y xbar_ce varían ±15%.
HREQ_LB_FACTOR = 0.85
HREQ_UB_FACTOR = 1.15
XBAR_LB_FACTOR = 0.85
XBAR_UB_FACTOR = 1.15


# ============================================================
# 1) PREPARACIÓN GENERAL IEEE57
# ============================================================

L_arcs = list(L_id)
G_IBR = list(M)

if isinstance(B, (int, float, np.floating)):
    print("⚠️ B estaba pisado como número. Lo reconstruyo desde df_load/df_lines.")

    if "df_load" in globals():
        B = sorted(df_load["Bus ID"].astype(int).unique().tolist())
    else:
        B = sorted(
            set(df_lines["from_bus"].astype(int))
            .union(set(df_lines["to_bus"].astype(int)))
        )

    print("✅ Nuevo B =", B[:10], "... |B| =", len(B))

lineas_entran = {bus: [] for bus in B}
lineas_salen  = {bus: [] for bus in B}

for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    if j in lineas_entran:
        lineas_entran[j].append(ell)

    if i in lineas_salen:
        lineas_salen[i].append(ell)

bus_slack = int(sorted(B)[0])

# ============================================================
# REFERENCIAS WELL-POSED DEL FW
# ============================================================
# Este bloque hace que el PADM-W + WCEP sea consistente con el FW
# well-posed. Si el FW anterior ya creó Hreq_fw, se usa directamente.
# Si no existe, se reconstruye con el criterio usado en el FW:
#   HREQ_ALPHA_WELLPOSED / HREQ_ALPHA_ORIGINAL = 0.28 / 0.35.
#
# Importante:
#   - NO se usa zH.
#   - Hreq_ce sigue estando en el RHS/b.
#   - Hreq_base ahora corresponde al Hreq well-posed, no al Hreq original.
# ============================================================

if "Hreq_fw" in globals():
    Hreq_ref = {y: float(Hreq_fw[y]) for y in Y}
    HREQ_REFERENCE_LABEL = "Hreq_fw existente del FW well-posed"
else:
    HREQ_ALPHA_ORIGINAL = 0.35
    HREQ_ALPHA_WELLPOSED = 0.28
    HREQ_SCALE_FACTOR = HREQ_ALPHA_WELLPOSED / HREQ_ALPHA_ORIGINAL
    Hreq_ref = {y: HREQ_SCALE_FACTOR * float(Hreq[y]) for y in Y}
    Hreq_fw = Hreq_ref.copy()
    HREQ_REFERENCE_LABEL = (
        f"Hreq_fw reconstruido automáticamente: "
        f"{HREQ_ALPHA_WELLPOSED}/{HREQ_ALPHA_ORIGINAL} = {HREQ_SCALE_FACTOR:.6f}"
    )

# ============================================================
# DEMANDA CORREGIDA DEL FW IEEE57
# ============================================================
# Si existe d_fw desde el FW corregido, se usa. Si no existe, se
# reconstruye con el mismo criterio usado en el FW:
#   demand_scale_factor = min(1, 0.98 * sum(Pmax) / demanda_pico_original)

DEMAND_SCALE_MODE = globals().get("DEMAND_SCALE_MODE", "auto_capacity")
DEMAND_SCALE_MANUAL = float(globals().get("DEMAND_SCALE_MANUAL", 1.0))
DEMAND_CAP_MARGIN = float(globals().get("DEMAND_CAP_MARGIN", 0.98))

if "d_fw" not in globals():
    print("⚠️ No existe d_fw desde el FW corregido. Lo reconstruyo automáticamente.")

    d_original = {k: float(v) for k, v in d.items()}

    demanda_original_ty = {}
    for t in T:
        for y in Y:
            demanda_original_ty[(t, y)] = sum(
                d_original[(i, t, y)]
                for i in B
                if (i, t, y) in d_original
            )

    peak_dem_original = max(demanda_original_ty.values())
    cap_total_pmax = sum(float(Pmax[g]) for g in G)

    if DEMAND_SCALE_MODE == "none":
        demand_scale_factor = 1.0
    elif DEMAND_SCALE_MODE == "manual":
        demand_scale_factor = float(DEMAND_SCALE_MANUAL)
    elif DEMAND_SCALE_MODE == "auto_capacity":
        if peak_dem_original <= 1e-9:
            demand_scale_factor = 1.0
        else:
            demand_scale_factor = min(
                1.0,
                DEMAND_CAP_MARGIN * cap_total_pmax / peak_dem_original
            )
    else:
        raise ValueError("DEMAND_SCALE_MODE debe ser 'auto_capacity', 'manual' o 'none'.")

    d_fw = {}
    for i in B:
        for t in T:
            for y in Y:
                if (i, t, y) not in d_original:
                    raise KeyError(f"Falta d[{i},{t},{y}] en el diccionario original.")
                d_fw[(i, t, y)] = demand_scale_factor * d_original[(i, t, y)]

    print(f"Factor demanda reconstruido = {demand_scale_factor:.8f}")
else:
    print("✅ Usando d_fw existente desde el FW corregido.")

d_ref = d_fw
DEMAND_REFERENCE_LABEL = "d_fw"

Hreq_base = {y: float(Hreq_ref[y]) for y in Y}
Pmax_sync_base = {g: float(Pmax[g]) for g in GS}
xbar_base = {m: float(xbar[m]) for m in M}

print("\n" + "="*70)
print("REFERENCIAS USADAS POR PADM-W + WCEP IEEE57")
print("="*70)
print("Referencia demanda:", DEMAND_REFERENCE_LABEL)
print("Referencia Hreq   :", HREQ_REFERENCE_LABEL)
print("Pmax síncrono queda fijo")
print("Cotas Hreq_ce: ±15% alrededor de Hreq_base/well-posed")
print("Cotas xbar_ce: ±15% alrededor de xbar_base")
for y in Y:
    print(f"Año {y}: Hreq original = {float(Hreq[y]):.6f} | Hreq_base/well-posed = {float(Hreq_base[y]):.6f}")

gamma_pos = [float(gamma_base[m]) for m in M if float(gamma_base[m]) > 1e-9]

if len(gamma_pos) == 0:
    UB_H = 1e8
else:
    UB_H = 2.0 * max(float(cinv[m]) for m in M) / min(gamma_pos)

UB_sync = 2.0 * max(max(float(cg[g]) for g in G), float(VOLL))
UB_xbar = 2.0 * max([float(cinv[m]) for m in M], default=1.0)

print("\n" + "="*70)
print("COTAS DUALES IEEE57")
print("="*70)
print(f"UB_H    = {UB_H:.6f}")
print(f"UB_sync = {UB_sync:.6f}")
print(f"UB_xbar = {UB_xbar:.6f}")
print(f"Bus slack usado: {bus_slack}")


# ============================================================
# 2) REPRESENTACIÓN SIMBÓLICA
# ============================================================

def key_p(g,t,y):       return ("p", g, t, y)
def key_u(g,t,y):       return ("u", g, t, y)
def key_s(i,t,y):       return ("s", i, t, y)
def key_xcap(m,y):      return ("xcap", m, y)
def key_hsyn(m,t,y):    return ("h_syn", m, t, y)
def key_rffr(m,t,y):    return ("r_ffr", m, t, y)
def key_fpos(ell,t,y):  return ("f_pos", ell, t, y)
def key_fneg(ell,t,y):  return ("f_neg", ell, t, y)
def key_thp(i,t,y):     return ("theta_p", i, t, y)
def key_thm(i,t,y):     return ("theta_m", i, t, y)

def param_coeff(family, idx, mult=1.0):
    return ("PARAM", family, idx, float(mult))

def is_param_coeff(c):
    return isinstance(c, tuple) and len(c) == 4 and c[0] == "PARAM"

def coeff_value(c, theta_vals):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * float(theta_vals[fam][idx])
    return float(c)

def coeff_expr_in_theta_model(c, theta_vars):
    if is_param_coeff(c):
        _, fam, idx, mult = c
        return mult * theta_vars[fam][idx]
    return float(c)

def rhs_const(val):
    return [("CONST", None, None, float(val))]

def rhs_param(family, idx, mult=1.0):
    return [("PARAM", family, idx, float(mult))]

def rhs_has_param(rhs_terms):
    return any(typ == "PARAM" for typ, fam, idx, mult in rhs_terms)

def rhs_eval(rhs_terms, theta_vals=None):
    out = 0.0

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            out += float(mult)
        elif typ == "PARAM":
            if theta_vals is None:
                raise ValueError("rhs_eval requiere theta_vals para RHS con parámetros.")
            out += float(mult) * float(theta_vals[fam][idx])
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return out

def rhs_expr_in_theta_model(rhs_terms, theta_vars):
    expr = gp.LinExpr()

    for typ, fam, idx, mult in rhs_terms:
        if typ == "CONST":
            expr += float(mult)
        elif typ == "PARAM":
            expr += float(mult) * theta_vars[fam][idx]
        else:
            raise ValueError(f"Tipo RHS no reconocido: {typ}")

    return expr

def add_to_terms(terms, key, coeff):
    if key not in terms:
        terms[key] = []
    terms[key].append(coeff)

def neg_coeff(c):
    if is_param_coeff(c):
        return ("PARAM", c[1], c[2], -c[3])
    return -float(c)

def terms_has_param(terms):
    for k, coeffs in terms.items():
        for c in coeffs:
            if is_param_coeff(c):
                return True
    return False

def terms_expr_z_model(terms, z_vars, theta_vals):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        for c in coeffs:
            expr += coeff_value(c, theta_vals) * z_vars[k]

    return expr

def terms_expr_theta_model_with_fixed_z(terms, z_vals, theta_vars):
    expr = gp.LinExpr()

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            expr += zval * coeff_expr_in_theta_model(c, theta_vars)

    return expr

def terms_eval(terms, z_vals, theta_vals):
    out = 0.0

    for k, coeffs in terms.items():
        zval = float(z_vals.get(k, 0.0))

        if abs(zval) <= 1e-12:
            continue

        for c in coeffs:
            out += coeff_value(c, theta_vals) * zval

    return out


# ============================================================
# 3) CREAR FILAS PRIMALES Ax >= b
# ============================================================

rows = {}
row_order = []

def add_row(name, terms, rhs_terms):
    if name in rows:
        raise ValueError(f"Fila repetida: {name}")

    rows[name] = {
        "terms": terms,
        "rhs": rhs_terms
    }

    row_order.append(name)

# ------------------------------------------------------------
# A.1 Balance nodal
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}

            for g in G_en_bus.get(i, []):
                add_to_terms(terms, key_p(g,t,y), 1.0)

            add_to_terms(terms, key_s(i,t,y), 1.0)

            for ell in lineas_entran[i]:
                add_to_terms(terms, key_fpos(ell,t,y), 1.0)
                add_to_terms(terms, key_fneg(ell,t,y), -1.0)

            for ell in lineas_salen[i]:
                add_to_terms(terms, key_fpos(ell,t,y), -1.0)
                add_to_terms(terms, key_fneg(ell,t,y), 1.0)

            rhs_val = float(d_ref[(i,t,y)])

            add_row(f"bal_p_{i}_{t}_{y}", terms, rhs_const(rhs_val))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"bal_n_{i}_{t}_{y}", terms_n, rhs_const(-rhs_val))

# ------------------------------------------------------------
# A.2 ENS <= demanda
# ------------------------------------------------------------
for i in B:
    for t in T:
        for y in Y:
            terms = {}
            add_to_terms(terms, key_s(i,t,y), -1.0)
            add_row(f"ens_ub_{i}_{t}_{y}", terms, rhs_const(-float(d_ref[(i,t,y)])))

# ------------------------------------------------------------
# A.3 Flujo DC
# ------------------------------------------------------------
for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    for t in T:
        for y in Y:
            Bij = float(Bline[ell])

            terms = {}
            add_to_terms(terms, key_fpos(ell,t,y), 1.0)
            add_to_terms(terms, key_fneg(ell,t,y), -1.0)

            add_to_terms(terms, key_thp(i,t,y), -Bij)
            add_to_terms(terms, key_thm(i,t,y),  Bij)
            add_to_terms(terms, key_thp(j,t,y),  Bij)
            add_to_terms(terms, key_thm(j,t,y), -Bij)

            add_row(f"dc_p_{ell}_{t}_{y}", terms, rhs_const(0.0))

            terms_n = {k: [neg_coeff(c) for c in coeffs] for k, coeffs in terms.items()}
            add_row(f"dc_n_{ell}_{t}_{y}", terms_n, rhs_const(0.0))

# ------------------------------------------------------------
# A.4 Límites térmicos fijos
# ------------------------------------------------------------
for ell in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[ell])

            terms_lb = {}
            add_to_terms(terms_lb, key_fpos(ell,t,y), 1.0)
            add_to_terms(terms_lb, key_fneg(ell,t,y), -1.0)
            add_row(f"flb_{ell}_{t}_{y}", terms_lb, rhs_const(-Fij))

            terms_ub = {}
            add_to_terms(terms_ub, key_fpos(ell,t,y), -1.0)
            add_to_terms(terms_ub, key_fneg(ell,t,y), 1.0)
            add_row(f"fub_{ell}_{t}_{y}", terms_ub, rhs_const(-Fij))

# ------------------------------------------------------------
# A.5 Slack bus
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_p = {}
        add_to_terms(terms_p, key_thp(bus_slack,t,y), 1.0)
        add_to_terms(terms_p, key_thm(bus_slack,t,y), -1.0)
        add_row(f"slack_p_{t}_{y}", terms_p, rhs_const(0.0))

        terms_n = {}
        add_to_terms(terms_n, key_thp(bus_slack,t,y), -1.0)
        add_to_terms(terms_n, key_thm(bus_slack,t,y), 1.0)
        add_row(f"slack_n_{t}_{y}", terms_n, rhs_const(0.0))

print(f"Bus slack usado: {bus_slack}")

# ------------------------------------------------------------
# A.6 Límites generación fijos p <= P_disp
# ------------------------------------------------------------
for g in G:
    for t in T:
        cap_base = float(P_disp.get((g,t), Pmax[g]))

        for y in Y:
            terms = {}
            add_to_terms(terms, key_p(g,t,y), -1.0)
            add_row(f"p_cap_{g}_{t}_{y}", terms, rhs_const(-cap_base))

# ------------------------------------------------------------
# A.7 Síncronos
# ------------------------------------------------------------
for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            terms_lb = {}
            add_to_terms(terms_lb, key_p(g,t,y), 1.0)
            add_to_terms(terms_lb, key_u(g,t,y), -pmin_g)
            add_row(f"sync_lb_{g}_{t}_{y}", terms_lb, rhs_const(0.0))

            terms_ub = {}
            add_to_terms(terms_ub, key_u(g,t,y), float(Pmax_sync_base[g]))
            add_to_terms(terms_ub, key_p(g,t,y), -1.0)
            add_row(f"sync_ub_{g}_{t}_{y}", terms_ub, rhs_const(0.0))

            terms_u = {}
            add_to_terms(terms_u, key_u(g,t,y), -1.0)
            add_row(f"u_ub_{g}_{t}_{y}", terms_u, rhs_const(-1.0))

# ------------------------------------------------------------
# A.8-A.10 Inversión, h_syn, FFR
# ------------------------------------------------------------
for m in M:
    for y in Y:
        terms_x = {}
        add_to_terms(terms_x, key_xcap(m,y), -1.0)
        add_row(f"x_ub_{m}_{y}", terms_x, rhs_param("xbar_ce", m, -1.0))

        for t in T:
            terms_h = {}
            add_to_terms(terms_h, key_xcap(m,y), float(gamma_base[m]))
            add_to_terms(terms_h, key_hsyn(m,t,y), -1.0)
            add_row(f"hsyn_lim_{m}_{t}_{y}", terms_h, rhs_const(0.0))

            terms_r = {}
            add_to_terms(terms_r, key_xcap(m,y), float(kappa_base[m]))
            add_to_terms(terms_r, key_rffr(m,t,y), -1.0)
            add_row(f"ffr_lim_{m}_{t}_{y}", terms_r, rhs_const(0.0))

# ------------------------------------------------------------
# A.11 Seguridad: Hreq fijo en b, Rreq fijo
# ------------------------------------------------------------
for t in T:
    for y in Y:
        terms_H = {}

        for g in GS:
            add_to_terms(terms_H, key_u(g,t,y), float(Hg.get(g, 0.0)))

        for m in M:
            add_to_terms(terms_H, key_hsyn(m,t,y), 1.0)

        add_row(f"sys_H_{t}_{y}", terms_H, rhs_param("Hreq_ce", y, 1.0))

        terms_R = {}

        for m in M:
            add_to_terms(terms_R, key_rffr(m,t,y), 1.0)

        add_row(f"sys_R_{t}_{y}", terms_R, rhs_const(float(Rreq[y])))


# ============================================================
# 4) VARIABLES PRIMALES Y COSTOS c
# ============================================================

primal_keys = []

for g in G:
    for t in T:
        for y in Y:
            primal_keys.append(key_p(g,t,y))

for g in GS:
    for t in T:
        for y in Y:
            primal_keys.append(key_u(g,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_s(i,t,y))

for m in M:
    for y in Y:
        primal_keys.append(key_xcap(m,y))

for m in M:
    for t in T:
        for y in Y:
            primal_keys.append(key_hsyn(m,t,y))
            primal_keys.append(key_rffr(m,t,y))

for ell in L_arcs:
    for t in T:
        for y in Y:
            primal_keys.append(key_fpos(ell,t,y))
            primal_keys.append(key_fneg(ell,t,y))

for i in B:
    for t in T:
        for y in Y:
            primal_keys.append(key_thp(i,t,y))
            primal_keys.append(key_thm(i,t,y))

c_map = {}

for g in G:
    for t in T:
        for y in Y:
            c_map[key_p(g,t,y)] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map[key_s(i,t,y)] = float(VOLL)

for m in M:
    for y in Y:
        c_map[key_xcap(m,y)] = float(cinv[m])

for k in primal_keys:
    c_map.setdefault(k, 0.0)


# ============================================================
# 5) CLASIFICACIÓN ACOPLADAS VS SEPARABLES
# ============================================================

coupled_primal_rows = [
    name for name in row_order
    if terms_has_param(rows[name]["terms"]) or rhs_has_param(rows[name]["rhs"])
]

separable_primal_rows = [
    name for name in row_order
    if name not in set(coupled_primal_rows)
]

coupled_dual_cols = set()

for name in row_order:
    terms = rows[name]["terms"]

    for k, coeffs in terms.items():
        if any(is_param_coeff(c) for c in coeffs):
            coupled_dual_cols.add(k)

coupled_dual_cols = sorted(list(coupled_dual_cols), key=str)

separable_dual_cols = [
    k for k in primal_keys
    if k not in set(coupled_dual_cols)
]

print("\n" + "="*70)
print("CLASIFICACIÓN PADM-W IEEE57 - FORMULACIÓN ORIGINAL")
print("="*70)
print(f"N° filas primales acopladas   : {len(coupled_primal_rows)}")
print(f"N° filas primales separables  : {len(separable_primal_rows)}")
print(f"N° columnas duales acopladas  : {len(coupled_dual_cols)}")
print(f"N° columnas duales separables : {len(separable_dual_cols)}")

bad_rows = [
    name for name in coupled_primal_rows
    if not (name.startswith("sys_H_") or name.startswith("x_ub_"))
]

if bad_rows:
    print("\n⚠️ Hay filas acopladas no esperadas:")
    print(bad_rows[:30])
else:
    print("\n✅ Filas acopladas esperadas: sys_H_ y x_ub_.")


# ============================================================
# 6) FUNCIONES PADM-W
# ============================================================

def initial_theta_vals():
    return {
        "Hreq_ce": {y: float(Hreq_base[y]) for y in Y},
        "xbar_ce": {m: float(xbar_base[m]) for m in M},
    }

def compute_J(theta_vals):
    J = 0.0

    for y in Y:
        base = float(Hreq_base[y])
        val = float(theta_vals["Hreq_ce"][y])
        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_H * abs(val - base) / abs(base)
        else:
            J += alpha_H * abs(val - base)

    for m in M:
        base = float(xbar_base[m])
        val = float(theta_vals["xbar_ce"][m])
        if USE_PERCENT_OBJECTIVE and abs(base) > 1e-9:
            J += alpha_Xbar * abs(val - base) / abs(base)
        else:
            J += alpha_Xbar * abs(val - base)

    return J

def compute_ctx(z_vals):
    return sum(
        float(c_map[k]) * float(z_vals.get(k, 0.0))
        for k in primal_keys
    )

def compute_bty(theta_vals, ydual_vals):
    out = 0.0

    for name in row_order:
        out += rhs_eval(rows[name]["rhs"], theta_vals) * float(ydual_vals.get(name, 0.0))

    return out

def compute_primal_residuals(theta_vals, z_vals):
    res = {}

    for name in row_order:
        lhs = terms_eval(rows[name]["terms"], z_vals, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)
        res[name] = max(0.0, rhs - lhs)

    return res

def compute_dual_residuals(theta_vals, ydual_vals):
    res = {}

    for k in primal_keys:
        lhs = 0.0

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            yd = float(ydual_vals.get(name, 0.0))

            if abs(yd) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd

        rhs = float(c_map.get(k, 0.0))
        res[k] = max(0.0, lhs - rhs)

    return res

def compute_sd_residual(theta_vals, z_vals, ydual_vals):
    return max(0.0, compute_ctx(z_vals) - compute_bty(theta_vals, ydual_vals))

def max_residuals(theta_vals, z_vals, ydual_vals):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)
    sd = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return {
        "max_primal_all": max(pr.values()) if pr else 0.0,
        "max_dual_all": max(dr.values()) if dr else 0.0,
        "sd": sd,
        "max_primal_coupled": max([pr[name] for name in coupled_primal_rows], default=0.0),
        "max_dual_coupled": max([dr[k] for k in coupled_dual_cols], default=0.0),
    }

def compute_phi(theta_vals, z_vals, ydual_vals, rho, mu, use_true_objective=True):
    Jval = compute_J(theta_vals) if use_true_objective else 0.0

    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    pen_p = sum(float(rho[name]) * pr[name] for name in coupled_primal_rows)
    pen_d = sum(float(mu[k]) * dr[k] for k in coupled_dual_cols)

    sd_penalty = compute_sd_residual(theta_vals, z_vals, ydual_vals)

    return Jval + pen_p + pen_d + 1e3 * sd_penalty

def diff_theta(theta_a, theta_b):
    vals = []

    for y in Y:
        vals.append(abs(theta_a["Hreq_ce"][y] - theta_b["Hreq_ce"][y]))

    for m in M:
        vals.append(abs(theta_a["xbar_ce"][m] - theta_b["xbar_ce"][m]))

    return max(vals) if vals else 0.0

def diff_dict(a, b):
    keys = set(a.keys()) | set(b.keys())
    return max(
        [abs(float(a.get(k,0.0)) - float(b.get(k,0.0))) for k in keys],
        default=0.0
    )


# ============================================================
# 7) SUBPROBLEMAS PADM-W
# ============================================================

def add_z_variables(model):
    z = {}

    for g in G:
        for t in T:
            for y in Y:
                z[key_p(g,t,y)] = model.addVar(lb=0.0, name=f"p[{g},{t},{y}]")

    for g in GS:
        for t in T:
            for y in Y:
                z[key_u(g,t,y)] = model.addVar(lb=0.0, ub=1.0, name=f"u[{g},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_s(i,t,y)] = model.addVar(lb=0.0, name=f"s[{i},{t},{y}]")

    for m in M:
        for y in Y:
            z[key_xcap(m,y)] = model.addVar(lb=0.0, name=f"xcap[{m},{y}]")

    for m in M:
        for t in T:
            for y in Y:
                z[key_hsyn(m,t,y)] = model.addVar(lb=0.0, name=f"h_syn[{m},{t},{y}]")
                z[key_rffr(m,t,y)] = model.addVar(lb=0.0, name=f"r_ffr[{m},{t},{y}]")

    for ell in L_arcs:
        for t in T:
            for y in Y:
                z[key_fpos(ell,t,y)] = model.addVar(lb=0.0, name=f"f_pos[{ell},{t},{y}]")
                z[key_fneg(ell,t,y)] = model.addVar(lb=0.0, name=f"f_neg[{ell},{t},{y}]")

    for i in B:
        for t in T:
            for y in Y:
                z[key_thp(i,t,y)] = model.addVar(lb=0.0, name=f"theta_p[{i},{t},{y}]")
                z[key_thm(i,t,y)] = model.addVar(lb=0.0, name=f"theta_m[{i},{t},{y}]")

    return z

def add_dual_variables(model):
    yd = {}

    for name in row_order:
        ub_here = GRB.INFINITY

        if name.startswith("sys_H_"):
            ub_here = UB_H
        elif name.startswith("x_ub_"):
            ub_here = UB_xbar

        yd[name] = model.addVar(lb=0.0, ub=ub_here, name=f"ydual[{name}]")

    return yd

def add_theta_variables(model, warm_theta=None):
    theta = {
        "Hreq_ce": {},
        "xbar_ce": {},
    }

    for y in Y:
        theta["Hreq_ce"][y] = model.addVar(
            lb=HREQ_LB_FACTOR * float(Hreq_base[y]),
            ub=HREQ_UB_FACTOR * float(Hreq_base[y]),
            name=f"Hreq_ce[{y}]"
        )

    for m in M:
        theta["xbar_ce"][m] = model.addVar(
            lb=XBAR_LB_FACTOR * float(xbar_base[m]),
            ub=XBAR_UB_FACTOR * float(xbar_base[m]),
            name=f"xbar_ce[{m}]"
        )

    if warm_theta is not None:
        for y in Y:
            theta["Hreq_ce"][y].Start = float(warm_theta["Hreq_ce"][y])
        for m in M:
            theta["xbar_ce"][m].Start = float(warm_theta["xbar_ce"][m])

    return theta

def extract_theta(theta_vars):
    return {
        "Hreq_ce": {y: float(theta_vars["Hreq_ce"][y].X) for y in Y},
        "xbar_ce": {m: float(theta_vars["xbar_ce"][m].X) for m in M},
    }

def add_separable_primal_constraints_y(model, z_vars):
    theta_base = initial_theta_vals()

    for name in separable_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z_vars, theta_base)
        rhs = rhs_eval(rows[name]["rhs"], theta_base)
        model.addConstr(lhs >= rhs, name=f"Y_primal[{name}]")

def add_D_constraint_y(model, z_vars):
    for y in Y:
        if y >= y_star:
            model.addConstr(
                gp.quicksum(
                    float(eg[g]) * z_vars[key_p(g,t,y)]
                    for g in GE
                    for t in T
                )
                <= float(epsilon_CO2_wcep),
                name=f"D_CO2_cap_{y}"
            )

def add_separable_dual_constraints_y(model, ydual_vars):
    theta_base = initial_theta_vals()

    for k in separable_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_base) * ydual_vars[name]

        model.addConstr(lhs <= float(c_map.get(k, 0.0)), name=f"Y_dual[{k}]")

def add_strong_duality_y(model, theta_vals, z_vars, ydual_vars):
    ctx = gp.quicksum(float(c_map[k]) * z_vars[k] for k in primal_keys)

    bty = gp.LinExpr()

    for name in row_order:
        bty += rhs_eval(rows[name]["rhs"], theta_vals) * ydual_vars[name]

    model.addConstr(ctx <= bty, name="Y_StrongDuality")

def solve_theta_subproblem(
    z_vals,
    ydual_vals,
    rho,
    mu,
    warm_theta=None,
    log=False,
    use_true_objective=True
):
    m = gp.Model("PADM_theta_subproblem_IEEE57_wellposed_original")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    theta = add_theta_variables(m, warm_theta=warm_theta)

    obj = gp.LinExpr()

    if use_true_objective:
        for y in Y:
            dvar = m.addVar(lb=0.0, name=f"dHreq[{y}]")
            diff = theta["Hreq_ce"][y] - float(Hreq_base[y])
            m.addConstr(dvar >= diff, name=f"dHreq_pos[{y}]")
            m.addConstr(dvar >= -diff, name=f"dHreq_neg[{y}]")
            base = abs(float(Hreq_base[y]))
            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_H) * dvar / base
            else:
                obj += float(alpha_H) * dvar

        for mm in M:
            dvar = m.addVar(lb=0.0, name=f"dXbar[{mm}]")
            diff = theta["xbar_ce"][mm] - float(xbar_base[mm])
            m.addConstr(dvar >= diff, name=f"dXbar_pos[{mm}]")
            m.addConstr(dvar >= -diff, name=f"dXbar_neg[{mm}]")
            base = abs(float(xbar_base[mm]))
            if USE_PERCENT_OBJECTIVE and base > 1e-9:
                obj += float(alpha_Xbar) * dvar / base
            else:
                obj += float(alpha_Xbar) * dvar

    for name in coupled_primal_rows:
        lhs = terms_expr_theta_model_with_fixed_z(rows[name]["terms"], z_vals, theta)
        rhs = rhs_expr_in_theta_model(rows[name]["rhs"], theta)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            ydval = float(ydual_vals.get(name, 0.0))

            if abs(ydval) <= 1e-12:
                continue

            for c in terms[k]:
                lhs += ydval * coeff_expr_in_theta_model(c, theta)

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    # Dualidad fuerte dura en theta:
    #   c^T z fijo <= b(theta)^T y fijo
    ctx_fixed = sum(float(c_map[k]) * float(z_vals.get(k, 0.0)) for k in primal_keys)

    bty_theta = gp.LinExpr()

    for name in row_order:
        ydval = float(ydual_vals.get(name, 0.0))

        if abs(ydval) <= 1e-12:
            continue

        bty_theta += ydval * rhs_expr_in_theta_model(rows[name]["rhs"], theta)

    # En este caso theta=(Hreq_ce,xbar_ce) modifica b, por lo que b(theta)^T lambda
    # sí depende de theta. La dualidad fuerte se puede imponer dura en theta.
    m.addConstr(ctx_fixed <= bty_theta, name="Theta_StrongDuality")

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema THETA falló.")
        print(f"Status = {m.Status}")
        return None, m.Status, None

    theta_vals = extract_theta(theta)
    return theta_vals, m.Status, float(m.ObjVal)

def solve_y_subproblem(theta_vals, rho, mu, warm_z=None, warm_ydual=None, log=False):
    m = gp.Model("PADM_y_subproblem_IEEE57_wellposed_original")
    m.Params.OutputFlag = 1 if log else 0

    if TIME_LIMIT_SUBPROBLEM is not None:
        m.Params.TimeLimit = TIME_LIMIT_SUBPROBLEM

    z = add_z_variables(m)
    yd = add_dual_variables(m)

    m.update()

    if warm_z is not None:
        for k, v in z.items():
            if k in warm_z:
                v.Start = float(warm_z[k])

    if warm_ydual is not None:
        for name, v in yd.items():
            if name in warm_ydual:
                v.Start = float(warm_ydual[name])

    add_separable_primal_constraints_y(m, z)
    add_D_constraint_y(m, z)
    add_separable_dual_constraints_y(m, yd)
    add_strong_duality_y(m, theta_vals, z, yd)

    obj = gp.LinExpr()

    for name in coupled_primal_rows:
        lhs = terms_expr_z_model(rows[name]["terms"], z, theta_vals)
        rhs = rhs_eval(rows[name]["rhs"], theta_vals)

        v = m.addVar(lb=0.0, name=f"viol_primal[{name}]")
        m.addConstr(v >= rhs - lhs, name=f"def_viol_primal[{name}]")

        obj += float(rho[name]) * v

    for k in coupled_dual_cols:
        lhs = gp.LinExpr()

        for name in row_order:
            terms = rows[name]["terms"]

            if k not in terms:
                continue

            for c in terms[k]:
                lhs += coeff_value(c, theta_vals) * yd[name]

        v = m.addVar(lb=0.0, name=f"viol_dual[{k}]")
        m.addConstr(v >= lhs - float(c_map.get(k, 0.0)), name=f"def_viol_dual[{k}]")

        obj += float(mu[k]) * v

    if Y_OBJECTIVE_MODE == "ctx":
        obj += TIEBREAKER_CTX_WEIGHT * gp.quicksum(
            float(c_map[k]) * z[k]
            for k in primal_keys
        )

    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT] or m.SolCount == 0:
        print("\n⚠️ Subproblema Y falló.")
        print(f"Status = {m.Status}")
        return None, None, m.Status, None

    z_vals = {k: float(v.X) for k, v in z.items()}
    yd_vals = {name: float(v.X) for name, v in yd.items()}

    return z_vals, yd_vals, m.Status, float(m.ObjVal)

def initialize_penalties():
    rho = {name: float(RHO0) for name in coupled_primal_rows}
    mu = {k: float(MU0) for k in coupled_dual_cols}
    return rho, mu

def update_penalties(theta_vals, z_vals, ydual_vals, rho, mu):
    pr = compute_primal_residuals(theta_vals, z_vals)
    dr = compute_dual_residuals(theta_vals, ydual_vals)

    n_rho = 0
    n_mu = 0

    for name in coupled_primal_rows:
        if pr[name] > FEAS_TOL:
            rho[name] = min(MAX_PENALTY, PENALTY_FACTOR * rho[name])
            n_rho += 1

    for k in coupled_dual_cols:
        if dr[k] > FEAS_TOL:
            mu[k] = min(MAX_PENALTY, PENALTY_FACTOR * mu[k])
            n_mu += 1

    return n_rho, n_mu


# ============================================================
# 8) LOOP PADM-W
# ============================================================

def run_padm(
    use_true_objective=True,
    theta_init=None,
    z_init=None,
    ydual_init=None,
    label="PADM"
):
    print("\n" + "="*70)
    print(f"INICIO {label}")
    print("="*70)
    print(f"use_true_objective = {use_true_objective}")

    rho, mu = initialize_penalties()

    theta_vals = copy.deepcopy(theta_init) if theta_init is not None else initial_theta_vals()

    if z_init is None or ydual_init is None:
        print("\nInicializando Y=(z,lambda)...")

        z_vals, ydual_vals, st_y, obj_y = solve_y_subproblem(
            theta_vals,
            rho,
            mu,
            warm_z=None,
            warm_ydual=None,
            log=False
        )

        if z_vals is None:
            print("❌ No se pudo inicializar Y.")

            return {
                "best_feasible": None,
                "theta": theta_vals,
                "z": None,
                "ydual": None,
                "history": [],
                "last_residuals": None,
                "rho": rho,
                "mu": mu,
            }

    else:
        z_vals = z_init.copy()
        ydual_vals = ydual_init.copy()

    history = []
    best_feasible = None

    res0 = max_residuals(theta_vals, z_vals, ydual_vals)

    print("\nChequeo inicial:")
    print(f"J(theta)       = {compute_J(theta_vals):.8f}")
    print(f"Max primal     = {res0['max_primal_all']:.3e}")
    print(f"Max dual       = {res0['max_dual_all']:.3e}")
    print(f"Strong duality = {res0['sd']:.3e}")

    for k_outer in range(MAX_OUTER):

        print("\n" + "#"*70)
        print(f"{label} | OUTER k = {k_outer}")
        print("#"*70)

        phi_prev = compute_phi(
            theta_vals,
            z_vals,
            ydual_vals,
            rho,
            mu,
            use_true_objective=use_true_objective
        )

        for l_inner in range(MAX_INNER):

            theta_old = copy.deepcopy(theta_vals)
            z_old = z_vals.copy()
            yd_old = ydual_vals.copy()

            theta_new, st_theta, obj_theta = solve_theta_subproblem(
                z_vals,
                ydual_vals,
                rho,
                mu,
                warm_theta=theta_vals,
                log=False,
                use_true_objective=use_true_objective
            )

            if theta_new is None:
                print("⚠️ Falló subproblema theta.")
                break

            theta_vals = theta_new

            z_new, yd_new, st_y, obj_y = solve_y_subproblem(
                theta_vals,
                rho,
                mu,
                warm_z=z_vals,
                warm_ydual=ydual_vals,
                log=False
            )

            if z_new is None:
                print("⚠️ Falló subproblema Y.")
                break

            z_vals = z_new
            ydual_vals = yd_new

            phi_now = compute_phi(
                theta_vals,
                z_vals,
                ydual_vals,
                rho,
                mu,
                use_true_objective=use_true_objective
            )

            res = max_residuals(theta_vals, z_vals, ydual_vals)

            dtheta = diff_theta(theta_vals, theta_old)
            dz = diff_dict(z_vals, z_old)
            dyd = diff_dict(ydual_vals, yd_old)
            dphi = abs(phi_now - phi_prev) / max(1.0, abs(phi_prev))

            if PADM_OUTPUT:
                print(
                    f"k={k_outer:02d}, l={l_inner:02d} | "
                    f"Phi={phi_now:.6e}, J={compute_J(theta_vals):.6e}, "
                    f"p_all={res['max_primal_all']:.2e}, "
                    f"d_all={res['max_dual_all']:.2e}, "
                    f"sd={res['sd']:.2e}, "
                    f"p_coup={res['max_primal_coupled']:.2e}, "
                    f"d_coup={res['max_dual_coupled']:.2e}, "
                    f"dtheta={dtheta:.2e}, dz={dz:.2e}, dyd={dyd:.2e}, dphi={dphi:.2e}"
                )

            history.append({
                "outer": k_outer,
                "inner": l_inner,
                "phi": phi_now,
                "J": compute_J(theta_vals),
                "max_primal_all": res["max_primal_all"],
                "max_dual_all": res["max_dual_all"],
                "sd": res["sd"],
                "max_primal_coupled": res["max_primal_coupled"],
                "max_dual_coupled": res["max_dual_coupled"],
                "dtheta": dtheta,
                "dz": dz,
                "dyd": dyd,
                "dphi": dphi,
                "rho_max": max(rho.values()) if rho else 0.0,
                "mu_max": max(mu.values()) if mu else 0.0,
            })

            if dphi <= INNER_TOL and dtheta <= INNER_TOL and dz <= INNER_TOL and dyd <= INNER_TOL:
                print(f"Inner loop converge en l={l_inner}.")
                break

            phi_prev = phi_now

        res = max_residuals(theta_vals, z_vals, ydual_vals)

        feasible = (
            res["max_primal_all"] <= FEAS_TOL
            and res["max_dual_all"] <= FEAS_TOL
            and res["sd"] <= FEAS_TOL
        )

        print("\n" + "-"*70)
        print(f"{label} | Chequeo fin outer k={k_outer}")
        print("-"*70)
        print(f"J(theta)             = {compute_J(theta_vals):.8f}")
        print(f"Max primal all       = {res['max_primal_all']:.3e}")
        print(f"Max dual all         = {res['max_dual_all']:.3e}")
        print(f"Strong duality       = {res['sd']:.3e}")
        print(f"Max primal coupled   = {res['max_primal_coupled']:.3e}")
        print(f"Max dual coupled     = {res['max_dual_coupled']:.3e}")
        print(f"rho_max              = {max(rho.values()) if rho else 0.0:.3e}")
        print(f"mu_max               = {max(mu.values()) if mu else 0.0:.3e}")

        if feasible:
            print(f"\n✅ {label} encontró solución factible primal-dual.")

            best_feasible = {
                "theta": copy.deepcopy(theta_vals),
                "z": z_vals.copy(),
                "ydual": ydual_vals.copy(),
                "J": compute_J(theta_vals),
                "residuals": res,
                "outer": k_outer,
                "history": history,
            }

            break

        n_rho, n_mu = update_penalties(theta_vals, z_vals, ydual_vals, rho, mu)

        print("\nActualización penalizaciones:")
        print(f"rho actualizadas: {n_rho}")
        print(f"mu actualizadas : {n_mu}")

    if best_feasible is None:
        print(f"\n⚠️ {label} terminó sin certificar factibilidad.")

    return {
        "best_feasible": best_feasible,
        "theta": theta_vals,
        "z": z_vals,
        "ydual": ydual_vals,
        "history": history,
        "last_residuals": max_residuals(theta_vals, z_vals, ydual_vals) if z_vals is not None else None,
        "rho": rho,
        "mu": mu,
    }


# ============================================================
# 9) EJECUTAR PADM-W
# ============================================================

PADM_WARMSTART = None

if RUN_PADM_W:

    res_ws = run_padm(
        use_true_objective=False,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM-W ETAPA 1: FACTIBILIDAD"
    )

    if res_ws["best_feasible"] is not None:

        theta_start = res_ws["best_feasible"]["theta"]
        z_start = res_ws["best_feasible"]["z"]
        yd_start = res_ws["best_feasible"]["ydual"]

        print("\n" + "="*70)
        print("WARMSTART FACTIBLE ENCONTRADO. INICIANDO ETAPA 2 CON J REAL.")
        print("="*70)

        res_final = run_padm(
            use_true_objective=True,
            theta_init=theta_start,
            z_init=z_start,
            ydual_init=yd_start,
            label="PADM-W ETAPA 2: OBJETIVO REAL"
        )

        if res_final["best_feasible"] is not None:
            final_solution = res_final["best_feasible"]
            final_label = "PADM-W ETAPA 2"
        else:
            print("\n⚠️ Etapa 2 no certificó factibilidad. Se conserva warmstart factible de etapa 1.")
            final_solution = res_ws["best_feasible"]
            final_label = "PADM-W ETAPA 1"

    else:
        print("\n⚠️ Etapa 1 no encontró punto factible.")
        final_solution = None
        final_label = "PADM-W FAILED"
        res_final = res_ws

else:
    res_final = run_padm(
        use_true_objective=True,
        theta_init=None,
        z_init=None,
        ydual_init=None,
        label="PADM"
    )

    if res_final["best_feasible"] is not None:
        final_solution = res_final["best_feasible"]
        final_label = "PADM"
    else:
        final_solution = None
        final_label = "PADM FAILED"

print("\n" + "="*70)
print("RESULTADOS FINALES PADM-W IEEE57")
print("="*70)

if final_solution is not None:
    sol_theta = final_solution["theta"]
    sol_z = final_solution["z"]
    sol_yd = final_solution["ydual"]

    print("✅ SOLUCIÓN FACTIBLE CERTIFICADA")
    print(f"Método final     : {final_label}")
    print(f"Outer encontrado : {final_solution['outer']}")
    print(f"J                : {final_solution['J']:.8f}")

    PADM_WARMSTART = {
        "theta": copy.deepcopy(sol_theta),
        "z": sol_z.copy(),
        "ydual": sol_yd.copy(),
        "label": final_label,
        "J": final_solution["J"],
        "residuals": final_solution["residuals"],
    }

else:
    print("⚠️ NO SE CERTIFICÓ FACTIBILIDAD EN PADM-W.")

    sol_theta = res_final["theta"]
    sol_z = res_final["z"]
    sol_yd = res_final["ydual"]

    if sol_z is not None and sol_yd is not None:
        res = max_residuals(sol_theta, sol_z, sol_yd)

        PADM_WARMSTART = {
            "theta": copy.deepcopy(sol_theta),
            "z": sol_z.copy(),
            "ydual": sol_yd.copy(),
            "label": "PADM_LAST_POINT_NOT_CERTIFIED",
            "J": compute_J(sol_theta),
            "residuals": res,
        }

        print("⚠️ Se guardó el último punto de PADM-W como Start no certificado.")
    else:
        PADM_WARMSTART = None
        print("❌ No hay punto PADM-W disponible para warm start.")

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("CHECK PADM_WARMSTART IEEE57")
    print("="*70)

    sol_theta = PADM_WARMSTART["theta"]
    sol_z = PADM_WARMSTART["z"]
    sol_yd = PADM_WARMSTART["ydual"]

    res_check = max_residuals(sol_theta, sol_z, sol_yd)

    print(f"Origen          = {PADM_WARMSTART['label']}")
    print(f"J(theta)        = {PADM_WARMSTART['J']:.8f}")
    print(f"Max primal      = {res_check['max_primal_all']:.6e}")
    print(f"Max dual        = {res_check['max_dual_all']:.6e}")
    print(f"Strong duality  = {res_check['sd']:.6e}")


# ============================================================
# 10) WCEP FINAL CON DUALIDAD FUERTE Y WARM START PADM-W
# ============================================================

print("\n" + "="*70)
print("CONSTRUYENDO WCEP FINAL IEEE57 CON DUALIDAD FUERTE")
print("="*70)

w = gp.Model("WCEP_IEEE57_wellposed_DF_warmstart_PADM_original")
w.Params.NonConvex = 2
w.Params.MIPGap = WCEP_MIPGAP
w.Params.OutputFlag = WCEP_OUTPUT
w.Params.TimeLimit = WCEP_TIME_LIMIT

# ------------------------------------------------------------
# Variables primales lower-level
# ------------------------------------------------------------

p = w.addVars(G, T, Y, lb=0.0, name="p")
u = w.addVars(GS, T, Y, lb=0.0, ub=1.0, name="u")
s = w.addVars(B, T, Y, lb=0.0, name="s")

xcap = w.addVars(M, Y, lb=0.0, name="xcap")
h_syn = w.addVars(M, T, Y, lb=0.0, name="h_syn")
r_ffr = w.addVars(M, T, Y, lb=0.0, name="r_ffr")

f_pos = w.addVars(L_arcs, T, Y, lb=0.0, name="f_pos")
f_neg = w.addVars(L_arcs, T, Y, lb=0.0, name="f_neg")

theta_p = w.addVars(B, T, Y, lb=0.0, name="theta_p")
theta_m = w.addVars(B, T, Y, lb=0.0, name="theta_m")

# ------------------------------------------------------------
# Variables mutables upper-level
# ------------------------------------------------------------

Hreq_ce = w.addVars(Y, lb=0.0, name="Hreq_ce")
xbar_ce = w.addVars(M, lb=0.0, name="xbar_ce")

for y in Y:
    base = Hreq_base[y]
    w.addConstr(Hreq_ce[y] >= HREQ_LB_FACTOR * base, name=f"adm_lb_Hreq[{y}]")
    w.addConstr(Hreq_ce[y] <= HREQ_UB_FACTOR * base, name=f"adm_ub_Hreq[{y}]")

for m in M:
    base = xbar_base[m]
    w.addConstr(xbar_ce[m] >= XBAR_LB_FACTOR * base, name=f"adm_lb_xbar[{m}]")
    w.addConstr(xbar_ce[m] <= XBAR_UB_FACTOR * base, name=f"adm_ub_xbar[{m}]")

# ------------------------------------------------------------
# Factibilidad primal Ax >= b
# ------------------------------------------------------------

R_primal = {}
RHS_expr = {}

for i in B:
    for t in T:
        for y in Y:
            expr = (
                gp.quicksum(p[g, t, y] for g in G_en_bus.get(i, []))
                + s[i, t, y]
                + gp.quicksum(
                    f_pos[ell, t, y] - f_neg[ell, t, y]
                    for ell in lineas_entran[i]
                )
                - gp.quicksum(
                    f_pos[ell, t, y] - f_neg[ell, t, y]
                    for ell in lineas_salen[i]
                )
            )

            rhs_val = float(d_ref[(i, t, y)])

            name = f"bal_p_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

            name = f"bal_n_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= -rhs_val, name=name)
            RHS_expr[name] = -rhs_val

for i in B:
    for t in T:
        for y in Y:
            rhs_val = -float(d_ref[(i, t, y)])
            name = f"ens_ub_{i}_{t}_{y}"
            R_primal[name] = w.addConstr(-s[i, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for ell in L_arcs:
    i = from_bus_line[ell]
    j = to_bus_line[ell]

    for t in T:
        for y in Y:
            Bij = float(Bline[ell])

            flow = f_pos[ell, t, y] - f_neg[ell, t, y]
            theta_i = theta_p[i, t, y] - theta_m[i, t, y]
            theta_j = theta_p[j, t, y] - theta_m[j, t, y]

            expr = flow - Bij * (theta_i - theta_j)

            name = f"dc_p_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

            name = f"dc_n_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(-expr >= 0.0, name=name)
            RHS_expr[name] = 0.0

for ell in L_arcs:
    for t in T:
        for y in Y:
            Fij = float(F0[ell])
            flow = f_pos[ell, t, y] - f_neg[ell, t, y]

            name = f"flb_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

            name = f"fub_{ell}_{t}_{y}"
            R_primal[name] = w.addConstr(-flow >= -Fij, name=name)
            RHS_expr[name] = -Fij

for t in T:
    for y in Y:
        expr_slack = theta_p[bus_slack, t, y] - theta_m[bus_slack, t, y]

        name = f"slack_p_{t}_{y}"
        R_primal[name] = w.addConstr(expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

        name = f"slack_n_{t}_{y}"
        R_primal[name] = w.addConstr(-expr_slack >= 0.0, name=name)
        RHS_expr[name] = 0.0

for g in G:
    for t in T:
        cap_base = float(P_disp.get((g, t), Pmax[g]))

        for y in Y:
            rhs_val = -cap_base
            name = f"p_cap_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-p[g, t, y] >= rhs_val, name=name)
            RHS_expr[name] = rhs_val

for g in GS:
    pmin_g = float(Pmin.get(g, 0.0))

    for t in T:
        for y in Y:
            name = f"sync_lb_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                p[g, t, y] - pmin_g * u[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"sync_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(Pmax_sync_base[g]) * u[g, t, y] - p[g, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"u_ub_{g}_{t}_{y}"
            R_primal[name] = w.addConstr(-u[g, t, y] >= -1.0, name=name)
            RHS_expr[name] = -1.0

for m in M:
    for y in Y:
        name = f"x_ub_{m}_{y}"
        R_primal[name] = w.addConstr(-xcap[m, y] >= -xbar_ce[m], name=name)
        RHS_expr[name] = -xbar_ce[m]

        for t in T:
            name = f"hsyn_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(gamma_base[m]) * xcap[m, y] - h_syn[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

            name = f"ffr_lim_{m}_{t}_{y}"
            R_primal[name] = w.addConstr(
                float(kappa_base[m]) * xcap[m, y] - r_ffr[m, t, y] >= 0.0,
                name=name
            )
            RHS_expr[name] = 0.0

for t in T:
    for y in Y:
        lhs_H = (
            gp.quicksum(float(Hg.get(g, 0.0)) * u[g, t, y] for g in GS)
            + gp.quicksum(h_syn[m, t, y] for m in M)
        )

        name = f"sys_H_{t}_{y}"
        R_primal[name] = w.addConstr(lhs_H >= Hreq_ce[y], name=name)
        RHS_expr[name] = Hreq_ce[y]

        name = f"sys_R_{t}_{y}"
        R_primal[name] = w.addConstr(
            gp.quicksum(r_ffr[m, t, y] for m in M) >= float(Rreq[y]),
            name=name
        )
        RHS_expr[name] = float(Rreq[y])

# ------------------------------------------------------------
# Restricción D no dualizada
# ------------------------------------------------------------

for y in Y:
    if y >= y_star:
        w.addConstr(
            gp.quicksum(float(eg[g]) * p[g, t, y] for g in GE for t in T)
            <= epsilon_CO2_wcep,
            name=f"D_CO2_cap_{y}"
        )

w.update()

# ------------------------------------------------------------
# Variables duales
# ------------------------------------------------------------

ydual = {}

for name in R_primal.keys():
    ub_here = GRB.INFINITY

    if name.startswith("sys_H_"):
        ub_here = UB_H
    elif name.startswith("x_ub_"):
        ub_here = UB_xbar

    ydual[name] = w.addVar(
        lb=0.0,
        ub=ub_here,
        name=f"ydual[{name}]"
    )

w.update()

# ------------------------------------------------------------
# Costos del forward
# ------------------------------------------------------------

c_map_w = {}

for g in G:
    for t in T:
        for y in Y:
            c_map_w[p[g, t, y]] = float(cg[g])

for i in B:
    for t in T:
        for y in Y:
            c_map_w[s[i, t, y]] = float(VOLL)

for m in M:
    for y in Y:
        c_map_w[xcap[m, y]] = float(cinv[m])

primal_vars_w = (
    list(p.values())
    + list(u.values())
    + list(s.values())
    + list(xcap.values())
    + list(h_syn.values())
    + list(r_ffr.values())
    + list(f_pos.values())
    + list(f_neg.values())
    + list(theta_p.values())
    + list(theta_m.values())
)

# ------------------------------------------------------------
# Factibilidad dual A^T y <= c
# ------------------------------------------------------------

for v in primal_vars_w:
    lhs = gp.QuadExpr()

    for name, constr in R_primal.items():
        coeff = 0.0

        if isinstance(constr, gp.Constr):
            val = w.getCoeff(constr, v)

            if abs(val) > 1e-9:
                coeff = val

        elif isinstance(constr, gp.QConstr):
            qrow = w.getQCRow(constr)

            lin = qrow.getLinExpr()
            for jj in range(lin.size()):
                if lin.getVar(jj).sameAs(v):
                    coeff += lin.getCoeff(jj)

            for jj in range(qrow.size()):
                v1 = qrow.getVar1(jj)
                v2 = qrow.getVar2(jj)
                cq = qrow.getCoeff(jj)

                if v1.sameAs(v):
                    coeff += cq * v2
                elif v2.sameAs(v):
                    coeff += cq * v1

        if isinstance(coeff, (gp.Var, gp.LinExpr, gp.QuadExpr)):
            lhs += coeff * ydual[name]
        else:
            if abs(coeff) > 1e-9:
                lhs += coeff * ydual[name]

    w.addQConstr(
        lhs <= c_map_w.get(v, 0.0),
        name=f"df_{v.VarName}"
    )

# ------------------------------------------------------------
# Dualidad fuerte
# ------------------------------------------------------------

ctx = gp.quicksum(c_map_w.get(v, 0.0) * v for v in primal_vars_w)

bty = gp.QuadExpr()

for name in R_primal.keys():
    rhs = RHS_expr[name]

    if isinstance(rhs, (gp.Var, gp.LinExpr, gp.QuadExpr)):
        bty += rhs * ydual[name]
    else:
        bty += float(rhs) * ydual[name]

w.addQConstr(
    ctx - bty <= 0.0,
    name="StrongDuality"
)

# ------------------------------------------------------------
# Objetivo L1
# ------------------------------------------------------------

dHreq = w.addVars(Y, lb=0.0, name="dHreq")
dXbar = w.addVars(M, lb=0.0, name="dXbar")

for y in Y:
    base = Hreq_base[y]
    diff = Hreq_ce[y] - base
    w.addConstr(dHreq[y] >= diff, name=f"dHreq_pos[{y}]")
    w.addConstr(dHreq[y] >= -diff, name=f"dHreq_neg[{y}]")

for m in M:
    base = xbar_base[m]
    diff = xbar_ce[m] - base
    w.addConstr(dXbar[m] >= diff, name=f"dXbar_pos[{m}]")
    w.addConstr(dXbar[m] >= -diff, name=f"dXbar_neg[{m}]")

if USE_PERCENT_OBJECTIVE:
    obj_wcep = gp.LinExpr()
    for y in Y:
        base = abs(float(Hreq_base[y]))
        if base > 1e-9:
            obj_wcep += alpha_H * dHreq[y] / base
        else:
            obj_wcep += alpha_H * dHreq[y]
    for m in M:
        base = abs(float(xbar_base[m]))
        if base > 1e-9:
            obj_wcep += alpha_Xbar * dXbar[m] / base
        else:
            obj_wcep += alpha_Xbar * dXbar[m]
    w.setObjective(obj_wcep, GRB.MINIMIZE)
else:
    w.setObjective(
        alpha_H * dHreq.sum() + alpha_Xbar * dXbar.sum(),
        GRB.MINIMIZE
    )

# ============================================================
# 11) APLICAR WARM START DEL PADM-W
# ============================================================

def safe_start(var, value):
    try:
        if value is not None and np.isfinite(float(value)):
            var.Start = float(value)
    except Exception:
        pass

if PADM_WARMSTART is not None:
    print("\n" + "="*70)
    print("APLICANDO PADM_WARMSTART AL WCEP FINAL IEEE57")
    print("="*70)
    print("Origen:", PADM_WARMSTART.get("label", "PADM"))

    ws_theta = PADM_WARMSTART["theta"]
    ws_z = PADM_WARMSTART["z"]
    ws_yd = PADM_WARMSTART["ydual"]

    for y in Y:
        safe_start(Hreq_ce[y], ws_theta["Hreq_ce"][y])
        safe_start(dHreq[y], abs(ws_theta["Hreq_ce"][y] - Hreq_base[y]))

    for m in M:
        safe_start(xbar_ce[m], ws_theta["xbar_ce"][m])
        safe_start(dXbar[m], abs(ws_theta["xbar_ce"][m] - xbar_base[m]))

    for g in G:
        for t in T:
            for y in Y:
                safe_start(p[g, t, y], ws_z.get(("p", g, t, y), None))

    for g in GS:
        for t in T:
            for y in Y:
                safe_start(u[g, t, y], ws_z.get(("u", g, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(s[i, t, y], ws_z.get(("s", i, t, y), None))

    for m in M:
        for y in Y:
            safe_start(xcap[m, y], ws_z.get(("xcap", m, y), None))

    for m in M:
        for t in T:
            for y in Y:
                safe_start(h_syn[m, t, y], ws_z.get(("h_syn", m, t, y), None))
                safe_start(r_ffr[m, t, y], ws_z.get(("r_ffr", m, t, y), None))

    for ell in L_arcs:
        for t in T:
            for y in Y:
                safe_start(f_pos[ell, t, y], ws_z.get(("f_pos", ell, t, y), None))
                safe_start(f_neg[ell, t, y], ws_z.get(("f_neg", ell, t, y), None))

    for i in B:
        for t in T:
            for y in Y:
                safe_start(theta_p[i, t, y], ws_z.get(("theta_p", i, t, y), None))
                safe_start(theta_m[i, t, y], ws_z.get(("theta_m", i, t, y), None))

    for name, var in ydual.items():
        safe_start(var, ws_yd.get(name, None))

    print("✅ Warm start aplicado al WCEP IEEE57.")
else:
    print("\n⚠️ No existe PADM_WARMSTART. El WCEP se resolverá sin Start.")


# ============================================================
# 12) OPTIMIZAR WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESOLVIENDO WCEP FINAL IEEE57")
print("="*70)

w.optimize()

# Registro de tiempos/GAP WCEP
runtime_wcep = float(w.Runtime)
status_wcep = int(w.Status)
sol_count_wcep = int(w.SolCount)

print("\n" + "="*70)
print("REGISTRO DE OPTIMIZACIÓN WCEP IEEE57")
print("="*70)
print(f"Status Gurobi = {status_wcep}")
print(f"Tiempo optimize() = {runtime_wcep:.2f} s = {runtime_wcep/60:.2f} min = {runtime_wcep/3600:.2f} h")
print(f"Soluciones encontradas = {sol_count_wcep}")

if sol_count_wcep > 0:
    print(f"Mejor objetivo encontrado = {float(w.ObjVal):.8f}")
    print(f"Mejor bound               = {float(w.ObjBound):.8f}")
    print(f"MIPGap                    = {float(w.MIPGap):.6e}")
    print(f"MIPGap (%)                = {100*float(w.MIPGap):.4f}%")
else:
    print("No se encontró solución factible.")

total_script_time = time.time() - t0_total_script
print(f"Tiempo total script = {total_script_time:.2f} s = {total_script_time/60:.2f} min = {total_script_time/3600:.2f} h")


# ============================================================
# 13) RESULTADOS WCEP FINAL
# ============================================================

print("\n" + "="*70)
print("RESULTADOS WCEP FINAL IEEE57")
print("="*70)
print(f"Status = {w.Status}")

if w.Status == GRB.OPTIMAL:
    print(f"✅ WCEP OPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status == GRB.SUBOPTIMAL:
    print(f"⚠️ WCEP SUBOPTIMAL. Obj = {w.ObjVal:.8f}")

elif w.Status in [GRB.INFEASIBLE, GRB.INF_OR_UNBD]:
    print("❌ WCEP infactible o INF_OR_UNBD.")
else:
    if w.SolCount > 0:
        print(f"⚠️ WCEP terminó con status {w.Status}, pero tiene solución.")
        print(f"Obj = {w.ObjVal:.8f}")
    else:
        print(f"⚠️ WCEP terminó con status {w.Status} y sin solución.")

if w.SolCount > 0:

    tol = 1e-6

    print("\n" + "="*70)
    print("CAMBIOS ÓPTIMOS EN PARÁMETROS MUTABLES")
    print("="*70)

    print("\n--- Hreq_ce ---")
    print(f"Cotas usadas: {HREQ_LB_FACTOR:.2f}*Hreq_base <= Hreq_ce <= {HREQ_UB_FACTOR:.2f}*Hreq_base")
    hubo_cambio = False
    for y in Y:
        val = float(Hreq_ce[y].X)
        base = float(Hreq_base[y])
        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"Año {y}: {base:.6f} -> {val:.6f}   ({pct:+.2f}%)")
    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n--- xbar_ce ---")
    print(f"Cotas usadas: {XBAR_LB_FACTOR:.2f}*xbar_base <= xbar_ce <= {XBAR_UB_FACTOR:.2f}*xbar_base")
    hubo_cambio = False
    for m in M:
        val = float(xbar_ce[m].X)
        base = float(xbar_base[m])
        if abs(val - base) > tol:
            hubo_cambio = True
            pct = 100.0 * (val / base - 1.0) if abs(base) > 1e-9 else float("inf")
            print(f"{m}: {base:.6f} -> {val:.6f}   ({pct:+.2f}%)")
    if not hubo_cambio:
        print("Sin cambios relevantes.")

    print("\n" + "="*70)
    print("REVISIÓN DUALIDAD FUERTE")
    print("="*70)

    ctx_val = ctx.getValue()
    bty_val = bty.getValue()

    print(f"c^T x  = {ctx_val:.8f}")
    print(f"b^T y  = {bty_val:.8f}")
    print(f"gap    = {ctx_val - bty_val:.6e}")

    print("\n" + "="*70)
    print("EMISIONES ANUALES DE LA SOLUCIÓN WCEP")
    print("="*70)

    for y in Y:
        emis_y = sum(
            float(eg[g]) * float(p[g, t, y].X)
            for g in GE
            for t in T
        )

        if y >= y_star:
            holgura = epsilon_CO2_wcep - emis_y
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"límite D = {epsilon_CO2_wcep:.6f} | "
                f"holgura = {holgura:.6f}"
            )
        else:
            print(
                f"Año {y}: emisiones = {emis_y:.6f} | "
                f"sin restricción D"
            )

    print("\n" + "="*70)
    print("CHEQUEO H Y R")
    print("="*70)

    for y in Y:
        for t in T:
            H_sync_val = sum(
                float(Hg.get(g, 0.0)) * float(u[g, t, y].X)
                for g in GS
            )

            H_syn_val = sum(
                float(h_syn[m, t, y].X)
                for m in M
            )

            H_total = H_sync_val + H_syn_val
            H_lim = float(Hreq_base[y])

            R_total = sum(
                float(r_ffr[m, t, y].X)
                for m in M
            )

            R_lim = float(Rreq[y])

            print(
                f"(y={y}, t={t}) | "
                f"H_total={H_total:.6f}, Hreq_base={H_lim:.6f}, margen_H={H_total - H_lim:.6e} | "
                f"R_total={R_total:.6f}, Rreq={R_lim:.6f}, margen_R={R_total - R_lim:.6e}"
            )

    print("\n" + "="*70)
    print("REVISIÓN DE DUALES ACOTADAS")
    print("="*70)

    hits_H = 0
    hits_xbar = 0

    for name, var in ydual.items():
        if name.startswith("sys_H_"):
            if abs(var.X - UB_H) <= 1e-6 * max(1.0, UB_H):
                hits_H += 1

        elif name.startswith("x_ub_"):
            if abs(var.X - UB_xbar) <= 1e-6 * max(1.0, UB_xbar):
                hits_xbar += 1

    print(f"Duales sys_H pegadas a cota : {hits_H}")
    print(f"Duales x_ub pegadas a cota  : {hits_xbar}")

else:
    print("\nNo hay solución WCEP disponible para reportar.")

✅ Usando d_fw existente desde el FW corregido.

REFERENCIAS USADAS POR PADM-W + WCEP IEEE57
Referencia demanda: d_fw
Referencia Hreq   : Hreq_fw existente del FW well-posed
Pmax síncrono queda fijo
Cotas Hreq_ce: ±15% alrededor de Hreq_base/well-posed
Cotas xbar_ce: ±15% alrededor de xbar_base
Año 1: Hreq original = 10.714900 | Hreq_base/well-posed = 8.571920
Año 2: Hreq original = 11.250645 | Hreq_base/well-posed = 9.000516
Año 3: Hreq original = 11.786390 | Hreq_base/well-posed = 9.429112

COTAS DUALES IEEE57
UB_H    = 324200000.000000
UB_sync = 10000.000000
UB_xbar = 3242000.000000
Bus slack usado: 1
Bus slack usado: 1

CLASIFICACIÓN PADM-W IEEE57 - FORMULACIÓN ORIGINAL
N° filas primales acopladas   : 39
N° filas primales separables  : 6492
N° columnas duales acopladas  : 0
N° columnas duales separables : 4443

✅ Filas acopladas esperadas: sys_H_ y x_ub_.

INICIO PADM-W ETAPA 1: FACTIBILIDAD
use_true_objective = False

Inicializando Y=(z,lambda)...

Chequeo inicial:
J(theta)       =